In [2]:
import os
os.chdir("/content/drive/MyDrive/毕设/data")

# mins

In [4]:
#!/usr/bin/env python3
"""
resample_1min_to_15min.py
─────────────────────────
Reads 1-minute OHLCV CSVs from processed/, resamples to 15-minute bars,
computes 53 technical indicators (matching the 1-hour feature set), and
saves both the resampled OHLCV and the full feature files.

Usage:
    python resample_1min_to_15min.py
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────
CONFIG = {
    "PROCESSED_DIR": "processed",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "RESAMPLE_FREQ": "15min",
}

# Possible names for the timestamp column in source CSVs
TIMESTAMP_ALIASES = ["timestamp", "ts_event", "datetime", "date", "time", "ts"]


# ──────────────────────────────────────────────────────────────────────
# 1. Discover reference indicator columns from an existing 1hour file
# ──────────────────────────────────────────────────────────────────────
def get_reference_indicator_cols(features_dir: str) -> list[str]:
    ohlcv_cols = {"timestamp", "open", "high", "low", "close", "volume"}
    for fname in sorted(os.listdir(features_dir)):
        if fname.endswith("_1hour_features.csv"):
            path = os.path.join(features_dir, fname)
            df = pd.read_csv(path, nrows=0)
            ind_cols = [c for c in df.columns if c.lower() not in ohlcv_cols]
            if ind_cols:
                print(f"[ref] Using {fname} as reference → {len(ind_cols)} indicator columns")
                return ind_cols
    return []


# ──────────────────────────────────────────────────────────────────────
# 2. Detect and normalize the timestamp column
# ──────────────────────────────────────────────────────────────────────
def find_time_column(df: pd.DataFrame) -> str | None:
    """Find the timestamp column regardless of naming convention."""
    cols_lower = {c.lower(): c for c in df.columns}
    for alias in TIMESTAMP_ALIASES:
        if alias in cols_lower:
            return cols_lower[alias]
    return None


def normalize_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find the time column, parse as datetime, set as index named 'timestamp'.
    Strips timezone to naive UTC for clean resampling.
    """
    time_col = find_time_column(df)

    if time_col is not None:
        df = df.copy()
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df[time_col] = df[time_col].dt.tz_localize(None)
        df = df.set_index(time_col)
        df.index.name = "timestamp"
    else:
        df = df.copy()
        df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
        df.index.name = "timestamp"

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError(f"Could not create DatetimeIndex. Columns: {list(df.columns)}")

    return df


# ──────────────────────────────────────────────────────────────────────
# 3. Resample 1-min → 15-min OHLCV
# ──────────────────────────────────────────────────────────────────────
def resample_ohlcv(df_1min: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Standard OHLCV aggregation on a DatetimeIndex dataframe."""
    df_1min = normalize_timestamps(df_1min)

    # Lowercase column names
    df_1min.columns = [c.lower() for c in df_1min.columns]

    agg_rules = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    }
    agg_rules = {k: v for k, v in agg_rules.items() if k in df_1min.columns}

    df_15 = df_1min[list(agg_rules.keys())].resample(freq).agg(agg_rules)

    # Drop non-trading periods (all NaN) and rows without close
    df_15.dropna(how="all", inplace=True)
    df_15.dropna(subset=["close"], inplace=True)

    return df_15


# ──────────────────────────────────────────────────────────────────────
# 4. Compute technical indicators via pandas_ta
# ──────────────────────────────────────────────────────────────────────
def compute_indicators_pandas_ta(df: pd.DataFrame, ref_cols: list[str]) -> pd.DataFrame:
    try:
        import pandas_ta as ta
    except ImportError:
        print("[WARN] pandas_ta not installed – pip install pandas_ta")
        return compute_indicators_manual(df, ref_cols)

    work = df[["open", "high", "low", "close", "volume"]].copy()

    # Reverting to original call for pandas_ta accessor.
    # The previous error indicates that `ta.strategy()` directly on the module is incorrect.
    # The root cause might be an issue with the pandas_ta accessor registration itself due to dependency conflicts.
    work.ta.strategy("All")

    ta_cols_lower = {c.lower(): c for c in work.columns}
    matched = {}
    unmatched = []

    for ref in ref_cols:
        rl = ref.lower()
        if rl in ta_cols_lower:
            matched[ref] = ta_cols_lower[rl]
        else:
            found = False
            for tc_lower, tc_orig in ta_cols_lower.items():
                if rl.replace("_", "") == tc_lower.replace("_", ""):
                    matched[ref] = tc_orig
                    found = True
                    break
            if not found:
                unmatched.append(ref)

    result = df[["open", "high", "low", "close", "volume"]].copy()
    for ref_name, ta_name in matched.items():
        result[ref_name] = work[ta_name]

    if unmatched:
        manual = _compute_manual_indicators(df)
        manual_lower = {c.lower(): c for c in manual.columns}
        still_missing = []
        for ref in unmatched:
            rl = ref.lower()
            if rl in manual_lower:
                result[ref] = manual[manual_lower[rl]]
            else:
                still_missing.append(ref)
        if still_missing:
            print(f"  [WARN] {len(still_missing)} indicators could not be matched: "
                  f"{still_missing[:10]}{'...' if len(still_missing) > 10 else ''}")

    return result


def _compute_manual_indicators(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    c, h, l, v = df["close"], df["high"], df["low"], df["volume"]
    o = df["open"]

    # Moving Averages
    for w in [5, 10, 20, 50, 100, 200]:
        out[f"sma_{w}"] = c.rolling(w).mean()
        out[f"ema_{w}"] = c.ewm(span=w, adjust=False).mean()

    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]

    # RSI
    for w in [7, 14, 21]:
        delta = c.diff()
        gain = delta.clip(lower=0).rolling(w).mean()
        loss = (-delta.clip(upper=0)).rolling(w).mean()
        rs = gain / loss.replace(0, np.nan)
        out[f"rsi_{w}"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    for w in [20]:
        mid = c.rolling(w).mean()
        std = c.rolling(w).std()
        out[f"bb_upper_{w}"] = mid + 2 * std
        out[f"bb_middle_{w}"] = mid
        out[f"bb_lower_{w}"] = mid - 2 * std
        out[f"bb_bandwidth_{w}"] = (out[f"bb_upper_{w}"] - out[f"bb_lower_{w}"]) / mid
        out[f"bb_percent_{w}"] = (c - out[f"bb_lower_{w}"]) / (out[f"bb_upper_{w}"] - out[f"bb_lower_{w}"]).replace(0, np.nan)

    out["bb_upper"] = out.get("bb_upper_20", c)
    out["bb_middle"] = out.get("bb_middle_20", c)
    out["bb_lower"] = out.get("bb_lower_20", c)

    # ATR
    for w in [7, 14, 21]:
        tr = pd.concat([
            h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()
        ], axis=1).max(axis=1)
        out[f"atr_{w}"] = tr.rolling(w).mean()
    out["true_range"] = pd.concat([
        h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()
    ], axis=1).max(axis=1)

    # Stochastic
    for w in [14]:
        low_min = l.rolling(w).min()
        high_max = h.rolling(w).max()
        out[f"stoch_k_{w}"] = 100 * (c - low_min) / (high_max - low_min).replace(0, np.nan)
        out[f"stoch_d_{w}"] = out[f"stoch_k_{w}"].rolling(3).mean()

    # ADX
    for w in [14]:
        plus_dm = h.diff().clip(lower=0)
        minus_dm = (-l.diff()).clip(upper=0)
        tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
        atr = tr.rolling(w).mean()
        plus_di = 100 * (plus_dm.rolling(w).mean() / atr.replace(0, np.nan))
        minus_di = 100 * (minus_dm.rolling(w).mean() / atr.replace(0, np.nan))
        dx = 100 * ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
        out[f"adx_{w}"] = dx.rolling(w).mean()
        out[f"plus_di_{w}"] = plus_di
        out[f"minus_di_{w}"] = minus_di

    # CCI
    for w in [14, 20]:
        tp = (h + l + c) / 3
        out[f"cci_{w}"] = (tp - tp.rolling(w).mean()) / (0.015 * tp.rolling(w).std())

    # Williams %R
    for w in [14]:
        high_max = h.rolling(w).max()
        low_min = l.rolling(w).min()
        out[f"willr_{w}"] = -100 * (high_max - c) / (high_max - low_min).replace(0, np.nan)

    # MFI
    for w in [14]:
        tp = (h + l + c) / 3
        mf = tp * v
        pos_mf = pd.Series(np.where(tp > tp.shift(1), mf, 0), index=df.index).rolling(w).sum()
        neg_mf = pd.Series(np.where(tp < tp.shift(1), mf, 0), index=df.index).rolling(w).sum()
        mfi = 100 - (100 / (1 + pos_mf / neg_mf.replace(0, np.nan)))
        out[f"mfi_{w}"] = mfi

    # OBV
    obv = pd.Series(np.where(c > c.shift(1), v, np.where(c < c.shift(1), -v, 0)),
                    index=df.index).cumsum()
    out["obv"] = obv

    # VWAP
    tp = (h + l + c) / 3
    out["vwap"] = (tp * v).cumsum() / v.cumsum().replace(0, np.nan)

    # Volatility / momentum
    for w in [5, 10, 20]:
        out[f"volatility_{w}"] = c.pct_change().rolling(w).std()
        out[f"return_{w}"] = c.pct_change(w)
        out[f"log_return_{w}"] = np.log(c / c.shift(w))

    out["return_1"] = c.pct_change(1)
    out["log_return_1"] = np.log(c / c.shift(1))

    # Price ratios / spreads
    out["hl_spread"] = (h - l) / c.replace(0, np.nan)
    out["oc_spread"] = (c - o) / o.replace(0, np.nan)

    # Volume indicators
    for w in [5, 10, 20]:
        out[f"volume_sma_{w}"] = v.rolling(w).mean()
    out["volume_ratio"] = v / v.rolling(20).mean().replace(0, np.nan)

    # Time features
    if isinstance(df.index, pd.DatetimeIndex):
        out["hour"] = df.index.hour
        out["day_of_week"] = df.index.dayofweek
        out["minute"] = df.index.minute
    else:
        out["hour"] = 0
        out["day_of_week"] = 0
        out["minute"] = 0

    return out


def compute_indicators_manual(df: pd.DataFrame, ref_cols: list[str]) -> pd.DataFrame:
    manual = _compute_manual_indicators(df)
    result = df[["open", "high", "low", "close", "volume"]].copy()

    manual_lower = {c.lower(): c for c in manual.columns}
    matched = 0
    for ref in ref_cols:
        rl = ref.lower()
        if rl in manual_lower:
            result[ref] = manual[manual_lower[rl]]
            matched += 1
        else:
            for ml, mc in manual_lower.items():
                if rl.replace("_", "") == ml.replace("_", ""):
                    result[ref] = manual[mc]
                    matched += 1
                    break

    if matched < len(ref_cols) * 0.5:
        print(f"  [INFO] Low match rate ({matched}/{len(ref_cols)}), adding all manual indicators")
        for col in manual.columns:
            if col not in result.columns:
                result[col] = manual[col]

    return result


# ──────────────────────────────────────────────────────────────────────
# 5. Main pipeline
# ──────────────────────────────────────────────────────────────────────
def process_ticker(ticker: str, config: dict, ref_cols: list[str]) -> dict:
    proc_dir = config["PROCESSED_DIR"]
    feat_dir = config["FEATURES_DIR"]
    freq = config["RESAMPLE_FREQ"]

    in_path = os.path.join(proc_dir, f"{ticker}_1min.csv")
    if not os.path.exists(in_path):
        print(f"[SKIP] {in_path} not found")
        return {}

    # ── Read ────────────────────────────────────────────────────────
    df_1min = pd.read_csv(in_path)
    n_1min = len(df_1min)

    # ── Resample ────────────────────────────────────────────────────
    df_15 = resample_ohlcv(df_1min, freq)
    n_15 = len(df_15)

    if n_15 == 0:
        print(f"[ERROR] {ticker}: resample produced 0 rows!")
        return {}

    # Sanity check
    expected = n_1min // 15
    if n_15 < expected * 0.5:
        print(f"  [WARN] {ticker}: got {n_15} 15min bars from {n_1min} 1min bars "
              f"(expected ~{expected})")

    # Save resampled OHLCV
    out_ohlcv = os.path.join(proc_dir, f"{ticker}_15min.csv")
    df_15.to_csv(out_ohlcv)

    # ── Compute indicators ──────────────────────────────────────────
    if ref_cols:
        df_feat = compute_indicators_pandas_ta(df_15, ref_cols)
    else:
        manual = _compute_manual_indicators(df_15)
        df_feat = pd.concat([df_15[["open", "high", "low", "close", "volume"]], manual], axis=1)

    # Drop duplicate columns
    df_feat = df_feat.loc[:, ~df_feat.columns.duplicated()]

    n_features = len([c for c in df_feat.columns
                      if c.lower() not in {"open", "high", "low", "close", "volume", "timestamp"}])

    # Save features
    out_feat = os.path.join(feat_dir, f"{ticker}_15min_features.csv")
    df_feat.index.name = "timestamp"
    df_feat.to_csv(out_feat)

    print(f"  {ticker}: {n_1min:>9,} 1min rows → {n_15:>7,} 15min rows → {n_features} indicator cols")
    return {"ticker": ticker, "n_1min": n_1min, "n_15min": n_15, "n_features": n_features}


def main():
    config = CONFIG
    proc_dir = config["PROCESSED_DIR"]
    feat_dir = config["FEATURES_DIR"]

    if not os.path.isdir(proc_dir):
        print(f"[ERROR] Processed dir not found: {proc_dir}")
        sys.exit(1)
    os.makedirs(feat_dir, exist_ok=True)

    ref_cols = get_reference_indicator_cols(feat_dir)
    if not ref_cols:
        print("[INFO] No 1hour_features reference found – will compute all indicators manually")

    print(f"\nResampling 1min → {config['RESAMPLE_FREQ']} + computing indicators")
    print("=" * 75)
    results = []
    for ticker in config["TICKERS"]:
        info = process_ticker(ticker, config, ref_cols)
        if info:
            results.append(info)

    print("=" * 75)
    if results:
        print(f"Done: {len(results)} tickers processed")
        total_1 = sum(r["n_1min"] for r in results)
        total_15 = sum(r["n_15min"] for r in results)
        print(f"Total: {total_1:,} 1min rows → {total_15:,} 15min rows")
    else:
        print("No tickers processed. Check that processed/{TICKER}_1min.csv files exist.")


if __name__ == "__main__":
    main()

[ref] Using AAPL_1hour_features.csv as reference → 41 indicator columns

Resampling 1min → 15min + computing indicators
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  AAPL:   574,798 1min rows →  43,349 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  MSFT:   498,006 1min rows →  42,922 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  GOOGL:   404,100 1min rows →  39,076 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  GOOG:   372,249 1min rows →  38,153 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  NVDA:   511,269 1min 

In [5]:
import pandas as pd
import os

# Check 1min source files
print("=== 1min source files ===")
for f in sorted(os.listdir("processed")):
    if "_1min" in f:
        df = pd.read_csv(f"processed/{f}", nrows=5)
        full = pd.read_csv(f"processed/{f}")
        print(f"{f}: {len(full)} rows, cols={list(df.columns)}")
        print(f"  first row: {df.iloc[0].to_dict()}")
        print()

# Check 15min feature files
print("=== 15min feature files ===")
for f in sorted(os.listdir("features")):
    if "_15min" in f:
        df = pd.read_csv(f"features/{f}")
        print(f"{f}: {len(df)} rows, {len(df.columns)} cols")
        if len(df) > 0:
            print(f"  first row index/timestamp: {df.iloc[0, 0]}")

=== 1min source files ===
AAPL_1min.csv: 574798 rows, cols=['ts_event', 'open', 'high', 'low', 'close', 'volume']
  first row: {'ts_event': '2020-01-02 09:00:00+00:00', 'open': 73.8125, 'high': 73.8125, 'low': 73.77, 'close': 73.8125, 'volume': 15872.0}

GOOGL_1min.csv: 404100 rows, cols=['ts_event', 'open', 'high', 'low', 'close', 'volume']
  first row: {'ts_event': '2020-01-02 09:00:00+00:00', 'open': 67.65, 'high': 67.65, 'low': 67.65, 'close': 67.65, 'volume': 200.0}

GOOG_1min.csv: 372249 rows, cols=['ts_event', 'open', 'high', 'low', 'close', 'volume']
  first row: {'ts_event': '2020-01-02 09:00:00+00:00', 'open': 67.35, 'high': 67.35, 'low': 67.35, 'close': 67.35, 'volume': 200.0}

MSFT_1min.csv: 498006 rows, cols=['ts_event', 'open', 'high', 'low', 'close', 'volume']
  first row: {'ts_event': '2020-01-02 09:01:00+00:00', 'open': 158.9, 'high': 158.9, 'low': 158.9, 'close': 158.9, 'volume': 2000.0}

NVDA_1min.csv: 511269 rows, cols=['ts_event', 'open', 'high', 'low', 'close', 'v

# mo


In [7]:
"""
multiscale_cnn.py
=================
Multi-Scale CNN for turning point detection.

Short branch: 30×15min bars (7.5h) — immediate reversal patterns
Long branch:  48×1hour bars (2 days) — bull/bear dynamics, momentum decay,
              volume divergence, volatility contraction

Three evaluation levels:
  1. Detection metrics (precision/recall/AUC) vs single-scale CNN
  2. MFE/MAE analysis — do multi-scale candidates have better trade quality?
  3. Meta-label integration — LightGBM filter on multi-scale candidates

Usage (Colab A100):
    !pip install lightgbm --quiet
    !python multiscale_cnn.py
"""

import os, warnings, time, json, math
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# ─── Reproducibility ───
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────── CONFIG ───────────────────────────

CFG = {
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "SHORT_FREQ": "15min",
    "LONG_FREQ": "1hour",
    "SHORT_WIN": 30,
    "LONG_WIN": 48,
    # Zigzag
    "ZIGZAG_ATR_MULT": 2.0,
    "ZIGZAG_ATR_PERIOD": 14,
    "MIN_BARS_BETWEEN": 6,
    "TP_PROXIMITY": 3,
    "NEG_RATIO": 3,
    # Training
    "BATCH": 256,
    "LR": 1e-3,
    "WD": 1e-4,
    "EPOCHS": 100,
    "PATIENCE": 20,
    "SCHED_PATIENCE": 10,
    "GRAD_CLIP": 1.0,
    # Eval
    "DET_THRESHOLDS": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "MFE_MAX_BARS": [12, 24, 48],
    # Meta-label
    "META_TP": 0.005,
    "META_SL": 0.003,
    "META_MB": 48,
    "LGB_PARAMS": {
        "objective": "binary", "metric": "binary_logloss",
        "n_estimators": 500, "max_depth": 5, "learning_rate": 0.05,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "min_child_samples": 20, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "verbose": -1, "random_state": 42,
    },
    "LGB_THRESHOLDS": [0.45, 0.50, 0.55, 0.60],
    # Single-scale baselines (from previous runs)
    "SINGLE_BASELINES": {
        "bottom_auc": 0.822, "top_auc": 0.798,
        "bottom_prec_07": 0.085, "top_prec_07": 0.081,
        "bottom_mfe_mae_24": 0.69, "top_mfe_mae_24": 0.78,
        "bottom_lgb_wr_05": 0.818, "top_lgb_wr_05": 0.459,
    },
    "OUTPUT_DIR": "results",
    # Set to a list of tasks to skip L1/L2 for (loads saved predictions).
    # E.g. ["bottom"] to skip bottom but run top fully.
    # Set to True to skip all, False to skip none.
    "SKIP_L1_L2": ["bottom", "top"],
}


# ═══════════════════════════════════════════════════════════════
#  DATA LOADING
# ═══════════════════════════════════════════════════════════════

def load_csv(ticker, freq, cfg):
    for f in [freq, "1hour" if freq == "1h" else freq]:
        p = os.path.join(cfg["FEATURES_DIR"], f"{ticker}_{f}_features.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            for col in ["timestamp", "ts_event", "datetime", "date"]:
                if col in df.columns:
                    df["timestamp"] = pd.to_datetime(df[col], utc=True)
                    break
            if "timestamp" not in df.columns:
                first = df.columns[0]
                if first not in df.select_dtypes(include=[np.number]).columns:
                    df["timestamp"] = pd.to_datetime(df[first], utc=True)
            df = df.sort_values("timestamp").reset_index(drop=True)
            return df
    raise FileNotFoundError(f"No {freq} features for {ticker}")


def numeric_cols(df, exclude=None):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    if exclude:
        ex |= set(c.lower() for c in exclude)
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


# ═══════════════════════════════════════════════════════════════
#  1-HOUR DYNAMICS FEATURES
# ═══════════════════════════════════════════════════════════════

def add_dynamics_features(df):
    """Add 8 'dynamics' columns to 1hour DataFrame. Uses only past data."""
    close = df["close"].values.astype(float)
    open_ = df["open"].values.astype(float) if "open" in df.columns else close
    high = df["high"].values.astype(float) if "high" in df.columns else close
    low = df["low"].values.astype(float) if "low" in df.columns else close
    vol = df["volume"].values.astype(float) if "volume" in df.columns else np.ones(len(df))
    n = len(df)

    # 1. buyer_seller_ratio (rolling 12)
    up_bar = (close > open_).astype(float)
    down_bar = (close < open_).astype(float)
    bsr = np.full(n, 1.0)
    for i in range(12, n):
        seg_up = up_bar[i - 12:i]
        seg_dn = down_bar[i - 12:i]
        seg_vol = vol[i - 12:i]
        up_vol = np.mean(seg_vol[seg_up == 1]) if seg_up.sum() > 0 else 0
        dn_vol = np.mean(seg_vol[seg_dn == 1]) if seg_dn.sum() > 0 else 1e-10
        bsr[i] = up_vol / (dn_vol + 1e-10)
    df["dyn_buyer_seller_ratio"] = bsr

    # 2. rsi_slope_12
    rsi_col = None
    for c in df.columns:
        if c.lower() in ("rsi_14", "rsi"):
            rsi_col = c
            break
    rsi_slope = np.zeros(n)
    if rsi_col is not None:
        rsi = df[rsi_col].values.astype(float)
        x = np.arange(12, dtype=float)
        x_mean = x.mean()
        x_var = ((x - x_mean) ** 2).sum()
        for i in range(12, n):
            seg = rsi[i - 12:i]
            if np.any(np.isnan(seg)):
                continue
            rsi_slope[i] = np.sum((x - x_mean) * (seg - seg.mean())) / (x_var + 1e-10)
    df["dyn_rsi_slope_12"] = rsi_slope

    # 3. macd_accel
    macd_hist_col = None
    for c in df.columns:
        cl = c.lower()
        if "macd" in cl and ("hist" in cl or "diff" in cl):
            macd_hist_col = c
            break
    macd_accel = np.zeros(n)
    if macd_hist_col is not None:
        mh = df[macd_hist_col].values.astype(float)
        for i in range(3, n):
            if not np.isnan(mh[i]) and not np.isnan(mh[i - 3]):
                macd_accel[i] = mh[i] - mh[i - 3]
    df["dyn_macd_accel"] = macd_accel

    # 4. volume_trend (slope of log(vol) over 12 bars)
    vol_trend = np.zeros(n)
    log_vol = np.log(vol + 1)
    x12 = np.arange(12, dtype=float)
    x12m = x12.mean()
    x12v = ((x12 - x12m) ** 2).sum()
    for i in range(12, n):
        seg = log_vol[i - 12:i]
        vol_trend[i] = np.sum((x12 - x12m) * (seg - seg.mean())) / (x12v + 1e-10)
    df["dyn_volume_trend"] = vol_trend

    # 5. atr_change_12
    atr_col = None
    for c in df.columns:
        if c.lower() in ("atr_14", "atr"):
            atr_col = c
            break
    atr_change = np.zeros(n)
    if atr_col is not None:
        atr = df[atr_col].values.astype(float)
        for i in range(12, n):
            if atr[i - 12] > 0 and not np.isnan(atr[i]) and not np.isnan(atr[i - 12]):
                atr_change[i] = (atr[i] - atr[i - 12]) / (atr[i - 12] + 1e-10)
    df["dyn_atr_change_12"] = atr_change

    # 6. lower_shadow_ratio (rolling 6)
    body_low = np.minimum(open_, close)
    full_range = high - low + 1e-10
    lower_shadow = (body_low - low) / full_range
    lsr = np.zeros(n)
    for i in range(6, n):
        lsr[i] = np.mean(lower_shadow[i - 6:i])
    df["dyn_lower_shadow_ratio"] = lsr

    # 7. price_position_48
    pp48 = np.full(n, 0.5)
    for i in range(48, n):
        hh = np.max(high[i - 48:i + 1])
        ll = np.min(low[i - 48:i + 1])
        rng = hh - ll
        pp48[i] = (close[i] - ll) / (rng + 1e-10) if rng > 0 else 0.5
    df["dyn_price_position_48"] = pp48

    # 8. consecutive_down
    consec = np.zeros(n)
    for i in range(1, n):
        if close[i] < close[i - 1]:
            consec[i] = consec[i - 1] + 1
        else:
            consec[i] = 0
    df["dyn_consecutive_down"] = consec

    return df


# ═══════════════════════════════════════════════════════════════
#  ZIGZAG LABELLING (ATR-adaptive, on 15min)
# ═══════════════════════════════════════════════════════════════

def compute_atr(high, low, close, period=14):
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i] - low[i],
                     abs(high[i] - close[i - 1]),
                     abs(low[i] - close[i - 1]))
    atr = np.full(n, np.nan)
    if n >= period:
        atr[period - 1] = np.mean(tr[:period])
        for i in range(period, n):
            atr[i] = (atr[i - 1] * (period - 1) + tr[i]) / period
    return atr


def zigzag_atr(high, low, close, atr_mult=2.0, atr_period=14,
               min_bars=6):
    """ATR-adaptive zigzag. Returns (bottom_indices, top_indices)."""
    n = len(close)
    atr = compute_atr(high, low, close, atr_period)
    # Fill NaN ATR with first valid
    first_valid = np.where(~np.isnan(atr))[0]
    if len(first_valid) > 0:
        atr[:first_valid[0]] = atr[first_valid[0]]
    atr = np.nan_to_num(atr, nan=close.mean() * 0.01)

    bottoms, tops = [], []
    direction = 0  # 0=undecided, 1=looking for top, -1=looking for bottom
    last_high_idx = 0
    last_low_idx = 0
    last_high = high[0]
    last_low = low[0]
    last_tp_idx = -min_bars * 2

    for i in range(1, n):
        threshold = atr[i] * atr_mult

        if direction >= 0:  # looking for top or undecided
            if high[i] > last_high:
                last_high = high[i]
                last_high_idx = i
            if last_high - low[i] >= threshold and i - last_tp_idx >= min_bars:
                if last_high_idx != 0:
                    tops.append(last_high_idx)
                    last_tp_idx = last_high_idx
                direction = -1
                last_low = low[i]
                last_low_idx = i

        if direction <= 0:  # looking for bottom or undecided
            if low[i] < last_low:
                last_low = low[i]
                last_low_idx = i
            if high[i] - last_low >= threshold and i - last_tp_idx >= min_bars:
                if last_low_idx != 0:
                    bottoms.append(last_low_idx)
                    last_tp_idx = last_low_idx
                direction = 1
                last_high = high[i]
                last_high_idx = i

    return np.array(bottoms, dtype=int), np.array(tops, dtype=int)


def make_labels(n, tp_indices, proximity=3):
    """Binary labels: 1 if within `proximity` bars of a TP."""
    labels = np.zeros(n, dtype=np.float32)
    for idx in tp_indices:
        lo = max(0, idx - proximity)
        hi = min(n, idx + proximity + 1)
        labels[lo:hi] = 1.0
    return labels


# ═══════════════════════════════════════════════════════════════
#  ALIGNMENT: 15min → 1hour mapping
# ═══════════════════════════════════════════════════════════════

def build_hour_index_map(ts_15min, ts_1hour):
    """
    For each 15min timestamp, find the index of the latest 1hour bar
    at or before that time. Returns array of 1hour indices (or -1 if none).
    """
    ts_h = ts_1hour.values.astype(np.int64)
    ts_s = ts_15min.values.astype(np.int64)
    hour_idx = np.full(len(ts_s), -1, dtype=int)
    j = 0
    for i in range(len(ts_s)):
        while j < len(ts_h) - 1 and ts_h[j + 1] <= ts_s[i]:
            j += 1
        if ts_h[j] <= ts_s[i]:
            hour_idx[i] = j
    return hour_idx


# ═══════════════════════════════════════════════════════════════
#  DATASET CONSTRUCTION
# ═══════════════════════════════════════════════════════════════

def build_paired_windows(cfg):
    """
    Build aligned (short_window, long_window, label, meta) for all tickers.
    Returns dict with train/val/test splits.
    """
    print(f"\n  Building paired windows...")
    short_win = cfg["SHORT_WIN"]
    long_win = cfg["LONG_WIN"]
    prox = cfg["TP_PROXIMITY"]

    all_data = {"short": [], "long": [], "bottom_label": [], "top_label": [],
                "ticker": [], "bar_idx": [], "timestamp": []}

    for ticker in cfg["TICKERS"]:
        try:
            df_s = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            df_l = load_csv(ticker, cfg["LONG_FREQ"], cfg)
        except FileNotFoundError as e:
            print(f"    [SKIP] {ticker}: {e}")
            continue

        # Dynamics features on 1hour
        df_l = add_dynamics_features(df_l)

        # Feature columns
        s_cols = numeric_cols(df_s)
        l_cols = numeric_cols(df_l)

        s_feat = df_s[s_cols].values.astype(np.float32)
        l_feat = df_l[l_cols].values.astype(np.float32)

        # NaN → 0
        s_feat = np.nan_to_num(s_feat, nan=0.0, posinf=0.0, neginf=0.0)
        l_feat = np.nan_to_num(l_feat, nan=0.0, posinf=0.0, neginf=0.0)

        # Zigzag labels on 15min
        close_s = df_s["close"].values.astype(float)
        high_s = df_s["high"].values.astype(float) if "high" in df_s.columns else close_s
        low_s = df_s["low"].values.astype(float) if "low" in df_s.columns else close_s

        bottoms, tops = zigzag_atr(high_s, low_s, close_s,
                                   cfg["ZIGZAG_ATR_MULT"],
                                   cfg["ZIGZAG_ATR_PERIOD"],
                                   cfg["MIN_BARS_BETWEEN"])
        bottom_labels = make_labels(len(df_s), bottoms, prox)
        top_labels = make_labels(len(df_s), tops, prox)

        # Alignment map
        hour_idx_map = build_hour_index_map(df_s["timestamp"], df_l["timestamp"])

        # Build windows
        n_short = len(df_s)
        count = 0
        for i in range(short_win, n_short):
            hi = hour_idx_map[i]
            if hi < long_win:
                continue  # not enough 1hour history

            short_window = s_feat[i - short_win:i]  # (30, N_s)
            long_window = l_feat[hi - long_win + 1:hi + 1]  # (48, N_l)

            if short_window.shape[0] != short_win or long_window.shape[0] != long_win:
                continue

            all_data["short"].append(short_window)
            all_data["long"].append(long_window)
            all_data["bottom_label"].append(bottom_labels[i])
            all_data["top_label"].append(top_labels[i])
            all_data["ticker"].append(ticker)
            all_data["bar_idx"].append(i)
            all_data["timestamp"].append(str(df_s["timestamp"].iloc[i]))
            count += 1

        n_bot = int(bottom_labels[short_win:].sum())
        n_top = int(top_labels[short_win:].sum())
        print(f"    {ticker}: {count:,} samples, "
              f"bottoms={n_bot} tops={n_top}  "
              f"s_feat={s_feat.shape[1]} l_feat={l_feat.shape[1]}")

    # Convert to arrays
    X_short = np.array(all_data["short"], dtype=np.float32)
    X_long = np.array(all_data["long"], dtype=np.float32)
    y_bottom = np.array(all_data["bottom_label"], dtype=np.float32)
    y_top = np.array(all_data["top_label"], dtype=np.float32)
    tickers = np.array(all_data["ticker"])
    bar_indices = np.array(all_data["bar_idx"])
    timestamps = np.array(all_data["timestamp"])

    n = len(y_bottom)
    print(f"\n  Total: {n:,} samples")
    print(f"  Short shape: {X_short.shape}  Long shape: {X_long.shape}")
    print(f"  Bottom pos rate: {y_bottom.mean():.4f}  "
          f"Top pos rate: {y_top.mean():.4f}")

    # Time-based split (data already sorted by time within each ticker,
    # but we sort globally by bar_idx proxy — use index order since tickers processed sequentially)
    # Actually split per-ticker to preserve time ordering
    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)

    for ticker in cfg["TICKERS"]:
        t_mask = tickers == ticker
        t_idx = np.where(t_mask)[0]
        if len(t_idx) == 0:
            continue
        n_t = len(t_idx)
        tr_end = int(n_t * 0.70)
        va_end = int(n_t * 0.85)
        train_mask[t_idx[:tr_end]] = True
        val_mask[t_idx[tr_end:va_end]] = True
        test_mask[t_idx[va_end:]] = True

    print(f"  Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  "
          f"Test: {test_mask.sum():,}")

    return {
        "X_short": X_short, "X_long": X_long,
        "y_bottom": y_bottom, "y_top": y_top,
        "tickers": tickers, "bar_indices": bar_indices,
        "timestamps": timestamps,
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "n_short_feat": X_short.shape[2], "n_long_feat": X_long.shape[2],
    }


# ═══════════════════════════════════════════════════════════════
#  NORMALIZE + UNDERSAMPLE
# ═══════════════════════════════════════════════════════════════

def normalize_windows(X_train, *others):
    """Fit scaler on train, transform all. X shape: (N, T, F)."""
    n, t, f = X_train.shape
    flat = X_train.reshape(-1, f)
    mu = np.nanmean(flat, axis=0)
    sigma = np.nanstd(flat, axis=0) + 1e-8
    mu = np.nan_to_num(mu, nan=0.0)
    sigma = np.where(np.isnan(sigma) | (sigma < 1e-8), 1.0, sigma)

    results = []
    for arr in (X_train,) + others:
        normed = (arr - mu) / sigma
        np.nan_to_num(normed, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        results.append(normed)
    return tuple(results) + (mu, sigma)


def undersample_negatives(X_s, X_l, y, neg_ratio=3, seed=42):
    """Undersample negatives to neg_ratio:1."""
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    n_neg_keep = min(n_pos * neg_ratio, len(neg_idx))
    rng = np.random.RandomState(seed)
    neg_keep = rng.choice(neg_idx, size=n_neg_keep, replace=False)
    keep = np.sort(np.concatenate([pos_idx, neg_keep]))
    return X_s[keep], X_l[keep], y[keep]


# ═══════════════════════════════════════════════════════════════
#  MODEL
# ═══════════════════════════════════════════════════════════════

class MultiScaleCNN(nn.Module):
    def __init__(self, n_short_feat, n_long_feat):
        super().__init__()
        # Short branch (15min, 30 bars)
        self.short_branch = nn.Sequential(
            nn.Conv1d(n_short_feat, 32, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        # Long branch (1hour, 48 bars)
        self.long_branch = nn.Sequential(
            nn.Conv1d(n_long_feat, 32, kernel_size=7, padding=3),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        # Fusion
        self.head = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x_short, x_long):
        # Input: (B, T, F) → Conv1d wants (B, F, T)
        s = self.short_branch(x_short.transpose(1, 2))
        l = self.long_branch(x_long.transpose(1, 2))
        fused = torch.cat([s, l], dim=1)
        return self.head(fused).squeeze(-1)


class PairedDataset(Dataset):
    def __init__(self, X_short, X_long, y):
        self.xs = torch.tensor(X_short, dtype=torch.float32)
        self.xl = torch.tensor(X_long, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.xs[i], self.xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  TRAINING
# ═══════════════════════════════════════════════════════════════

def train_model(X_s_tr, X_l_tr, y_tr, X_s_va, X_l_va, y_va,
                n_short_feat, n_long_feat, cfg, task):
    """Train MultiScaleCNN for one task."""
    print(f"\n    Training MultiScaleCNN ({task})...")
    # Undersample training
    X_s_tr_u, X_l_tr_u, y_tr_u = undersample_negatives(
        X_s_tr, X_l_tr, y_tr, cfg["NEG_RATIO"])
    print(f"    After undersample: {len(y_tr_u):,} "
          f"(pos={y_tr_u.sum():.0f}, neg={len(y_tr_u)-y_tr_u.sum():.0f})")

    model = MultiScaleCNN(n_short_feat, n_long_feat).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Params: {n_params:,}")

    pos_weight = torch.tensor(
        [(1 - y_tr_u.mean()) / (y_tr_u.mean() + 1e-8)],
        dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["LR"],
                                  weight_decay=cfg["WD"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=cfg["SCHED_PATIENCE"])

    train_dl = DataLoader(PairedDataset(X_s_tr_u, X_l_tr_u, y_tr_u),
                          batch_size=cfg["BATCH"], shuffle=True,
                          num_workers=0, pin_memory=True)
    val_dl = DataLoader(PairedDataset(X_s_va, X_l_va, y_va),
                        batch_size=cfg["BATCH"] * 4, shuffle=False,
                        num_workers=0, pin_memory=True)

    best_loss = float("inf")
    best_state = None
    patience_ctr = 0
    best_epoch = 0

    for epoch in range(cfg["EPOCHS"]):
        model.train()
        t_loss = 0.0
        for xs, xl, yb in train_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xs, xl)
            if torch.isnan(logits).any():
                continue
            loss = criterion(logits, yb)
            if torch.isnan(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["GRAD_CLIP"])
            optimizer.step()
            t_loss += loss.item() * len(yb)
        t_loss /= max(len(y_tr_u), 1)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for xs, xl, yb in val_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                logits = model(xs, xl)
                v_loss += criterion(logits, yb).item() * len(yb)
        v_loss /= max(len(y_va), 1)

        scheduler.step(v_loss)
        is_best = v_loss < best_loss
        if is_best:
            best_loss = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            best_epoch = epoch + 1
        else:
            patience_ctr += 1

        lr = optimizer.param_groups[0]["lr"]
        if (epoch + 1) % 10 == 0 or is_best or patience_ctr == cfg["PATIENCE"]:
            tag = "  *best" if is_best else ""
            print(f"      E{epoch+1:3d} t={t_loss:.6f} v={v_loss:.6f} "
                  f"lr={lr:.1e} p={patience_ctr}{tag}")
        if patience_ctr >= cfg["PATIENCE"]:
            print(f"      Early stop (best={best_epoch})")
            break

    if best_state:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    return model


def predict(model, X_s, X_l, batch_size=1024):
    model.eval()
    dl = DataLoader(PairedDataset(X_s, X_l, np.zeros(len(X_s))),
                    batch_size=batch_size, shuffle=False,
                    num_workers=0, pin_memory=True)
    preds = []
    with torch.no_grad():
        for xs, xl, _ in dl:
            logits = model(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy()
            probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -20, 20)))
            preds.append(np.nan_to_num(probs, nan=0.5))
    return np.concatenate(preds)


# ═══════════════════════════════════════════════════════════════
#  LEVEL 1: DETECTION METRICS
# ═══════════════════════════════════════════════════════════════

def eval_detection(probs, y_true, thresholds, baseline_rate):
    """Precision, recall, F1 at each threshold + AUC."""
    auc = roc_auc_score(y_true, probs) if y_true.sum() > 0 and (1 - y_true).sum() > 0 else 0.5
    results = {"auc": auc, "n": len(y_true),
               "n_pos": int(y_true.sum()),
               "baseline_rate": baseline_rate}
    for th in thresholds:
        pred = (probs >= th).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        n_pred = tp + fp
        # z-score: precision vs baseline rate
        se = np.sqrt(baseline_rate * (1 - baseline_rate) / n_pred) if n_pred > 0 and 0 < baseline_rate < 1 else 1
        z = (prec - baseline_rate) / se if se > 0 and n_pred > 0 else 0
        results[th] = {"precision": prec, "recall": rec, "f1": f1,
                       "n_pred": n_pred, "tp": tp, "z": z}
    return results


def print_detection(results, task, single_auc):
    print(f"\n  {task.upper()} — Detection Metrics  "
          f"(AUC={results['auc']:.3f}  single-scale={single_auc:.3f}  "
          f"Δ={results['auc']-single_auc:+.3f})")
    print(f"  {'Th':>5} {'Prec':>8} {'Recall':>8} {'F1':>8} "
          f"{'N_pred':>8} {'TP':>6} {'z':>7}")
    print(f"  {'-'*52}")
    for th in sorted(k for k in results if isinstance(k, float)):
        r = results[th]
        sig = "*" if abs(r["z"]) > 1.96 else ""
        print(f"  {th:>5.1f} {r['precision']*100:>7.2f}% "
              f"{r['recall']*100:>7.1f}% {r['f1']*100:>7.2f}% "
              f"{r['n_pred']:>8} {r['tp']:>6} {r['z']:>+6.2f}{sig}")


# ═══════════════════════════════════════════════════════════════
#  LEVEL 2: MFE/MAE ANALYSIS
# ═══════════════════════════════════════════════════════════════

def compute_mfe_mae_for_candidates(cfg, probs, tickers, bar_indices,
                                    test_mask, task, threshold=0.5):
    """Compute MFE/MAE for multi-scale CNN candidates."""
    direction = "long" if task == "bottom" else "short"
    cand_mask = test_mask & (probs >= threshold)
    cand_idx = np.where(cand_mask)[0]

    if len(cand_idx) == 0:
        return {}

    # Group by ticker
    results = {}
    for mb in cfg["MFE_MAX_BARS"]:
        all_mfe, all_mae, all_probs = [], [], []
        for ticker in cfg["TICKERS"]:
            try:
                df = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            except FileNotFoundError:
                continue
            close = df["close"].values.astype(float)
            high = df["high"].values.astype(float) if "high" in df.columns else close
            low = df["low"].values.astype(float) if "low" in df.columns else close

            t_cand = cand_idx[(tickers[cand_idx] == ticker)]
            for ci in t_cand:
                bi = bar_indices[ci]
                if bi + 1 >= len(close):
                    continue
                end = min(bi + mb + 1, len(close))
                if bi + 1 >= end:
                    continue
                seg_h = high[bi + 1:end]
                seg_l = low[bi + 1:end]
                entry = close[bi]
                if direction == "long":
                    mfe = max((np.max(seg_h) - entry) / entry, 0)
                    mae = max((entry - np.min(seg_l)) / entry, 0)
                else:
                    mfe = max((entry - np.min(seg_l)) / entry, 0)
                    mae = max((np.max(seg_h) - entry) / entry, 0)
                all_mfe.append(mfe)
                all_mae.append(mae)
                all_probs.append(probs[ci])

        all_mfe = np.array(all_mfe)
        all_mae = np.array(all_mae)
        all_probs = np.array(all_probs)
        mfe_med = np.median(all_mfe) if len(all_mfe) > 0 else 0
        mae_med = np.median(all_mae) if len(all_mae) > 0 else 0
        ratio = mfe_med / mae_med if mae_med > 0 else 999

        results[mb] = {
            "n": len(all_mfe), "mfe_med": float(mfe_med),
            "mae_med": float(mae_med), "ratio": float(ratio),
            "mfe": all_mfe, "mae": all_mae, "probs": all_probs,
        }

        print(f"    MB={mb}: N={len(all_mfe):,}  "
              f"MFE_med={mfe_med*100:.3f}%  MAE_med={mae_med*100:.3f}%  "
              f"ratio={ratio:.3f}")

    # Probability bucketed analysis (using MB=24)
    best_mb = 24 if 24 in results else list(results.keys())[0]
    d = results[best_mb]
    if d["n"] > 0:
        print(f"\n    Prob-bucketed MFE/MAE (MB={best_mb}):")
        print(f"    {'Bucket':>12} {'N':>6} {'MFE_med':>9} {'MAE_med':>9} "
              f"{'Ratio':>7}")
        print(f"    {'-'*48}")
        buckets = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]
        for lo, hi in buckets:
            mask = (d["probs"] >= lo) & (d["probs"] < hi)
            n_b = mask.sum()
            if n_b < 5:
                print(f"    [{lo:.1f}-{hi:.1f}) {n_b:>6}  —")
                continue
            mfe_b = np.median(d["mfe"][mask])
            mae_b = np.median(d["mae"][mask])
            r_b = mfe_b / mae_b if mae_b > 0 else 999
            print(f"    [{lo:.1f}-{hi:.1f}) {n_b:>6} "
                  f"{mfe_b*100:>8.3f}% {mae_b*100:>8.3f}% {r_b:>7.3f}")

    return results


# ═══════════════════════════════════════════════════════════════
#  LEVEL 3: META-LABEL (LightGBM)
# ═══════════════════════════════════════════════════════════════

def triple_barrier_vectorized(close_arr, high_arr, low_arr,
                              indices, direction, tp_pct, sl_pct, max_bars):
    """
    Vectorized triple-barrier labelling — processes ALL candidates simultaneously.

    For each horizon step h=1..max_bars, computes TP/SL hits across all
    still-open positions in a single numpy pass.  Turns O(N * max_bars) Python
    iterations into O(max_bars) numpy vectorised operations.

    Returns: labels (int array), pnls (float array)
    """
    n_price = len(close_arr)
    n_cand = len(indices)

    entries = close_arr[indices]                       # (n_cand,)
    labels = np.zeros(n_cand, dtype=np.int32)
    pnls = np.zeros(n_cand, dtype=np.float64)
    still_open = np.ones(n_cand, dtype=bool)

    for h in range(1, max_bars + 1):
        if not still_open.any():
            break

        future_idx = indices + h
        valid = still_open & (future_idx < n_price)
        if not valid.any():
            continue

        vi = np.where(valid)[0]                        # indices into candidates
        fi = future_idx[vi]                            # indices into price arrays
        ent = entries[vi]

        if direction == "long":
            tp_hit = (high_arr[fi] - ent) / ent >= tp_pct
            sl_hit = (ent - low_arr[fi]) / ent >= sl_pct
        else:
            tp_hit = (ent - low_arr[fi]) / ent >= tp_pct
            sl_hit = (high_arr[fi] - ent) / ent >= sl_pct

        # Both hit same bar → conservative: SL wins
        both = tp_hit & sl_hit
        pure_tp = tp_hit & ~sl_hit
        pure_sl = sl_hit & ~tp_hit

        # TP exits
        tp_idx = vi[pure_tp]
        labels[tp_idx] = 1
        pnls[tp_idx] = tp_pct
        still_open[tp_idx] = False

        # SL exits (including both-hit)
        sl_idx = vi[pure_sl | both]
        labels[sl_idx] = 0
        pnls[sl_idx] = -sl_pct
        still_open[sl_idx] = False

    # Timeout exits — mark-to-market PnL
    to_mask = still_open
    if to_mask.any():
        to_idx = np.where(to_mask)[0]
        exit_idx = np.minimum(indices[to_idx] + max_bars, n_price - 1)
        exit_p = close_arr[exit_idx]
        if direction == "long":
            pnls[to_idx] = (exit_p - entries[to_idx]) / entries[to_idx]
        else:
            pnls[to_idx] = (entries[to_idx] - exit_p) / entries[to_idx]
        # labels already 0

    return labels, pnls


def build_meta_features(X_short, X_long, probs):
    """Flatten last few bars + CNN prob as LGB features."""
    n = len(probs)
    # Short: last 5 bars flattened
    s_last = X_short[:, -5:, :].reshape(n, -1)
    # Long: last 3 bars flattened
    l_last = X_long[:, -3:, :].reshape(n, -1)
    # CNN prob
    prob_col = probs.reshape(-1, 1)
    return np.nan_to_num(np.hstack([s_last, l_last, prob_col]),
                         nan=0.0, posinf=0.0, neginf=0.0)


def run_meta_label(cfg, probs, X_short, X_long, tickers, bar_indices,
                   test_mask, task, threshold):
    """Level 3: LightGBM meta-label on multi-scale CNN candidates.

    Fixed: pre-loads ticker CSVs once (8 reads, not 50k), then uses
    vectorized triple-barrier labelling per ticker.
    """
    if not HAS_LGB:
        print("    [SKIP] LightGBM not installed")
        return {}

    direction = "long" if task == "bottom" else "short"
    cand_mask = test_mask & (probs >= threshold)
    cand_idx = np.where(cand_mask)[0]
    if len(cand_idx) < 50:
        print(f"    [SKIP] Only {len(cand_idx)} candidates")
        return {}

    print(f"    Candidates: {len(cand_idx):,}")

    # ── Step A: Pre-load all ticker price data ONCE ──
    t0 = time.time()
    ticker_data = {}  # ticker → (close, high, low)
    for ticker in cfg["TICKERS"]:
        try:
            df = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            close = df["close"].values.astype(np.float64)
            high = df["high"].values.astype(np.float64) if "high" in df.columns else close.copy()
            low = df["low"].values.astype(np.float64) if "low" in df.columns else close.copy()
            ticker_data[ticker] = (close, high, low)
        except FileNotFoundError:
            pass
    print(f"    Loaded {len(ticker_data)} ticker CSVs  ({time.time()-t0:.2f}s)")

    # ── Step B: Vectorized triple-barrier labelling per ticker ──
    t1 = time.time()
    all_labels = np.full(len(cand_idx), -1, dtype=np.int32)
    all_pnls = np.zeros(len(cand_idx), dtype=np.float64)
    valid_mask = np.zeros(len(cand_idx), dtype=bool)

    cand_tickers = tickers[cand_idx]
    cand_bars = bar_indices[cand_idx]

    for ticker, (close, high, low) in ticker_data.items():
        t_mask = cand_tickers == ticker
        if not t_mask.any():
            continue
        t_positions = np.where(t_mask)[0]       # positions within cand_idx
        t_bar_idx = cand_bars[t_positions]       # bar indices into price arrays

        # Filter out-of-range
        ok = (t_bar_idx >= 1) & (t_bar_idx < len(close) - 1)
        if not ok.any():
            continue
        t_positions = t_positions[ok]
        t_bar_idx = t_bar_idx[ok]

        labs, pnls_v = triple_barrier_vectorized(
            close, high, low, t_bar_idx, direction,
            cfg["META_TP"], cfg["META_SL"], cfg["META_MB"])

        all_labels[t_positions] = labs
        all_pnls[t_positions] = pnls_v
        valid_mask[t_positions] = True

    # Keep only valid candidates
    valid_cand = cand_idx[valid_mask]
    labels = all_labels[valid_mask].astype(np.float32)
    pnls = all_pnls[valid_mask].astype(np.float32)
    print(f"    Triple-barrier labelling: {len(labels):,} trades  ({time.time()-t1:.2f}s)")

    if len(valid_cand) < 50:
        print(f"    [SKIP] Only {len(valid_cand)} valid candidates")
        return {}

    # ── Step C: Build features ──
    t2 = time.time()
    X_meta = build_meta_features(X_short[valid_cand],
                                 X_long[valid_cand],
                                 probs[valid_cand])
    print(f"    Feature matrix: {X_meta.shape}  ({time.time()-t2:.2f}s)")

    # ── Step D: Train/test split + LightGBM ──
    t3 = time.time()
    n_c = len(labels)
    split = int(n_c * 0.7)
    X_tr, y_tr, pnl_tr = X_meta[:split], labels[:split], pnls[:split]
    X_te, y_te, pnl_te = X_meta[split:], labels[split:], pnls[split:]

    base_rate = float(y_te.mean())
    print(f"    Meta-label: train={len(y_tr):,} test={len(y_te):,} "
          f"base_wr={base_rate*100:.1f}%")

    val_split = int(len(y_tr) * 0.8)
    params = dict(cfg["LGB_PARAMS"])
    n_est = params.pop("n_estimators", 500)
    model = lgb.LGBMClassifier(n_estimators=n_est, **params)
    model.fit(X_tr[:val_split], y_tr[:val_split],
              eval_set=[(X_tr[val_split:], y_tr[val_split:])],
              callbacks=[lgb.early_stopping(30, verbose=False),
                         lgb.log_evaluation(0)])
    print(f"    LightGBM training  ({time.time()-t3:.2f}s)")

    # ── Step E: Evaluate ──
    t4 = time.time()
    lgb_probs = model.predict_proba(X_te)[:, 1]

    results = {"base_rate": base_rate, "n_test": len(y_te)}
    print(f"\n    {'Th':>5} {'WR':>8} {'N':>6} {'PF':>7} {'TotPnL':>9}")
    print(f"    {'-'*42}")
    for th in cfg["LGB_THRESHOLDS"]:
        m = lgb_probs >= th
        n_t = int(m.sum())
        if n_t == 0:
            results[th] = {"wr": 0, "n": 0, "pf": 0, "pnl": 0}
            continue
        wr = float(y_te[m].mean())
        ps = pnl_te[m]
        gp = ps[ps > 0].sum()
        gl = abs(ps[ps < 0].sum())
        pf = gp / gl if gl > 0 else (999 if gp > 0 else 0)
        results[th] = {"wr": wr, "n": n_t,
                       "pf": float(pf), "pnl": float(ps.sum())}
        print(f"    {th:>5.2f} {wr*100:>7.1f}% {n_t:>6} "
              f"{pf:>7.2f} {ps.sum()*100:>8.3f}%")
    print(f"    Evaluation  ({time.time()-t4:.2f}s)")

    # ── 保存 Meta LGB 权重 ──
    import joblib
    meta_dir = os.path.join(cfg["OUTPUT_DIR"],
                            f"models/layer3/trade_filter/lgb_{task}_v1")
    os.makedirs(meta_dir, exist_ok=True)
    joblib.dump(model, os.path.join(meta_dir, "weights.joblib"))
    with open(os.path.join(meta_dir, "meta.json"), "w") as _mf:
        json.dump({
            "adapter": "lightgbm",
            "output": "probability",
            "task": task,
            "direction": "long" if task == "bottom" else "short",
            "metrics": {
                "base_wr": float(base_rate),
                "wr_at_0.45": float(results.get(0.45, {}).get("wr", 0)),
                "wr_at_0.50": float(results.get(0.50, {}).get("wr", 0)),
                "n_test": len(y_te),
            }
        }, _mf, indent=2)
    print(f"    Saved Meta LGB ({task}): {meta_dir}/weights.joblib")

    return results


# ═══════════════════════════════════════════════════════════════
#  SAVE PREDICTIONS
# ═══════════════════════════════════════════════════════════════

def save_predictions(probs, tickers, bar_indices, timestamps,
                     test_mask, cfg, task):
    rows = []
    test_idx = np.where(test_mask)[0]
    for i in test_idx:
        rows.append({
            "ticker": tickers[i],
            "bar_index": int(bar_indices[i]),
            "timestamp": timestamps[i],
            "prob": float(probs[i]),
        })
    df = pd.DataFrame(rows)
    path = os.path.join(cfg["OUTPUT_DIR"],
                        f"multiscale_cnn_{task}_predictions.csv")
    df.to_csv(path, index=False)
    print(f"  Saved: {path} ({len(df)} rows)")


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    cfg = CFG
    print("=" * 78)
    print("  Multi-Scale CNN for Turning Point Detection")
    print(f"  Short: {cfg['SHORT_WIN']}×{cfg['SHORT_FREQ']}  "
          f"Long: {cfg['LONG_WIN']}×{cfg['LONG_FREQ']}")
    print(f"  Device: {DEVICE}")
    print("=" * 78)

    os.makedirs(cfg["OUTPUT_DIR"], exist_ok=True)

    # ── Build data ──
    data = build_paired_windows(cfg)

    X_s = data["X_short"]
    X_l = data["X_long"]
    tr = data["train_mask"]
    va = data["val_mask"]
    te = data["test_mask"]

    # Normalize short and long branches separately
    X_s_tr, X_s_va, X_s_te, _, _ = normalize_windows(
        X_s[tr], X_s[va], X_s[te])
    X_l_tr, X_l_va, X_l_te, _, _ = normalize_windows(
        X_l[tr], X_l[va], X_l[te])

    # Reassemble full arrays (needed for prediction indexing)
    X_s_full = np.zeros_like(X_s)
    X_l_full = np.zeros_like(X_l)
    X_s_full[tr] = X_s_tr
    X_s_full[va] = X_s_va
    X_s_full[te] = X_s_te
    X_l_full[tr] = X_l_tr
    X_l_full[va] = X_l_va
    X_l_full[te] = X_l_te

    all_results = {}

    for task in ["bottom", "top"]:
        y_col = data["y_bottom"] if task == "bottom" else data["y_top"]
        print(f"\n{'#'*78}")
        print(f"  TASK: {task.upper()}")
        print(f"{'#'*78}")

        y_tr = y_col[tr]
        y_va = y_col[va]
        y_te = y_col[te]

        det = {}
        mfe_results = {}
        train_time = 0.0

        skip_cfg = cfg.get("SKIP_L1_L2", False)
        skip_this = (skip_cfg is True or
                     (isinstance(skip_cfg, list) and task in skip_cfg))

        if skip_this:
            # ── Load saved predictions instead of training ──
            pred_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}_predictions.csv")
            ckpt_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}.pt")
            if not os.path.exists(pred_path):
                print(f"  [ERROR] {pred_path} not found, cannot skip L1/L2")
                continue

            print(f"  Loading saved predictions: {pred_path}")
            pred_df = pd.read_csv(pred_path)
            probs = np.zeros(len(y_col))
            # Map predictions back to test indices
            test_idx_arr = np.where(te)[0]
            test_tickers = data["tickers"][test_idx_arr]
            test_bars = data["bar_indices"][test_idx_arr]
            for _, row in pred_df.iterrows():
                matches = ((test_tickers == row["ticker"]) &
                           (test_bars == int(row["bar_index"])))
                mi = np.where(matches)[0]
                if len(mi) > 0:
                    probs[test_idx_arr[mi[0]]] = row["prob"]
            print(f"  Loaded {len(pred_df):,} predictions, "
                  f"mapped to {(probs[te] > 0).sum():,} test samples")
            print(f"\n  ═══ SKIPPING LEVELS 1-2 (already completed) ═══")

        else:
            # ── Train ──
            t0 = time.time()
            model = train_model(X_s_tr, X_l_tr, y_tr,
                                X_s_va, X_l_va, y_va,
                                data["n_short_feat"], data["n_long_feat"],
                                cfg, task)
            train_time = time.time() - t0
            print(f"    Training time: {train_time:.1f}s")

            # Save model
            ckpt_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"    Saved: {ckpt_path}")

            # ── Predict on full dataset ──
            probs = np.zeros(len(y_col))
            probs[te] = predict(model, X_s_te, X_l_te)
            probs[va] = predict(model, X_s_va, X_l_va)

            # ── Save predictions ──
            save_predictions(probs, data["tickers"], data["bar_indices"],
                             data["timestamps"], te, cfg, task)

            # ════════════════════════════════════════════
            # LEVEL 1: Detection Metrics
            # ════════════════════════════════════════════
            print(f"\n  ═══ LEVEL 1: DETECTION METRICS ═══")

            baseline_rate = y_te.mean()
            det = eval_detection(probs[te], y_te, cfg["DET_THRESHOLDS"],
                                 baseline_rate)
            single_auc = cfg["SINGLE_BASELINES"][f"{task}_auc"]
            print_detection(det, task, single_auc)

            # ════════════════════════════════════════════
            # LEVEL 2: MFE/MAE Analysis
            # ════════════════════════════════════════════
            print(f"\n  ═══ LEVEL 2: MFE/MAE ANALYSIS ═══")

            mfe_results = compute_mfe_mae_for_candidates(
                cfg, probs, data["tickers"], data["bar_indices"],
                te, task, threshold=0.5)

        # ════════════════════════════════════════════
        # LEVEL 3: Meta-Label
        # ════════════════════════════════════════════
        print(f"\n  ═══ LEVEL 3: META-LABEL INTEGRATION ═══")

        # Pick threshold with best MFE/MAE ratio and N>500
        best_th = 0.5
        if mfe_results:
            mb24 = mfe_results.get(24, {})
            if mb24 and mb24.get("n", 0) > 0:
                # Try higher thresholds
                for test_th in [0.5, 0.6, 0.7]:
                    cand_n = (te & (probs >= test_th)).sum()
                    if cand_n > 500:
                        best_th = test_th

        print(f"  Using CNN threshold={best_th} for meta-label")
        t_l3 = time.time()
        meta = run_meta_label(cfg, probs, X_s_full, X_l_full,
                              data["tickers"], data["bar_indices"],
                              te, task, best_th)
        print(f"  Level 3 total: {time.time()-t_l3:.2f}s")

        all_results[task] = {
            "detection": {
                "auc": det.get("auc", 0),
                "n": det.get("n", 0),
                "thresholds": {
                    str(k): v for k, v in det.items()
                    if isinstance(k, float)
                },
            } if det else {},
            "mfe_mae": {
                str(k): {kk: float(vv) if not isinstance(vv, np.ndarray) else None
                         for kk, vv in v.items()}
                for k, v in mfe_results.items()
            } if mfe_results else {},
            "meta_label": {
                str(k): (v if not isinstance(v, dict)
                         else {kk: float(vv) if isinstance(vv, (float, np.floating))
                               else int(vv) if isinstance(vv, (int, np.integer))
                               else vv
                               for kk, vv in v.items()})
                for k, v in meta.items()
            } if meta else {},
            "train_time": train_time,
        }

    # ═══════════════════════════════════════════════════════════
    #  MERGE WITH PREVIOUSLY SAVED RESULTS (for skipped levels)
    # ═══════════════════════════════════════════════════════════
    prev_json = os.path.join(cfg["OUTPUT_DIR"], "multiscale_cnn_results.json")
    if os.path.exists(prev_json):
        try:
            with open(prev_json) as f:
                prev = json.load(f)
            for task in ["bottom", "top"]:
                if task in prev and task in all_results:
                    cur = all_results[task]
                    old = prev[task]
                    # Fill in missing sections from previous run
                    if not cur.get("detection") and old.get("detection"):
                        cur["detection"] = old["detection"]
                        print(f"  [INFO] Loaded {task} L1 detection from previous run")
                    if not cur.get("mfe_mae") and old.get("mfe_mae"):
                        cur["mfe_mae"] = old["mfe_mae"]
                        print(f"  [INFO] Loaded {task} L2 MFE/MAE from previous run")
        except Exception:
            pass

    # ═══════════════════════════════════════════════════════════
    #  META-LABEL SUMMARY (key results from Level 3)
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ LEVEL 3 META-LABEL SUMMARY")
    print(f"{'='*78}")
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        meta = r.get("meta_label", {})
        if not meta:
            print(f"\n  {task.upper()}: not available")
            continue
        base = meta.get("base_rate", 0)
        n_test = meta.get("n_test", 0)
        direction = "LONG" if task == "bottom" else "SHORT"
        print(f"\n  {task.upper()} ({direction}):  base_wr={base*100:.1f}%  n_test={n_test}")
        print(f"  {'Th':>6} {'WR':>8} {'N':>7} {'PF':>7} {'TotPnL':>10} {'Δ vs base':>10}")
        print(f"  {'-'*52}")
        for th_key in sorted(k for k in meta if k not in ("base_rate", "n_test")):
            th_v = meta[th_key]
            if not isinstance(th_v, dict):
                continue
            wr = th_v.get("wr", 0)
            n_t = th_v.get("n", 0)
            pf = th_v.get("pf", 0)
            pnl = th_v.get("pnl", 0)
            delta = wr - base if n_t > 0 else 0
            mark = "✓" if wr > 0.55 and n_t >= 100 else ""
            print(f"  {th_key:>6} {wr*100:>7.1f}% {n_t:>7} {pf:>7.2f} "
                  f"{pnl*100:>9.1f}% {delta*100:>+9.1f}pp {mark}")

    # ═══════════════════════════════════════════════════════════
    #  FINAL COMPARISON TABLE
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ SINGLE-SCALE vs MULTI-SCALE CNN COMPARISON")
    print(f"{'='*78}")

    sb = cfg["SINGLE_BASELINES"]
    print(f"\n  ┌{'─'*68}┐")
    print(f"  │ {'Metric':<24} │ {'Single-Scale':>13} │ "
          f"{'Multi-Scale':>12} │ {'Δ':>10} │")
    print(f"  ├{'─'*68}┤")

    rows = []
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        det = r.get("detection", {})
        mfe = r.get("mfe_mae", {})
        meta = r.get("meta_label", {})

        # AUC
        ms_auc = det.get("auc", 0)
        ss_auc = sb[f"{task}_auc"]
        rows.append((f"{task.title()} AUC", ss_auc, ms_auc))

        # MFE/MAE @24
        ms_ratio = 0
        if "24" in mfe and mfe["24"].get("ratio") is not None:
            ms_ratio = mfe["24"]["ratio"]
        ss_ratio = sb[f"{task}_mfe_mae_24"]
        rows.append((f"{task.title()} MFE/MAE @24", ss_ratio, ms_ratio))

        # LGB WR @0.5
        ms_wr = 0
        if meta:
            th_data = meta.get("0.5") or meta.get(0.5)
            if th_data and isinstance(th_data, dict):
                ms_wr = th_data.get("wr", 0)
        ss_wr = sb[f"{task}_lgb_wr_05"]
        rows.append((f"{task.title()} LGB WR @0.5", ss_wr, ms_wr))

    for name, ss, ms in rows:
        ss_s = f"{ss:.3f}" if ss < 1 else f"{ss*100:.1f}%"
        if ms == 0 and ss != 0:
            # Metric wasn't computed this run and not loaded from previous
            print(f"  │ {name:<24} │ {ss_s:>13} │ {'—':>12} │ {'—':>10} │")
        else:
            delta = ms - ss
            ms_s = f"{ms:.3f}" if ms < 1 else f"{ms*100:.1f}%"
            d_s = f"{delta:+.3f}" if abs(delta) < 1 else f"{delta*100:+.1f}pp"
            print(f"  │ {name:<24} │ {ss_s:>13} │ {ms_s:>12} │ {d_s:>10} │")

    print(f"  └{'─'*68}┘")

    # Conclusions
    print(f"\n  CONCLUSIONS:")
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        det = r.get("detection", {})
        ms_auc = det.get("auc", 0)
        ss_auc = sb[f"{task}_auc"]

        if ms_auc == 0:
            print(f"    — {task.upper()}: AUC not available (L1 skipped or not run)")
        elif ms_auc > ss_auc + 0.01:
            print(f"    ✓ {task.upper()}: Multi-scale improves AUC "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")
        elif ms_auc > ss_auc - 0.01:
            print(f"    ~ {task.upper()}: Multi-scale AUC comparable "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")
        else:
            print(f"    ✗ {task.upper()}: Multi-scale AUC declined "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")

        mfe = r.get("mfe_mae", {})
        if "24" in mfe and mfe["24"].get("ratio"):
            ratio = mfe["24"]["ratio"]
            ss_r = sb[f"{task}_mfe_mae_24"]
            if ratio > ss_r:
                print(f"    ✓ {task.upper()}: MFE/MAE improved "
                      f"({ss_r:.3f} → {ratio:.3f})")

    # Save
    json_path = os.path.join(cfg["OUTPUT_DIR"],
                             "multiscale_cnn_results.json")
    # Clean for JSON
    save = {}
    for task, r in all_results.items():
        save[task] = {
            "detection": r.get("detection", {}),
            "mfe_mae": r.get("mfe_mae", {}),
            "meta_label": r.get("meta_label", {}),
            "train_time": r.get("train_time", 0),
        }
    with open(json_path, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"\n  Saved: {json_path}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

  Multi-Scale CNN for Turning Point Detection
  Short: 30×15min  Long: 48×1hour
  Device: cuda

  Building paired windows...
    AAPL: 42,919 samples, bottoms=17195 tops=17203  s_feat=66 l_feat=53
    MSFT: 42,496 samples, bottoms=17327 tops=17347  s_feat=66 l_feat=53
    GOOGL: 38,701 samples, bottoms=15460 tops=15496  s_feat=66 l_feat=53
    GOOG: 37,779 samples, bottoms=15212 tops=15246  s_feat=66 l_feat=53
    NVDA: 42,281 samples, bottoms=16570 tops=16572  s_feat=66 l_feat=53
    TSLA: 42,977 samples, bottoms=17486 tops=17504  s_feat=66 l_feat=53
    SPY: 42,817 samples, bottoms=17039 tops=17076  s_feat=66 l_feat=53
    QQQ: 42,914 samples, bottoms=16596 tops=16621  s_feat=66 l_feat=53

  Total: 332,884 samples
  Short shape: (332884, 30, 66)  Long shape: (332884, 48, 53)
  Bottom pos rate: 0.3957  Top pos rate: 0.3962
  Train: 233,014  Val: 49,933  Test: 49,937

##############################################################################
  TASK: BOTTOM
#########################

In [7]:
"""
vol_prediction_v2.py  (FIXED — regression on log(RV))
=====================================================
Four-model volatility regime prediction comparison.
All LSTM models use REGRESSION on log(RV), NOT binary classification.
Direction is DERIVED: pred_rv > current_rv → "up".

Models:
  A: Original Vol LSTM (must reproduce ~71% DA)
  B: Multi-Scale Vol LSTM (1hour blocks + daily context)
  C: LightGBM (block features) — both cls & reg, pick best
  D: LightGBM (block + daily)  — both cls & reg, pick best

Usage:
    !pip install lightgbm --quiet
    !python vol_prediction_v2.py
"""

import os, time, json, warnings, math
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, precision_score,
                             recall_score, f1_score, r2_score,
                             mean_squared_error)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[WARN] lightgbm not installed — Models C/D skipped")

try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    HAS_ARCH = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────── CONFIG ───────────────────────────
# Model A matches original vol_prediction.py EXACTLY
CFG = {
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "VOL_DATA_DIR": "vol_data",
    "BLOCK_SIZE": 12,
    # Model A — original config
    "SEQ_LEN": 10,
    "HIDDEN": 32,
    "NUM_LAYERS": 1,
    "DROPOUT": 0.1,
    "LR": 5e-4,
    "WD": 1e-4,
    "BATCH": 64,
    "EPOCHS": 200,
    "PATIENCE": 30,
    "SCHED_PATIENCE": 10,
    # Model B
    "DAILY_LOOKBACK": 20,
    # Splits
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.85,  # train+val = 85%, test = 15%
}


# ═══════════════════════════════════════════════════════════════
#  UTILITIES
# ═══════════════════════════════════════════════════════════════

def load_features_csv(ticker, freq):
    p = os.path.join(CFG["FEATURES_DIR"], f"{ticker}_{freq}_features.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    df = pd.read_csv(p)
    for col in ["timestamp", "ts_event", "datetime", "date"]:
        if col in df.columns:
            df["timestamp"] = pd.to_datetime(df[col], utc=True)
            break
    if "timestamp" not in df.columns:
        first = df.columns[0]
        if first not in df.select_dtypes(include=[np.number]).columns:
            df["timestamp"] = pd.to_datetime(df[first], utc=True)
    return df.sort_values("timestamp").reset_index(drop=True)


def numeric_cols(df):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


# ═══════════════════════════════════════════════════════════════
#  DATA: Build blocks from 1hour features
# ═══════════════════════════════════════════════════════════════

def compute_blocks_from_1h(df_1h, block_size=12):
    """Build non-overlapping blocks. Returns DataFrame with rv, log_rv, etc."""
    close = df_1h["close"].values.astype(np.float64)
    ts = df_1h["timestamp"].values
    lr = np.concatenate([[0], np.diff(np.log(close + 1e-10))])
    n_blocks = len(close) // block_size
    rows = []
    for b in range(n_blocks):
        s, e = b * block_size, (b + 1) * block_size
        rv = np.std(lr[s:e]) * np.sqrt(block_size)
        rows.append({
            "block_idx": b, "bar_start": s, "bar_end": e,
            "rv": rv, "log_rv": np.log(rv + 1e-10),
            "ts_start": ts[s], "ts_end": ts[min(e - 1, len(close) - 1)],
        })
    return pd.DataFrame(rows)


def load_or_build_vol_data():
    """
    Try to load pre-split CSVs from vol_data/.
    If not found, build from 1hour features and save.
    Returns: dict[ticker] → {"blocks": df, "block_feats": array, "feat_names": list,
                              "train_idx", "val_idx", "test_idx"}
    """
    BS = CFG["BLOCK_SIZE"]
    os.makedirs(CFG["VOL_DATA_DIR"], exist_ok=True)
    all_data = {}

    # Check if pre-split CSVs exist
    first_ticker = CFG["TICKERS"][0]
    pre_split_exists = os.path.exists(
        os.path.join(CFG["VOL_DATA_DIR"], f"{first_ticker}_vol_train.csv"))

    for ticker in CFG["TICKERS"]:
        try:
            df_1h = load_features_csv(ticker, "1hour")
        except FileNotFoundError:
            print(f"    [SKIP] {ticker}: no 1hour data")
            continue

        fc = numeric_cols(df_1h)
        feat_data = np.nan_to_num(df_1h[fc].values.astype(np.float32), nan=0.0)

        # Build blocks
        dfb = compute_blocks_from_1h(df_1h, BS)
        nb = len(dfb)

        # Block-level mean features
        block_feats = np.zeros((nb, len(fc)), dtype=np.float32)
        for i in range(nb):
            s, e = dfb.loc[i, "bar_start"], dfb.loc[i, "bar_end"]
            block_feats[i] = np.mean(feat_data[s:e], axis=0)

        # Per-ticker chronological split: 70/15/15
        tr_end = int(nb * CFG["TRAIN_RATIO"])
        va_end = int(nb * CFG["VAL_RATIO"])

        # Save pre-split CSVs if they don't exist
        if not pre_split_exists:
            for split_name, s, e in [("train", 0, tr_end),
                                      ("val", tr_end, va_end),
                                      ("test", va_end, nb)]:
                sp = os.path.join(CFG["VOL_DATA_DIR"],
                                  f"{ticker}_vol_{split_name}.csv")
                dfb.iloc[s:e].to_csv(sp, index=False)

        all_data[ticker] = {
            "blocks": dfb,
            "block_feats": block_feats,
            "feat_names": fc,
            "train_end": tr_end,
            "val_end": va_end,
            "n_blocks": nb,
        }
        print(f"    {ticker}: {nb} blocks  "
              f"train={tr_end} val={va_end-tr_end} test={nb-va_end}  "
              f"feat={len(fc)}")

    if not pre_split_exists:
        print(f"  Saved pre-split CSVs to {CFG['VOL_DATA_DIR']}/")

    return all_data


# ═══════════════════════════════════════════════════════════════
#  DAILY DATA
# ═══════════════════════════════════════════════════════════════

def compute_daily_bars(df_1h):
    """Aggregate 1hour → daily with derived features."""
    df = df_1h.copy()
    df["_date"] = df["timestamp"].dt.date

    has_vol = "volume" in df.columns
    agg = {"open": ("open", "first"), "high": ("high", "max"),
           "low": ("low", "min"), "close": ("close", "last"),
           "n_bars": ("close", "count")}
    if has_vol:
        agg["volume"] = ("volume", "sum")
    daily = df.groupby("_date").agg(**agg).reset_index()
    daily.rename(columns={"_date": "date"}, inplace=True)

    c = daily["close"].values.astype(np.float64)
    h = daily["high"].values.astype(np.float64)
    lo = daily["low"].values.astype(np.float64)
    o = daily["open"].values.astype(np.float64)
    n = len(daily)

    daily["daily_return"] = (c - o) / (o + 1e-10)
    daily["daily_range"] = (h - lo) / (c + 1e-10)

    if has_vol:
        v = daily["volume"].values.astype(np.float64)
        daily["vol_change"] = np.concatenate([[1.0], v[1:] / (v[:-1] + 1e-10)])
    else:
        daily["vol_change"] = 1.0

    # Intraday RV per day
    drv = []
    for _, grp in df.groupby("_date"):
        cc = grp["close"].values.astype(np.float64)
        if len(cc) > 1:
            lr = np.diff(np.log(cc + 1e-10))
            drv.append(np.std(lr) * np.sqrt(len(cc)))
        else:
            drv.append(0.0)
    daily["daily_rv"] = drv

    # RSI-14
    delta = np.diff(c, prepend=c[0])
    gain = np.where(delta > 0, delta, 0.0)
    loss_arr = np.where(delta < 0, -delta, 0.0)
    ag = np.zeros(n); al = np.zeros(n)
    if n > 14:
        ag[14] = np.mean(gain[1:15]); al[14] = np.mean(loss_arr[1:15])
        for i in range(15, n):
            ag[i] = (ag[i-1]*13 + gain[i]) / 14
            al[i] = (al[i-1]*13 + loss_arr[i]) / 14
    rs = ag / (al + 1e-10)
    daily["rsi_14"] = 100 - 100 / (1 + rs)

    # ATR-14
    tr = np.zeros(n)
    for i in range(1, n):
        tr[i] = max(h[i]-lo[i], abs(h[i]-c[i-1]), abs(lo[i]-c[i-1]))
    tr[0] = h[0] - lo[0]
    atr = np.zeros(n)
    if n > 14:
        atr[14] = np.mean(tr[1:15])
        for i in range(15, n):
            atr[i] = (atr[i-1]*13 + tr[i]) / 14
    daily["atr_14"] = atr

    # BB bandwidth 20
    bb = np.zeros(n)
    for i in range(20, n):
        sma = np.mean(c[i-20:i]); std = np.std(c[i-20:i])
        bb[i] = (4 * std) / (sma + 1e-10)
    daily["bb_bandwidth_20"] = bb
    daily["log_close"] = np.log(c + 1e-10)

    return daily


# ═══════════════════════════════════════════════════════════════
#  SEQUENCE BUILDERS (REGRESSION targets)
# ═══════════════════════════════════════════════════════════════

def build_lstm_sequences(all_data, seq_len):
    """
    Build sequences for LSTM regression.
    Input: [block_feats | log_rv] for seq_len blocks.
    Target: log(rv) of the NEXT block.
    Also store current_rv for direction derivation.
    """
    X_all, y_all, cur_rv_all, meta_all = [], [], [], []
    n_feat = None

    for ticker, d in all_data.items():
        dfb = d["blocks"]
        bf = d["block_feats"]
        nb = d["n_blocks"]
        log_rvs = dfb["log_rv"].values.astype(np.float32)
        raw_rvs = dfb["rv"].values.astype(np.float64)

        if n_feat is None:
            n_feat = bf.shape[1] + 1  # +1 for log_rv
            print(f"  LSTM input dim: {bf.shape[1]} numeric + 1 log_rv = {n_feat}")

        # Append log_rv as extra column
        bf_ext = np.column_stack([bf, log_rvs.reshape(-1, 1)])

        for i in range(seq_len, nb - 1):
            # Input: blocks [i-seq_len .. i-1]
            seq = bf_ext[i - seq_len:i]
            # Target: log(rv) of block i
            target = log_rvs[i]
            # Current rv (block i-1) for direction derivation
            cur = raw_rvs[i - 1]

            X_all.append(seq)
            y_all.append(target)
            cur_rv_all.append(cur)
            meta_all.append((ticker, i))

    X = np.array(X_all, dtype=np.float32)
    y = np.array(y_all, dtype=np.float32)
    cur = np.array(cur_rv_all, dtype=np.float64)
    return X, y, cur, meta_all, n_feat


def build_multiscale_sequences(all_data, all_daily, seq_len, daily_lb):
    """Build paired sequences for Model B (short + long branch, regression)."""
    Xs_all, Xl_all, y_all, cur_all, meta_all = [], [], [], [], []
    n_sfeat = None; n_dfeat = None

    dcols = ["daily_rv", "daily_range", "daily_return", "vol_change",
             "rsi_14", "atr_14", "bb_bandwidth_20", "log_close"]

    for ticker, d in all_data.items():
        dfb = d["blocks"]
        bf = d["block_feats"]
        nb = d["n_blocks"]
        log_rvs = dfb["log_rv"].values.astype(np.float32)
        raw_rvs = dfb["rv"].values.astype(np.float64)

        bf_ext = np.column_stack([bf, log_rvs.reshape(-1, 1)])
        if n_sfeat is None:
            n_sfeat = bf_ext.shape[1]

        daily = all_daily.get(ticker)
        if daily is None:
            continue
        dc_valid = [c for c in dcols if c in daily.columns]
        if n_dfeat is None:
            n_dfeat = len(dc_valid)
        df_vals = np.nan_to_num(daily[dc_valid].values.astype(np.float32), nan=0.0)
        ddates = daily["date"].values

        for i in range(seq_len, nb - 1):
            short_seq = bf_ext[i - seq_len:i]
            target = log_rvs[i]
            cur_rv = raw_rvs[i - 1]

            # Align to daily
            block_end = pd.Timestamp(dfb.loc[i, "ts_end"])
            bd = block_end.date() if hasattr(block_end, 'date') else block_end
            day_idx = -1
            for di in range(len(ddates) - 1, -1, -1):
                if ddates[di] <= bd:
                    day_idx = di; break
            if day_idx < daily_lb:
                continue
            long_seq = df_vals[day_idx - daily_lb + 1:day_idx + 1]
            if len(long_seq) != daily_lb:
                continue

            Xs_all.append(short_seq)
            Xl_all.append(long_seq)
            y_all.append(target)
            cur_all.append(cur_rv)
            meta_all.append((ticker, i))

    return (np.array(Xs_all, dtype=np.float32),
            np.array(Xl_all, dtype=np.float32),
            np.array(y_all, dtype=np.float32),
            np.array(cur_all, dtype=np.float64),
            meta_all, n_sfeat, n_dfeat)


def build_lgb_features(all_data, all_daily, include_daily=False):
    """Flat features for LightGBM. Returns X, y_reg (log_rv), y_cls (direction), cur_rv, meta, fnames."""
    SL = CFG["SEQ_LEN"]
    DAILY_LB = 5

    X_rows, y_reg, y_cls, cur_rv_all, meta = [], [], [], [], []
    key_cols_names = ["close", "volume", "atr_14", "rsi_14"]

    for ticker, d in all_data.items():
        dfb = d["blocks"]
        bf = d["block_feats"]
        fc = d["feat_names"]
        nb = d["n_blocks"]
        rvs = dfb["rv"].values.astype(np.float64)
        log_rvs = dfb["log_rv"].values.astype(np.float64)

        fc_list = list(fc)
        key_idxs = {}
        for kc in key_cols_names:
            matches = [j for j, c in enumerate(fc_list) if kc.lower() in c.lower()]
            if matches:
                key_idxs[kc] = matches[0]

        daily = all_daily.get(ticker) if include_daily else None
        daily_cols = ["daily_rv", "daily_range", "daily_return", "vol_change"]
        if daily is not None:
            dc_valid = [c for c in daily_cols if c in daily.columns]
            df_vals = np.nan_to_num(daily[dc_valid].values.astype(np.float32), nan=0.0)
            ddates = daily["date"].values
            n_dc = len(dc_valid)
        else:
            n_dc = 0

        spy_data = all_data.get("SPY")
        spy_rvs = spy_data["blocks"]["rv"].values if spy_data else None

        for i in range(SL, nb - 1):
            feats = {}
            # A) Per-block features (last SL blocks)
            for off in range(SL):
                bi = i - SL + off
                prefix = f"b{off}"
                feats[f"{prefix}_rv"] = rvs[bi]
                feats[f"{prefix}_log_rv"] = log_rvs[bi]
                for kc, ki in key_idxs.items():
                    feats[f"{prefix}_{kc}"] = float(bf[bi, ki])

            # B) RV rolling stats
            rv_win = rvs[max(0, i-SL):i]
            rv4 = rvs[max(0, i-4):i]
            feats["rv_mean_4"] = np.mean(rv4) if len(rv4) else 0
            feats["rv_std_4"] = np.std(rv4) if len(rv4) > 1 else 0
            feats["rv_mean_8"] = np.mean(rv_win) if len(rv_win) else 0
            feats["rv_std_8"] = np.std(rv_win) if len(rv_win) > 1 else 0
            feats["rv_ratio"] = rvs[i-1] / (feats["rv_mean_8"] + 1e-10)
            if len(rv_win) >= 3:
                feats["rv_trend"] = float(np.polyfit(np.arange(len(rv_win)), rv_win, 1)[0])
            else:
                feats["rv_trend"] = 0.0
            streak_up = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] > rvs[k-1]: streak_up += 1
                else: break
            feats["rv_streak_up"] = streak_up
            streak_dn = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] < rvs[k-1]: streak_dn += 1
                else: break
            feats["rv_streak_down"] = streak_dn
            feats["current_log_rv"] = log_rvs[i-1]

            # C) Current block tech indicators
            for j, cn in enumerate(fc_list):
                feats[f"tech_{cn}"] = float(bf[i-1, j])

            # D) SPY cross-asset
            if ticker != "SPY" and spy_rvs is not None and i < len(spy_rvs):
                feats["spy_rv_cur"] = spy_rvs[max(0, i-1)]
                spy4 = spy_rvs[max(0, i-4):i]
                feats["spy_rv_mean_4"] = np.mean(spy4) if len(spy4) else 0
            else:
                feats["spy_rv_cur"] = 0; feats["spy_rv_mean_4"] = 0

            # E) Daily (Model D)
            if include_daily and daily is not None:
                bets = pd.Timestamp(dfb.loc[i, "ts_end"])
                bd = bets.date() if hasattr(bets, 'date') else bets
                day_idx = -1
                for di in range(len(ddates)-1, -1, -1):
                    if ddates[di] <= bd: day_idx = di; break
                if day_idx >= DAILY_LB:
                    for d_off in range(DAILY_LB):
                        di2 = day_idx - DAILY_LB + 1 + d_off
                        for ci, cn in enumerate(dc_valid[:n_dc]):
                            feats[f"d{d_off}_{cn}"] = float(df_vals[di2, ci])
                    drv_w = df_vals[day_idx-DAILY_LB+1:day_idx+1, 0]
                    feats["daily_rv_mean_5"] = float(np.mean(drv_w))
                    feats["daily_rv_std_5"] = float(np.std(drv_w))
                    if len(drv_w) >= 3:
                        feats["daily_rv_trend"] = float(
                            np.polyfit(np.arange(len(drv_w)), drv_w, 1)[0])
                    else:
                        feats["daily_rv_trend"] = 0.0

            X_rows.append(feats)
            y_reg.append(log_rvs[i])  # regression target
            # classification target
            y_cls.append(1 if rvs[i] > rvs[i-1] else 0)
            cur_rv_all.append(rvs[i-1])
            meta.append((ticker, i))

    if not X_rows:
        return np.array([]), np.array([]), np.array([]), np.array([]), meta, []
    fnames = list(X_rows[0].keys())
    X = np.array([[r.get(fn, 0.0) for fn in fnames] for r in X_rows], dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return (X, np.array(y_reg, dtype=np.float32),
            np.array(y_cls, dtype=np.float32),
            np.array(cur_rv_all, dtype=np.float64), meta, fnames)


# ═══════════════════════════════════════════════════════════════
#  MODELS
# ═══════════════════════════════════════════════════════════════

class VolLSTM_A(nn.Module):
    """Model A: Original Vol LSTM — REGRESSION on log(RV).
    LSTM(hidden=32, layers=1) → Linear(32,32) → ReLU → Dropout(0.1) → Linear(32,1)
    """
    def __init__(self, n_feat):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, 32, 1, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class VolLSTM_B(nn.Module):
    """Model B: Multi-Scale Vol LSTM — REGRESSION on log(RV).
    Short: LSTM(32,1) → 32-dim.  Long: LSTM(32,1) → 32-dim.
    Fusion: concat → Linear(64,32) → ReLU → Dropout(0.1) → Linear(32,1).
    """
    def __init__(self, n_sfeat, n_dfeat):
        super().__init__()
        self.short_lstm = nn.LSTM(n_sfeat, 32, 1, batch_first=True)
        self.long_lstm = nn.LSTM(n_dfeat, 32, 1, batch_first=True)
        self.fuse = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1),
        )
    def forward(self, xs, xl):
        so, _ = self.short_lstm(xs)
        lo, _ = self.long_lstm(xl)
        return self.fuse(torch.cat([so[:, -1, :], lo[:, -1, :]], dim=1)).squeeze(-1)


class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class PairDS(Dataset):
    def __init__(self, Xs, Xl, y):
        self.Xs = torch.tensor(Xs, dtype=torch.float32)
        self.Xl = torch.tensor(Xl, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.Xs[i], self.Xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  SPLIT HELPERS
# ═══════════════════════════════════════════════════════════════

def split3(n):
    """Return (train_end, val_end) indices."""
    return int(n * CFG["TRAIN_RATIO"]), int(n * CFG["VAL_RATIO"])


def normalize_from_train(X, tr_end):
    """Normalize using training set statistics only. Handles 2D or 3D."""
    if X.ndim == 3:
        flat = X[:tr_end].reshape(-1, X.shape[2])
    else:
        flat = X[:tr_end]
    mu = flat.mean(0); sig = flat.std(0) + 1e-8
    X_out = (X - mu) / sig
    return np.nan_to_num(X_out, nan=0.0, posinf=0.0, neginf=0.0)


# ═══════════════════════════════════════════════════════════════
#  TRAINING
# ═══════════════════════════════════════════════════════════════

def train_regression_lstm(model, X_tr, y_tr, X_va, y_va, tag="A"):
    """Train LSTM with MSE loss on log(RV)."""
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=CFG["SCHED_PATIENCE"], factor=0.5)

    tr_dl = DataLoader(SeqDS(X_tr, y_tr), batch_size=CFG["BATCH"], shuffle=True)
    va_dl = DataLoader(SeqDS(X_va, y_va), batch_size=512)

    best_vl = float("inf"); best_st = None; pat = 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); tl = 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
        tl /= len(y_tr)

        model.eval(); vl = 0
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xb), yb).item() * len(yb)
        vl /= len(y_va)
        sched.step(vl)

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0; best_ep = ep + 1
        else:
            pat += 1
        if (ep+1) % 40 == 0 or pat == CFG["PATIENCE"]:
            print(f"    [{tag}] E{ep+1:3d} t={tl:.6f} v={vl:.6f} p={pat}")
        if pat >= CFG["PATIENCE"]:
            print(f"    [{tag}] Early stop (best={best_ep})")
            break

    if best_st:
        model.load_state_dict(best_st)
    model.to(DEVICE).eval()
    return model


def predict_lstm(model, X):
    dl = DataLoader(SeqDS(X, np.zeros(len(X))), batch_size=512)
    preds = []
    with torch.no_grad():
        for xb, _ in dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)


def train_regression_lstm_b(model, Xs_tr, Xl_tr, y_tr, Xs_va, Xl_va, y_va):
    """Train Model B with MSE loss."""
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=CFG["SCHED_PATIENCE"], factor=0.5)

    tr_dl = DataLoader(PairDS(Xs_tr, Xl_tr, y_tr), batch_size=CFG["BATCH"], shuffle=True)
    va_dl = DataLoader(PairDS(Xs_va, Xl_va, y_va), batch_size=512)

    best_vl = float("inf"); best_st = None; pat = 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); tl = 0
        for xs, xl, yb in tr_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xs, xl), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
        tl /= len(y_tr)

        model.eval(); vl = 0
        with torch.no_grad():
            for xs, xl, yb in va_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xs, xl), yb).item() * len(yb)
        vl /= len(y_va)
        sched.step(vl)

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0; best_ep = ep + 1
        else:
            pat += 1
        if (ep+1) % 40 == 0 or pat == CFG["PATIENCE"]:
            print(f"    [B] E{ep+1:3d} t={tl:.6f} v={vl:.6f} p={pat}")
        if pat >= CFG["PATIENCE"]:
            print(f"    [B] Early stop (best={best_ep})")
            break

    if best_st:
        model.load_state_dict(best_st)
    model.to(DEVICE).eval()
    return model


def predict_lstm_b(model, Xs, Xl):
    dl = DataLoader(PairDS(Xs, Xl, np.zeros(len(Xs))), batch_size=512)
    preds = []
    with torch.no_grad():
        for xs, xl, _ in dl:
            preds.append(model(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)


# ═══════════════════════════════════════════════════════════════
#  EVALUATION
# ═══════════════════════════════════════════════════════════════

def eval_regression(pred_log, true_log, cur_rv, meta_te, name):
    """
    Evaluate regression model.
    Direction: pred_rv > cur_rv → 1.  True direction: true_rv > cur_rv → 1.
    """
    pred_rv = np.exp(pred_log)
    true_rv = np.exp(true_log)

    pred_dir = (pred_rv > cur_rv).astype(int)
    true_dir = (true_rv > cur_rv).astype(int)

    da = (pred_dir == true_dir).mean()
    r2 = r2_score(true_rv, pred_rv)
    rmse = np.sqrt(mean_squared_error(true_rv, pred_rv))
    n = len(true_dir)

    # z-score
    se = np.sqrt(0.5 * 0.5 / n)
    z = (da - 0.5) / se
    p_val = 2 * (1 - __import__('scipy').stats.norm.cdf(abs(z))) if abs(z) > 0 else 1.0

    # F1 for high-vol class
    f1 = f1_score(true_dir, pred_dir, zero_division=0)

    result = {"name": name, "da": float(da), "r2": float(r2),
              "rmse": float(rmse), "z": float(z), "p": float(p_val),
              "f1": float(f1), "n": n}

    # Per-ticker
    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                pt[ticker] = {
                    "da": float((pred_dir[mask] == true_dir[mask]).mean()),
                    "n": int(mask.sum()),
                }
        result["per_ticker"] = pt

    return result


def eval_classification(pred_prob, true_dir, meta_te, name):
    """Evaluate binary classifier."""
    preds = (pred_prob >= 0.5).astype(int)
    da = (preds == true_dir).mean()
    auc = roc_auc_score(true_dir, pred_prob) if true_dir.sum() > 0 and (1-true_dir).sum() > 0 else 0.5
    f1 = f1_score(true_dir, preds, zero_division=0)
    n = len(true_dir)
    z = (da - 0.5) / np.sqrt(0.5*0.5/n)

    result = {"name": name, "da": float(da), "r2": float(0),
              "rmse": float(0), "z": float(z), "f1": float(f1), "n": n}
    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                pt[ticker] = {"da": float((preds[mask] == true_dir[mask]).mean()),
                              "n": int(mask.sum())}
        result["per_ticker"] = pt
    return result


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("  Volatility Prediction v2 — REGRESSION on log(RV)")
    print(f"  Device: {DEVICE}")
    print("=" * 70)
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    os.makedirs(CFG["VOL_DATA_DIR"], exist_ok=True)
    t_start = time.time()

    # ── Load / build data ──
    print("\n  Loading data...")
    all_data = load_or_build_vol_data()
    all_daily = {}
    for ticker in CFG["TICKERS"]:
        try:
            df_1h = load_features_csv(ticker, "1hour")
            all_daily[ticker] = compute_daily_bars(df_1h)
        except FileNotFoundError:
            pass

    # ══════════════════════════════════════════════════════
    #  MODEL A: Original Vol LSTM (regression)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  MODEL A: Original Vol LSTM (regression on log(RV))")
    print("=" * 70)

    t0 = time.time()
    X_a, y_a, cur_a, meta_a, n_feat_a = build_lstm_sequences(all_data, CFG["SEQ_LEN"])
    print(f"  Samples: {len(y_a):,}  features: {n_feat_a}")

    tr_a, va_a = split3(len(y_a))
    X_a_n = normalize_from_train(X_a, tr_a)

    model_a = VolLSTM_A(n_feat_a).to(DEVICE)
    print(f"  Architecture: LSTM({n_feat_a}, h=32, L=1) → Linear(32,32) "
          f"→ ReLU → Drop(0.1) → Linear(32,1)")
    print(f"  Loss: MSE on log(RV)   LR={CFG['LR']}   "
          f"seq_len={CFG['SEQ_LEN']}   epochs={CFG['EPOCHS']}")

    model_a = train_regression_lstm(
        model_a, X_a_n[:tr_a], y_a[:tr_a],
        X_a_n[tr_a:va_a], y_a[tr_a:va_a], tag="A")

    pred_a = predict_lstm(model_a, X_a_n[va_a:])
    res_a = eval_regression(pred_a, y_a[va_a:], cur_a[va_a:],
                            meta_a[va_a:], "A: Original LSTM")
    print(f"\n  Model A Test: DA={res_a['da']*100:.1f}%  R²={res_a['r2']:.3f}  "
          f"RMSE={res_a['rmse']:.6f}  z={res_a['z']:+.2f}  ({time.time()-t0:.1f}s)")

    if abs(res_a['da'] - 0.713) > 0.015:
        print(f"  ⚠ WARNING: DA={res_a['da']*100:.1f}%, expected ~71.3%")

    # Save Model A
    sp = os.path.join(CFG["RESULTS_DIR"], "vol_lstm_regression.pt")
    torch.save(model_a.state_dict(), sp)
    print(f"  Saved: {sp}")

    # ══════════════════════════════════════════════════════
    #  MODEL B: Multi-Scale Vol LSTM (regression)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  MODEL B: Multi-Scale Vol LSTM (regression)")
    print("=" * 70)

    t0 = time.time()
    Xs_b, Xl_b, y_b, cur_b, meta_b, n_sf, n_df = build_multiscale_sequences(
        all_data, all_daily, CFG["SEQ_LEN"], CFG["DAILY_LOOKBACK"])
    print(f"  Samples: {len(y_b):,}  short_feat={n_sf}  daily_feat={n_df}")

    res_b = None
    if len(y_b) > 100:
        tr_b, va_b = split3(len(y_b))
        Xs_b_n = normalize_from_train(Xs_b, tr_b)
        Xl_b_n = normalize_from_train(Xl_b, tr_b)

        model_b = VolLSTM_B(n_sf, n_df).to(DEVICE)
        model_b = train_regression_lstm_b(
            model_b, Xs_b_n[:tr_b], Xl_b_n[:tr_b], y_b[:tr_b],
            Xs_b_n[tr_b:va_b], Xl_b_n[tr_b:va_b], y_b[tr_b:va_b])

        pred_b = predict_lstm_b(model_b, Xs_b_n[va_b:], Xl_b_n[va_b:])
        res_b = eval_regression(pred_b, y_b[va_b:], cur_b[va_b:],
                                meta_b[va_b:], "B: Multi-Scale LSTM")
        print(f"\n  Model B Test: DA={res_b['da']*100:.1f}%  R²={res_b['r2']:.3f}  "
              f"RMSE={res_b['rmse']:.6f}  ({time.time()-t0:.1f}s)")

        sp2 = os.path.join(CFG["RESULTS_DIR"], "vol_multiscale_lstm_regression.pt")
        torch.save(model_b.state_dict(), sp2)
        print(f"  Saved: {sp2}")
    else:
        print("  [SKIP] Insufficient daily-aligned samples")

    # ══════════════════════════════════════════════════════
    #  NAIVE BASELINES
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  NAIVE BASELINES")
    print("=" * 70)

    te_true_log = y_a[va_a:]
    te_cur_rv = cur_a[va_a:]
    te_true_rv = np.exp(te_true_log)
    te_true_dir = (te_true_rv > te_cur_rv).astype(int)
    n_te = len(te_true_dir)

    # Naive RW: predict next_rv = current_rv → always "same" → direction = 0
    naive_rw_pred = te_cur_rv.copy()
    naive_rw_dir = np.zeros(n_te, dtype=int)  # pred = cur → pred_dir always 0
    naive_rw_da = (naive_rw_dir == te_true_dir).mean()
    naive_rw_r2 = r2_score(te_true_rv, naive_rw_pred)
    naive_rw_rmse = np.sqrt(mean_squared_error(te_true_rv, naive_rw_pred))

    # Historical mean: predict next_rv = mean(train_rv)
    train_rvs = np.exp(y_a[:tr_a])
    hist_mean = train_rvs.mean()
    hist_pred = np.full(n_te, hist_mean)
    hist_dir = (hist_mean > te_cur_rv).astype(int)
    hist_da = (hist_dir == te_true_dir).mean()
    hist_r2 = r2_score(te_true_rv, hist_pred)
    hist_rmse = np.sqrt(mean_squared_error(te_true_rv, hist_pred))

    print(f"  Naive (RW — predict current):   DA={naive_rw_da*100:.1f}%  "
          f"R²={naive_rw_r2:.3f}  RMSE={naive_rw_rmse:.6f}")
    print(f"  Historical Mean (={hist_mean:.6f}):  DA={hist_da*100:.1f}%  "
          f"R²={hist_r2:.3f}  RMSE={hist_rmse:.6f}")

    # GARCH
    garch_res = None
    if HAS_ARCH:
        print("  Fitting GARCH(1,1)...")
        try:
            train_rvs_ts = np.exp(y_a[:tr_a]) * 10000  # scale for GARCH
            am = arch_model(train_rvs_ts, vol='GARCH', p=1, q=1, mean='Constant')
            garch_fit = am.fit(disp='off')
            garch_forecasts = []
            all_rvs_scaled = np.exp(y_a) * 10000
            for i in range(va_a, len(y_a)):
                window = all_rvs_scaled[:i]
                am_i = arch_model(window, vol='GARCH', p=1, q=1, mean='Constant')
                fit_i = am_i.fit(disp='off', last_obs=len(window))
                fc = fit_i.forecast(horizon=1)
                garch_forecasts.append(fc.variance.values[-1, 0])
            garch_pred_rv = np.sqrt(np.array(garch_forecasts)) / 10000
            garch_dir = (garch_pred_rv > te_cur_rv).astype(int)
            garch_da = (garch_dir == te_true_dir).mean()
            garch_r2 = r2_score(te_true_rv, garch_pred_rv)
            garch_rmse = np.sqrt(mean_squared_error(te_true_rv, garch_pred_rv))
            garch_res = {"da": float(garch_da), "r2": float(garch_r2),
                         "rmse": float(garch_rmse)}
            print(f"  GARCH(1,1):   DA={garch_da*100:.1f}%  "
                  f"R²={garch_r2:.3f}  RMSE={garch_rmse:.6f}")
        except Exception as e:
            print(f"  GARCH failed: {e}")
    else:
        print("  [SKIP] GARCH — arch package not installed")

    # ══════════════════════════════════════════════════════
    #  MODELS C & D: LightGBM
    # ══════════════════════════════════════════════════════
    res_c = None; res_d = None
    mdl_c = None; mdl_d = None; fn_c = []; fn_d = []

    if HAS_LGB:
        for tag, include_daily in [("C", False), ("D", True)]:
            print(f"\n{'='*70}")
            print(f"  MODEL {tag}: LightGBM ({'block+daily' if include_daily else 'block'})")
            print(f"{'='*70}")

            t0 = time.time()
            X_lg, y_reg_lg, y_cls_lg, cur_lg, meta_lg, fnames_lg = build_lgb_features(
                all_data, all_daily, include_daily=include_daily)
            print(f"  Samples: {len(y_cls_lg):,}  features: {len(fnames_lg)}")

            if len(X_lg) == 0:
                continue

            tr_lg, va_lg = split3(len(y_cls_lg))
            meta_te_lg = meta_lg[va_lg:]

            # ── Classification approach ──
            mdl_cls = lgb.LGBMClassifier(
                n_estimators=1000, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
                reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
            mdl_cls.fit(X_lg[:tr_lg], y_cls_lg[:tr_lg],
                        eval_set=[(X_lg[tr_lg:va_lg], y_cls_lg[tr_lg:va_lg])],
                        callbacks=[lgb.early_stopping(50, verbose=False),
                                   lgb.log_evaluation(0)])
            cls_probs = mdl_cls.predict_proba(X_lg[va_lg:])[:, 1]
            cls_preds = (cls_probs >= 0.5).astype(int)
            cls_da = (cls_preds == y_cls_lg[va_lg:]).mean()

            # ── Regression approach ──
            mdl_reg = lgb.LGBMRegressor(
                n_estimators=1000, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
                reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
            mdl_reg.fit(X_lg[:tr_lg], y_reg_lg[:tr_lg],
                        eval_set=[(X_lg[tr_lg:va_lg], y_reg_lg[tr_lg:va_lg])],
                        callbacks=[lgb.early_stopping(50, verbose=False),
                                   lgb.log_evaluation(0)])
            reg_pred_log = mdl_reg.predict(X_lg[va_lg:])
            reg_pred_rv = np.exp(reg_pred_log)
            reg_dir = (reg_pred_rv > cur_lg[va_lg:]).astype(int)
            te_dir = y_cls_lg[va_lg:].astype(int)
            reg_da = (reg_dir == te_dir).mean()

            print(f"  Classification DA: {cls_da*100:.1f}%")
            print(f"  Regression DA:     {reg_da*100:.1f}%")

            # Pick the better one
            if reg_da >= cls_da:
                print(f"  → Using REGRESSION approach")
                res_lg = eval_regression(reg_pred_log,
                                         y_reg_lg[va_lg:], cur_lg[va_lg:],
                                         meta_te_lg,
                                         f"{tag}: LightGBM ({'block+daily' if include_daily else 'block'}) [reg]")
                best_mdl = mdl_reg
            else:
                print(f"  → Using CLASSIFICATION approach")
                res_lg = eval_classification(cls_probs, te_dir, meta_te_lg,
                                             f"{tag}: LightGBM ({'block+daily' if include_daily else 'block'}) [cls]")
                best_mdl = mdl_cls

            print(f"  Best DA: {res_lg['da']*100:.1f}%  ({time.time()-t0:.1f}s)")

            if tag == "C":
                res_c = res_lg; mdl_c = best_mdl; fn_c = fnames_lg
            else:
                res_d = res_lg; mdl_d = best_mdl; fn_d = fnames_lg

    # ══════════════════════════════════════════════════════
    #  COMPARISON TABLE
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ VOLATILITY PREDICTION COMPARISON (regression on log(RV))")
    print(f"{'='*70}")

    all_results = {}

    # Naive baselines
    all_results["Naive (RW)"] = {
        "name": "Naive (RW)", "da": float(naive_rw_da),
        "r2": float(naive_rw_r2), "rmse": float(naive_rw_rmse),
        "f1": 0, "n": n_te,
    }
    all_results["Historical Mean"] = {
        "name": "Historical Mean", "da": float(hist_da),
        "r2": float(hist_r2), "rmse": float(hist_rmse),
        "f1": 0, "n": n_te,
    }
    if garch_res:
        all_results["GARCH(1,1)"] = {
            "name": "GARCH(1,1)", "da": garch_res["da"],
            "r2": garch_res["r2"], "rmse": garch_res["rmse"],
            "f1": 0, "n": n_te,
        }

    all_results["A: Original LSTM"] = res_a
    if res_b:
        all_results["B: Multi-Scale LSTM"] = res_b
    if res_c:
        all_results[res_c["name"]] = res_c
    if res_d:
        all_results[res_d["name"]] = res_d

    print(f"\n  ┌{'─'*71}┐")
    print(f"  │ {'Model':<36} │ {'DA':>7} │ {'R²':>7} │ "
          f"{'RMSE':>8} │ {'N':>5} │")
    print(f"  ├{'─'*71}┤")

    best_da = 0; best_name = ""
    for name, r in all_results.items():
        da = r["da"]; r2 = r.get("r2", 0); rmse = r.get("rmse", 0); n = r.get("n", 0)
        if da > best_da and "Naive" not in name and "Historical" not in name:
            best_da = da; best_name = name
        r2_s = f"{r2:>7.3f}" if r2 != 0 else "    —"
        rmse_s = f"{rmse:>8.6f}" if rmse != 0 else "      —"
        marker = " ★" if name == best_name else ""
        print(f"  │ {name:<36} │ {da*100:>6.1f}% │ {r2_s} │ "
              f"{rmse_s} │ {n:>5} │{marker}")
    print(f"  └{'─'*71}┘")

    # ══════════════════════════════════════════════════════
    #  PER-TICKER (best model)
    # ══════════════════════════════════════════════════════
    best_r = all_results.get(best_name, {})
    pt = best_r.get("per_ticker", {})
    if pt:
        print(f"\n  Per-Ticker ({best_name}):")
        print(f"  {'Ticker':<8} {'DA':>8} {'N':>6}")
        print(f"  {'-'*24}")
        das = []
        for ticker in CFG["TICKERS"]:
            td = pt.get(ticker, {})
            if td:
                print(f"  {ticker:<8} {td['da']*100:>7.1f}% {td['n']:>6}")
                das.append(td['da'])
        if das:
            print(f"  {'Average':<8} {np.mean(das)*100:>7.1f}%")

    # ══════════════════════════════════════════════════════
    #  FEATURE IMPORTANCE (best LightGBM)
    # ══════════════════════════════════════════════════════
    best_lgb = mdl_d if mdl_d else mdl_c
    best_lgb_fn = fn_d if mdl_d else fn_c
    best_lgb_tag = "D" if mdl_d else "C"

    if best_lgb is not None:
        imp = best_lgb.feature_importances_
        fi = sorted(zip(best_lgb_fn, imp), key=lambda x: -x[1])

        print(f"\n  Feature Importance (Model {best_lgb_tag} — top 15):")
        print(f"  {'#':>4} {'Feature':<40} {'Imp':>8}")
        print(f"  {'-'*55}")
        for rank, (fn, fi_v) in enumerate(fi[:15], 1):
            cat = ""
            if "rv" in fn.lower() and not fn.startswith("tech_"):
                cat = " [RV]"
            elif fn.startswith("d") and fn[1:2].isdigit():
                cat = " [daily]"
            elif fn.startswith("tech_"):
                cat = " [tech]"
            elif fn.startswith("b") and fn[1:2].isdigit():
                cat = " [block]"
            elif "spy" in fn.lower():
                cat = " [cross]"
            print(f"  {rank:>4} {fn:<40} {fi_v:>8}{cat}")

        # Category totals
        cats = {"RV sequence": 0, "Block per-bar": 0, "Tech indicators": 0,
                "Daily": 0, "Cross-asset": 0, "Other": 0}
        for fn, fi_v in fi:
            if ("rv" in fn.lower() or fn.startswith("rv_")) and not fn.startswith("tech_"):
                cats["RV sequence"] += fi_v
            elif fn.startswith("b") and fn[1:2].isdigit() and "rv" not in fn:
                cats["Block per-bar"] += fi_v
            elif fn.startswith("tech_"):
                cats["Tech indicators"] += fi_v
            elif fn.startswith("d") and fn[1:2].isdigit():
                cats["Daily"] += fi_v
            elif "spy" in fn.lower():
                cats["Cross-asset"] += fi_v
            else:
                cats["Other"] += fi_v
        total = sum(cats.values()) + 1e-10
        print(f"\n  Category breakdown:")
        for cat, val in sorted(cats.items(), key=lambda x: -x[1]):
            if val > 0:
                print(f"    {cat:<20} {val/total*100:>5.1f}%")

    # ══════════════════════════════════════════════════════
    #  ANALYSIS
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ ANALYSIS")
    print(f"{'='*70}")

    print(f"\n  Naive RW baseline:       DA={naive_rw_da*100:.1f}%")
    print(f"  Historical Mean baseline: DA={hist_da*100:.1f}%  ← TRUE baseline (mean reversion)")
    if garch_res:
        print(f"  GARCH(1,1):              DA={garch_res['da']*100:.1f}%")

    da_a = res_a["da"]
    print(f"\n  Model A (original LSTM): DA={da_a*100:.1f}%")
    print(f"    vs Historical Mean: {(da_a - hist_da)*100:+.1f}pp")
    print(f"    vs Naive RW:        {(da_a - naive_rw_da)*100:+.1f}pp")

    if res_b:
        da_b = res_b["da"]
        print(f"\n  Model B (multi-scale):  DA={da_b*100:.1f}%")
        print(f"    vs Model A: {(da_b - da_a)*100:+.1f}pp")
        if da_b > da_a + 0.01:
            print(f"    → Daily context improves vol prediction ✓")
        elif da_b > da_a - 0.01:
            print(f"    → Comparable to single-scale baseline")
        else:
            print(f"    → Multi-scale did not help")

    if res_c:
        print(f"\n  Model C (LGB block):    DA={res_c['da']*100:.1f}%")
    if res_d:
        print(f"  Model D (LGB full):     DA={res_d['da']*100:.1f}%")
        if res_c:
            d = res_d["da"] - res_c["da"]
            print(f"    Daily features add: {d*100:+.1f}pp")

    # Best overall
    all_das = [(n, r["da"]) for n, r in all_results.items()
               if "Naive" not in n and "Historical" not in n and "GARCH" not in n]
    if all_das:
        bn, bd = max(all_das, key=lambda x: x[1])
        print(f"\n  ★ Best model: {bn}  DA={bd*100:.1f}%")
        if bd > hist_da + 0.01:
            print(f"    Beats Historical Mean baseline by {(bd-hist_da)*100:+.1f}pp ✓")
        else:
            print(f"    Does not clearly beat Historical Mean baseline")

    elapsed = time.time() - t_start
    print(f"\n  Total runtime: {elapsed:.1f}s")

    # ── Save ──
    jp = os.path.join(CFG["RESULTS_DIR"], "vol_prediction_v2_results.json")
    save = {}
    for name, r in all_results.items():
        save[name] = {k: v for k, v in r.items()
                      if isinstance(v, (int, float, str, dict))}
    save["naive_baselines"] = {
        "random_walk_da": float(naive_rw_da),
        "historical_mean_da": float(hist_da),
        "historical_mean_value": float(hist_mean),
    }
    if garch_res:
        save["garch"] = garch_res
    save["config"] = {
        "seq_len": CFG["SEQ_LEN"], "hidden": CFG["HIDDEN"],
        "layers": CFG["NUM_LAYERS"], "dropout": CFG["DROPOUT"],
        "lr": CFG["LR"], "loss": "MSE on log(RV)",
    }
    with open(jp, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"  Saved: {jp}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

  Volatility Prediction v2 — REGRESSION on log(RV)
  Device: cuda

  Loading data...
    AAPL: 980 blocks  train=686 val=147 test=147  feat=45
    MSFT: 980 blocks  train=686 val=147 test=147  feat=45
    GOOGL: 950 blocks  train=665 val=142 test=143  feat=45
    GOOG: 952 blocks  train=666 val=143 test=143  feat=45
    NVDA: 979 blocks  train=685 val=147 test=147  feat=45
    TSLA: 980 blocks  train=686 val=147 test=147  feat=45
    SPY: 980 blocks  train=686 val=147 test=147  feat=45
    QQQ: 980 blocks  train=686 val=147 test=147  feat=45

  MODEL A: Original Vol LSTM (regression on log(RV))
  LSTM input dim: 45 numeric + 1 log_rv = 46
  Samples: 7,693  features: 46
  Architecture: LSTM(46, h=32, L=1) → Linear(32,32) → ReLU → Drop(0.1) → Linear(32,1)
  Loss: MSE on log(RV)   LR=0.0005   seq_len=10   epochs=200
    [A] E 38 t=0.251884 v=0.412444 p=30
    [A] Early stop (best=8)

  Model A Test: DA=69.2%  R²=0.526  RMSE=0.005941  z=+13.01  (5.5s)
  ⚠ WARNING: DA=69.2%, expected ~71.3%

In [6]:
"""
vol_prediction_v3.py — Model E: LightGBM + LSTM Embeddings
===========================================================
Combines Model D's 102 flat features with Model B's Multi-Scale LSTM
internal embeddings (32+32=64 dims) for ~166 total features.

Prerequisite: vol_prediction_v2.py must have been run first to produce
  results/vol_multiscale_lstm_regression.pt (Model B weights)

Usage:
    !python vol_prediction_v3.py
"""

import os, sys, time, json, warnings, math
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, r2_score, mean_squared_error,
                             f1_score, precision_score, recall_score)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[ERROR] lightgbm required"); sys.exit(1)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = {
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "VOL_DATA_DIR": "vol_data",
    "BLOCK_SIZE": 12,
    "SEQ_LEN": 10,
    "DAILY_LOOKBACK": 20,
    "LR": 5e-4,
    "WD": 1e-4,
    "BATCH": 64,
    "EPOCHS": 200,
    "PATIENCE": 30,
    "SCHED_PATIENCE": 10,
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.85,
}


# ═══════════════════════════════════════════════════════════════
#  UTILITIES (same as v2)
# ═══════════════════════════════════════════════════════════════

def load_features_csv(ticker, freq):
    p = os.path.join(CFG["FEATURES_DIR"], f"{ticker}_{freq}_features.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    df = pd.read_csv(p)
    for col in ["timestamp", "ts_event", "datetime", "date"]:
        if col in df.columns:
            df["timestamp"] = pd.to_datetime(df[col], utc=True)
            break
    if "timestamp" not in df.columns:
        first = df.columns[0]
        if first not in df.select_dtypes(include=[np.number]).columns:
            df["timestamp"] = pd.to_datetime(df[first], utc=True)
    return df.sort_values("timestamp").reset_index(drop=True)


def numeric_cols(df):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


def split3(n):
    return int(n * CFG["TRAIN_RATIO"]), int(n * CFG["VAL_RATIO"])


# ═══════════════════════════════════════════════════════════════
#  DATA PIPELINE (matches v2 exactly)
# ═══════════════════════════════════════════════════════════════

def compute_blocks(df_1h, block_size=12):
    close = df_1h["close"].values.astype(np.float64)
    ts = df_1h["timestamp"].values
    lr = np.concatenate([[0], np.diff(np.log(close + 1e-10))])
    n_blocks = len(close) // block_size
    rows = []
    for b in range(n_blocks):
        s, e = b * block_size, (b + 1) * block_size
        rv = np.std(lr[s:e]) * np.sqrt(block_size)
        rows.append({"block_idx": b, "bar_start": s, "bar_end": e,
                      "rv": rv, "log_rv": np.log(rv + 1e-10),
                      "ts_start": ts[s], "ts_end": ts[min(e-1, len(close)-1)]})
    return pd.DataFrame(rows)


def compute_daily_bars(df_1h):
    df = df_1h.copy()
    df["_date"] = df["timestamp"].dt.date
    has_vol = "volume" in df.columns
    agg = {"open": ("open", "first"), "high": ("high", "max"),
           "low": ("low", "min"), "close": ("close", "last"),
           "n_bars": ("close", "count")}
    if has_vol:
        agg["volume"] = ("volume", "sum")
    daily = df.groupby("_date").agg(**agg).reset_index()
    daily.rename(columns={"_date": "date"}, inplace=True)

    c = daily["close"].values.astype(np.float64)
    h = daily["high"].values.astype(np.float64)
    lo = daily["low"].values.astype(np.float64)
    o = daily["open"].values.astype(np.float64)
    n = len(daily)

    daily["daily_return"] = (c - o) / (o + 1e-10)
    daily["daily_range"] = (h - lo) / (c + 1e-10)

    if has_vol:
        v = daily["volume"].values.astype(np.float64)
        daily["vol_change"] = np.concatenate([[1.0], v[1:] / (v[:-1] + 1e-10)])
    else:
        daily["vol_change"] = 1.0

    drv = []
    for _, grp in df.groupby("_date"):
        cc = grp["close"].values.astype(np.float64)
        if len(cc) > 1:
            lr_ = np.diff(np.log(cc + 1e-10))
            drv.append(np.std(lr_) * np.sqrt(len(cc)))
        else:
            drv.append(0.0)
    daily["daily_rv"] = drv

    # RSI-14
    delta = np.diff(c, prepend=c[0])
    gain = np.where(delta > 0, delta, 0.0)
    loss_a = np.where(delta < 0, -delta, 0.0)
    ag = np.zeros(n); al = np.zeros(n)
    if n > 14:
        ag[14] = np.mean(gain[1:15]); al[14] = np.mean(loss_a[1:15])
        for i in range(15, n):
            ag[i] = (ag[i-1]*13 + gain[i]) / 14
            al[i] = (al[i-1]*13 + loss_a[i]) / 14
    daily["rsi_14"] = 100 - 100 / (1 + ag / (al + 1e-10))

    # ATR-14
    tr = np.zeros(n)
    for i in range(1, n):
        tr[i] = max(h[i]-lo[i], abs(h[i]-c[i-1]), abs(lo[i]-c[i-1]))
    tr[0] = h[0] - lo[0]
    atr = np.zeros(n)
    if n > 14:
        atr[14] = np.mean(tr[1:15])
        for i in range(15, n):
            atr[i] = (atr[i-1]*13 + tr[i]) / 14
    daily["atr_14"] = atr

    # BB bandwidth
    bb = np.zeros(n)
    for i in range(20, n):
        sma = np.mean(c[i-20:i]); std = np.std(c[i-20:i])
        bb[i] = (4 * std) / (sma + 1e-10)
    daily["bb_bandwidth_20"] = bb
    daily["log_close"] = np.log(c + 1e-10)

    return daily


def load_all_data():
    """Load and build blocks + daily for all tickers."""
    BS = CFG["BLOCK_SIZE"]
    all_blocks = {}   # ticker → (dfb, block_feats, feat_names)
    all_daily = {}

    for ticker in CFG["TICKERS"]:
        try:
            df_1h = load_features_csv(ticker, "1hour")
        except FileNotFoundError:
            continue
        fc = numeric_cols(df_1h)
        fv = np.nan_to_num(df_1h[fc].values.astype(np.float32), nan=0.0)
        dfb = compute_blocks(df_1h, BS)
        nb = len(dfb)
        bf = np.zeros((nb, len(fc)), dtype=np.float32)
        for i in range(nb):
            s, e = dfb.loc[i, "bar_start"], dfb.loc[i, "bar_end"]
            bf[i] = np.mean(fv[s:e], axis=0)
        all_blocks[ticker] = (dfb, bf, fc)
        all_daily[ticker] = compute_daily_bars(df_1h)

    return all_blocks, all_daily


# ═══════════════════════════════════════════════════════════════
#  BUILD ALIGNED DATASETS
# ═══════════════════════════════════════════════════════════════

def build_model_b_sequences(all_blocks, all_daily):
    """
    Build short+long sequences for Model B. Returns arrays + (ticker, block_idx) keys.
    Matches v2 logic exactly: short = 10 blocks of (feat+log_rv), long = 20 days.
    """
    SL = CFG["SEQ_LEN"]; DL = CFG["DAILY_LOOKBACK"]
    dcols = ["daily_rv", "daily_range", "daily_return", "vol_change",
             "rsi_14", "atr_14", "bb_bandwidth_20", "log_close"]

    Xs, Xl, keys = [], [], []
    n_sf = n_df = None

    for ticker, (dfb, bf, fc) in all_blocks.items():
        log_rvs = dfb["log_rv"].values.astype(np.float32)
        bf_ext = np.column_stack([bf, log_rvs.reshape(-1, 1)])
        if n_sf is None:
            n_sf = bf_ext.shape[1]

        daily = all_daily.get(ticker)
        if daily is None:
            continue
        dc_v = [c for c in dcols if c in daily.columns]
        if n_df is None:
            n_df = len(dc_v)
        df_vals = np.nan_to_num(daily[dc_v].values.astype(np.float32), nan=0.0)
        ddates = daily["date"].values
        nb = len(dfb)

        for i in range(SL, nb - 1):
            short_seq = bf_ext[i - SL:i]

            block_end = pd.Timestamp(dfb.loc[i, "ts_end"])
            bd = block_end.date() if hasattr(block_end, 'date') else block_end
            day_idx = -1
            for di in range(len(ddates) - 1, -1, -1):
                if ddates[di] <= bd:
                    day_idx = di; break
            if day_idx < DL:
                continue
            long_seq = df_vals[day_idx - DL + 1:day_idx + 1]
            if len(long_seq) != DL:
                continue

            Xs.append(short_seq)
            Xl.append(long_seq)
            keys.append((ticker, i))

    return (np.array(Xs, dtype=np.float32),
            np.array(Xl, dtype=np.float32),
            keys, n_sf, n_df)


def build_model_d_features(all_blocks, all_daily):
    """
    Build Model D flat features (block + daily). Returns X, y_reg, y_cls, cur_rv, keys, fnames.
    Matches v2 logic exactly.
    """
    SL = CFG["SEQ_LEN"]; DAILY_LB = 5
    key_cols_names = ["close", "volume", "atr_14", "rsi_14"]
    daily_cols = ["daily_rv", "daily_range", "daily_return", "vol_change"]

    X_rows, y_reg, y_cls, cur_rv, keys = [], [], [], [], []

    for ticker, (dfb, bf, fc) in all_blocks.items():
        rvs = dfb["rv"].values.astype(np.float64)
        log_rvs = dfb["log_rv"].values.astype(np.float64)
        fc_list = list(fc)
        nb = len(dfb)

        key_idxs = {}
        for kc in key_cols_names:
            matches = [j for j, c in enumerate(fc_list) if kc.lower() in c.lower()]
            if matches:
                key_idxs[kc] = matches[0]

        daily = all_daily.get(ticker)
        if daily is not None:
            dc_v = [c for c in daily_cols if c in daily.columns]
            df_vals = np.nan_to_num(daily[dc_v].values.astype(np.float32), nan=0.0)
            ddates = daily["date"].values
            n_dc = len(dc_v)
        else:
            dc_v = []; n_dc = 0; df_vals = None; ddates = None

        spy_data = all_blocks.get("SPY")
        spy_rvs = spy_data[0]["rv"].values if spy_data else None

        for i in range(SL, nb - 1):
            feats = {}
            for off in range(SL):
                bi = i - SL + off
                px = f"b{off}"
                feats[f"{px}_rv"] = rvs[bi]
                feats[f"{px}_log_rv"] = log_rvs[bi]
                for kc, ki in key_idxs.items():
                    feats[f"{px}_{kc}"] = float(bf[bi, ki])

            rv_win = rvs[max(0, i-SL):i]
            rv4 = rvs[max(0, i-4):i]
            feats["rv_mean_4"] = np.mean(rv4) if len(rv4) else 0
            feats["rv_std_4"] = np.std(rv4) if len(rv4) > 1 else 0
            feats["rv_mean_8"] = np.mean(rv_win) if len(rv_win) else 0
            feats["rv_std_8"] = np.std(rv_win) if len(rv_win) > 1 else 0
            feats["rv_ratio"] = rvs[i-1] / (feats["rv_mean_8"] + 1e-10)
            if len(rv_win) >= 3:
                feats["rv_trend"] = float(np.polyfit(np.arange(len(rv_win)), rv_win, 1)[0])
            else:
                feats["rv_trend"] = 0.0
            su = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] > rvs[k-1]: su += 1
                else: break
            feats["rv_streak_up"] = su
            sd = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] < rvs[k-1]: sd += 1
                else: break
            feats["rv_streak_down"] = sd
            feats["current_log_rv"] = log_rvs[i-1]

            for j, cn in enumerate(fc_list):
                feats[f"tech_{cn}"] = float(bf[i-1, j])

            if ticker != "SPY" and spy_rvs is not None and i < len(spy_rvs):
                feats["spy_rv_cur"] = spy_rvs[max(0, i-1)]
                s4 = spy_rvs[max(0, i-4):i]
                feats["spy_rv_mean_4"] = np.mean(s4) if len(s4) else 0
            else:
                feats["spy_rv_cur"] = 0; feats["spy_rv_mean_4"] = 0

            # Daily features
            if daily is not None and df_vals is not None:
                bets = pd.Timestamp(dfb.loc[i, "ts_end"])
                bd = bets.date() if hasattr(bets, 'date') else bets
                day_idx = -1
                for di in range(len(ddates)-1, -1, -1):
                    if ddates[di] <= bd: day_idx = di; break
                if day_idx >= DAILY_LB:
                    for d_off in range(DAILY_LB):
                        di2 = day_idx - DAILY_LB + 1 + d_off
                        for ci, cn in enumerate(dc_v[:n_dc]):
                            feats[f"d{d_off}_{cn}"] = float(df_vals[di2, ci])
                    drv_w = df_vals[day_idx-DAILY_LB+1:day_idx+1, 0]
                    feats["daily_rv_mean_5"] = float(np.mean(drv_w))
                    feats["daily_rv_std_5"] = float(np.std(drv_w))
                    if len(drv_w) >= 3:
                        feats["daily_rv_trend"] = float(
                            np.polyfit(np.arange(len(drv_w)), drv_w, 1)[0])
                    else:
                        feats["daily_rv_trend"] = 0.0

            X_rows.append(feats)
            y_reg.append(log_rvs[i])
            y_cls.append(1 if rvs[i] > rvs[i-1] else 0)
            cur_rv.append(rvs[i-1])
            keys.append((ticker, i))

    if not X_rows:
        return np.array([]), np.array([]), np.array([]), np.array([]), keys, []
    fnames = list(X_rows[0].keys())
    X = np.array([[r.get(fn, 0.0) for fn in fnames] for r in X_rows], dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return (X, np.array(y_reg, dtype=np.float32),
            np.array(y_cls, dtype=np.float32),
            np.array(cur_rv, dtype=np.float64), keys, fnames)


# ═══════════════════════════════════════════════════════════════
#  MODEL B ARCHITECTURE (must match v2 exactly)
# ═══════════════════════════════════════════════════════════════

class VolLSTM_B(nn.Module):
    def __init__(self, n_sfeat, n_dfeat):
        super().__init__()
        self.short_lstm = nn.LSTM(n_sfeat, 32, 1, batch_first=True)
        self.long_lstm = nn.LSTM(n_dfeat, 32, 1, batch_first=True)
        self.fuse = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, xs, xl):
        so, _ = self.short_lstm(xs)
        lo, _ = self.long_lstm(xl)
        return self.fuse(torch.cat([so[:, -1, :], lo[:, -1, :]], dim=1)).squeeze(-1)

    def extract_embeddings(self, xs, xl):
        """Extract 32-dim embeddings from each branch (before fusion)."""
        so, _ = self.short_lstm(xs)
        lo, _ = self.long_lstm(xl)
        return so[:, -1, :], lo[:, -1, :]  # (batch,32), (batch,32)


class PairDS(Dataset):
    def __init__(self, Xs, Xl):
        self.Xs = torch.tensor(Xs, dtype=torch.float32)
        self.Xl = torch.tensor(Xl, dtype=torch.float32)
    def __len__(self): return len(self.Xs)
    def __getitem__(self, i): return self.Xs[i], self.Xl[i]


class PairLabelDS(Dataset):
    def __init__(self, Xs, Xl, y):
        self.Xs = torch.tensor(Xs, dtype=torch.float32)
        self.Xl = torch.tensor(Xl, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.Xs[i], self.Xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  RETRAIN MODEL B IF NEEDED
# ═══════════════════════════════════════════════════════════════

def retrain_model_b(Xs, Xl, y_reg, tr, va, n_sf, n_df):
    """Retrain Model B with same config as v2 (regression on log(RV), MSE)."""
    print("  Retraining Model B (regression on log(RV))...")

    # Normalize
    flat_s = Xs[:tr].reshape(-1, n_sf)
    mu_s, sig_s = flat_s.mean(0), flat_s.std(0) + 1e-8
    Xs_n = np.nan_to_num((Xs - mu_s) / sig_s, nan=0.0, posinf=0.0, neginf=0.0)

    flat_l = Xl[:tr].reshape(-1, n_df)
    mu_l, sig_l = flat_l.mean(0), flat_l.std(0) + 1e-8
    Xl_n = np.nan_to_num((Xl - mu_l) / sig_l, nan=0.0, posinf=0.0, neginf=0.0)

    model = VolLSTM_B(n_sf, n_df).to(DEVICE)
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=CFG["SCHED_PATIENCE"], factor=0.5)

    tr_dl = DataLoader(PairLabelDS(Xs_n[:tr], Xl_n[:tr], y_reg[:tr]),
                       batch_size=CFG["BATCH"], shuffle=True)
    va_dl = DataLoader(PairLabelDS(Xs_n[tr:va], Xl_n[tr:va], y_reg[tr:va]),
                       batch_size=512)

    best_vl = float("inf"); best_st = None; pat = 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); tl = 0
        for xs, xl, yb in tr_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xs, xl), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
        tl /= tr

        model.eval(); vl = 0
        with torch.no_grad():
            for xs, xl, yb in va_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xs, xl), yb).item() * len(yb)
        vl /= (va - tr)
        sched.step(vl)

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if (ep+1) % 40 == 0 or pat == CFG["PATIENCE"]:
            print(f"    E{ep+1:3d} t={tl:.6f} v={vl:.6f} p={pat}")
        if pat >= CFG["PATIENCE"]:
            print(f"    Early stop")
            break

    if best_st:
        model.load_state_dict(best_st)
    model.to(DEVICE).eval()

    sp = os.path.join(CFG["RESULTS_DIR"], "vol_multiscale_lstm_regression.pt")
    torch.save(model.state_dict(), sp)
    print(f"  Saved: {sp}")
    return model, Xs_n, Xl_n


# ═══════════════════════════════════════════════════════════════
#  EVALUATION
# ═══════════════════════════════════════════════════════════════

def eval_regression(pred_log, true_log, cur_rv, meta_te, name):
    pred_rv = np.exp(pred_log)
    true_rv = np.exp(true_log)
    pred_dir = (pred_rv > cur_rv).astype(int)
    true_dir = (true_rv > cur_rv).astype(int)

    da = (pred_dir == true_dir).mean()
    r2 = r2_score(true_rv, pred_rv)
    rmse = np.sqrt(mean_squared_error(true_rv, pred_rv))
    n = len(true_dir)
    z = (da - 0.5) / np.sqrt(0.5*0.5/n)
    from scipy.stats import norm
    p_val = 2 * (1 - norm.cdf(abs(z)))
    f1 = f1_score(true_dir, pred_dir, zero_division=0)

    result = {"name": name, "da": float(da), "r2": float(r2),
              "rmse": float(rmse), "z": float(z), "p": float(p_val),
              "f1": float(f1), "n": n}

    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                td_da = float((pred_dir[mask] == true_dir[mask]).mean())
                tn = int(mask.sum())
                tz = (td_da - 0.5) / np.sqrt(0.5*0.5/tn)
                pt[ticker] = {"da": td_da, "n": tn, "z": float(tz)}
        result["per_ticker"] = pt
    return result


def eval_classification(pred_prob, true_dir, meta_te, name):
    preds = (pred_prob >= 0.5).astype(int)
    da = (preds == true_dir).mean()
    n = len(true_dir)
    z = (da - 0.5) / np.sqrt(0.5*0.5/n)
    auc = roc_auc_score(true_dir, pred_prob) if true_dir.sum() > 0 and (1-true_dir).sum() > 0 else 0.5
    f1 = f1_score(true_dir, preds, zero_division=0)
    result = {"name": name, "da": float(da), "r2": 0.0, "rmse": 0.0,
              "z": float(z), "f1": float(f1), "n": n, "auc": float(auc)}
    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                pt[ticker] = {"da": float((preds[mask] == true_dir[mask]).mean()),
                              "n": int(mask.sum())}
        result["per_ticker"] = pt
    return result


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("  Vol Prediction v3 — Model E: LightGBM + LSTM Embeddings")
    print(f"  Device: {DEVICE}")
    print("=" * 70)
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    t_start = time.time()

    # ══════════════════════════════════════════════════════
    #  STEP 1: Load all data
    # ══════════════════════════════════════════════════════
    print("\n  STEP 1: Loading data...")
    all_blocks, all_daily = load_all_data()
    print(f"  Tickers loaded: {list(all_blocks.keys())}")

    # Build Model B sequences (with keys for alignment)
    print("\n  Building Model B sequences...")
    Xs_b, Xl_b, keys_b, n_sf, n_df = build_model_b_sequences(all_blocks, all_daily)
    print(f"  Model B: {len(keys_b):,} samples  "
          f"short=({CFG['SEQ_LEN']},{n_sf})  long=({CFG['DAILY_LOOKBACK']},{n_df})")

    # Build Model D features (with keys for alignment)
    print("  Building Model D features...")
    X_d, y_reg_d, y_cls_d, cur_d, keys_d, fnames_d = build_model_d_features(
        all_blocks, all_daily)
    print(f"  Model D: {len(keys_d):,} samples  {len(fnames_d)} features")

    # ── ALIGN datasets using (ticker, block_idx) keys ──
    print("\n  Aligning datasets...")
    key_set_b = set(keys_b)
    key_set_d = set(keys_d)
    common_keys = sorted(key_set_b & key_set_d)
    print(f"  Model B: {len(keys_b):,}  Model D: {len(keys_d):,}  "
          f"Common: {len(common_keys):,}")

    # Build lookup indices
    b_idx_map = {k: i for i, k in enumerate(keys_b)}
    d_idx_map = {k: i for i, k in enumerate(keys_d)}

    b_indices = np.array([b_idx_map[k] for k in common_keys])
    d_indices = np.array([d_idx_map[k] for k in common_keys])

    Xs_aligned = Xs_b[b_indices]
    Xl_aligned = Xl_b[b_indices]
    X_d_aligned = X_d[d_indices]
    y_reg_aligned = y_reg_d[d_indices]
    y_cls_aligned = y_cls_d[d_indices]
    cur_aligned = cur_d[d_indices]
    meta_aligned = common_keys

    n_total = len(common_keys)
    tr, va = split3(n_total)
    print(f"  Aligned: {n_total:,}  train={tr} val={va-tr} test={n_total-va}")

    # ══════════════════════════════════════════════════════
    #  STEP 2: Load / retrain Model B
    # ══════════════════════════════════════════════════════
    print("\n  STEP 2: Loading Model B checkpoint...")
    ckpt_path = os.path.join(CFG["RESULTS_DIR"], "vol_multiscale_lstm_regression.pt")

    model_b = VolLSTM_B(n_sf, n_df).to(DEVICE)
    retrained = False

    if os.path.exists(ckpt_path):
        try:
            state = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
            model_b.load_state_dict(state)
            print(f"  ✓ Loaded: {ckpt_path}")
        except Exception as e:
            print(f"  ✗ Failed to load: {e}")
            print(f"  Retraining Model B...")
            model_b, Xs_n_cache, Xl_n_cache = retrain_model_b(
                Xs_aligned, Xl_aligned, y_reg_aligned, tr, va, n_sf, n_df)
            retrained = True
    else:
        print(f"  ✗ Checkpoint not found: {ckpt_path}")
        print(f"  Retraining Model B...")
        model_b, Xs_n_cache, Xl_n_cache = retrain_model_b(
            Xs_aligned, Xl_aligned, y_reg_aligned, tr, va, n_sf, n_df)
        retrained = True

    model_b.eval()

    # Normalize sequences for embedding extraction
    if not retrained:
        flat_s = Xs_aligned[:tr].reshape(-1, n_sf)
        mu_s, sig_s = flat_s.mean(0), flat_s.std(0) + 1e-8
        Xs_n = np.nan_to_num((Xs_aligned - mu_s) / sig_s, nan=0.0, posinf=0.0, neginf=0.0)

        flat_l = Xl_aligned[:tr].reshape(-1, n_df)
        mu_l, sig_l = flat_l.mean(0), flat_l.std(0) + 1e-8
        Xl_n = np.nan_to_num((Xl_aligned - mu_l) / sig_l, nan=0.0, posinf=0.0, neginf=0.0)
    else:
        Xs_n = Xs_n_cache
        Xl_n = Xl_n_cache

    # ── Extract embeddings ──
    print("  Extracting LSTM embeddings...")
    dl = DataLoader(PairDS(Xs_n, Xl_n), batch_size=512)
    all_short_emb, all_long_emb = [], []
    with torch.no_grad():
        for xs, xl in dl:
            se, le = model_b.extract_embeddings(xs.to(DEVICE), xl.to(DEVICE))
            all_short_emb.append(se.cpu().numpy())
            all_long_emb.append(le.cpu().numpy())
    short_emb = np.concatenate(all_short_emb)  # (N, 32)
    long_emb = np.concatenate(all_long_emb)    # (N, 32)

    # Normalize embeddings from training set only
    emb_mu_s = short_emb[:tr].mean(0); emb_sig_s = short_emb[:tr].std(0) + 1e-8
    emb_mu_l = long_emb[:tr].mean(0); emb_sig_l = long_emb[:tr].std(0) + 1e-8
    short_emb_n = np.nan_to_num((short_emb - emb_mu_s) / emb_sig_s, nan=0.0)
    long_emb_n = np.nan_to_num((long_emb - emb_mu_l) / emb_sig_l, nan=0.0)

    print(f"  Short emb: {short_emb_n.shape}  Long emb: {long_emb_n.shape}")

    # ── Evaluate Model B on test set ──
    pred_b_dl = DataLoader(PairDS(Xs_n[va:], Xl_n[va:]), batch_size=512)
    pred_b_list = []
    with torch.no_grad():
        for xs, xl in pred_b_dl:
            pred_b_list.append(model_b(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy())
    pred_b_log = np.concatenate(pred_b_list)
    res_b = eval_regression(pred_b_log, y_reg_aligned[va:], cur_aligned[va:],
                            meta_aligned[va:], "B: Multi-Scale LSTM")
    print(f"  Model B Test: DA={res_b['da']*100:.1f}%  R²={res_b['r2']:.3f}  "
          f"z={res_b['z']:+.2f}")

    # ══════════════════════════════════════════════════════
    #  STEP 3: Build Model E feature matrix
    # ══════════════════════════════════════════════════════
    print("\n  STEP 3: Building Model E features...")

    emb_short_names = [f"emb_short_{i}" for i in range(32)]
    emb_long_names = [f"emb_long_{i}" for i in range(32)]
    fnames_e = fnames_d + emb_short_names + emb_long_names

    X_e = np.column_stack([X_d_aligned, short_emb_n, long_emb_n])
    print(f"  Model E features: {X_d_aligned.shape[1]} (D) + 32 (short_emb) "
          f"+ 32 (long_emb) = {X_e.shape[1]}")

    # ══════════════════════════════════════════════════════
    #  STEP 4: Train Model E + re-run Model D (aligned)
    # ══════════════════════════════════════════════════════
    print("\n  STEP 4: Training LightGBM models...")

    te_y_reg = y_reg_aligned[va:]
    te_y_cls = y_cls_aligned[va:].astype(int)
    te_cur = cur_aligned[va:]
    meta_te = meta_aligned[va:]

    # ── Model D (aligned) — classification ──
    print("\n  Model D (aligned, classification)...")
    mdl_d_cls = lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_d_cls.fit(X_d_aligned[:tr], y_cls_aligned[:tr],
                  eval_set=[(X_d_aligned[tr:va], y_cls_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    d_cls_prob = mdl_d_cls.predict_proba(X_d_aligned[va:])[:, 1]
    d_cls_da = ((d_cls_prob >= 0.5).astype(int) == te_y_cls).mean()

    # ── Model D (aligned) — regression ──
    print("  Model D (aligned, regression)...")
    mdl_d_reg = lgb.LGBMRegressor(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_d_reg.fit(X_d_aligned[:tr], y_reg_aligned[:tr],
                  eval_set=[(X_d_aligned[tr:va], y_reg_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    d_reg_pred = mdl_d_reg.predict(X_d_aligned[va:])
    d_reg_dir = (np.exp(d_reg_pred) > te_cur).astype(int)
    d_reg_da = (d_reg_dir == te_y_cls).mean()

    print(f"  Model D cls DA: {d_cls_da*100:.1f}%  reg DA: {d_reg_da*100:.1f}%")
    if d_reg_da >= d_cls_da:
        res_d = eval_regression(d_reg_pred, te_y_reg, te_cur, meta_te,
                                "D: LightGBM (block+daily) [reg]")
        d_approach = "regression"
    else:
        res_d = eval_classification(d_cls_prob, te_y_cls, meta_te,
                                    "D: LightGBM (block+daily) [cls]")
        d_approach = "classification"
    print(f"  Model D best: {d_approach}  DA={res_d['da']*100:.1f}%")

    # ── Model E — classification ──
    print("\n  Model E (classification)...")
    mdl_e_cls = lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_e_cls.fit(X_e[:tr], y_cls_aligned[:tr],
                  eval_set=[(X_e[tr:va], y_cls_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    e_cls_prob = mdl_e_cls.predict_proba(X_e[va:])[:, 1]
    e_cls_da = ((e_cls_prob >= 0.5).astype(int) == te_y_cls).mean()

    # ── Model E — regression ──
    print("  Model E (regression)...")
    mdl_e_reg = lgb.LGBMRegressor(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_e_reg.fit(X_e[:tr], y_reg_aligned[:tr],
                  eval_set=[(X_e[tr:va], y_reg_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    e_reg_pred = mdl_e_reg.predict(X_e[va:])
    e_reg_dir = (np.exp(e_reg_pred) > te_cur).astype(int)
    e_reg_da = (e_reg_dir == te_y_cls).mean()

    print(f"  Model E cls DA: {e_cls_da*100:.1f}%  reg DA: {e_reg_da*100:.1f}%")
    if e_reg_da >= e_cls_da:
        res_e = eval_regression(e_reg_pred, te_y_reg, te_cur, meta_te,
                                "E: LightGBM + LSTM emb [reg]")
        best_e_mdl = mdl_e_reg
        e_approach = "regression"
    else:
        res_e = eval_classification(e_cls_prob, te_y_cls, meta_te,
                                    "E: LightGBM + LSTM emb [cls]")
        best_e_mdl = mdl_e_cls
        e_approach = "classification"
    print(f"  Model E best: {e_approach}  DA={res_e['da']*100:.1f}%")

    # ── Naive baselines on aligned test set ──
    true_rv_te = np.exp(te_y_reg)
    naive_rw_da = float((te_y_cls == 0).mean())  # predict "down" = predict cur
    train_rvs = np.exp(y_reg_aligned[:tr])
    hist_mean = float(train_rvs.mean())
    hist_dir = (hist_mean > te_cur).astype(int)
    hist_da = float((hist_dir == te_y_cls).mean())

    # ══════════════════════════════════════════════════════
    #  STEP 5: Comparison table
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ COMPARISON TABLE")
    print(f"{'='*70}")

    all_r = {
        "Naive (RW)": {"da": naive_rw_da, "r2": 0, "rmse": 0, "n": len(te_y_cls)},
        "Historical Mean": {"da": hist_da, "r2": 0, "rmse": 0, "n": len(te_y_cls)},
    }
    all_r["B: Multi-Scale LSTM"] = res_b
    all_r[res_d["name"]] = res_d
    all_r[res_e["name"]] = res_e

    print(f"\n  ┌{'─'*71}┐")
    print(f"  │ {'Model':<38} │ {'DA':>7} │ {'R²':>7} │ {'N':>6} │")
    print(f"  ├{'─'*71}┤")

    for name, r in all_r.items():
        da = r["da"]; r2v = r.get("r2", 0); n = r.get("n", 0)
        r2_s = f"{r2v:>7.3f}" if r2v != 0 else "     —"
        print(f"  │ {name:<38} │ {da*100:>6.1f}% │ {r2_s} │ {n:>6} │")

    delta = res_e["da"] - res_d["da"]
    marker = "✓ LSTM emb helps" if delta > 0.005 else ("~ marginal" if delta > -0.005 else "✗ no gain")
    print(f"  ├{'─'*71}┤")
    print(f"  │ {'Δ (E vs D)':<38} │ {delta*100:>+6.1f}p │ {'':>7} │ {marker:>6} │")
    print(f"  └{'─'*71}┘")

    # ══════════════════════════════════════════════════════
    #  STEP 6: Feature importance
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ FEATURE IMPORTANCE (Model E — top 20)")
    print(f"{'='*70}")

    imp = best_e_mdl.feature_importances_
    fi = sorted(zip(fnames_e, imp), key=lambda x: -x[1])

    print(f"\n  {'#':>4} {'Feature':<42} {'Imp':>8} {'Cat':>8}")
    print(f"  {'-'*65}")

    emb_in_top20 = 0
    for rank, (fn, fv) in enumerate(fi[:20], 1):
        if fn.startswith("emb_short_"):
            cat = "[emb_s]"; emb_in_top20 += 1
        elif fn.startswith("emb_long_"):
            cat = "[emb_l]"; emb_in_top20 += 1
        elif ("rv" in fn.lower() or fn.startswith("rv_")) and not fn.startswith("tech_"):
            cat = "[rv]"
        elif fn.startswith("d") and fn[1:2].isdigit():
            cat = "[daily]"
        elif fn.startswith("tech_"):
            cat = "[tech]"
        elif fn.startswith("b") and fn[1:2].isdigit():
            cat = "[block]"
        elif "spy" in fn.lower():
            cat = "[cross]"
        else:
            cat = "[other]"
        print(f"  {rank:>4} {fn:<42} {fv:>8} {cat:>8}")

    # Category breakdown
    cats = {"Block features": 0, "Daily features": 0, "Tech indicators": 0,
            "LSTM short emb": 0, "LSTM long emb": 0,
            "RV sequence": 0, "Cross-asset": 0, "Other": 0}
    for fn, fv in fi:
        if fn.startswith("emb_short_"):
            cats["LSTM short emb"] += fv
        elif fn.startswith("emb_long_"):
            cats["LSTM long emb"] += fv
        elif ("rv" in fn.lower() or fn.startswith("rv_")) and not fn.startswith("tech_"):
            cats["RV sequence"] += fv
        elif fn.startswith("d") and fn[1:2].isdigit():
            cats["Daily features"] += fv
        elif fn.startswith("tech_"):
            cats["Tech indicators"] += fv
        elif fn.startswith("b") and fn[1:2].isdigit():
            cats["Block features"] += fv
        elif "spy" in fn.lower():
            cats["Cross-asset"] += fv
        else:
            cats["Other"] += fv

    total_imp = sum(cats.values()) + 1e-10
    print(f"\n  Category importance breakdown:")
    for cat, val in sorted(cats.items(), key=lambda x: -x[1]):
        bar = "█" * int(val/total_imp*40)
        print(f"    {cat:<20} {val/total_imp*100:>5.1f}%  {bar}")

    lstm_total = (cats["LSTM short emb"] + cats["LSTM long emb"]) / total_imp * 100
    print(f"\n  LSTM embedding total: {lstm_total:.1f}%")
    print(f"  LSTM embedding features in top 20: {emb_in_top20}/20")

    if emb_in_top20 >= 3:
        print(f"  → LSTM captures patterns LightGBM can't learn from flat features ✓")
    elif emb_in_top20 >= 1:
        print(f"  → LSTM provides some complementary signal")
    else:
        print(f"  → Flat features already subsume LSTM's knowledge")

    # ══════════════════════════════════════════════════════
    #  STEP 7: Per-ticker breakdown
    # ══════════════════════════════════════════════════════
    pt_e = res_e.get("per_ticker", {})
    if pt_e:
        print(f"\n  Per-Ticker (Model E):")
        print(f"  {'Ticker':<8} {'DA':>8} {'N':>6} {'z':>7}")
        print(f"  {'-'*32}")
        das = []
        for ticker in CFG["TICKERS"]:
            td = pt_e.get(ticker, {})
            if td:
                z_s = f"{td.get('z', 0):+.2f}" if "z" in td else "—"
                print(f"  {ticker:<8} {td['da']*100:>7.1f}% {td['n']:>6} {z_s:>7}")
                das.append(td["da"])
        if das:
            print(f"  {'Average':<8} {np.mean(das)*100:>7.1f}%")

    # ══════════════════════════════════════════════════════
    #  ANALYSIS
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ ANALYSIS")
    print(f"{'='*70}")

    print(f"\n  Model B (LSTM only):         DA={res_b['da']*100:.1f}%")
    print(f"  Model D (LGB flat only):     DA={res_d['da']*100:.1f}%")
    print(f"  Model E (LGB + LSTM emb):    DA={res_e['da']*100:.1f}%")
    print(f"\n  Δ(E vs D): {delta*100:+.1f}pp")
    print(f"  Δ(E vs B): {(res_e['da']-res_b['da'])*100:+.1f}pp")

    if delta > 0.01:
        print(f"\n  → LSTM embeddings provide meaningful boost over flat features")
        print(f"    Sequential patterns captured by LSTM complement tabular features")
    elif delta > 0:
        print(f"\n  → Small positive contribution from LSTM embeddings")
        print(f"    Most vol prediction power comes from flat RV/tech features")
    else:
        print(f"\n  → LSTM embeddings do not improve over flat features")
        print(f"    LightGBM with engineered features already captures available signal")

    elapsed = time.time() - t_start
    print(f"\n  Total runtime: {elapsed:.1f}s")

    # ── Save ──
    jp = os.path.join(CFG["RESULTS_DIR"], "vol_prediction_v3_results.json")
    save = {}
    for name, r in all_r.items():
        save[name] = {k: v for k, v in r.items()
                      if isinstance(v, (int, float, str, dict))}
    save["config"] = {
        "model_d_features": len(fnames_d),
        "lstm_emb_features": 64,
        "total_features": len(fnames_e),
        "model_e_approach": e_approach,
        "model_d_approach": d_approach,
    }
    save["category_importance"] = {cat: float(val/total_imp)
                                   for cat, val in cats.items()}
    with open(jp, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"  Saved: {jp}")

    # ── 保存模型权重 ──
    import joblib

    # Model E（最终用的那个）
    e_dir = os.path.join(CFG["RESULTS_DIR"],
                         "models/layer1/volatility/lightgbm_v3")
    os.makedirs(e_dir, exist_ok=True)
    joblib.dump(best_e_mdl, os.path.join(e_dir, "weights.joblib"))
    with open(os.path.join(e_dir, "meta.json"), "w") as f:
        json.dump({
            "adapter": "lightgbm",
            "output": "probability" if e_approach == "classification" else "regression",
            "approach": e_approach,
            "features": len(fnames_e),
            "metrics": {"DA": float(res_e["da"])}
        }, f, indent=2)
    print(f"  Saved Model E: {e_dir}/weights.joblib")

    # Model D（flat features only，对比用）
    d_dir = os.path.join(CFG["RESULTS_DIR"],
                         "models/layer1/volatility/lightgbm_v3_flat")
    os.makedirs(d_dir, exist_ok=True)
    best_d_mdl = mdl_d_reg if d_approach == "regression" else mdl_d_cls
    joblib.dump(best_d_mdl, os.path.join(d_dir, "weights.joblib"))
    with open(os.path.join(d_dir, "meta.json"), "w") as f:
        json.dump({
            "adapter": "lightgbm",
            "output": "probability" if d_approach == "classification" else "regression",
            "approach": d_approach,
            "features": len(fnames_d),
            "metrics": {"DA": float(res_d["da"])}
        }, f, indent=2)
    print(f"  Saved Model D: {d_dir}/weights.joblib")

    print("\n  Done!")


if __name__ == "__main__":
    main()

  Vol Prediction v3 — Model E: LightGBM + LSTM Embeddings
  Device: cuda

  STEP 1: Loading data...
  Tickers loaded: ['AAPL', 'MSFT', 'GOOGL', 'GOOG', 'NVDA', 'TSLA', 'SPY', 'QQQ']

  Building Model B sequences...
  Model B: 7,578 samples  short=(10,46)  long=(20,8)
  Building Model D features...
  Model D: 7,693 samples  119 features

  Aligning datasets...
  Model B: 7,578  Model D: 7,693  Common: 7,578
  Aligned: 7,578  train=5304 val=1137 test=1137

  STEP 2: Loading Model B checkpoint...
  ✓ Loaded: results/vol_multiscale_lstm_regression.pt
  Extracting LSTM embeddings...
  Short emb: (7578, 32)  Long emb: (7578, 32)
  Model B Test: DA=80.7%  R²=0.747  z=+20.67

  STEP 3: Building Model E features...
  Model E features: 119 (D) + 32 (short_emb) + 32 (long_emb) = 183

  STEP 4: Training LightGBM models...

  Model D (aligned, classification)...
  Model D (aligned, regression)...
  Model D cls DA: 81.2%  reg DA: 83.6%
  Model D best: regression  DA=83.6%

  Model E (classification)

In [9]:
"""
meta_label_v2.py
================
Improved meta-label with MFE/MAE analysis and adaptive parameters.

Previous v1 issues:
  - 47 trades in 2 years (bottom), 0-2 trades (top)
  - TP/SL (0.5%/0.3%) didn't match 15min dynamics
  - CNN threshold=0.7 too strict
  - Same TP/SL for long and short

5-step pipeline:
  1. MFE/MAE analysis → understand price dynamics after CNN signals
  2. Auto-calculate optimal TP/SL from data (not guessed)
  3. Build features + labels with optimal parameters
  4. Train LightGBM (separate bottom/top + vol-gated top)
  5. Evaluate with baselines, per-ticker, feature importance
  6. 3-way comparison: fixed 0.5/0.3 vs MFE-optimized vs ATR-adaptive

Key metric: at threshold where N_trades > 100, win rate > 55%.

Usage (Colab):
    !pip install lightgbm --quiet
    !python meta_label_v2.py
"""

import os, warnings, time, json
import numpy as np
import pandas as pd
from scipy import stats as sp_stats

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print("  [WARN] lightgbm not installed: pip install lightgbm")
    HAS_LGB = False

# ─────────────────────────── CONFIG ───────────────────────────

CONFIG = {
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "15min",
    "CNN_BOTTOM_PATH": "results/cnn_bottom_predictions.csv",
    "CNN_TOP_PATH": "results/cnn_top_predictions.csv",
    "CNN_THRESHOLD": 0.5,
    "MAX_BARS_OPTIONS": [12, 24, 48],
    "TRAIN_RATIO": 0.7,
    # ATR-adaptive barriers (for comparison)
    "ATR_TP_MULT": 1.5,
    "ATR_SL_MULT": 1.0,
    "ATR_PERIOD": 14,
    # LightGBM
    "LGB_PARAMS": {
        "objective": "binary",
        "metric": "binary_logloss",
        "n_estimators": 500,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_samples": 20,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "verbose": -1,
        "random_state": 42,
    },
    "LGB_THRESHOLDS": [0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    "OUTPUT_DIR": "results",
}


# ═══════════════════════════════════════════════════════════════
#  UTILITIES
# ═══════════════════════════════════════════════════════════════

def load_features(ticker, cfg):
    for freq in [cfg["FREQ"], "1hour"]:
        p = os.path.join(cfg["FEATURES_DIR"], f"{ticker}_{freq}_features.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            for col in ["timestamp", "ts_event", "datetime", "date"]:
                if col in df.columns:
                    df["timestamp"] = pd.to_datetime(df[col], utc=True)
                    break
            if "timestamp" not in df.columns:
                first = df.columns[0]
                if first not in df.select_dtypes(include=[np.number]).columns:
                    df["timestamp"] = pd.to_datetime(df[first], utc=True)
            df = df.sort_values("timestamp").reset_index(drop=True)
            return df
    raise FileNotFoundError(f"No features for {ticker}")


def load_cnn_preds(cfg, task):
    path = cfg["CNN_BOTTOM_PATH"] if task == "bottom" else cfg["CNN_TOP_PATH"]
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def map_cnn_to_features(ticker_cnn, n_feat, test_frac=0.15):
    """Map CNN prediction rows → feature-file indices in test period."""
    n_cnn = len(ticker_cnn)
    test_start = int(n_feat * (1.0 - test_frac))
    test_idx = np.arange(test_start, n_feat)
    n_test = len(test_idx)
    if n_cnn == n_test:
        return test_idx
    elif n_cnn > 0:
        scale = n_test / n_cnn
        return test_idx[np.clip((np.arange(n_cnn) * scale).astype(int),
                                0, n_test - 1)]
    return np.array([], dtype=int)


def compute_atr(high, low, close, period=14):
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i] - low[i],
                     abs(high[i] - close[i - 1]),
                     abs(low[i] - close[i - 1]))
    atr = np.full(n, np.nan)
    if n >= period:
        atr[period - 1] = np.mean(tr[:period])
        for i in range(period, n):
            atr[i] = (atr[i - 1] * (period - 1) + tr[i]) / period
    return atr


def get_candidates(cfg, task):
    """Return list of (ticker, feature_df, cand_feat_idx, cand_probs)."""
    cnn_df = load_cnn_preds(cfg, task)
    th = cfg["CNN_THRESHOLD"]
    out = []
    for ticker in cfg["TICKERS"]:
        tc = cnn_df[cnn_df["ticker"] == ticker].reset_index(drop=True)
        if len(tc) == 0:
            continue
        try:
            df = load_features(ticker, cfg)
        except FileNotFoundError:
            continue
        feat_all = map_cnn_to_features(tc, len(df))
        mask = tc["cnn_prob"].values >= th
        pos = np.where(mask)[0]
        if len(pos) == 0:
            continue
        out.append((ticker, df, feat_all[pos], tc["cnn_prob"].values[mask]))
    return out


# ═══════════════════════════════════════════════════════════════
#  TRIPLE-BARRIER LABELLING
# ═══════════════════════════════════════════════════════════════

def triple_barrier(close, high, low, idx, direction,
                   tp_pct, sl_pct, max_bars):
    """Returns (label, exit_type, pnl_pct)."""
    entry = close[idx]
    n = len(close)
    for i in range(1, min(max_bars + 1, n - idx)):
        bi = idx + i
        if direction == "long":
            tp_hit = (high[bi] - entry) / entry >= tp_pct
            sl_hit = (entry - low[bi]) / entry >= sl_pct
        else:
            tp_hit = (entry - low[bi]) / entry >= tp_pct
            sl_hit = (high[bi] - entry) / entry >= sl_pct
        if tp_hit and sl_hit:
            return 0, "SL", -sl_pct
        elif tp_hit:
            return 1, "TP", tp_pct
        elif sl_hit:
            return 0, "SL", -sl_pct
    # timeout
    if idx + max_bars < n:
        ep = close[min(idx + max_bars, n - 1)]
        pnl = ((ep - entry) / entry if direction == "long"
               else (entry - ep) / entry)
    else:
        pnl = 0.0
    return 0, "timeout", pnl


# ═══════════════════════════════════════════════════════════════
#  STEP 1: MFE / MAE ANALYSIS
# ═══════════════════════════════════════════════════════════════

def compute_mfe_mae_single(close, high, low, idx, direction, max_bars):
    n = len(close)
    end = min(idx + max_bars + 1, n)
    if idx + 1 >= end:
        return 0.0, 0.0
    entry = close[idx]
    seg_h = high[idx + 1:end]
    seg_l = low[idx + 1:end]
    if len(seg_h) == 0:
        return 0.0, 0.0
    if direction == "long":
        mfe = max((np.max(seg_h) - entry) / entry, 0.0)
        mae = max((entry - np.min(seg_l)) / entry, 0.0)
    else:
        mfe = max((entry - np.min(seg_l)) / entry, 0.0)
        mae = max((np.max(seg_h) - entry) / entry, 0.0)
    return mfe, mae


def run_mfe_mae_analysis(cfg):
    """Step 1: Analyze MFE/MAE across all candidates."""
    print(f"\n{'='*78}")
    print(f"  STEP 1: MFE/MAE ANALYSIS  (CNN threshold={cfg['CNN_THRESHOLD']})")
    print(f"{'='*78}")

    results = {}
    for task in ["bottom", "top"]:
        direction = "long" if task == "bottom" else "short"
        print(f"\n  ── {task.upper()} ({direction}) ──")
        candidates = get_candidates(cfg, task)
        results[task] = {}

        for mb in cfg["MAX_BARS_OPTIONS"]:
            all_mfe, all_mae = [], []
            ticker_data = {}

            for ticker, df, cand_idx, cand_probs in candidates:
                close = df["close"].values
                high_ = df["high"].values if "high" in df.columns else close
                low_ = df["low"].values if "low" in df.columns else close
                t_mfe, t_mae = [], []
                for idx in cand_idx:
                    mfe, mae = compute_mfe_mae_single(
                        close, high_, low_, idx, direction, mb)
                    t_mfe.append(mfe)
                    t_mae.append(mae)
                all_mfe.extend(t_mfe)
                all_mae.extend(t_mae)
                if t_mfe:
                    ticker_data[ticker] = {
                        "mfe_med": float(np.median(t_mfe)),
                        "mae_med": float(np.median(t_mae)),
                        "n": len(t_mfe),
                    }

            all_mfe = np.array(all_mfe)
            all_mae = np.array(all_mae)
            results[task][mb] = {
                "mfe": all_mfe, "mae": all_mae,
                "ticker_data": ticker_data,
            }

            ptiles = [10, 25, 40, 50, 60, 75, 90]
            print(f"\n  {task.upper()} MFE (MAX_BARS={mb}, N={len(all_mfe):,}):")
            mfe_p = [np.percentile(all_mfe, p) * 100 for p in ptiles]
            print(f"    " + "  ".join(
                f"P{p}={v:.3f}%" for p, v in zip(ptiles, mfe_p)))

            print(f"  {task.upper()} MAE (MAX_BARS={mb}, N={len(all_mae):,}):")
            mae_p = [np.percentile(all_mae, p) * 100 for p in ptiles]
            print(f"    " + "  ".join(
                f"P{p}={v:.3f}%" for p, v in zip(ptiles, mae_p)))

            mfe_50 = np.percentile(all_mfe, 50)
            mae_50 = np.percentile(all_mae, 50)
            ratio = mfe_50 / mae_50 if mae_50 > 0 else 999
            print(f"  MFE/MAE ratio at median: {ratio:.2f}")

            print(f"\n  Per-ticker medians (MAX_BARS={mb}):")
            print(f"  {'Ticker':<8} {'MFE_med':>8} {'MAE_med':>8} "
                  f"{'Ratio':>6} {'N':>6}")
            print(f"  {'-'*42}")
            for t in sorted(ticker_data.keys()):
                d = ticker_data[t]
                r = d["mfe_med"] / d["mae_med"] if d["mae_med"] > 0 else 999
                print(f"  {t:<8} {d['mfe_med']*100:>7.3f}% "
                      f"{d['mae_med']*100:>7.3f}% {r:>6.2f} {d['n']:>6}")

    return results


# ═══════════════════════════════════════════════════════════════
#  STEP 2: AUTO-CALCULATE OPTIMAL TP/SL
# ═══════════════════════════════════════════════════════════════

def compute_base_wr(cfg, task, tp_pct, sl_pct, max_bars):
    """Base win rate across all CNN candidates (no model filtering)."""
    direction = "long" if task == "bottom" else "short"
    candidates = get_candidates(cfg, task)
    total, wins = 0, 0
    for ticker, df, cand_idx, _ in candidates:
        close = df["close"].values
        high_ = df["high"].values if "high" in df.columns else close
        low_ = df["low"].values if "low" in df.columns else close
        for idx in cand_idx:
            if idx < 1 or idx >= len(close) - 1:
                continue
            lab, _, _ = triple_barrier(close, high_, low_, idx,
                                       direction, tp_pct, sl_pct, max_bars)
            total += 1
            wins += lab
    return (wins / total if total > 0 else 0), total, wins


def run_optimal_params(cfg, mfe_mae):
    """Step 2: Grid search for optimal TP/SL per task."""
    print(f"\n{'='*78}")
    print(f"  STEP 2: AUTO-CALCULATE OPTIMAL TP/SL")
    print(f"{'='*78}")

    optimal = {}
    for task in ["bottom", "top"]:
        direction = "long" if task == "bottom" else "short"
        print(f"\n  ── {task.upper()} ({direction}) ──")

        best_edge = -999
        best_params = None
        rows = []

        for mb in cfg["MAX_BARS_OPTIONS"]:
            data = mfe_mae[task].get(mb)
            if data is None or len(data["mfe"]) == 0:
                continue
            mfe_arr = data["mfe"]
            mae_arr = data["mae"]

            tp_opts = [
                ("MFE_P25", float(np.percentile(mfe_arr, 25))),
                ("MFE_P40", float(np.percentile(mfe_arr, 40))),
                ("MFE_P50", float(np.percentile(mfe_arr, 50))),
            ]
            sl_opts = [
                ("MAE_P50", float(np.percentile(mae_arr, 50))),
                ("MAE_P60", float(np.percentile(mae_arr, 60))),
                ("MAE_P75", float(np.percentile(mae_arr, 75))),
            ]

            for tp_name, tp_pct in tp_opts:
                for sl_name, sl_pct in sl_opts:
                    if tp_pct <= 0 or sl_pct <= 0:
                        continue
                    be_wr = sl_pct / (tp_pct + sl_pct)
                    rr = tp_pct / sl_pct
                    wr, n_total, n_wins = compute_base_wr(
                        cfg, task, tp_pct, sl_pct, mb)
                    edge = wr - be_wr

                    rows.append({
                        "mb": mb, "tp_name": tp_name, "sl_name": sl_name,
                        "tp_pct": tp_pct, "sl_pct": sl_pct,
                        "rr": rr, "be_wr": be_wr,
                        "base_wr": wr, "edge": edge,
                        "n_total": n_total, "n_wins": n_wins,
                    })
                    if edge > best_edge and n_total >= 50:
                        best_edge = edge
                        best_params = rows[-1]

        rows.sort(key=lambda r: r["edge"], reverse=True)
        print(f"\n  TP/SL Grid ({task}):")
        print(f"  {'MB':>3} {'TP':>10} {'SL':>10} {'TP%':>7} {'SL%':>7} "
              f"{'RR':>5} {'BE_WR':>7} {'BaseWR':>7} {'Edge':>8} {'N':>6}")
        print(f"  {'-'*80}")
        for r in rows[:15]:
            star = " ★" if r is best_params else ""
            print(f"  {r['mb']:>3} {r['tp_name']:>10} {r['sl_name']:>10} "
                  f"{r['tp_pct']*100:>6.3f}% {r['sl_pct']*100:>6.3f}% "
                  f"{r['rr']:>5.2f} {r['be_wr']*100:>6.1f}% "
                  f"{r['base_wr']*100:>6.1f}% "
                  f"{r['edge']*100:>+7.1f}pp "
                  f"{r['n_total']:>6}{star}")

        if best_params:
            print(f"\n  ★ OPTIMAL {task.upper()}: "
                  f"TP={best_params['tp_pct']*100:.3f}% "
                  f"SL={best_params['sl_pct']*100:.3f}% "
                  f"MB={best_params['mb']} "
                  f"RR={best_params['rr']:.2f}:1")
            print(f"    Base WR={best_params['base_wr']*100:.1f}% "
                  f"BE={best_params['be_wr']*100:.1f}% "
                  f"Edge={best_params['edge']*100:+.1f}pp "
                  f"N={best_params['n_total']}")
            optimal[task] = best_params
        else:
            print(f"  [WARN] No positive edge, using MFE_P40/MAE_P60 from MB=24")
            d = mfe_mae[task].get(24, mfe_mae[task].get(
                cfg["MAX_BARS_OPTIONS"][0]))
            tp = float(np.percentile(d["mfe"], 40))
            sl = float(np.percentile(d["mae"], 60))
            optimal[task] = {
                "tp_pct": max(tp, 0.001), "sl_pct": max(sl, 0.001),
                "mb": 24, "rr": tp / sl if sl > 0 else 1.0,
                "be_wr": sl / (tp + sl) if (tp + sl) > 0 else 0.5,
                "base_wr": 0.5, "edge": 0.0, "n_total": 0,
            }

    return optimal


# ═══════════════════════════════════════════════════════════════
#  STEP 3: BUILD FEATURES AND LABELS
# ═══════════════════════════════════════════════════════════════

def build_candidate_features(df, cand_idx, cand_probs, tech_cols):
    """Build feature matrix for LightGBM at candidate indices."""
    close = df["close"].values
    high_ = df["high"].values if "high" in df.columns else close
    low_ = df["low"].values if "low" in df.columns else close
    vol = df["volume"].values if "volume" in df.columns else np.ones(len(df))

    rows = []
    for k, idx in enumerate(cand_idx):
        if idx < 20 or idx >= len(close):
            continue
        r = {}
        c = close[idx]

        # --- CNN features (2) ---
        r["cnn_prob"] = cand_probs[k]
        start = max(0, k - 200)
        recent = cand_probs[start:k + 1]
        r["cnn_prob_rank"] = np.sum(recent <= cand_probs[k]) / max(len(recent), 1)

        # --- Price action (8) ---
        for lb in [5, 10, 20]:
            r[f"ret_{lb}"] = (c - close[max(0, idx - lb)]) / (
                abs(close[max(0, idx - lb)]) + 1e-10)
        for lb in [5, 10]:
            seg = close[max(0, idx - lb):idx + 1]
            rets = (np.diff(seg) / (np.abs(seg[:-1]) + 1e-10)
                    if len(seg) > 1 else [0])
            r[f"rvol_{lb}"] = float(np.std(rets))
        h20 = np.max(high_[max(0, idx - 20):idx + 1])
        l20 = np.min(low_[max(0, idx - 20):idx + 1])
        r["dist_high_20"] = (c - h20) / (c + 1e-10)
        r["dist_low_20"] = (c - l20) / (c + 1e-10)
        r["bar_range"] = (high_[idx] - low_[idx]) / (c + 1e-10)

        # --- Technical indicators from CSV ---
        for col in tech_cols:
            v = df[col].iloc[idx]
            r[col] = float(v) if not pd.isna(v) else 0.0

        # --- Bollinger position ---
        bb_u = bb_l = None
        for col in tech_cols:
            cl = col.lower()
            if any(s in cl for s in ["bbu", "bb_upper", "upper_band"]):
                bb_u = df[col].iloc[idx]
            elif any(s in cl for s in ["bbl", "bb_lower", "lower_band"]):
                bb_l = df[col].iloc[idx]
        if bb_u is not None and bb_l is not None:
            rng = bb_u - bb_l
            r["bb_position"] = (c - bb_l) / rng if abs(rng) > 1e-10 else 0.5

        # --- Volume ratio ---
        vol_20 = np.mean(vol[max(0, idx - 20):idx]) if idx >= 1 else 1
        r["volume_ratio"] = vol[idx] / (vol_20 + 1e-10)

        # --- Time features (2) ---
        if "timestamp" in df.columns:
            ts = df["timestamp"].iloc[idx]
            r["hour"] = ts.hour
            r["day_of_week"] = ts.dayofweek

        r["_idx"] = idx
        rows.append(r)
    return pd.DataFrame(rows)


def build_dataset(cfg, task, tp_pct, sl_pct, max_bars, label_tag=""):
    """Build labeled dataset with given TP/SL/MAX_BARS."""
    direction = "long" if task == "bottom" else "short"
    print(f"\n  Building dataset: {task} {label_tag}"
          f"  TP={tp_pct*100:.3f}% SL={sl_pct*100:.3f}% MB={max_bars}")

    exclude = {"timestamp", "ts_event", "datetime", "date", "time",
               "unnamed: 0", "symbol", "ticker",
               "open", "high", "low", "close", "volume"}

    candidates = get_candidates(cfg, task)
    frames = []
    stats = {"total": 0, "tp": 0, "sl": 0, "timeout": 0}

    for ticker, df, cand_idx, cand_probs in candidates:
        close = df["close"].values
        high_ = df["high"].values if "high" in df.columns else close
        low_ = df["low"].values if "low" in df.columns else close

        tech_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                     if c.lower() not in exclude]
        feat_df = build_candidate_features(df, cand_idx, cand_probs, tech_cols)
        if len(feat_df) == 0:
            continue

        feat_set = set(feat_df["_idx"].values)
        labels, exits, pnls = [], [], []
        for idx in cand_idx:
            if idx not in feat_set or idx < 1 or idx >= len(close) - 1:
                labels.append(np.nan); exits.append("skip"); pnls.append(0)
                continue
            lab, ex, pnl = triple_barrier(close, high_, low_, idx,
                                          direction, tp_pct, sl_pct, max_bars)
            labels.append(lab); exits.append(ex); pnls.append(pnl)

        aligned_l, aligned_e, aligned_p = [], [], []
        for k_idx in range(len(cand_idx)):
            if cand_idx[k_idx] in feat_set:
                aligned_l.append(labels[k_idx])
                aligned_e.append(exits[k_idx])
                aligned_p.append(pnls[k_idx])

        ml = min(len(aligned_l), len(feat_df))
        feat_df = feat_df.iloc[:ml].copy()
        feat_df["label"] = aligned_l[:ml]
        feat_df["exit_type"] = aligned_e[:ml]
        feat_df["pnl_pct"] = aligned_p[:ml]
        feat_df["ticker"] = ticker
        feat_df = feat_df.dropna(subset=["label"]).reset_index(drop=True)
        feat_df["label"] = feat_df["label"].astype(int)

        nt = len(feat_df)
        n_tp = (feat_df["exit_type"] == "TP").sum()
        n_sl = (feat_df["exit_type"] == "SL").sum()
        n_to = (feat_df["exit_type"] == "timeout").sum()
        stats["total"] += nt; stats["tp"] += n_tp
        stats["sl"] += n_sl; stats["timeout"] += n_to
        wr = n_tp / nt if nt > 0 else 0
        print(f"    {ticker}: {nt:>5}  WR={wr*100:>5.1f}%  "
              f"TP={n_tp} SL={n_sl} TO={n_to}")
        frames.append(feat_df)

    if not frames:
        return pd.DataFrame(), stats
    combined = pd.concat(frames, ignore_index=True)
    t = stats["total"]
    wr = stats["tp"] / t if t > 0 else 0
    print(f"\n  TOTAL: {t:,}  base WR={wr*100:.1f}%  "
          f"(TP={stats['tp']} SL={stats['sl']} TO={stats['timeout']})")
    return combined, stats


def build_atr_dataset(cfg, task, max_bars):
    """Build ATR-adaptive barrier dataset for comparison."""
    direction = "long" if task == "bottom" else "short"
    exclude = {"timestamp", "ts_event", "datetime", "date", "time",
               "unnamed: 0", "symbol", "ticker",
               "open", "high", "low", "close", "volume"}
    candidates = get_candidates(cfg, task)
    frames = []
    stats = {"total": 0, "tp": 0, "sl": 0, "timeout": 0}

    for ticker, df, cand_idx, cand_probs in candidates:
        close = df["close"].values
        high_ = df["high"].values if "high" in df.columns else close
        low_ = df["low"].values if "low" in df.columns else close
        atr = compute_atr(high_, low_, close, cfg["ATR_PERIOD"])

        tech_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                     if c.lower() not in exclude]
        feat_df = build_candidate_features(df, cand_idx, cand_probs, tech_cols)
        if len(feat_df) == 0:
            continue
        feat_set = set(feat_df["_idx"].values)
        labels, exits, pnls = [], [], []
        for idx in cand_idx:
            if idx not in feat_set or idx < 1 or idx >= len(close) - 1:
                labels.append(np.nan); exits.append("skip"); pnls.append(0)
                continue
            a = atr[idx]
            if np.isnan(a) or a <= 0:
                a = close[idx] * 0.003
            tp = (a * cfg["ATR_TP_MULT"]) / close[idx]
            sl = (a * cfg["ATR_SL_MULT"]) / close[idx]
            lab, ex, pnl = triple_barrier(close, high_, low_, idx,
                                          direction, tp, sl, max_bars)
            labels.append(lab); exits.append(ex); pnls.append(pnl)

        aligned_l, aligned_e, aligned_p = [], [], []
        for k in range(len(cand_idx)):
            if cand_idx[k] in feat_set:
                aligned_l.append(labels[k])
                aligned_e.append(exits[k])
                aligned_p.append(pnls[k])
        ml = min(len(aligned_l), len(feat_df))
        feat_df = feat_df.iloc[:ml].copy()
        feat_df["label"] = aligned_l[:ml]
        feat_df["exit_type"] = aligned_e[:ml]
        feat_df["pnl_pct"] = aligned_p[:ml]
        feat_df["ticker"] = ticker
        feat_df = feat_df.dropna(subset=["label"]).reset_index(drop=True)
        feat_df["label"] = feat_df["label"].astype(int)
        nt = len(feat_df)
        stats["total"] += nt
        stats["tp"] += (feat_df["exit_type"] == "TP").sum()
        stats["sl"] += (feat_df["exit_type"] == "SL").sum()
        stats["timeout"] += (feat_df["exit_type"] == "timeout").sum()
        frames.append(feat_df)

    if not frames:
        return pd.DataFrame(), stats
    return pd.concat(frames, ignore_index=True), stats


# ═══════════════════════════════════════════════════════════════
#  STEP 4: TRAIN LightGBM
# ═══════════════════════════════════════════════════════════════

def get_lgb_cols(df):
    skip = {"label", "exit_type", "pnl_pct", "ticker", "task", "_idx"}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in skip]


def train_lgb(train_df, feat_cols, cfg, label):
    X = np.nan_to_num(train_df[feat_cols].values, nan=0.0,
                      posinf=0.0, neginf=0.0)
    y = train_df["label"].values
    vs = int(len(y) * 0.8)
    X_t, y_t = X[:vs], y[:vs]
    X_v, y_v = X[vs:], y[vs:]
    print(f"    LGB [{label}]: train={len(y_t)} val={len(y_v)} "
          f"pos_tr={y_t.mean():.3f} pos_va={y_v.mean():.3f}")
    params = dict(cfg["LGB_PARAMS"])
    n_est = params.pop("n_estimators", 500)
    model = lgb.LGBMClassifier(n_estimators=n_est, **params)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)],
              callbacks=[lgb.early_stopping(30, verbose=False),
                         lgb.log_evaluation(0)])
    print(f"    Best iter: {getattr(model, 'best_iteration_', n_est)}")
    return model


# ═══════════════════════════════════════════════════════════════
#  STEP 5: EVALUATE
# ═══════════════════════════════════════════════════════════════

def eval_thresholds(probs, y, pnls, thresholds, base_rate):
    out = {}
    for th in thresholds:
        m = probs >= th
        n = int(m.sum())
        if n == 0:
            out[th] = dict(win_rate=0, n_trades=0, n_wins=0,
                           profit_factor=0, total_pnl=0, avg_pnl=0,
                           z_score=0, p_value=1)
            continue
        ys, ps = y[m], pnls[m]
        nw = int(ys.sum())
        wr = nw / n
        gp = ps[ps > 0].sum()
        gl = abs(ps[ps < 0].sum())
        pf = gp / gl if gl > 0 else (999.0 if gp > 0 else 0.0)
        se = np.sqrt(base_rate * (1 - base_rate) / n) if 0 < base_rate < 1 else 1
        z = (wr - base_rate) / se if se > 0 else 0
        pv = 2 * (1 - sp_stats.norm.cdf(abs(z)))
        out[th] = dict(win_rate=wr, n_trades=n, n_wins=nw,
                       profit_factor=pf, total_pnl=float(ps.sum()),
                       avg_pnl=float(ps.sum()) / n,
                       z_score=z, p_value=pv)
    return out


def eval_baseline(test_df, mask, base_rate, name):
    y = test_df["label"].values
    pnls = test_df["pnl_pct"].values
    n = int(mask.sum())
    if n == 0:
        return dict(name=name, win_rate=0, n_trades=0, n_wins=0,
                    profit_factor=0, total_pnl=0, z_score=0, p_value=1)
    ys, ps = y[mask], pnls[mask]
    nw = int(ys.sum()); wr = nw / n
    gp = ps[ps > 0].sum(); gl = abs(ps[ps < 0].sum())
    pf = gp / gl if gl > 0 else (999.0 if gp > 0 else 0.0)
    se = np.sqrt(base_rate * (1 - base_rate) / n) if 0 < base_rate < 1 else 1
    z = (wr - base_rate) / se if se > 0 else 0
    pv = 2 * (1 - sp_stats.norm.cdf(abs(z)))
    return dict(name=name, win_rate=wr, n_trades=n, n_wins=nw,
                profit_factor=pf, total_pnl=float(ps.sum()),
                z_score=z, p_value=pv)


# ── Printing helpers ──

def print_threshold_table(results, task, base_rate, be_wr, barrier_tag):
    print(f"\n  ┌{'─'*80}┐")
    print(f"  │  {task.upper()} — LightGBM [{barrier_tag}]  "
          f"base={base_rate*100:.1f}%  BE={be_wr*100:.1f}%"
          f"{' '*(80-len(task)-len(barrier_tag)-42)}│")
    print(f"  ├{'─'*80}┤")
    print(f"  │ {'Th':>5} │ {'WinRate':>8} │ {'N_trade':>8} │ "
          f"{'N_win':>6} │ {'PF':>7} │ {'TotPnL':>9} │ "
          f"{'AvgPnL':>9} │ {'z':>7} │")
    print(f"  ├{'─'*80}┤")
    for th in sorted(results.keys()):
        r = results[th]
        sig = ("***" if r["p_value"] < 0.001 else "**" if r["p_value"] < 0.01
               else "*" if r["p_value"] < 0.05 else "")
        ws = f"{r['win_rate']*100:.1f}%" if r["n_trades"] > 0 else "—"
        print(f"  │ {th:>5.2f} │ {ws:>8} │ {r['n_trades']:>8} │ "
              f"{r['n_wins']:>6} │ {r['profit_factor']:>7.2f} │ "
              f"{r['total_pnl']*100:>8.3f}% │ "
              f"{r['avg_pnl']*100:>8.4f}% │ "
              f"{r['z_score']:>+6.2f}{sig:>1} │")
    print(f"  └{'─'*80}┘")


def print_baselines(baselines, task):
    print(f"\n  {task.upper()} — BASELINES:")
    print(f"  {'Name':<22} {'WR':>8} {'N':>7} {'PF':>7} "
          f"{'TotPnL':>9} {'z':>7}")
    print(f"  {'-'*62}")
    for b in baselines:
        sig = "*" if b["p_value"] < 0.05 else ""
        ws = f"{b['win_rate']*100:.1f}%" if b["n_trades"] > 0 else "—"
        print(f"  {b['name']:<22} {ws:>8} {b['n_trades']:>7} "
              f"{b['profit_factor']:>7.2f} "
              f"{b['total_pnl']*100:>8.3f}% "
              f"{b['z_score']:>+6.2f}{sig}")


def print_importance(model, feat_cols, top_n=20):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1]
    print(f"\n  TOP {top_n} FEATURES:")
    print(f"  {'#':>3} {'Feature':<35} {'Imp':>8}")
    print(f"  {'-'*50}")
    for i in range(min(top_n, len(feat_cols))):
        print(f"  {i+1:>3} {feat_cols[idx[i]]:<35} {imp[idx[i]]:>8}")


def print_per_ticker(test_df, probs, threshold, task):
    y = test_df["label"].values
    pnls = test_df["pnl_pct"].values
    print(f"\n  {task.upper()} — PER-TICKER @ threshold={threshold}:")
    print(f"  {'Ticker':<8} {'FiltWR':>8} {'N_filt':>7} {'BaseWR':>8} "
          f"{'N_all':>7} {'PF':>7} {'TotPnL':>9}")
    print(f"  {'-'*62}")
    for ticker in sorted(test_df["ticker"].unique()):
        mt = (test_df["ticker"] == ticker).values
        pt, yt, pnlt = probs[mt], y[mt], pnls[mt]
        bwr = yt.mean()
        fm = pt >= threshold
        nf = int(fm.sum())
        if nf == 0:
            print(f"  {ticker:<8} {'—':>8} {0:>7} "
                  f"{bwr*100:>7.1f}% {int(mt.sum()):>7}")
            continue
        fwr = yt[fm].mean()
        pf_ = pnlt[fm]
        gp = pf_[pf_ > 0].sum(); gl = abs(pf_[pf_ < 0].sum())
        pf = gp / gl if gl > 0 else 999.0
        print(f"  {ticker:<8} {fwr*100:>7.1f}% {nf:>7} "
              f"{bwr*100:>7.1f}% {int(mt.sum()):>7} "
              f"{pf:>7.2f} {pf_.sum()*100:>8.3f}%")


# ═══════════════════════════════════════════════════════════════
#  PIPELINE: run one barrier mode for one task
# ═══════════════════════════════════════════════════════════════

def run_barrier_mode(cfg, task, dataset, stats, tp_pct, sl_pct,
                     max_bars, barrier_tag):
    """Train LightGBM and evaluate for one barrier mode."""
    if len(dataset) < 50:
        print(f"  [SKIP] {barrier_tag}: only {len(dataset)} samples")
        return {}

    dataset = dataset.sort_values("_idx").reset_index(drop=True)
    n = len(dataset)
    sp = int(n * cfg["TRAIN_RATIO"])
    train_df = dataset.iloc[:sp].reset_index(drop=True)
    test_df = dataset.iloc[sp:].reset_index(drop=True)

    base_rate = test_df["label"].mean()
    be_wr = sl_pct / (tp_pct + sl_pct) if (tp_pct + sl_pct) > 0 else 0.5
    edge = base_rate - be_wr

    print(f"\n  [{barrier_tag}] Train={len(train_df):,} Test={len(test_df):,}")
    print(f"  Base WR={base_rate*100:.1f}%  BE={be_wr*100:.1f}%  "
          f"Edge={edge*100:+.1f}pp")

    feat_cols = get_lgb_cols(dataset)
    print(f"  Features: {len(feat_cols)}")

    if not HAS_LGB or len(train_df) < 50:
        return {"base_rate": float(base_rate), "be_wr": float(be_wr),
                "n_test": len(test_df)}

    model = train_lgb(train_df, feat_cols, cfg, f"{task}-{barrier_tag}")

    X_test = np.nan_to_num(test_df[feat_cols].values, nan=0.0,
                           posinf=0.0, neginf=0.0)
    probs = model.predict_proba(X_test)[:, 1]
    y_test = test_df["label"].values
    pnls_test = test_df["pnl_pct"].values

    lgb_res = eval_thresholds(probs, y_test, pnls_test,
                              cfg["LGB_THRESHOLDS"], base_rate)
    print_threshold_table(lgb_res, task, base_rate, be_wr, barrier_tag)

    # Baselines
    nt = len(test_df)
    baselines = [
        eval_baseline(test_df, np.ones(nt, dtype=bool), base_rate, "CNN alone"),
        eval_baseline(test_df, np.random.RandomState(42).random(nt) > 0.5,
                      base_rate, "Random 50%"),
    ]
    rsi_cols = [c for c in feat_cols if c.lower() in ("rsi_14", "rsi")]
    if rsi_cols:
        rv = test_df[rsi_cols[0]].values
        if task == "bottom":
            baselines.append(eval_baseline(test_df, rv < 30, base_rate, "RSI<30"))
        else:
            baselines.append(eval_baseline(test_df, rv > 70, base_rate, "RSI>70"))
    vol_cols = [c for c in feat_cols
                if "rvol" in c.lower() or (
                    "vol" in c.lower() and "volume" not in c.lower())]
    if vol_cols:
        vv = test_df[vol_cols[0]].values
        baselines.append(eval_baseline(test_df, vv > np.median(vv),
                                       base_rate, "High Vol"))
    print_baselines(baselines, task)

    print_importance(model, feat_cols)

    # Best threshold with N>100 and N>20
    best_100 = max(
        cfg["LGB_THRESHOLDS"],
        key=lambda th: (lgb_res[th]["win_rate"]
                        if lgb_res[th]["n_trades"] >= 100 else 0))
    best_20 = max(
        cfg["LGB_THRESHOLDS"],
        key=lambda th: (lgb_res[th]["win_rate"]
                        if lgb_res[th]["n_trades"] >= 20 else 0))

    print_per_ticker(test_df, probs, best_100 if lgb_res[best_100]["n_trades"] >= 100
                     else best_20, task)

    return {
        "base_rate": float(base_rate),
        "be_wr": float(be_wr),
        "edge_no_model": float(edge),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "tp_pct": float(tp_pct),
        "sl_pct": float(sl_pct),
        "max_bars": int(max_bars),
        "thresholds": {
            str(k): {kk: (float(vv) if isinstance(vv, (float, np.floating))
                          else int(vv) if isinstance(vv, (int, np.integer))
                          else vv)
                     for kk, vv in v.items()}
            for k, v in lgb_res.items()
        },
        "baselines": {b["name"]: {k: (float(v) if isinstance(v, (float, np.floating))
                                       else int(v) if isinstance(v, (int, np.integer))
                                       else v)
                                   for k, v in b.items() if k != "name"}
                      for b in baselines},
        "feature_importance": {
            feat_cols[i]: float(model.feature_importances_[i])
            for i in np.argsort(model.feature_importances_)[::-1][:20]
        },
        "best_th_100": float(best_100),
        "best_th_20": float(best_20),
    }


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    cfg = CONFIG
    print("=" * 78)
    print("  Meta-Label v2: MFE/MAE → Optimal TP/SL → LightGBM Filter")
    print(f"  CNN threshold: {cfg['CNN_THRESHOLD']}")
    print(f"  Tickers: {cfg['TICKERS']}")
    print("=" * 78)

    os.makedirs(cfg["OUTPUT_DIR"], exist_ok=True)

    # ──── STEP 1: MFE/MAE ────
    mfe_mae = run_mfe_mae_analysis(cfg)

    # ──── STEP 2: Optimal TP/SL ────
    optimal = run_optimal_params(cfg, mfe_mae)

    # ──── STEPS 3-5 per task: 3-way comparison ────
    all_results = {}
    for task in ["bottom", "top"]:
        direction = "LONG" if task == "bottom" else "SHORT"
        print(f"\n{'#'*78}")
        print(f"  TASK: {task.upper()} → {direction}")
        print(f"{'#'*78}")

        opt = optimal[task]
        opt_tp = opt["tp_pct"]
        opt_sl = opt["sl_pct"]
        opt_mb = opt.get("mb", 24)

        task_res = {}

        # ── Mode 1: MFE-optimized ──
        print(f"\n{'='*60}")
        print(f"  BARRIER MODE 1: MFE-OPTIMIZED")
        print(f"  TP={opt_tp*100:.3f}% SL={opt_sl*100:.3f}% MB={opt_mb} "
              f"RR={opt_tp/opt_sl:.2f}:1")
        print(f"{'='*60}")
        ds_opt, st_opt = build_dataset(cfg, task, opt_tp, opt_sl, opt_mb,
                                       label_tag="MFE-opt")
        task_res["mfe_optimized"] = run_barrier_mode(
            cfg, task, ds_opt, st_opt, opt_tp, opt_sl, opt_mb, "MFE-opt")

        # ── Mode 2: Fixed 0.5%/0.3% ──
        fix_tp, fix_sl, fix_mb = 0.005, 0.003, 48
        print(f"\n{'='*60}")
        print(f"  BARRIER MODE 2: FIXED")
        print(f"  TP={fix_tp*100:.1f}% SL={fix_sl*100:.1f}% MB={fix_mb} "
              f"RR={fix_tp/fix_sl:.2f}:1")
        print(f"{'='*60}")
        ds_fix, st_fix = build_dataset(cfg, task, fix_tp, fix_sl, fix_mb,
                                       label_tag="Fixed")
        task_res["fixed"] = run_barrier_mode(
            cfg, task, ds_fix, st_fix, fix_tp, fix_sl, fix_mb, "Fixed")

        # ── Mode 3: ATR-adaptive ──
        # Use median ATR from training data as proxy for be_wr calculation
        atr_tp_mult = cfg["ATR_TP_MULT"]
        atr_sl_mult = cfg["ATR_SL_MULT"]
        print(f"\n{'='*60}")
        print(f"  BARRIER MODE 3: ATR-ADAPTIVE")
        print(f"  TP={atr_tp_mult}×ATR  SL={atr_sl_mult}×ATR  MB={opt_mb} "
              f"RR={atr_tp_mult/atr_sl_mult:.2f}:1")
        print(f"{'='*60}")
        ds_atr, st_atr = build_atr_dataset(cfg, task, opt_mb)
        # For ATR, be_wr uses the multiplier ratio
        atr_tp_proxy = atr_tp_mult * 0.003  # rough proxy
        atr_sl_proxy = atr_sl_mult * 0.003
        task_res["atr"] = run_barrier_mode(
            cfg, task, ds_atr, st_atr,
            atr_tp_proxy, atr_sl_proxy, opt_mb, "ATR")

        # ── Vol-gated top model (top task only) ──
        if task == "top" and len(ds_opt) > 50:
            print(f"\n{'='*60}")
            print(f"  VOL-GATED TOP MODEL (MFE-opt, high vol only)")
            print(f"{'='*60}")
            vol_cols = [c for c in ds_opt.columns
                        if "rvol" in c.lower() or (
                            "vol" in c.lower() and "volume" not in c.lower()
                            and c.lower() not in ("volume_ratio",))]
            if vol_cols:
                vc = vol_cols[0]
                med = ds_opt[vc].median()
                ds_vg = ds_opt[ds_opt[vc] > med].copy()
                print(f"  Vol filter on '{vc}': {len(ds_vg)}/{len(ds_opt)} "
                      f"candidates retained")
                if len(ds_vg) > 50:
                    st_vg = {"total": len(ds_vg),
                             "tp": (ds_vg["exit_type"] == "TP").sum(),
                             "sl": (ds_vg["exit_type"] == "SL").sum(),
                             "timeout": (ds_vg["exit_type"] == "timeout").sum()}
                    task_res["vol_gated"] = run_barrier_mode(
                        cfg, task, ds_vg, st_vg, opt_tp, opt_sl, opt_mb,
                        "Vol-gated")

        all_results[task] = task_res

    # ═══════════════════════════════════════════════════════════
    #  FINAL SUMMARY: 3-WAY COMPARISON
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ FINAL SUMMARY: 3-WAY BARRIER COMPARISON")
    print(f"{'='*78}")

    for task in ["bottom", "top"]:
        direction = "LONG" if task == "bottom" else "SHORT"
        tr = all_results.get(task, {})
        if not tr:
            continue

        print(f"\n  {task.upper()} ({direction}):")
        print(f"  {'Mode':<16} │ {'TP/SL':>14} │ {'BaseWR':>7} │ {'BE':>6} │ "
              f"{'Edge':>7} │ {'BestWR':>7} │ {'@N≥100':>7} │ {'N_test':>7} │")
        print(f"  {'─'*16}─┼{'─'*16}┼{'─'*9}┼{'─'*8}┼"
              f"{'─'*9}┼{'─'*9}┼{'─'*9}┼{'─'*9}┤")

        for mode_name, mode_label in [("mfe_optimized", "MFE-opt"),
                                       ("fixed", "Fixed 0.5/0.3"),
                                       ("atr", "ATR-adaptive"),
                                       ("vol_gated", "Vol-gated")]:
            r = tr.get(mode_name, {})
            if not r or "base_rate" not in r:
                continue
            base = r["base_rate"]
            be = r.get("be_wr", 0)
            edge = base - be
            n_test = r.get("n_test", 0)
            tp_s = f"{r.get('tp_pct',0)*100:.2f}/{r.get('sl_pct',0)*100:.2f}%"

            # Best WR at any threshold with N>0
            ths = r.get("thresholds", {})
            best_wr = 0
            best_wr_100 = 0
            for th_k, th_v in ths.items():
                if th_v.get("n_trades", 0) > 0:
                    best_wr = max(best_wr, th_v["win_rate"])
                if th_v.get("n_trades", 0) >= 100:
                    best_wr_100 = max(best_wr_100, th_v["win_rate"])

            bw_s = f"{best_wr*100:.1f}%" if best_wr > 0 else "—"
            bw100_s = f"{best_wr_100*100:.1f}%" if best_wr_100 > 0 else "—"

            print(f"  {mode_label:<16} │ {tp_s:>14} │ "
                  f"{base*100:>6.1f}% │ {be*100:>5.1f}% │ "
                  f"{edge*100:>+6.1f}pp │ {bw_s:>7} │ "
                  f"{bw100_s:>7} │ {n_test:>7} │")

    # KEY METRIC
    print(f"\n  {'─'*60}")
    print(f"  KEY METRIC: Win Rate where N_trades ≥ 100")
    print(f"  Target: >55% win rate with >100 trades\n")

    for task in ["bottom", "top"]:
        tr = all_results.get(task, {})
        direction = "LONG" if task == "bottom" else "SHORT"
        for mode_name in ["mfe_optimized", "fixed", "atr", "vol_gated"]:
            r = tr.get(mode_name, {})
            if not r:
                continue
            ths = r.get("thresholds", {})
            for th_k in sorted(ths.keys()):
                th_v = ths[th_k]
                if th_v.get("n_trades", 0) >= 100:
                    wr = th_v["win_rate"]
                    nt = th_v["n_trades"]
                    mark = "✓" if wr > 0.55 else "~" if wr > 0.50 else "✗"
                    print(f"    {mark} {task.upper()} {mode_name:<16} "
                          f"th={th_k}: WR={wr*100:.1f}% N={nt}")

    # CONCLUSIONS
    print(f"\n  {'─'*60}")
    print(f"  CONCLUSIONS:")

    for task in ["bottom", "top"]:
        tr = all_results.get(task, {})
        r_opt = tr.get("mfe_optimized", {})
        if not r_opt:
            continue
        base = r_opt.get("base_rate", 0)
        be = r_opt.get("be_wr", 0)
        edge = base - be
        direction = "LONG" if task == "bottom" else "SHORT"

        if edge > 0.05:
            print(f"    ✓ {task.upper()} ({direction}): strong edge "
                  f"{edge*100:+.1f}pp over breakeven")
        elif edge > 0:
            print(f"    ~ {task.upper()} ({direction}): marginal edge "
                  f"{edge*100:+.1f}pp — LGB filter can help")
        else:
            print(f"    ✗ {task.upper()} ({direction}): negative edge "
                  f"{edge*100:+.1f}pp — signal not profitable without filter")

        # Check if LGB improves at N>=100
        ths = r_opt.get("thresholds", {})
        improved = False
        for th_k, th_v in ths.items():
            if th_v.get("n_trades", 0) >= 100:
                if th_v["win_rate"] > base + 0.02:
                    improved = True
                    print(f"      LGB filter improves: "
                          f"th={th_k} WR={th_v['win_rate']*100:.1f}% "
                          f"(+{(th_v['win_rate']-base)*100:.1f}pp) "
                          f"N={th_v['n_trades']}")
                    break
        if not improved:
            print(f"      LGB filter: no significant improvement at N≥100")

    # ── Save ──
    save = {}
    for task in all_results:
        save[task] = {}
        for mode in all_results[task]:
            r = all_results[task][mode]
            if not r:
                continue
            save[task][mode] = {
                k: v for k, v in r.items()
                if k != "feature_importance" or True
            }
    json_path = os.path.join(cfg["OUTPUT_DIR"], "meta_label_v2_results.json")
    with open(json_path, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"\n  Saved: {json_path}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

  Meta-Label v2: MFE/MAE → Optimal TP/SL → LightGBM Filter
  CNN threshold: 0.5
  Tickers: ['AAPL', 'MSFT', 'GOOGL', 'GOOG', 'NVDA', 'TSLA', 'SPY', 'QQQ']

  STEP 1: MFE/MAE ANALYSIS  (CNN threshold=0.5)

  ── BOTTOM (long) ──

  BOTTOM MFE (MAX_BARS=12, N=10,436):
    P10=0.042%  P25=0.151%  P40=0.314%  P50=0.454%  P60=0.630%  P75=0.961%  P90=1.695%
  BOTTOM MAE (MAX_BARS=12, N=10,436):
    P10=0.114%  P25=0.307%  P40=0.530%  P50=0.710%  P60=0.913%  P75=1.362%  P90=2.288%
  MFE/MAE ratio at median: 0.64

  Per-ticker medians (MAX_BARS=12):
  Ticker    MFE_med  MAE_med  Ratio      N
  ------------------------------------------
  AAPL       0.452%   0.605%   0.75   1351
  GOOG       0.508%   0.696%   0.73   1189
  GOOGL      0.469%   0.692%   0.68   1257
  MSFT       0.407%   0.655%   0.62   1217
  NVDA       0.645%   1.094%   0.59   1514
  QQQ        0.354%   0.581%   0.61   1340
  SPY        0.251%   0.487%   0.51   1164
  TSLA       0.789%   1.186%   0.67   1404

  BOTTOM MFE (MAX_BA

In [14]:
# -*- coding: utf-8 -*-
"""Untitled5.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1uwKMhUwOsUZ_ZTHM3y2bUq4xAQGdTDSO
"""

import os
os.chdir("/content/drive/MyDrive/毕设/data")

"""# mins"""

#!/usr/bin/env python3
"""
resample_1min_to_15min.py
─────────────────────────
Reads 1-minute OHLCV CSVs from processed/, resamples to 15-minute bars,
computes 53 technical indicators (matching the 1-hour feature set), and
saves both the resampled OHLCV and the full feature files.

Usage:
    python resample_1min_to_15min.py
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────
CONFIG = {
    "PROCESSED_DIR": "processed",
    "FEATURES_DIR": "features",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "RESAMPLE_FREQ": "15min",
}

# Possible names for the timestamp column in source CSVs
TIMESTAMP_ALIASES = ["timestamp", "ts_event", "datetime", "date", "time", "ts"]


# ──────────────────────────────────────────────────────────────────────
# 1. Discover reference indicator columns from an existing 1hour file
# ──────────────────────────────────────────────────────────────────────
def get_reference_indicator_cols(features_dir: str) -> list[str]:
    ohlcv_cols = {"timestamp", "open", "high", "low", "close", "volume"}
    for fname in sorted(os.listdir(features_dir)):
        if fname.endswith("_1hour_features.csv"):
            path = os.path.join(features_dir, fname)
            df = pd.read_csv(path, nrows=0)
            ind_cols = [c for c in df.columns if c.lower() not in ohlcv_cols]
            if ind_cols:
                print(f"[ref] Using {fname} as reference → {len(ind_cols)} indicator columns")
                return ind_cols
    return []


# ──────────────────────────────────────────────────────────────────────
# 2. Detect and normalize the timestamp column
# ──────────────────────────────────────────────────────────────────────
def find_time_column(df: pd.DataFrame) -> str | None:
    """Find the timestamp column regardless of naming convention."""
    cols_lower = {c.lower(): c for c in df.columns}
    for alias in TIMESTAMP_ALIASES:
        if alias in cols_lower:
            return cols_lower[alias]
    return None


def normalize_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find the time column, parse as datetime, set as index named 'timestamp'.
    Strips timezone to naive UTC for clean resampling.
    """
    time_col = find_time_column(df)

    if time_col is not None:
        df = df.copy()
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df[time_col] = df[time_col].dt.tz_localize(None)
        df = df.set_index(time_col)
        df.index.name = "timestamp"
    else:
        df = df.copy()
        df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
        df.index.name = "timestamp"

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError(f"Could not create DatetimeIndex. Columns: {list(df.columns)}")

    return df


# ──────────────────────────────────────────────────────────────────────
# 3. Resample 1-min → 15-min OHLCV
# ──────────────────────────────────────────────────────────────────────
def resample_ohlcv(df_1min: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Standard OHLCV aggregation on a DatetimeIndex dataframe."""
    df_1min = normalize_timestamps(df_1min)

    # Lowercase column names
    df_1min.columns = [c.lower() for c in df_1min.columns]

    agg_rules = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    }
    agg_rules = {k: v for k, v in agg_rules.items() if k in df_1min.columns}

    df_15 = df_1min[list(agg_rules.keys())].resample(freq).agg(agg_rules)

    # Drop non-trading periods (all NaN) and rows without close
    df_15.dropna(how="all", inplace=True)
    df_15.dropna(subset=["close"], inplace=True)

    return df_15


# ──────────────────────────────────────────────────────────────────────
# 4. Compute technical indicators via pandas_ta
# ──────────────────────────────────────────────────────────────────────
def compute_indicators_pandas_ta(df: pd.DataFrame, ref_cols: list[str]) -> pd.DataFrame:
    try:
        import pandas_ta as ta
    except ImportError:
        print("[WARN] pandas_ta not installed – pip install pandas_ta")
        return compute_indicators_manual(df, ref_cols)

    work = df[["open", "high", "low", "close", "volume"]].copy()

    # Reverting to original call for pandas_ta accessor.
    # The previous error indicates that `ta.strategy()` directly on the module is incorrect.
    # The root cause might be an issue with the pandas_ta accessor registration itself due to dependency conflicts.
    work.ta.strategy("All")

    ta_cols_lower = {c.lower(): c for c in work.columns}
    matched = {}
    unmatched = []

    for ref in ref_cols:
        rl = ref.lower()
        if rl in ta_cols_lower:
            matched[ref] = ta_cols_lower[rl]
        else:
            found = False
            for tc_lower, tc_orig in ta_cols_lower.items():
                if rl.replace("_", "") == tc_lower.replace("_", ""):
                    matched[ref] = tc_orig
                    found = True
                    break
            if not found:
                unmatched.append(ref)

    result = df[["open", "high", "low", "close", "volume"]].copy()
    for ref_name, ta_name in matched.items():
        result[ref_name] = work[ta_name]

    if unmatched:
        manual = _compute_manual_indicators(df)
        manual_lower = {c.lower(): c for c in manual.columns}
        still_missing = []
        for ref in unmatched:
            rl = ref.lower()
            if rl in manual_lower:
                result[ref] = manual[manual_lower[rl]]
            else:
                still_missing.append(ref)
        if still_missing:
            print(f"  [WARN] {len(still_missing)} indicators could not be matched: "
                  f"{still_missing[:10]}{'...' if len(still_missing) > 10 else ''}")

    return result


def _compute_manual_indicators(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    c, h, l, v = df["close"], df["high"], df["low"], df["volume"]
    o = df["open"]

    # Moving Averages
    for w in [5, 10, 20, 50, 100, 200]:
        out[f"sma_{w}"] = c.rolling(w).mean()
        out[f"ema_{w}"] = c.ewm(span=w, adjust=False).mean()

    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]

    # RSI
    for w in [7, 14, 21]:
        delta = c.diff()
        gain = delta.clip(lower=0).rolling(w).mean()
        loss = (-delta.clip(upper=0)).rolling(w).mean()
        rs = gain / loss.replace(0, np.nan)
        out[f"rsi_{w}"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    for w in [20]:
        mid = c.rolling(w).mean()
        std = c.rolling(w).std()
        out[f"bb_upper_{w}"] = mid + 2 * std
        out[f"bb_middle_{w}"] = mid
        out[f"bb_lower_{w}"] = mid - 2 * std
        out[f"bb_bandwidth_{w}"] = (out[f"bb_upper_{w}"] - out[f"bb_lower_{w}"]) / mid
        out[f"bb_percent_{w}"] = (c - out[f"bb_lower_{w}"]) / (out[f"bb_upper_{w}"] - out[f"bb_lower_{w}"]).replace(0, np.nan)

    out["bb_upper"] = out.get("bb_upper_20", c)
    out["bb_middle"] = out.get("bb_middle_20", c)
    out["bb_lower"] = out.get("bb_lower_20", c)

    # ATR
    for w in [7, 14, 21]:
        tr = pd.concat([
            h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()
        ], axis=1).max(axis=1)
        out[f"atr_{w}"] = tr.rolling(w).mean()
    out["true_range"] = pd.concat([
        h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()
    ], axis=1).max(axis=1)

    # Stochastic
    for w in [14]:
        low_min = l.rolling(w).min()
        high_max = h.rolling(w).max()
        out[f"stoch_k_{w}"] = 100 * (c - low_min) / (high_max - low_min).replace(0, np.nan)
        out[f"stoch_d_{w}"] = out[f"stoch_k_{w}"].rolling(3).mean()

    # ADX
    for w in [14]:
        plus_dm = h.diff().clip(lower=0)
        minus_dm = (-l.diff()).clip(upper=0)
        tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
        atr = tr.rolling(w).mean()
        plus_di = 100 * (plus_dm.rolling(w).mean() / atr.replace(0, np.nan))
        minus_di = 100 * (minus_dm.rolling(w).mean() / atr.replace(0, np.nan))
        dx = 100 * ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
        out[f"adx_{w}"] = dx.rolling(w).mean()
        out[f"plus_di_{w}"] = plus_di
        out[f"minus_di_{w}"] = minus_di

    # CCI
    for w in [14, 20]:
        tp = (h + l + c) / 3
        out[f"cci_{w}"] = (tp - tp.rolling(w).mean()) / (0.015 * tp.rolling(w).std())

    # Williams %R
    for w in [14]:
        high_max = h.rolling(w).max()
        low_min = l.rolling(w).min()
        out[f"willr_{w}"] = -100 * (high_max - c) / (high_max - low_min).replace(0, np.nan)

    # MFI
    for w in [14]:
        tp = (h + l + c) / 3
        mf = tp * v
        pos_mf = pd.Series(np.where(tp > tp.shift(1), mf, 0), index=df.index).rolling(w).sum()
        neg_mf = pd.Series(np.where(tp < tp.shift(1), mf, 0), index=df.index).rolling(w).sum()
        mfi = 100 - (100 / (1 + pos_mf / neg_mf.replace(0, np.nan)))
        out[f"mfi_{w}"] = mfi

    # OBV
    obv = pd.Series(np.where(c > c.shift(1), v, np.where(c < c.shift(1), -v, 0)),
                    index=df.index).cumsum()
    out["obv"] = obv

    # VWAP
    tp = (h + l + c) / 3
    out["vwap"] = (tp * v).cumsum() / v.cumsum().replace(0, np.nan)

    # Volatility / momentum
    for w in [5, 10, 20]:
        out[f"volatility_{w}"] = c.pct_change().rolling(w).std()
        out[f"return_{w}"] = c.pct_change(w)
        out[f"log_return_{w}"] = np.log(c / c.shift(w))

    out["return_1"] = c.pct_change(1)
    out["log_return_1"] = np.log(c / c.shift(1))

    # Price ratios / spreads
    out["hl_spread"] = (h - l) / c.replace(0, np.nan)
    out["oc_spread"] = (c - o) / o.replace(0, np.nan)

    # Volume indicators
    for w in [5, 10, 20]:
        out[f"volume_sma_{w}"] = v.rolling(w).mean()
    out["volume_ratio"] = v / v.rolling(20).mean().replace(0, np.nan)

    # Time features
    if isinstance(df.index, pd.DatetimeIndex):
        out["hour"] = df.index.hour
        out["day_of_week"] = df.index.dayofweek
        out["minute"] = df.index.minute
    else:
        out["hour"] = 0
        out["day_of_week"] = 0
        out["minute"] = 0

    return out


def compute_indicators_manual(df: pd.DataFrame, ref_cols: list[str]) -> pd.DataFrame:
    manual = _compute_manual_indicators(df)
    result = df[["open", "high", "low", "close", "volume"]].copy()

    manual_lower = {c.lower(): c for c in manual.columns}
    matched = 0
    for ref in ref_cols:
        rl = ref.lower()
        if rl in manual_lower:
            result[ref] = manual[manual_lower[rl]]
            matched += 1
        else:
            for ml, mc in manual_lower.items():
                if rl.replace("_", "") == ml.replace("_", ""):
                    result[ref] = manual[mc]
                    matched += 1
                    break

    if matched < len(ref_cols) * 0.5:
        print(f"  [INFO] Low match rate ({matched}/{len(ref_cols)}), adding all manual indicators")
        for col in manual.columns:
            if col not in result.columns:
                result[col] = manual[col]

    return result


# ──────────────────────────────────────────────────────────────────────
# 5. Main pipeline
# ──────────────────────────────────────────────────────────────────────
def process_ticker(ticker: str, config: dict, ref_cols: list[str]) -> dict:
    proc_dir = config["PROCESSED_DIR"]
    feat_dir = config["FEATURES_DIR"]
    freq = config["RESAMPLE_FREQ"]

    in_path = os.path.join(proc_dir, f"{ticker}_1min.csv")
    if not os.path.exists(in_path):
        print(f"[SKIP] {in_path} not found")
        return {}

    # ── Read ────────────────────────────────────────────────────────
    df_1min = pd.read_csv(in_path)
    n_1min = len(df_1min)

    # ── Resample ────────────────────────────────────────────────────
    df_15 = resample_ohlcv(df_1min, freq)
    n_15 = len(df_15)

    if n_15 == 0:
        print(f"[ERROR] {ticker}: resample produced 0 rows!")
        return {}

    # Sanity check
    expected = n_1min // 15
    if n_15 < expected * 0.5:
        print(f"  [WARN] {ticker}: got {n_15} 15min bars from {n_1min} 1min bars "
              f"(expected ~{expected})")

    # Save resampled OHLCV
    out_ohlcv = os.path.join(proc_dir, f"{ticker}_15min.csv")
    df_15.to_csv(out_ohlcv)

    # ── Compute indicators ──────────────────────────────────────────
    if ref_cols:
        df_feat = compute_indicators_pandas_ta(df_15, ref_cols)
    else:
        manual = _compute_manual_indicators(df_15)
        df_feat = pd.concat([df_15[["open", "high", "low", "close", "volume"]], manual], axis=1)

    # Drop duplicate columns
    df_feat = df_feat.loc[:, ~df_feat.columns.duplicated()]

    n_features = len([c for c in df_feat.columns
                      if c.lower() not in {"open", "high", "low", "close", "volume", "timestamp"}])

    # Save features
    out_feat = os.path.join(feat_dir, f"{ticker}_15min_features.csv")
    df_feat.index.name = "timestamp"
    df_feat.to_csv(out_feat)

    print(f"  {ticker}: {n_1min:>9,} 1min rows → {n_15:>7,} 15min rows → {n_features} indicator cols")
    return {"ticker": ticker, "n_1min": n_1min, "n_15min": n_15, "n_features": n_features}


def main():
    config = CONFIG
    proc_dir = config["PROCESSED_DIR"]
    feat_dir = config["FEATURES_DIR"]

    if not os.path.isdir(proc_dir):
        print(f"[ERROR] Processed dir not found: {proc_dir}")
        sys.exit(1)
    os.makedirs(feat_dir, exist_ok=True)

    ref_cols = get_reference_indicator_cols(feat_dir)
    if not ref_cols:
        print("[INFO] No 1hour_features reference found – will compute all indicators manually")

    print(f"\nResampling 1min → {config['RESAMPLE_FREQ']} + computing indicators")
    print("=" * 75)
    results = []
    for ticker in config["TICKERS"]:
        info = process_ticker(ticker, config, ref_cols)
        if info:
            results.append(info)

    print("=" * 75)
    if results:
        print(f"Done: {len(results)} tickers processed")
        total_1 = sum(r["n_1min"] for r in results)
        total_15 = sum(r["n_15min"] for r in results)
        print(f"Total: {total_1:,} 1min rows → {total_15:,} 15min rows")
    else:
        print("No tickers processed. Check that processed/{TICKER}_1min.csv files exist.")


if __name__ == "__main__":
    main()

import pandas as pd
import os

# Check 1min source files
print("=== 1min source files ===")
for f in sorted(os.listdir("processed")):
    if "_1min" in f:
        df = pd.read_csv(f"processed/{f}", nrows=5)
        full = pd.read_csv(f"processed/{f}")
        print(f"{f}: {len(full)} rows, cols={list(df.columns)}")
        print(f"  first row: {df.iloc[0].to_dict()}")
        print()

# Check 15min feature files
print("=== 15min feature files ===")
for f in sorted(os.listdir("features")):
    if "_15min" in f:
        df = pd.read_csv(f"features/{f}")
        print(f"{f}: {len(df)} rows, {len(df.columns)} cols")
        if len(df) > 0:
            print(f"  first row index/timestamp: {df.iloc[0, 0]}")

"""# mo

"""

"""
multiscale_cnn.py
=================
Multi-Scale CNN for turning point detection.

Short branch: 30×15min bars (7.5h) — immediate reversal patterns
Long branch:  48×1hour bars (2 days) — bull/bear dynamics, momentum decay,
              volume divergence, volatility contraction

Three evaluation levels:
  1. Detection metrics (precision/recall/AUC) vs single-scale CNN
  2. MFE/MAE analysis — do multi-scale candidates have better trade quality?
  3. Meta-label integration — LightGBM filter on multi-scale candidates

Usage (Colab A100):
    !pip install lightgbm --quiet
    !python multiscale_cnn.py
"""

import os, warnings, time, json, math
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# ─── Reproducibility ───
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────── CONFIG ───────────────────────────

CFG = {
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "SHORT_FREQ": "15min",
    "LONG_FREQ": "1hour",
    "SHORT_WIN": 30,
    "LONG_WIN": 48,
    # Zigzag
    "ZIGZAG_ATR_MULT": 2.0,
    "ZIGZAG_ATR_PERIOD": 14,
    "MIN_BARS_BETWEEN": 6,
    "TP_PROXIMITY": 3,
    "NEG_RATIO": 3,
    # Training
    "BATCH": 256,
    "LR": 1e-3,
    "WD": 1e-4,
    "EPOCHS": 100,
    "PATIENCE": 20,
    "SCHED_PATIENCE": 10,
    "GRAD_CLIP": 1.0,
    # Eval
    "DET_THRESHOLDS": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "MFE_MAX_BARS": [12, 24, 48],
    # Meta-label
    "META_TP": 0.005,
    "META_SL": 0.003,
    "META_MB": 48,
    "LGB_PARAMS": {
        "objective": "binary", "metric": "binary_logloss",
        "n_estimators": 500, "max_depth": 5, "learning_rate": 0.05,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "min_child_samples": 20, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "verbose": -1, "random_state": 42,
    },
    "LGB_THRESHOLDS": [0.45, 0.50, 0.55, 0.60],
    # Single-scale baselines (from previous runs)
    "SINGLE_BASELINES": {
        "bottom_auc": 0.822, "top_auc": 0.798,
        "bottom_prec_07": 0.085, "top_prec_07": 0.081,
        "bottom_mfe_mae_24": 0.69, "top_mfe_mae_24": 0.78,
        "bottom_lgb_wr_05": 0.818, "top_lgb_wr_05": 0.459,
    },
    "OUTPUT_DIR": "results",
    # Set to a list of tasks to skip L1/L2 for (loads saved predictions).
    # E.g. ["bottom"] to skip bottom but run top fully.
    # Set to True to skip all, False to skip none.
    "SKIP_L1_L2": ["bottom"],
}


# ═══════════════════════════════════════════════════════════════
#  DATA LOADING
# ═══════════════════════════════════════════════════════════════

def load_csv(ticker, freq, cfg):
    for f in [freq, "1hour" if freq == "1h" else freq]:
        p = os.path.join(cfg["FEATURES_DIR"], f"{ticker}_{f}_features.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            for col in ["timestamp", "ts_event", "datetime", "date"]:
                if col in df.columns:
                    df["timestamp"] = pd.to_datetime(df[col], utc=True)
                    break
            if "timestamp" not in df.columns:
                first = df.columns[0]
                if first not in df.select_dtypes(include=[np.number]).columns:
                    df["timestamp"] = pd.to_datetime(df[first], utc=True)
            df = df.sort_values("timestamp").reset_index(drop=True)
            return df
    raise FileNotFoundError(f"No {freq} features for {ticker}")


def numeric_cols(df, exclude=None):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    if exclude:
        ex |= set(c.lower() for c in exclude)
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


# ═══════════════════════════════════════════════════════════════
#  1-HOUR DYNAMICS FEATURES
# ═══════════════════════════════════════════════════════════════

def add_dynamics_features(df):
    """Add 8 'dynamics' columns to 1hour DataFrame. Uses only past data."""
    close = df["close"].values.astype(float)
    open_ = df["open"].values.astype(float) if "open" in df.columns else close
    high = df["high"].values.astype(float) if "high" in df.columns else close
    low = df["low"].values.astype(float) if "low" in df.columns else close
    vol = df["volume"].values.astype(float) if "volume" in df.columns else np.ones(len(df))
    n = len(df)

    # 1. buyer_seller_ratio (rolling 12)
    up_bar = (close > open_).astype(float)
    down_bar = (close < open_).astype(float)
    bsr = np.full(n, 1.0)
    for i in range(12, n):
        seg_up = up_bar[i - 12:i]
        seg_dn = down_bar[i - 12:i]
        seg_vol = vol[i - 12:i]
        up_vol = np.mean(seg_vol[seg_up == 1]) if seg_up.sum() > 0 else 0
        dn_vol = np.mean(seg_vol[seg_dn == 1]) if seg_dn.sum() > 0 else 1e-10
        bsr[i] = up_vol / (dn_vol + 1e-10)
    df["dyn_buyer_seller_ratio"] = bsr

    # 2. rsi_slope_12
    rsi_col = None
    for c in df.columns:
        if c.lower() in ("rsi_14", "rsi"):
            rsi_col = c
            break
    rsi_slope = np.zeros(n)
    if rsi_col is not None:
        rsi = df[rsi_col].values.astype(float)
        x = np.arange(12, dtype=float)
        x_mean = x.mean()
        x_var = ((x - x_mean) ** 2).sum()
        for i in range(12, n):
            seg = rsi[i - 12:i]
            if np.any(np.isnan(seg)):
                continue
            rsi_slope[i] = np.sum((x - x_mean) * (seg - seg.mean())) / (x_var + 1e-10)
    df["dyn_rsi_slope_12"] = rsi_slope

    # 3. macd_accel
    macd_hist_col = None
    for c in df.columns:
        cl = c.lower()
        if "macd" in cl and ("hist" in cl or "diff" in cl):
            macd_hist_col = c
            break
    macd_accel = np.zeros(n)
    if macd_hist_col is not None:
        mh = df[macd_hist_col].values.astype(float)
        for i in range(3, n):
            if not np.isnan(mh[i]) and not np.isnan(mh[i - 3]):
                macd_accel[i] = mh[i] - mh[i - 3]
    df["dyn_macd_accel"] = macd_accel

    # 4. volume_trend (slope of log(vol) over 12 bars)
    vol_trend = np.zeros(n)
    log_vol = np.log(vol + 1)
    x12 = np.arange(12, dtype=float)
    x12m = x12.mean()
    x12v = ((x12 - x12m) ** 2).sum()
    for i in range(12, n):
        seg = log_vol[i - 12:i]
        vol_trend[i] = np.sum((x12 - x12m) * (seg - seg.mean())) / (x12v + 1e-10)
    df["dyn_volume_trend"] = vol_trend

    # 5. atr_change_12
    atr_col = None
    for c in df.columns:
        if c.lower() in ("atr_14", "atr"):
            atr_col = c
            break
    atr_change = np.zeros(n)
    if atr_col is not None:
        atr = df[atr_col].values.astype(float)
        for i in range(12, n):
            if atr[i - 12] > 0 and not np.isnan(atr[i]) and not np.isnan(atr[i - 12]):
                atr_change[i] = (atr[i] - atr[i - 12]) / (atr[i - 12] + 1e-10)
    df["dyn_atr_change_12"] = atr_change

    # 6. lower_shadow_ratio (rolling 6)
    body_low = np.minimum(open_, close)
    full_range = high - low + 1e-10
    lower_shadow = (body_low - low) / full_range
    lsr = np.zeros(n)
    for i in range(6, n):
        lsr[i] = np.mean(lower_shadow[i - 6:i])
    df["dyn_lower_shadow_ratio"] = lsr

    # 7. price_position_48
    pp48 = np.full(n, 0.5)
    for i in range(48, n):
        hh = np.max(high[i - 48:i + 1])
        ll = np.min(low[i - 48:i + 1])
        rng = hh - ll
        pp48[i] = (close[i] - ll) / (rng + 1e-10) if rng > 0 else 0.5
    df["dyn_price_position_48"] = pp48

    # 8. consecutive_down
    consec = np.zeros(n)
    for i in range(1, n):
        if close[i] < close[i - 1]:
            consec[i] = consec[i - 1] + 1
        else:
            consec[i] = 0
    df["dyn_consecutive_down"] = consec

    return df


# ═══════════════════════════════════════════════════════════════
#  ZIGZAG LABELLING (ATR-adaptive, on 15min)
# ═══════════════════════════════════════════════════════════════

def compute_atr(high, low, close, period=14):
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i] - low[i],
                     abs(high[i] - close[i - 1]),
                     abs(low[i] - close[i - 1]))
    atr = np.full(n, np.nan)
    if n >= period:
        atr[period - 1] = np.mean(tr[:period])
        for i in range(period, n):
            atr[i] = (atr[i - 1] * (period - 1) + tr[i]) / period
    return atr


def zigzag_atr(high, low, close, atr_mult=2.0, atr_period=14,
               min_bars=6):
    """ATR-adaptive zigzag. Returns (bottom_indices, top_indices)."""
    n = len(close)
    atr = compute_atr(high, low, close, atr_period)
    # Fill NaN ATR with first valid
    first_valid = np.where(~np.isnan(atr))[0]
    if len(first_valid) > 0:
        atr[:first_valid[0]] = atr[first_valid[0]]
    atr = np.nan_to_num(atr, nan=close.mean() * 0.01)

    bottoms, tops = [], []
    direction = 0  # 0=undecided, 1=looking for top, -1=looking for bottom
    last_high_idx = 0
    last_low_idx = 0
    last_high = high[0]
    last_low = low[0]
    last_tp_idx = -min_bars * 2

    for i in range(1, n):
        threshold = atr[i] * atr_mult

        if direction >= 0:  # looking for top or undecided
            if high[i] > last_high:
                last_high = high[i]
                last_high_idx = i
            if last_high - low[i] >= threshold and i - last_tp_idx >= min_bars:
                if last_high_idx != 0:
                    tops.append(last_high_idx)
                    last_tp_idx = last_high_idx
                direction = -1
                last_low = low[i]
                last_low_idx = i

        if direction <= 0:  # looking for bottom or undecided
            if low[i] < last_low:
                last_low = low[i]
                last_low_idx = i
            if high[i] - last_low >= threshold and i - last_tp_idx >= min_bars:
                if last_low_idx != 0:
                    bottoms.append(last_low_idx)
                    last_tp_idx = last_low_idx
                direction = 1
                last_high = high[i]
                last_high_idx = i

    return np.array(bottoms, dtype=int), np.array(tops, dtype=int)


def make_labels(n, tp_indices, proximity=3):
    """Binary labels: 1 if within `proximity` bars of a TP."""
    labels = np.zeros(n, dtype=np.float32)
    for idx in tp_indices:
        lo = max(0, idx - proximity)
        hi = min(n, idx + proximity + 1)
        labels[lo:hi] = 1.0
    return labels


# ═══════════════════════════════════════════════════════════════
#  ALIGNMENT: 15min → 1hour mapping
# ═══════════════════════════════════════════════════════════════

def build_hour_index_map(ts_15min, ts_1hour):
    """
    For each 15min timestamp, find the index of the latest 1hour bar
    at or before that time. Returns array of 1hour indices (or -1 if none).
    """
    ts_h = ts_1hour.values.astype(np.int64)
    ts_s = ts_15min.values.astype(np.int64)
    hour_idx = np.full(len(ts_s), -1, dtype=int)
    j = 0
    for i in range(len(ts_s)):
        while j < len(ts_h) - 1 and ts_h[j + 1] <= ts_s[i]:
            j += 1
        if ts_h[j] <= ts_s[i]:
            hour_idx[i] = j
    return hour_idx


# ═══════════════════════════════════════════════════════════════
#  DATASET CONSTRUCTION
# ═══════════════════════════════════════════════════════════════

def build_paired_windows(cfg):
    """
    Build aligned (short_window, long_window, label, meta) for all tickers.
    Returns dict with train/val/test splits.
    """
    print(f"\n  Building paired windows...")
    short_win = cfg["SHORT_WIN"]
    long_win = cfg["LONG_WIN"]
    prox = cfg["TP_PROXIMITY"]

    all_data = {"short": [], "long": [], "bottom_label": [], "top_label": [],
                "ticker": [], "bar_idx": [], "timestamp": []}

    for ticker in cfg["TICKERS"]:
        try:
            df_s = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            df_l = load_csv(ticker, cfg["LONG_FREQ"], cfg)
        except FileNotFoundError as e:
            print(f"    [SKIP] {ticker}: {e}")
            continue

        # Dynamics features on 1hour
        df_l = add_dynamics_features(df_l)

        # Feature columns
        s_cols = numeric_cols(df_s)
        l_cols = numeric_cols(df_l)

        s_feat = df_s[s_cols].values.astype(np.float32)
        l_feat = df_l[l_cols].values.astype(np.float32)

        # NaN → 0
        s_feat = np.nan_to_num(s_feat, nan=0.0, posinf=0.0, neginf=0.0)
        l_feat = np.nan_to_num(l_feat, nan=0.0, posinf=0.0, neginf=0.0)

        # Zigzag labels on 15min
        close_s = df_s["close"].values.astype(float)
        high_s = df_s["high"].values.astype(float) if "high" in df_s.columns else close_s
        low_s = df_s["low"].values.astype(float) if "low" in df_s.columns else close_s

        bottoms, tops = zigzag_atr(high_s, low_s, close_s,
                                   cfg["ZIGZAG_ATR_MULT"],
                                   cfg["ZIGZAG_ATR_PERIOD"],
                                   cfg["MIN_BARS_BETWEEN"])
        bottom_labels = make_labels(len(df_s), bottoms, prox)
        top_labels = make_labels(len(df_s), tops, prox)

        # Alignment map
        hour_idx_map = build_hour_index_map(df_s["timestamp"], df_l["timestamp"])

        # Build windows
        n_short = len(df_s)
        count = 0
        for i in range(short_win, n_short):
            hi = hour_idx_map[i]
            if hi < long_win:
                continue  # not enough 1hour history

            short_window = s_feat[i - short_win:i]  # (30, N_s)
            long_window = l_feat[hi - long_win + 1:hi + 1]  # (48, N_l)

            if short_window.shape[0] != short_win or long_window.shape[0] != long_win:
                continue

            all_data["short"].append(short_window)
            all_data["long"].append(long_window)
            all_data["bottom_label"].append(bottom_labels[i])
            all_data["top_label"].append(top_labels[i])
            all_data["ticker"].append(ticker)
            all_data["bar_idx"].append(i)
            all_data["timestamp"].append(str(df_s["timestamp"].iloc[i]))
            count += 1

        n_bot = int(bottom_labels[short_win:].sum())
        n_top = int(top_labels[short_win:].sum())
        print(f"    {ticker}: {count:,} samples, "
              f"bottoms={n_bot} tops={n_top}  "
              f"s_feat={s_feat.shape[1]} l_feat={l_feat.shape[1]}")

    # Convert to arrays
    X_short = np.array(all_data["short"], dtype=np.float32)
    X_long = np.array(all_data["long"], dtype=np.float32)
    y_bottom = np.array(all_data["bottom_label"], dtype=np.float32)
    y_top = np.array(all_data["top_label"], dtype=np.float32)
    tickers = np.array(all_data["ticker"])
    bar_indices = np.array(all_data["bar_idx"])
    timestamps = np.array(all_data["timestamp"])

    n = len(y_bottom)
    print(f"\n  Total: {n:,} samples")
    print(f"  Short shape: {X_short.shape}  Long shape: {X_long.shape}")
    print(f"  Bottom pos rate: {y_bottom.mean():.4f}  "
          f"Top pos rate: {y_top.mean():.4f}")

    # Time-based split (data already sorted by time within each ticker,
    # but we sort globally by bar_idx proxy — use index order since tickers processed sequentially)
    # Actually split per-ticker to preserve time ordering
    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)

    for ticker in cfg["TICKERS"]:
        t_mask = tickers == ticker
        t_idx = np.where(t_mask)[0]
        if len(t_idx) == 0:
            continue
        n_t = len(t_idx)
        tr_end = int(n_t * 0.70)
        va_end = int(n_t * 0.85)
        train_mask[t_idx[:tr_end]] = True
        val_mask[t_idx[tr_end:va_end]] = True
        test_mask[t_idx[va_end:]] = True

    print(f"  Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  "
          f"Test: {test_mask.sum():,}")

    return {
        "X_short": X_short, "X_long": X_long,
        "y_bottom": y_bottom, "y_top": y_top,
        "tickers": tickers, "bar_indices": bar_indices,
        "timestamps": timestamps,
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "n_short_feat": X_short.shape[2], "n_long_feat": X_long.shape[2],
    }


# ═══════════════════════════════════════════════════════════════
#  NORMALIZE + UNDERSAMPLE
# ═══════════════════════════════════════════════════════════════

def normalize_windows(X_train, *others):
    """Fit scaler on train, transform all. X shape: (N, T, F)."""
    n, t, f = X_train.shape
    flat = X_train.reshape(-1, f)
    mu = np.nanmean(flat, axis=0)
    sigma = np.nanstd(flat, axis=0) + 1e-8
    mu = np.nan_to_num(mu, nan=0.0)
    sigma = np.where(np.isnan(sigma) | (sigma < 1e-8), 1.0, sigma)

    results = []
    for arr in (X_train,) + others:
        normed = (arr - mu) / sigma
        np.nan_to_num(normed, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        results.append(normed)
    return tuple(results) + (mu, sigma)


def undersample_negatives(X_s, X_l, y, neg_ratio=3, seed=42):
    """Undersample negatives to neg_ratio:1."""
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    n_neg_keep = min(n_pos * neg_ratio, len(neg_idx))
    rng = np.random.RandomState(seed)
    neg_keep = rng.choice(neg_idx, size=n_neg_keep, replace=False)
    keep = np.sort(np.concatenate([pos_idx, neg_keep]))
    return X_s[keep], X_l[keep], y[keep]


# ═══════════════════════════════════════════════════════════════
#  MODEL
# ═══════════════════════════════════════════════════════════════

class MultiScaleCNN(nn.Module):
    def __init__(self, n_short_feat, n_long_feat):
        super().__init__()
        # Short branch (15min, 30 bars)
        self.short_branch = nn.Sequential(
            nn.Conv1d(n_short_feat, 32, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        # Long branch (1hour, 48 bars)
        self.long_branch = nn.Sequential(
            nn.Conv1d(n_long_feat, 32, kernel_size=7, padding=3),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        # Fusion
        self.head = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x_short, x_long):
        # Input: (B, T, F) → Conv1d wants (B, F, T)
        s = self.short_branch(x_short.transpose(1, 2))
        l = self.long_branch(x_long.transpose(1, 2))
        fused = torch.cat([s, l], dim=1)
        return self.head(fused).squeeze(-1)


class PairedDataset(Dataset):
    def __init__(self, X_short, X_long, y):
        self.xs = torch.tensor(X_short, dtype=torch.float32)
        self.xl = torch.tensor(X_long, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.xs[i], self.xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  TRAINING
# ═══════════════════════════════════════════════════════════════

def train_model(X_s_tr, X_l_tr, y_tr, X_s_va, X_l_va, y_va,
                n_short_feat, n_long_feat, cfg, task):
    """Train MultiScaleCNN for one task."""
    print(f"\n    Training MultiScaleCNN ({task})...")
    # Undersample training
    X_s_tr_u, X_l_tr_u, y_tr_u = undersample_negatives(
        X_s_tr, X_l_tr, y_tr, cfg["NEG_RATIO"])
    print(f"    After undersample: {len(y_tr_u):,} "
          f"(pos={y_tr_u.sum():.0f}, neg={len(y_tr_u)-y_tr_u.sum():.0f})")

    model = MultiScaleCNN(n_short_feat, n_long_feat).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Params: {n_params:,}")

    pos_weight = torch.tensor(
        [(1 - y_tr_u.mean()) / (y_tr_u.mean() + 1e-8)],
        dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["LR"],
                                  weight_decay=cfg["WD"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=cfg["SCHED_PATIENCE"])

    train_dl = DataLoader(PairedDataset(X_s_tr_u, X_l_tr_u, y_tr_u),
                          batch_size=cfg["BATCH"], shuffle=True,
                          num_workers=0, pin_memory=True)
    val_dl = DataLoader(PairedDataset(X_s_va, X_l_va, y_va),
                        batch_size=cfg["BATCH"] * 4, shuffle=False,
                        num_workers=0, pin_memory=True)

    best_loss = float("inf")
    best_state = None
    patience_ctr = 0
    best_epoch = 0

    for epoch in range(cfg["EPOCHS"]):
        model.train()
        t_loss = 0.0
        for xs, xl, yb in train_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xs, xl)
            if torch.isnan(logits).any():
                continue
            loss = criterion(logits, yb)
            if torch.isnan(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["GRAD_CLIP"])
            optimizer.step()
            t_loss += loss.item() * len(yb)
        t_loss /= max(len(y_tr_u), 1)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for xs, xl, yb in val_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                logits = model(xs, xl)
                v_loss += criterion(logits, yb).item() * len(yb)
        v_loss /= max(len(y_va), 1)

        scheduler.step(v_loss)
        is_best = v_loss < best_loss
        if is_best:
            best_loss = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            best_epoch = epoch + 1
        else:
            patience_ctr += 1

        lr = optimizer.param_groups[0]["lr"]
        if (epoch + 1) % 10 == 0 or is_best or patience_ctr == cfg["PATIENCE"]:
            tag = "  *best" if is_best else ""
            print(f"      E{epoch+1:3d} t={t_loss:.6f} v={v_loss:.6f} "
                  f"lr={lr:.1e} p={patience_ctr}{tag}")
        if patience_ctr >= cfg["PATIENCE"]:
            print(f"      Early stop (best={best_epoch})")
            break

    if best_state:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    return model


def predict(model, X_s, X_l, batch_size=1024):
    model.eval()
    dl = DataLoader(PairedDataset(X_s, X_l, np.zeros(len(X_s))),
                    batch_size=batch_size, shuffle=False,
                    num_workers=0, pin_memory=True)
    preds = []
    with torch.no_grad():
        for xs, xl, _ in dl:
            logits = model(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy()
            probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -20, 20)))
            preds.append(np.nan_to_num(probs, nan=0.5))
    return np.concatenate(preds)


# ═══════════════════════════════════════════════════════════════
#  LEVEL 1: DETECTION METRICS
# ═══════════════════════════════════════════════════════════════

def eval_detection(probs, y_true, thresholds, baseline_rate):
    """Precision, recall, F1 at each threshold + AUC."""
    auc = roc_auc_score(y_true, probs) if y_true.sum() > 0 and (1 - y_true).sum() > 0 else 0.5
    results = {"auc": auc, "n": len(y_true),
               "n_pos": int(y_true.sum()),
               "baseline_rate": baseline_rate}
    for th in thresholds:
        pred = (probs >= th).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        n_pred = tp + fp
        # z-score: precision vs baseline rate
        se = np.sqrt(baseline_rate * (1 - baseline_rate) / n_pred) if n_pred > 0 and 0 < baseline_rate < 1 else 1
        z = (prec - baseline_rate) / se if se > 0 and n_pred > 0 else 0
        results[th] = {"precision": prec, "recall": rec, "f1": f1,
                       "n_pred": n_pred, "tp": tp, "z": z}
    return results


def print_detection(results, task, single_auc):
    print(f"\n  {task.upper()} — Detection Metrics  "
          f"(AUC={results['auc']:.3f}  single-scale={single_auc:.3f}  "
          f"Δ={results['auc']-single_auc:+.3f})")
    print(f"  {'Th':>5} {'Prec':>8} {'Recall':>8} {'F1':>8} "
          f"{'N_pred':>8} {'TP':>6} {'z':>7}")
    print(f"  {'-'*52}")
    for th in sorted(k for k in results if isinstance(k, float)):
        r = results[th]
        sig = "*" if abs(r["z"]) > 1.96 else ""
        print(f"  {th:>5.1f} {r['precision']*100:>7.2f}% "
              f"{r['recall']*100:>7.1f}% {r['f1']*100:>7.2f}% "
              f"{r['n_pred']:>8} {r['tp']:>6} {r['z']:>+6.2f}{sig}")


# ═══════════════════════════════════════════════════════════════
#  LEVEL 2: MFE/MAE ANALYSIS
# ═══════════════════════════════════════════════════════════════

def compute_mfe_mae_for_candidates(cfg, probs, tickers, bar_indices,
                                    test_mask, task, threshold=0.5):
    """Compute MFE/MAE for multi-scale CNN candidates."""
    direction = "long" if task == "bottom" else "short"
    cand_mask = test_mask & (probs >= threshold)
    cand_idx = np.where(cand_mask)[0]

    if len(cand_idx) == 0:
        return {}

    # Group by ticker
    results = {}
    for mb in cfg["MFE_MAX_BARS"]:
        all_mfe, all_mae, all_probs = [], [], []
        for ticker in cfg["TICKERS"]:
            try:
                df = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            except FileNotFoundError:
                continue
            close = df["close"].values.astype(float)
            high = df["high"].values.astype(float) if "high" in df.columns else close
            low = df["low"].values.astype(float) if "low" in df.columns else close

            t_cand = cand_idx[(tickers[cand_idx] == ticker)]
            for ci in t_cand:
                bi = bar_indices[ci]
                if bi + 1 >= len(close):
                    continue
                end = min(bi + mb + 1, len(close))
                if bi + 1 >= end:
                    continue
                seg_h = high[bi + 1:end]
                seg_l = low[bi + 1:end]
                entry = close[bi]
                if direction == "long":
                    mfe = max((np.max(seg_h) - entry) / entry, 0)
                    mae = max((entry - np.min(seg_l)) / entry, 0)
                else:
                    mfe = max((entry - np.min(seg_l)) / entry, 0)
                    mae = max((np.max(seg_h) - entry) / entry, 0)
                all_mfe.append(mfe)
                all_mae.append(mae)
                all_probs.append(probs[ci])

        all_mfe = np.array(all_mfe)
        all_mae = np.array(all_mae)
        all_probs = np.array(all_probs)
        mfe_med = np.median(all_mfe) if len(all_mfe) > 0 else 0
        mae_med = np.median(all_mae) if len(all_mae) > 0 else 0
        ratio = mfe_med / mae_med if mae_med > 0 else 999

        results[mb] = {
            "n": len(all_mfe), "mfe_med": float(mfe_med),
            "mae_med": float(mae_med), "ratio": float(ratio),
            "mfe": all_mfe, "mae": all_mae, "probs": all_probs,
        }

        print(f"    MB={mb}: N={len(all_mfe):,}  "
              f"MFE_med={mfe_med*100:.3f}%  MAE_med={mae_med*100:.3f}%  "
              f"ratio={ratio:.3f}")

    # Probability bucketed analysis (using MB=24)
    best_mb = 24 if 24 in results else list(results.keys())[0]
    d = results[best_mb]
    if d["n"] > 0:
        print(f"\n    Prob-bucketed MFE/MAE (MB={best_mb}):")
        print(f"    {'Bucket':>12} {'N':>6} {'MFE_med':>9} {'MAE_med':>9} "
              f"{'Ratio':>7}")
        print(f"    {'-'*48}")
        buckets = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]
        for lo, hi in buckets:
            mask = (d["probs"] >= lo) & (d["probs"] < hi)
            n_b = mask.sum()
            if n_b < 5:
                print(f"    [{lo:.1f}-{hi:.1f}) {n_b:>6}  —")
                continue
            mfe_b = np.median(d["mfe"][mask])
            mae_b = np.median(d["mae"][mask])
            r_b = mfe_b / mae_b if mae_b > 0 else 999
            print(f"    [{lo:.1f}-{hi:.1f}) {n_b:>6} "
                  f"{mfe_b*100:>8.3f}% {mae_b*100:>8.3f}% {r_b:>7.3f}")

    return results


# ═══════════════════════════════════════════════════════════════
#  LEVEL 3: META-LABEL (LightGBM)
# ═══════════════════════════════════════════════════════════════

def triple_barrier_vectorized(close_arr, high_arr, low_arr,
                              indices, direction, tp_pct, sl_pct, max_bars):
    """
    Vectorized triple-barrier labelling — processes ALL candidates simultaneously.

    For each horizon step h=1..max_bars, computes TP/SL hits across all
    still-open positions in a single numpy pass.  Turns O(N * max_bars) Python
    iterations into O(max_bars) numpy vectorised operations.

    Returns: labels (int array), pnls (float array)
    """
    n_price = len(close_arr)
    n_cand = len(indices)

    entries = close_arr[indices]                       # (n_cand,)
    labels = np.zeros(n_cand, dtype=np.int32)
    pnls = np.zeros(n_cand, dtype=np.float64)
    still_open = np.ones(n_cand, dtype=bool)

    for h in range(1, max_bars + 1):
        if not still_open.any():
            break

        future_idx = indices + h
        valid = still_open & (future_idx < n_price)
        if not valid.any():
            continue

        vi = np.where(valid)[0]                        # indices into candidates
        fi = future_idx[vi]                            # indices into price arrays
        ent = entries[vi]

        if direction == "long":
            tp_hit = (high_arr[fi] - ent) / ent >= tp_pct
            sl_hit = (ent - low_arr[fi]) / ent >= sl_pct
        else:
            tp_hit = (ent - low_arr[fi]) / ent >= tp_pct
            sl_hit = (high_arr[fi] - ent) / ent >= sl_pct

        # Both hit same bar → conservative: SL wins
        both = tp_hit & sl_hit
        pure_tp = tp_hit & ~sl_hit
        pure_sl = sl_hit & ~tp_hit

        # TP exits
        tp_idx = vi[pure_tp]
        labels[tp_idx] = 1
        pnls[tp_idx] = tp_pct
        still_open[tp_idx] = False

        # SL exits (including both-hit)
        sl_idx = vi[pure_sl | both]
        labels[sl_idx] = 0
        pnls[sl_idx] = -sl_pct
        still_open[sl_idx] = False

    # Timeout exits — mark-to-market PnL
    to_mask = still_open
    if to_mask.any():
        to_idx = np.where(to_mask)[0]
        exit_idx = np.minimum(indices[to_idx] + max_bars, n_price - 1)
        exit_p = close_arr[exit_idx]
        if direction == "long":
            pnls[to_idx] = (exit_p - entries[to_idx]) / entries[to_idx]
        else:
            pnls[to_idx] = (entries[to_idx] - exit_p) / entries[to_idx]
        # labels already 0

    return labels, pnls


def build_meta_features(X_short, X_long, probs):
    """Flatten last few bars + CNN prob as LGB features."""
    n = len(probs)
    # Short: last 5 bars flattened
    s_last = X_short[:, -5:, :].reshape(n, -1)
    # Long: last 3 bars flattened
    l_last = X_long[:, -3:, :].reshape(n, -1)
    # CNN prob
    prob_col = probs.reshape(-1, 1)
    return np.nan_to_num(np.hstack([s_last, l_last, prob_col]),
                         nan=0.0, posinf=0.0, neginf=0.0)


def run_meta_label(cfg, probs, X_short, X_long, tickers, bar_indices,
                   test_mask, task, threshold):
    """Level 3: LightGBM meta-label on multi-scale CNN candidates.

    Fixed: pre-loads ticker CSVs once (8 reads, not 50k), then uses
    vectorized triple-barrier labelling per ticker.
    """
    if not HAS_LGB:
        print("    [SKIP] LightGBM not installed")
        return {}

    direction = "long" if task == "bottom" else "short"
    cand_mask = test_mask & (probs >= threshold)
    cand_idx = np.where(cand_mask)[0]
    if len(cand_idx) < 50:
        print(f"    [SKIP] Only {len(cand_idx)} candidates")
        return {}

    print(f"    Candidates: {len(cand_idx):,}")

    # ── Step A: Pre-load all ticker price data ONCE ──
    t0 = time.time()
    ticker_data = {}  # ticker → (close, high, low)
    for ticker in cfg["TICKERS"]:
        try:
            df = load_csv(ticker, cfg["SHORT_FREQ"], cfg)
            close = df["close"].values.astype(np.float64)
            high = df["high"].values.astype(np.float64) if "high" in df.columns else close.copy()
            low = df["low"].values.astype(np.float64) if "low" in df.columns else close.copy()
            ticker_data[ticker] = (close, high, low)
        except FileNotFoundError:
            pass
    print(f"    Loaded {len(ticker_data)} ticker CSVs  ({time.time()-t0:.2f}s)")

    # ── Step B: Vectorized triple-barrier labelling per ticker ──
    t1 = time.time()
    all_labels = np.full(len(cand_idx), -1, dtype=np.int32)
    all_pnls = np.zeros(len(cand_idx), dtype=np.float64)
    valid_mask = np.zeros(len(cand_idx), dtype=bool)

    cand_tickers = tickers[cand_idx]
    cand_bars = bar_indices[cand_idx]

    for ticker, (close, high, low) in ticker_data.items():
        t_mask = cand_tickers == ticker
        if not t_mask.any():
            continue
        t_positions = np.where(t_mask)[0]       # positions within cand_idx
        t_bar_idx = cand_bars[t_positions]       # bar indices into price arrays

        # Filter out-of-range
        ok = (t_bar_idx >= 1) & (t_bar_idx < len(close) - 1)
        if not ok.any():
            continue
        t_positions = t_positions[ok]
        t_bar_idx = t_bar_idx[ok]

        labs, pnls_v = triple_barrier_vectorized(
            close, high, low, t_bar_idx, direction,
            cfg["META_TP"], cfg["META_SL"], cfg["META_MB"])

        all_labels[t_positions] = labs
        all_pnls[t_positions] = pnls_v
        valid_mask[t_positions] = True

    # Keep only valid candidates
    valid_cand = cand_idx[valid_mask]
    labels = all_labels[valid_mask].astype(np.float32)
    pnls = all_pnls[valid_mask].astype(np.float32)
    print(f"    Triple-barrier labelling: {len(labels):,} trades  ({time.time()-t1:.2f}s)")

    if len(valid_cand) < 50:
        print(f"    [SKIP] Only {len(valid_cand)} valid candidates")
        return {}

    # ── Step C: Build features ──
    t2 = time.time()
    X_meta = build_meta_features(X_short[valid_cand],
                                 X_long[valid_cand],
                                 probs[valid_cand])
    print(f"    Feature matrix: {X_meta.shape}  ({time.time()-t2:.2f}s)")

    # ── Step D: Train/test split + LightGBM ──
    t3 = time.time()
    n_c = len(labels)
    split = int(n_c * 0.7)
    X_tr, y_tr, pnl_tr = X_meta[:split], labels[:split], pnls[:split]
    X_te, y_te, pnl_te = X_meta[split:], labels[split:], pnls[split:]

    base_rate = float(y_te.mean())
    print(f"    Meta-label: train={len(y_tr):,} test={len(y_te):,} "
          f"base_wr={base_rate*100:.1f}%")

    val_split = int(len(y_tr) * 0.8)
    params = dict(cfg["LGB_PARAMS"])
    n_est = params.pop("n_estimators", 500)
    model = lgb.LGBMClassifier(n_estimators=n_est, **params)
    model.fit(X_tr[:val_split], y_tr[:val_split],
              eval_set=[(X_tr[val_split:], y_tr[val_split:])],
              callbacks=[lgb.early_stopping(30, verbose=False),
                         lgb.log_evaluation(0)])
    print(f"    LightGBM training  ({time.time()-t3:.2f}s)")

    # ── Step E: Evaluate ──
    t4 = time.time()
    lgb_probs = model.predict_proba(X_te)[:, 1]

    results = {"base_rate": base_rate, "n_test": len(y_te)}
    print(f"\n    {'Th':>5} {'WR':>8} {'N':>6} {'PF':>7} {'TotPnL':>9}")
    print(f"    {'-'*42}")
    for th in cfg["LGB_THRESHOLDS"]:
        m = lgb_probs >= th
        n_t = int(m.sum())
        if n_t == 0:
            results[th] = {"wr": 0, "n": 0, "pf": 0, "pnl": 0}
            continue
        wr = float(y_te[m].mean())
        ps = pnl_te[m]
        gp = ps[ps > 0].sum()
        gl = abs(ps[ps < 0].sum())
        pf = gp / gl if gl > 0 else (999 if gp > 0 else 0)
        results[th] = {"wr": wr, "n": n_t,
                       "pf": float(pf), "pnl": float(ps.sum())}
        print(f"    {th:>5.2f} {wr*100:>7.1f}% {n_t:>6} "
              f"{pf:>7.2f} {ps.sum()*100:>8.3f}%")
    print(f"    Evaluation  ({time.time()-t4:.2f}s)")

    return results


# ═══════════════════════════════════════════════════════════════
#  SAVE PREDICTIONS
# ═══════════════════════════════════════════════════════════════

def save_predictions(probs, tickers, bar_indices, timestamps,
                     test_mask, cfg, task):
    rows = []
    test_idx = np.where(test_mask)[0]
    for i in test_idx:
        rows.append({
            "ticker": tickers[i],
            "bar_index": int(bar_indices[i]),
            "timestamp": timestamps[i],
            "prob": float(probs[i]),
        })
    df = pd.DataFrame(rows)
    path = os.path.join(cfg["OUTPUT_DIR"],
                        f"multiscale_cnn_{task}_predictions.csv")
    df.to_csv(path, index=False)
    print(f"  Saved: {path} ({len(df)} rows)")


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def load_checkpoint(ckpt_path: str, device=None):
    """
    Load a saved MultiScaleCNN checkpoint.

    Returns:
        model       – MultiScaleCNN, eval mode, on `device`
        s_mu, s_sigma – short-branch scaler arrays (shape: [n_short_feat])
        l_mu, l_sigma – long-branch scaler arrays  (shape: [n_long_feat])
        meta        – dict with task, short_win, long_win, n_*_feat

    Usage:
        model, s_mu, s_sigma, l_mu, l_sigma, meta = load_checkpoint("results/multiscale_cnn_bottom.pt")
        # Normalize your own windows before predicting:
        X_s_norm = (X_s - s_mu) / s_sigma
        X_l_norm = (X_l - l_mu) / l_sigma
        probs = predict(model, X_s_norm, X_l_norm)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    raw = torch.load(ckpt_path, map_location="cpu", weights_only=True)

    # Backward-compatible: support old plain state_dict format
    if "model_state_dict" not in raw:
        raise ValueError(
            f"Checkpoint {ckpt_path} is in old format (plain state_dict). "
            "Re-train to generate a full checkpoint with scaler parameters."
        )

    n_sf = raw["n_short_feat"]
    n_lf = raw["n_long_feat"]
    model = MultiScaleCNN(n_sf, n_lf)
    model.load_state_dict(raw["model_state_dict"])
    model.to(device).eval()

    meta = {
        "task": raw.get("task"),
        "short_win": raw.get("short_win"),
        "long_win": raw.get("long_win"),
        "n_short_feat": n_sf,
        "n_long_feat": n_lf,
    }
    # Return scalers as numpy arrays (same type as normalize_windows output)
    to_np = lambda t: t.numpy() if isinstance(t, torch.Tensor) else t
    return (model,
            to_np(raw["s_mu"]), to_np(raw["s_sigma"]),
            to_np(raw["l_mu"]), to_np(raw["l_sigma"]),
            meta)


def main():
    cfg = CFG
    print("=" * 78)
    print("  Multi-Scale CNN for Turning Point Detection")
    print(f"  Short: {cfg['SHORT_WIN']}×{cfg['SHORT_FREQ']}  "
          f"Long: {cfg['LONG_WIN']}×{cfg['LONG_FREQ']}")
    print(f"  Device: {DEVICE}")
    print("=" * 78)

    os.makedirs(cfg["OUTPUT_DIR"], exist_ok=True)

    # ── Build data ──
    data = build_paired_windows(cfg)

    X_s = data["X_short"]
    X_l = data["X_long"]
    tr = data["train_mask"]
    va = data["val_mask"]
    te = data["test_mask"]

    # Normalize short and long branches separately
    X_s_tr, X_s_va, X_s_te, s_mu, s_sigma = normalize_windows(
        X_s[tr], X_s[va], X_s[te])
    X_l_tr, X_l_va, X_l_te, l_mu, l_sigma = normalize_windows(
        X_l[tr], X_l[va], X_l[te])

    # Reassemble full arrays (needed for prediction indexing)
    X_s_full = np.zeros_like(X_s)
    X_l_full = np.zeros_like(X_l)
    X_s_full[tr] = X_s_tr
    X_s_full[va] = X_s_va
    X_s_full[te] = X_s_te
    X_l_full[tr] = X_l_tr
    X_l_full[va] = X_l_va
    X_l_full[te] = X_l_te

    all_results = {}

    for task in ["bottom", "top"]:
        y_col = data["y_bottom"] if task == "bottom" else data["y_top"]
        print(f"\n{'#'*78}")
        print(f"  TASK: {task.upper()}")
        print(f"{'#'*78}")

        y_tr = y_col[tr]
        y_va = y_col[va]
        y_te = y_col[te]

        det = {}
        mfe_results = {}
        train_time = 0.0

        skip_cfg = cfg.get("SKIP_L1_L2", False)
        skip_this = (skip_cfg is True or
                     (isinstance(skip_cfg, list) and task in skip_cfg))

        if skip_this:
            # ── Load saved predictions instead of training ──
            pred_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}_predictions.csv")
            ckpt_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}.pt")
            if not os.path.exists(pred_path):
                print(f"  [ERROR] {pred_path} not found, cannot skip L1/L2")
                continue

            print(f"  Loading saved predictions: {pred_path}")
            pred_df = pd.read_csv(pred_path)
            probs = np.zeros(len(y_col))
            # Map predictions back to test indices
            test_idx_arr = np.where(te)[0]
            test_tickers = data["tickers"][test_idx_arr]
            test_bars = data["bar_indices"][test_idx_arr]
            for _, row in pred_df.iterrows():
                matches = ((test_tickers == row["ticker"]) &
                           (test_bars == int(row["bar_index"])))
                mi = np.where(matches)[0]
                if len(mi) > 0:
                    probs[test_idx_arr[mi[0]]] = row["prob"]
            print(f"  Loaded {len(pred_df):,} predictions, "
                  f"mapped to {(probs[te] > 0).sum():,} test samples")
            print(f"\n  ═══ SKIPPING LEVELS 1-2 (already completed) ═══")

        else:
            # ── Train ──
            t0 = time.time()
            model = train_model(X_s_tr, X_l_tr, y_tr,
                                X_s_va, X_l_va, y_va,
                                data["n_short_feat"], data["n_long_feat"],
                                cfg, task)
            train_time = time.time() - t0
            print(f"    Training time: {train_time:.1f}s")

            # Save model (full checkpoint: weights + scalers + arch params)
            ckpt_path = os.path.join(cfg["OUTPUT_DIR"],
                                     f"multiscale_cnn_{task}.pt")
            checkpoint = {
                "model_state_dict": model.state_dict(),
                "n_short_feat": data["n_short_feat"],
                "n_long_feat": data["n_long_feat"],
                "short_win": cfg["SHORT_WIN"],
                "long_win": cfg["LONG_WIN"],
                # Convert numpy arrays to tensors for weights_only=True compat
                "s_mu": torch.tensor(s_mu, dtype=torch.float32),
                "s_sigma": torch.tensor(s_sigma, dtype=torch.float32),
                "l_mu": torch.tensor(l_mu, dtype=torch.float32),
                "l_sigma": torch.tensor(l_sigma, dtype=torch.float32),
                "task": task,
            }
            torch.save(checkpoint, ckpt_path)
            # Validate checkpoint: ensure weights are readable and NaN-free
            _ckpt_check = torch.load(ckpt_path, map_location="cpu",
                                     weights_only=True)
            _nan_found = any(
                torch.isnan(v).any()
                for v in _ckpt_check["model_state_dict"].values()
                if v.is_floating_point()
            )
            if _nan_found:
                print(f"    [ERROR] NaN weights in checkpoint! Saved but invalid.")
            else:
                print(f"    Saved & verified: {ckpt_path}")

            # ── Predict on full dataset ──
            probs = np.zeros(len(y_col))
            probs[te] = predict(model, X_s_te, X_l_te)
            probs[va] = predict(model, X_s_va, X_l_va)

            # ── Save predictions ──
            save_predictions(probs, data["tickers"], data["bar_indices"],
                             data["timestamps"], te, cfg, task)

            # ════════════════════════════════════════════
            # LEVEL 1: Detection Metrics
            # ════════════════════════════════════════════
            print(f"\n  ═══ LEVEL 1: DETECTION METRICS ═══")

            baseline_rate = y_te.mean()
            det = eval_detection(probs[te], y_te, cfg["DET_THRESHOLDS"],
                                 baseline_rate)
            single_auc = cfg["SINGLE_BASELINES"][f"{task}_auc"]
            print_detection(det, task, single_auc)

            # ════════════════════════════════════════════
            # LEVEL 2: MFE/MAE Analysis
            # ════════════════════════════════════════════
            print(f"\n  ═══ LEVEL 2: MFE/MAE ANALYSIS ═══")

            mfe_results = compute_mfe_mae_for_candidates(
                cfg, probs, data["tickers"], data["bar_indices"],
                te, task, threshold=0.5)

        # ════════════════════════════════════════════
        # LEVEL 3: Meta-Label
        # ════════════════════════════════════════════
        print(f"\n  ═══ LEVEL 3: META-LABEL INTEGRATION ═══")

        # Pick threshold with best MFE/MAE ratio and N>500
        best_th = 0.5
        if mfe_results:
            mb24 = mfe_results.get(24, {})
            if mb24 and mb24.get("n", 0) > 0:
                # Try higher thresholds
                for test_th in [0.5, 0.6, 0.7]:
                    cand_n = (te & (probs >= test_th)).sum()
                    if cand_n > 500:
                        best_th = test_th

        print(f"  Using CNN threshold={best_th} for meta-label")
        t_l3 = time.time()
        meta = run_meta_label(cfg, probs, X_s_full, X_l_full,
                              data["tickers"], data["bar_indices"],
                              te, task, best_th)
        print(f"  Level 3 total: {time.time()-t_l3:.2f}s")

        all_results[task] = {
            "detection": {
                "auc": det.get("auc", 0),
                "n": det.get("n", 0),
                "thresholds": {
                    str(k): v for k, v in det.items()
                    if isinstance(k, float)
                },
            } if det else {},
            "mfe_mae": {
                str(k): {kk: float(vv) if not isinstance(vv, np.ndarray) else None
                         for kk, vv in v.items()}
                for k, v in mfe_results.items()
            } if mfe_results else {},
            "meta_label": {
                str(k): (v if not isinstance(v, dict)
                         else {kk: float(vv) if isinstance(vv, (float, np.floating))
                               else int(vv) if isinstance(vv, (int, np.integer))
                               else vv
                               for kk, vv in v.items()})
                for k, v in meta.items()
            } if meta else {},
            "train_time": train_time,
        }

    # ═══════════════════════════════════════════════════════════
    #  MERGE WITH PREVIOUSLY SAVED RESULTS (for skipped levels)
    # ═══════════════════════════════════════════════════════════
    prev_json = os.path.join(cfg["OUTPUT_DIR"], "multiscale_cnn_results.json")
    if os.path.exists(prev_json):
        try:
            with open(prev_json) as f:
                prev = json.load(f)
            for task in ["bottom", "top"]:
                if task in prev and task in all_results:
                    cur = all_results[task]
                    old = prev[task]
                    # Fill in missing sections from previous run
                    if not cur.get("detection") and old.get("detection"):
                        cur["detection"] = old["detection"]
                        print(f"  [INFO] Loaded {task} L1 detection from previous run")
                    if not cur.get("mfe_mae") and old.get("mfe_mae"):
                        cur["mfe_mae"] = old["mfe_mae"]
                        print(f"  [INFO] Loaded {task} L2 MFE/MAE from previous run")
        except Exception:
            pass

    # ═══════════════════════════════════════════════════════════
    #  META-LABEL SUMMARY (key results from Level 3)
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ LEVEL 3 META-LABEL SUMMARY")
    print(f"{'='*78}")
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        meta = r.get("meta_label", {})
        if not meta:
            print(f"\n  {task.upper()}: not available")
            continue
        base = meta.get("base_rate", 0)
        n_test = meta.get("n_test", 0)
        direction = "LONG" if task == "bottom" else "SHORT"
        print(f"\n  {task.upper()} ({direction}):  base_wr={base*100:.1f}%  n_test={n_test}")
        print(f"  {'Th':>6} {'WR':>8} {'N':>7} {'PF':>7} {'TotPnL':>10} {'Δ vs base':>10}")
        print(f"  {'-'*52}")
        for th_key in sorted(k for k in meta if k not in ("base_rate", "n_test")):
            th_v = meta[th_key]
            if not isinstance(th_v, dict):
                continue
            wr = th_v.get("wr", 0)
            n_t = th_v.get("n", 0)
            pf = th_v.get("pf", 0)
            pnl = th_v.get("pnl", 0)
            delta = wr - base if n_t > 0 else 0
            mark = "✓" if wr > 0.55 and n_t >= 100 else ""
            print(f"  {th_key:>6} {wr*100:>7.1f}% {n_t:>7} {pf:>7.2f} "
                  f"{pnl*100:>9.1f}% {delta*100:>+9.1f}pp {mark}")

    # ═══════════════════════════════════════════════════════════
    #  FINAL COMPARISON TABLE
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ SINGLE-SCALE vs MULTI-SCALE CNN COMPARISON")
    print(f"{'='*78}")

    sb = cfg["SINGLE_BASELINES"]
    print(f"\n  ┌{'─'*68}┐")
    print(f"  │ {'Metric':<24} │ {'Single-Scale':>13} │ "
          f"{'Multi-Scale':>12} │ {'Δ':>10} │")
    print(f"  ├{'─'*68}┤")

    rows = []
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        det = r.get("detection", {})
        mfe = r.get("mfe_mae", {})
        meta = r.get("meta_label", {})

        # AUC
        ms_auc = det.get("auc", 0)
        ss_auc = sb[f"{task}_auc"]
        rows.append((f"{task.title()} AUC", ss_auc, ms_auc))

        # MFE/MAE @24
        ms_ratio = 0
        if "24" in mfe and mfe["24"].get("ratio") is not None:
            ms_ratio = mfe["24"]["ratio"]
        ss_ratio = sb[f"{task}_mfe_mae_24"]
        rows.append((f"{task.title()} MFE/MAE @24", ss_ratio, ms_ratio))

        # LGB WR @0.5
        ms_wr = 0
        if meta:
            th_data = meta.get("0.5") or meta.get(0.5)
            if th_data and isinstance(th_data, dict):
                ms_wr = th_data.get("wr", 0)
        ss_wr = sb[f"{task}_lgb_wr_05"]
        rows.append((f"{task.title()} LGB WR @0.5", ss_wr, ms_wr))

    for name, ss, ms in rows:
        ss_s = f"{ss:.3f}" if ss < 1 else f"{ss*100:.1f}%"
        if ms == 0 and ss != 0:
            # Metric wasn't computed this run and not loaded from previous
            print(f"  │ {name:<24} │ {ss_s:>13} │ {'—':>12} │ {'—':>10} │")
        else:
            delta = ms - ss
            ms_s = f"{ms:.3f}" if ms < 1 else f"{ms*100:.1f}%"
            d_s = f"{delta:+.3f}" if abs(delta) < 1 else f"{delta*100:+.1f}pp"
            print(f"  │ {name:<24} │ {ss_s:>13} │ {ms_s:>12} │ {d_s:>10} │")

    print(f"  └{'─'*68}┘")

    # Conclusions
    print(f"\n  CONCLUSIONS:")
    for task in ["bottom", "top"]:
        r = all_results.get(task, {})
        det = r.get("detection", {})
        ms_auc = det.get("auc", 0)
        ss_auc = sb[f"{task}_auc"]

        if ms_auc == 0:
            print(f"    — {task.upper()}: AUC not available (L1 skipped or not run)")
        elif ms_auc > ss_auc + 0.01:
            print(f"    ✓ {task.upper()}: Multi-scale improves AUC "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")
        elif ms_auc > ss_auc - 0.01:
            print(f"    ~ {task.upper()}: Multi-scale AUC comparable "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")
        else:
            print(f"    ✗ {task.upper()}: Multi-scale AUC declined "
                  f"({ss_auc:.3f} → {ms_auc:.3f})")

        mfe = r.get("mfe_mae", {})
        if "24" in mfe and mfe["24"].get("ratio"):
            ratio = mfe["24"]["ratio"]
            ss_r = sb[f"{task}_mfe_mae_24"]
            if ratio > ss_r:
                print(f"    ✓ {task.upper()}: MFE/MAE improved "
                      f"({ss_r:.3f} → {ratio:.3f})")

    # Save
    json_path = os.path.join(cfg["OUTPUT_DIR"],
                             "multiscale_cnn_results.json")
    # Clean for JSON
    save = {}
    for task, r in all_results.items():
        save[task] = {
            "detection": r.get("detection", {}),
            "mfe_mae": r.get("mfe_mae", {}),
            "meta_label": r.get("meta_label", {}),
            "train_time": r.get("train_time", 0),
        }
    with open(json_path, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"\n  Saved: {json_path}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

"""
vol_prediction_v2.py  (FIXED — regression on log(RV))
=====================================================
Four-model volatility regime prediction comparison.
All LSTM models use REGRESSION on log(RV), NOT binary classification.
Direction is DERIVED: pred_rv > current_rv → "up".

Models:
  A: Original Vol LSTM (must reproduce ~71% DA)
  B: Multi-Scale Vol LSTM (1hour blocks + daily context)
  C: LightGBM (block features) — both cls & reg, pick best
  D: LightGBM (block + daily)  — both cls & reg, pick best

Usage:
    !pip install lightgbm --quiet
    !python vol_prediction_v2.py
"""

import os, time, json, warnings, math
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, precision_score,
                             recall_score, f1_score, r2_score,
                             mean_squared_error)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[WARN] lightgbm not installed — Models C/D skipped")

try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    HAS_ARCH = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────── CONFIG ───────────────────────────
# Model A matches original vol_prediction.py EXACTLY
CFG = {
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "VOL_DATA_DIR": "vol_data",
    "BLOCK_SIZE": 12,
    # Model A — original config
    "SEQ_LEN": 10,
    "HIDDEN": 32,
    "NUM_LAYERS": 1,
    "DROPOUT": 0.1,
    "LR": 5e-4,
    "WD": 1e-4,
    "BATCH": 64,
    "EPOCHS": 200,
    "PATIENCE": 30,
    "SCHED_PATIENCE": 10,
    # Model B
    "DAILY_LOOKBACK": 20,
    # Splits
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.85,  # train+val = 85%, test = 15%
}


# ═══════════════════════════════════════════════════════════════
#  UTILITIES
# ═══════════════════════════════════════════════════════════════

def load_features_csv(ticker, freq):
    p = os.path.join(CFG["FEATURES_DIR"], f"{ticker}_{freq}_features.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    df = pd.read_csv(p)
    for col in ["timestamp", "ts_event", "datetime", "date"]:
        if col in df.columns:
            df["timestamp"] = pd.to_datetime(df[col], utc=True)
            break
    if "timestamp" not in df.columns:
        first = df.columns[0]
        if first not in df.select_dtypes(include=[np.number]).columns:
            df["timestamp"] = pd.to_datetime(df[first], utc=True)
    return df.sort_values("timestamp").reset_index(drop=True)


def numeric_cols(df):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


# ═══════════════════════════════════════════════════════════════
#  DATA: Build blocks from 1hour features
# ═══════════════════════════════════════════════════════════════

def compute_blocks_from_1h(df_1h, block_size=12):
    """Build non-overlapping blocks. Returns DataFrame with rv, log_rv, etc."""
    close = df_1h["close"].values.astype(np.float64)
    ts = df_1h["timestamp"].values
    lr = np.concatenate([[0], np.diff(np.log(close + 1e-10))])
    n_blocks = len(close) // block_size
    rows = []
    for b in range(n_blocks):
        s, e = b * block_size, (b + 1) * block_size
        rv = np.std(lr[s:e]) * np.sqrt(block_size)
        rows.append({
            "block_idx": b, "bar_start": s, "bar_end": e,
            "rv": rv, "log_rv": np.log(rv + 1e-10),
            "ts_start": ts[s], "ts_end": ts[min(e - 1, len(close) - 1)],
        })
    return pd.DataFrame(rows)


def load_or_build_vol_data():
    """
    Try to load pre-split CSVs from vol_data/.
    If not found, build from 1hour features and save.
    Returns: dict[ticker] → {"blocks": df, "block_feats": array, "feat_names": list,
                              "train_idx", "val_idx", "test_idx"}
    """
    BS = CFG["BLOCK_SIZE"]
    os.makedirs(CFG["VOL_DATA_DIR"], exist_ok=True)
    all_data = {}

    # Check if pre-split CSVs exist
    first_ticker = CFG["TICKERS"][0]
    pre_split_exists = os.path.exists(
        os.path.join(CFG["VOL_DATA_DIR"], f"{first_ticker}_vol_train.csv"))

    for ticker in CFG["TICKERS"]:
        try:
            df_1h = load_features_csv(ticker, "1hour")
        except FileNotFoundError:
            print(f"    [SKIP] {ticker}: no 1hour data")
            continue

        fc = numeric_cols(df_1h)
        feat_data = np.nan_to_num(df_1h[fc].values.astype(np.float32), nan=0.0)

        # Build blocks
        dfb = compute_blocks_from_1h(df_1h, BS)
        nb = len(dfb)

        # Block-level mean features
        block_feats = np.zeros((nb, len(fc)), dtype=np.float32)
        for i in range(nb):
            s, e = dfb.loc[i, "bar_start"], dfb.loc[i, "bar_end"]
            block_feats[i] = np.mean(feat_data[s:e], axis=0)

        # Per-ticker chronological split: 70/15/15
        tr_end = int(nb * CFG["TRAIN_RATIO"])
        va_end = int(nb * CFG["VAL_RATIO"])

        # Save pre-split CSVs if they don't exist
        if not pre_split_exists:
            for split_name, s, e in [("train", 0, tr_end),
                                      ("val", tr_end, va_end),
                                      ("test", va_end, nb)]:
                sp = os.path.join(CFG["VOL_DATA_DIR"],
                                  f"{ticker}_vol_{split_name}.csv")
                dfb.iloc[s:e].to_csv(sp, index=False)

        all_data[ticker] = {
            "blocks": dfb,
            "block_feats": block_feats,
            "feat_names": fc,
            "train_end": tr_end,
            "val_end": va_end,
            "n_blocks": nb,
        }
        print(f"    {ticker}: {nb} blocks  "
              f"train={tr_end} val={va_end-tr_end} test={nb-va_end}  "
              f"feat={len(fc)}")

    if not pre_split_exists:
        print(f"  Saved pre-split CSVs to {CFG['VOL_DATA_DIR']}/")

    return all_data


# ═══════════════════════════════════════════════════════════════
#  DAILY DATA
# ═══════════════════════════════════════════════════════════════

def compute_daily_bars(df_1h):
    """Aggregate 1hour → daily with derived features."""
    df = df_1h.copy()
    df["_date"] = df["timestamp"].dt.date

    has_vol = "volume" in df.columns
    agg = {"open": ("open", "first"), "high": ("high", "max"),
           "low": ("low", "min"), "close": ("close", "last"),
           "n_bars": ("close", "count")}
    if has_vol:
        agg["volume"] = ("volume", "sum")
    daily = df.groupby("_date").agg(**agg).reset_index()
    daily.rename(columns={"_date": "date"}, inplace=True)

    c = daily["close"].values.astype(np.float64)
    h = daily["high"].values.astype(np.float64)
    lo = daily["low"].values.astype(np.float64)
    o = daily["open"].values.astype(np.float64)
    n = len(daily)

    daily["daily_return"] = (c - o) / (o + 1e-10)
    daily["daily_range"] = (h - lo) / (c + 1e-10)

    if has_vol:
        v = daily["volume"].values.astype(np.float64)
        daily["vol_change"] = np.concatenate([[1.0], v[1:] / (v[:-1] + 1e-10)])
    else:
        daily["vol_change"] = 1.0

    # Intraday RV per day
    drv = []
    for _, grp in df.groupby("_date"):
        cc = grp["close"].values.astype(np.float64)
        if len(cc) > 1:
            lr = np.diff(np.log(cc + 1e-10))
            drv.append(np.std(lr) * np.sqrt(len(cc)))
        else:
            drv.append(0.0)
    daily["daily_rv"] = drv

    # RSI-14
    delta = np.diff(c, prepend=c[0])
    gain = np.where(delta > 0, delta, 0.0)
    loss_arr = np.where(delta < 0, -delta, 0.0)
    ag = np.zeros(n); al = np.zeros(n)
    if n > 14:
        ag[14] = np.mean(gain[1:15]); al[14] = np.mean(loss_arr[1:15])
        for i in range(15, n):
            ag[i] = (ag[i-1]*13 + gain[i]) / 14
            al[i] = (al[i-1]*13 + loss_arr[i]) / 14
    rs = ag / (al + 1e-10)
    daily["rsi_14"] = 100 - 100 / (1 + rs)

    # ATR-14
    tr = np.zeros(n)
    for i in range(1, n):
        tr[i] = max(h[i]-lo[i], abs(h[i]-c[i-1]), abs(lo[i]-c[i-1]))
    tr[0] = h[0] - lo[0]
    atr = np.zeros(n)
    if n > 14:
        atr[14] = np.mean(tr[1:15])
        for i in range(15, n):
            atr[i] = (atr[i-1]*13 + tr[i]) / 14
    daily["atr_14"] = atr

    # BB bandwidth 20
    bb = np.zeros(n)
    for i in range(20, n):
        sma = np.mean(c[i-20:i]); std = np.std(c[i-20:i])
        bb[i] = (4 * std) / (sma + 1e-10)
    daily["bb_bandwidth_20"] = bb
    daily["log_close"] = np.log(c + 1e-10)

    return daily


# ═══════════════════════════════════════════════════════════════
#  SEQUENCE BUILDERS (REGRESSION targets)
# ═══════════════════════════════════════════════════════════════

def build_lstm_sequences(all_data, seq_len):
    """
    Build sequences for LSTM regression.
    Input: [block_feats | log_rv] for seq_len blocks.
    Target: log(rv) of the NEXT block.
    Also store current_rv for direction derivation.
    """
    X_all, y_all, cur_rv_all, meta_all = [], [], [], []
    n_feat = None

    for ticker, d in all_data.items():
        dfb = d["blocks"]
        bf = d["block_feats"]
        nb = d["n_blocks"]
        log_rvs = dfb["log_rv"].values.astype(np.float32)
        raw_rvs = dfb["rv"].values.astype(np.float64)

        if n_feat is None:
            n_feat = bf.shape[1] + 1  # +1 for log_rv
            print(f"  LSTM input dim: {bf.shape[1]} numeric + 1 log_rv = {n_feat}")

        # Append log_rv as extra column
        bf_ext = np.column_stack([bf, log_rvs.reshape(-1, 1)])

        for i in range(seq_len, nb - 1):
            # Input: blocks [i-seq_len .. i-1]
            seq = bf_ext[i - seq_len:i]
            # Target: log(rv) of block i
            target = log_rvs[i]
            # Current rv (block i-1) for direction derivation
            cur = raw_rvs[i - 1]

            X_all.append(seq)
            y_all.append(target)
            cur_rv_all.append(cur)
            meta_all.append((ticker, i))

    X = np.array(X_all, dtype=np.float32)
    y = np.array(y_all, dtype=np.float32)
    cur = np.array(cur_rv_all, dtype=np.float64)
    return X, y, cur, meta_all, n_feat


def build_multiscale_sequences(all_data, all_daily, seq_len, daily_lb):
    """Build paired sequences for Model B (short + long branch, regression)."""
    Xs_all, Xl_all, y_all, cur_all, meta_all = [], [], [], [], []
    n_sfeat = None; n_dfeat = None

    dcols = ["daily_rv", "daily_range", "daily_return", "vol_change",
             "rsi_14", "atr_14", "bb_bandwidth_20", "log_close"]

    for ticker, d in all_data.items():
        dfb = d["blocks"]
        bf = d["block_feats"]
        nb = d["n_blocks"]
        log_rvs = dfb["log_rv"].values.astype(np.float32)
        raw_rvs = dfb["rv"].values.astype(np.float64)

        bf_ext = np.column_stack([bf, log_rvs.reshape(-1, 1)])
        if n_sfeat is None:
            n_sfeat = bf_ext.shape[1]

        daily = all_daily.get(ticker)
        if daily is None:
            continue
        dc_valid = [c for c in dcols if c in daily.columns]
        if n_dfeat is None:
            n_dfeat = len(dc_valid)
        df_vals = np.nan_to_num(daily[dc_valid].values.astype(np.float32), nan=0.0)
        ddates = daily["date"].values

        for i in range(seq_len, nb - 1):
            short_seq = bf_ext[i - seq_len:i]
            target = log_rvs[i]
            cur_rv = raw_rvs[i - 1]

            # Align to daily
            block_end = pd.Timestamp(dfb.loc[i, "ts_end"])
            bd = block_end.date() if hasattr(block_end, 'date') else block_end
            day_idx = -1
            for di in range(len(ddates) - 1, -1, -1):
                if ddates[di] <= bd:
                    day_idx = di; break
            if day_idx < daily_lb:
                continue
            long_seq = df_vals[day_idx - daily_lb + 1:day_idx + 1]
            if len(long_seq) != daily_lb:
                continue

            Xs_all.append(short_seq)
            Xl_all.append(long_seq)
            y_all.append(target)
            cur_all.append(cur_rv)
            meta_all.append((ticker, i))

    return (np.array(Xs_all, dtype=np.float32),
            np.array(Xl_all, dtype=np.float32),
            np.array(y_all, dtype=np.float32),
            np.array(cur_all, dtype=np.float64),
            meta_all, n_sfeat, n_dfeat)


def build_lgb_features(all_data, all_daily, include_daily=False):
    """Flat features for LightGBM. Returns X, y_reg (log_rv), y_cls (direction), cur_rv, meta, fnames."""
    SL = CFG["SEQ_LEN"]
    DAILY_LB = 5

    X_rows, y_reg, y_cls, cur_rv_all, meta = [], [], [], [], []
    key_cols_names = ["close", "volume", "atr_14", "rsi_14"]

    for ticker, d in all_data.items():
        dfb = d["blocks"]
        bf = d["block_feats"]
        fc = d["feat_names"]
        nb = d["n_blocks"]
        rvs = dfb["rv"].values.astype(np.float64)
        log_rvs = dfb["log_rv"].values.astype(np.float64)

        fc_list = list(fc)
        key_idxs = {}
        for kc in key_cols_names:
            matches = [j for j, c in enumerate(fc_list) if kc.lower() in c.lower()]
            if matches:
                key_idxs[kc] = matches[0]

        daily = all_daily.get(ticker) if include_daily else None
        daily_cols = ["daily_rv", "daily_range", "daily_return", "vol_change"]
        if daily is not None:
            dc_valid = [c for c in daily_cols if c in daily.columns]
            df_vals = np.nan_to_num(daily[dc_valid].values.astype(np.float32), nan=0.0)
            ddates = daily["date"].values
            n_dc = len(dc_valid)
        else:
            n_dc = 0

        spy_data = all_data.get("SPY")
        spy_rvs = spy_data["blocks"]["rv"].values if spy_data else None

        for i in range(SL, nb - 1):
            feats = {}
            # A) Per-block features (last SL blocks)
            for off in range(SL):
                bi = i - SL + off
                prefix = f"b{off}"
                feats[f"{prefix}_rv"] = rvs[bi]
                feats[f"{prefix}_log_rv"] = log_rvs[bi]
                for kc, ki in key_idxs.items():
                    feats[f"{prefix}_{kc}"] = float(bf[bi, ki])

            # B) RV rolling stats
            rv_win = rvs[max(0, i-SL):i]
            rv4 = rvs[max(0, i-4):i]
            feats["rv_mean_4"] = np.mean(rv4) if len(rv4) else 0
            feats["rv_std_4"] = np.std(rv4) if len(rv4) > 1 else 0
            feats["rv_mean_8"] = np.mean(rv_win) if len(rv_win) else 0
            feats["rv_std_8"] = np.std(rv_win) if len(rv_win) > 1 else 0
            feats["rv_ratio"] = rvs[i-1] / (feats["rv_mean_8"] + 1e-10)
            if len(rv_win) >= 3:
                feats["rv_trend"] = float(np.polyfit(np.arange(len(rv_win)), rv_win, 1)[0])
            else:
                feats["rv_trend"] = 0.0
            streak_up = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] > rvs[k-1]: streak_up += 1
                else: break
            feats["rv_streak_up"] = streak_up
            streak_dn = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] < rvs[k-1]: streak_dn += 1
                else: break
            feats["rv_streak_down"] = streak_dn
            feats["current_log_rv"] = log_rvs[i-1]

            # C) Current block tech indicators
            for j, cn in enumerate(fc_list):
                feats[f"tech_{cn}"] = float(bf[i-1, j])

            # D) SPY cross-asset
            if ticker != "SPY" and spy_rvs is not None and i < len(spy_rvs):
                feats["spy_rv_cur"] = spy_rvs[max(0, i-1)]
                spy4 = spy_rvs[max(0, i-4):i]
                feats["spy_rv_mean_4"] = np.mean(spy4) if len(spy4) else 0
            else:
                feats["spy_rv_cur"] = 0; feats["spy_rv_mean_4"] = 0

            # E) Daily (Model D)
            if include_daily and daily is not None:
                bets = pd.Timestamp(dfb.loc[i, "ts_end"])
                bd = bets.date() if hasattr(bets, 'date') else bets
                day_idx = -1
                for di in range(len(ddates)-1, -1, -1):
                    if ddates[di] <= bd: day_idx = di; break
                if day_idx >= DAILY_LB:
                    for d_off in range(DAILY_LB):
                        di2 = day_idx - DAILY_LB + 1 + d_off
                        for ci, cn in enumerate(dc_valid[:n_dc]):
                            feats[f"d{d_off}_{cn}"] = float(df_vals[di2, ci])
                    drv_w = df_vals[day_idx-DAILY_LB+1:day_idx+1, 0]
                    feats["daily_rv_mean_5"] = float(np.mean(drv_w))
                    feats["daily_rv_std_5"] = float(np.std(drv_w))
                    if len(drv_w) >= 3:
                        feats["daily_rv_trend"] = float(
                            np.polyfit(np.arange(len(drv_w)), drv_w, 1)[0])
                    else:
                        feats["daily_rv_trend"] = 0.0

            X_rows.append(feats)
            y_reg.append(log_rvs[i])  # regression target
            # classification target
            y_cls.append(1 if rvs[i] > rvs[i-1] else 0)
            cur_rv_all.append(rvs[i-1])
            meta.append((ticker, i))

    if not X_rows:
        return np.array([]), np.array([]), np.array([]), np.array([]), meta, []
    fnames = list(X_rows[0].keys())
    X = np.array([[r.get(fn, 0.0) for fn in fnames] for r in X_rows], dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return (X, np.array(y_reg, dtype=np.float32),
            np.array(y_cls, dtype=np.float32),
            np.array(cur_rv_all, dtype=np.float64), meta, fnames)


# ═══════════════════════════════════════════════════════════════
#  MODELS
# ═══════════════════════════════════════════════════════════════

class VolLSTM_A(nn.Module):
    """Model A: Original Vol LSTM — REGRESSION on log(RV).
    LSTM(hidden=32, layers=1) → Linear(32,32) → ReLU → Dropout(0.1) → Linear(32,1)
    """
    def __init__(self, n_feat):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, 32, 1, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class VolLSTM_B(nn.Module):
    """Model B: Multi-Scale Vol LSTM — REGRESSION on log(RV).
    Short: LSTM(32,1) → 32-dim.  Long: LSTM(32,1) → 32-dim.
    Fusion: concat → Linear(64,32) → ReLU → Dropout(0.1) → Linear(32,1).
    """
    def __init__(self, n_sfeat, n_dfeat):
        super().__init__()
        self.short_lstm = nn.LSTM(n_sfeat, 32, 1, batch_first=True)
        self.long_lstm = nn.LSTM(n_dfeat, 32, 1, batch_first=True)
        self.fuse = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1),
        )
    def forward(self, xs, xl):
        so, _ = self.short_lstm(xs)
        lo, _ = self.long_lstm(xl)
        return self.fuse(torch.cat([so[:, -1, :], lo[:, -1, :]], dim=1)).squeeze(-1)


class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class PairDS(Dataset):
    def __init__(self, Xs, Xl, y):
        self.Xs = torch.tensor(Xs, dtype=torch.float32)
        self.Xl = torch.tensor(Xl, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.Xs[i], self.Xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  SPLIT HELPERS
# ═══════════════════════════════════════════════════════════════

def split3(n):
    """Return (train_end, val_end) indices."""
    return int(n * CFG["TRAIN_RATIO"]), int(n * CFG["VAL_RATIO"])


def normalize_from_train(X, tr_end):
    """Normalize using training set statistics only. Handles 2D or 3D."""
    if X.ndim == 3:
        flat = X[:tr_end].reshape(-1, X.shape[2])
    else:
        flat = X[:tr_end]
    mu = flat.mean(0); sig = flat.std(0) + 1e-8
    X_out = (X - mu) / sig
    return np.nan_to_num(X_out, nan=0.0, posinf=0.0, neginf=0.0)


# ═══════════════════════════════════════════════════════════════
#  TRAINING
# ═══════════════════════════════════════════════════════════════

def train_regression_lstm(model, X_tr, y_tr, X_va, y_va, tag="A"):
    """Train LSTM with MSE loss on log(RV)."""
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=CFG["SCHED_PATIENCE"], factor=0.5)

    tr_dl = DataLoader(SeqDS(X_tr, y_tr), batch_size=CFG["BATCH"], shuffle=True)
    va_dl = DataLoader(SeqDS(X_va, y_va), batch_size=512)

    best_vl = float("inf"); best_st = None; pat = 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); tl = 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
        tl /= len(y_tr)

        model.eval(); vl = 0
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xb), yb).item() * len(yb)
        vl /= len(y_va)
        sched.step(vl)

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0; best_ep = ep + 1
        else:
            pat += 1
        if (ep+1) % 40 == 0 or pat == CFG["PATIENCE"]:
            print(f"    [{tag}] E{ep+1:3d} t={tl:.6f} v={vl:.6f} p={pat}")
        if pat >= CFG["PATIENCE"]:
            print(f"    [{tag}] Early stop (best={best_ep})")
            break

    if best_st:
        model.load_state_dict(best_st)
    model.to(DEVICE).eval()
    return model


def predict_lstm(model, X):
    dl = DataLoader(SeqDS(X, np.zeros(len(X))), batch_size=512)
    preds = []
    with torch.no_grad():
        for xb, _ in dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)


def train_regression_lstm_b(model, Xs_tr, Xl_tr, y_tr, Xs_va, Xl_va, y_va):
    """Train Model B with MSE loss."""
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=CFG["SCHED_PATIENCE"], factor=0.5)

    tr_dl = DataLoader(PairDS(Xs_tr, Xl_tr, y_tr), batch_size=CFG["BATCH"], shuffle=True)
    va_dl = DataLoader(PairDS(Xs_va, Xl_va, y_va), batch_size=512)

    best_vl = float("inf"); best_st = None; pat = 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); tl = 0
        for xs, xl, yb in tr_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xs, xl), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
        tl /= len(y_tr)

        model.eval(); vl = 0
        with torch.no_grad():
            for xs, xl, yb in va_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xs, xl), yb).item() * len(yb)
        vl /= len(y_va)
        sched.step(vl)

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0; best_ep = ep + 1
        else:
            pat += 1
        if (ep+1) % 40 == 0 or pat == CFG["PATIENCE"]:
            print(f"    [B] E{ep+1:3d} t={tl:.6f} v={vl:.6f} p={pat}")
        if pat >= CFG["PATIENCE"]:
            print(f"    [B] Early stop (best={best_ep})")
            break

    if best_st:
        model.load_state_dict(best_st)
    model.to(DEVICE).eval()
    return model


def predict_lstm_b(model, Xs, Xl):
    dl = DataLoader(PairDS(Xs, Xl, np.zeros(len(Xs))), batch_size=512)
    preds = []
    with torch.no_grad():
        for xs, xl, _ in dl:
            preds.append(model(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)


# ═══════════════════════════════════════════════════════════════
#  EVALUATION
# ═══════════════════════════════════════════════════════════════

def eval_regression(pred_log, true_log, cur_rv, meta_te, name):
    """
    Evaluate regression model.
    Direction: pred_rv > cur_rv → 1.  True direction: true_rv > cur_rv → 1.
    """
    pred_rv = np.exp(pred_log)
    true_rv = np.exp(true_log)

    pred_dir = (pred_rv > cur_rv).astype(int)
    true_dir = (true_rv > cur_rv).astype(int)

    da = (pred_dir == true_dir).mean()
    r2 = r2_score(true_rv, pred_rv)
    rmse = np.sqrt(mean_squared_error(true_rv, pred_rv))
    n = len(true_dir)

    # z-score
    se = np.sqrt(0.5 * 0.5 / n)
    z = (da - 0.5) / se
    p_val = 2 * (1 - __import__('scipy').stats.norm.cdf(abs(z))) if abs(z) > 0 else 1.0

    # F1 for high-vol class
    f1 = f1_score(true_dir, pred_dir, zero_division=0)

    result = {"name": name, "da": float(da), "r2": float(r2),
              "rmse": float(rmse), "z": float(z), "p": float(p_val),
              "f1": float(f1), "n": n}

    # Per-ticker
    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                pt[ticker] = {
                    "da": float((pred_dir[mask] == true_dir[mask]).mean()),
                    "n": int(mask.sum()),
                }
        result["per_ticker"] = pt

    return result


def eval_classification(pred_prob, true_dir, meta_te, name):
    """Evaluate binary classifier."""
    preds = (pred_prob >= 0.5).astype(int)
    da = (preds == true_dir).mean()
    auc = roc_auc_score(true_dir, pred_prob) if true_dir.sum() > 0 and (1-true_dir).sum() > 0 else 0.5
    f1 = f1_score(true_dir, preds, zero_division=0)
    n = len(true_dir)
    z = (da - 0.5) / np.sqrt(0.5*0.5/n)

    result = {"name": name, "da": float(da), "r2": float(0),
              "rmse": float(0), "z": float(z), "f1": float(f1), "n": n}
    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                pt[ticker] = {"da": float((preds[mask] == true_dir[mask]).mean()),
                              "n": int(mask.sum())}
        result["per_ticker"] = pt
    return result


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("  Volatility Prediction v2 — REGRESSION on log(RV)")
    print(f"  Device: {DEVICE}")
    print("=" * 70)
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    os.makedirs(CFG["VOL_DATA_DIR"], exist_ok=True)
    t_start = time.time()

    # ── Load / build data ──
    print("\n  Loading data...")
    all_data = load_or_build_vol_data()
    all_daily = {}
    for ticker in CFG["TICKERS"]:
        try:
            df_1h = load_features_csv(ticker, "1hour")
            all_daily[ticker] = compute_daily_bars(df_1h)
        except FileNotFoundError:
            pass

    # ══════════════════════════════════════════════════════
    #  MODEL A: Original Vol LSTM (regression)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  MODEL A: Original Vol LSTM (regression on log(RV))")
    print("=" * 70)

    t0 = time.time()
    X_a, y_a, cur_a, meta_a, n_feat_a = build_lstm_sequences(all_data, CFG["SEQ_LEN"])
    print(f"  Samples: {len(y_a):,}  features: {n_feat_a}")

    tr_a, va_a = split3(len(y_a))
    X_a_n = normalize_from_train(X_a, tr_a)

    model_a = VolLSTM_A(n_feat_a).to(DEVICE)
    print(f"  Architecture: LSTM({n_feat_a}, h=32, L=1) → Linear(32,32) "
          f"→ ReLU → Drop(0.1) → Linear(32,1)")
    print(f"  Loss: MSE on log(RV)   LR={CFG['LR']}   "
          f"seq_len={CFG['SEQ_LEN']}   epochs={CFG['EPOCHS']}")

    model_a = train_regression_lstm(
        model_a, X_a_n[:tr_a], y_a[:tr_a],
        X_a_n[tr_a:va_a], y_a[tr_a:va_a], tag="A")

    pred_a = predict_lstm(model_a, X_a_n[va_a:])
    res_a = eval_regression(pred_a, y_a[va_a:], cur_a[va_a:],
                            meta_a[va_a:], "A: Original LSTM")
    print(f"\n  Model A Test: DA={res_a['da']*100:.1f}%  R²={res_a['r2']:.3f}  "
          f"RMSE={res_a['rmse']:.6f}  z={res_a['z']:+.2f}  ({time.time()-t0:.1f}s)")

    if abs(res_a['da'] - 0.713) > 0.015:
        print(f"  ⚠ WARNING: DA={res_a['da']*100:.1f}%, expected ~71.3%")

    # Save Model A
    sp = os.path.join(CFG["RESULTS_DIR"], "vol_lstm_regression.pt")
    torch.save(model_a.state_dict(), sp)
    print(f"  Saved: {sp}")

    # ══════════════════════════════════════════════════════
    #  MODEL B: Multi-Scale Vol LSTM (regression)
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  MODEL B: Multi-Scale Vol LSTM (regression)")
    print("=" * 70)

    t0 = time.time()
    Xs_b, Xl_b, y_b, cur_b, meta_b, n_sf, n_df = build_multiscale_sequences(
        all_data, all_daily, CFG["SEQ_LEN"], CFG["DAILY_LOOKBACK"])
    print(f"  Samples: {len(y_b):,}  short_feat={n_sf}  daily_feat={n_df}")

    res_b = None
    if len(y_b) > 100:
        tr_b, va_b = split3(len(y_b))
        Xs_b_n = normalize_from_train(Xs_b, tr_b)
        Xl_b_n = normalize_from_train(Xl_b, tr_b)

        model_b = VolLSTM_B(n_sf, n_df).to(DEVICE)
        model_b = train_regression_lstm_b(
            model_b, Xs_b_n[:tr_b], Xl_b_n[:tr_b], y_b[:tr_b],
            Xs_b_n[tr_b:va_b], Xl_b_n[tr_b:va_b], y_b[tr_b:va_b])

        pred_b = predict_lstm_b(model_b, Xs_b_n[va_b:], Xl_b_n[va_b:])
        res_b = eval_regression(pred_b, y_b[va_b:], cur_b[va_b:],
                                meta_b[va_b:], "B: Multi-Scale LSTM")
        print(f"\n  Model B Test: DA={res_b['da']*100:.1f}%  R²={res_b['r2']:.3f}  "
              f"RMSE={res_b['rmse']:.6f}  ({time.time()-t0:.1f}s)")

        sp2 = os.path.join(CFG["RESULTS_DIR"], "vol_multiscale_lstm_regression.pt")
        torch.save(model_b.state_dict(), sp2)
        print(f"  Saved: {sp2}")
    else:
        print("  [SKIP] Insufficient daily-aligned samples")

    # ══════════════════════════════════════════════════════
    #  NAIVE BASELINES
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  NAIVE BASELINES")
    print("=" * 70)

    te_true_log = y_a[va_a:]
    te_cur_rv = cur_a[va_a:]
    te_true_rv = np.exp(te_true_log)
    te_true_dir = (te_true_rv > te_cur_rv).astype(int)
    n_te = len(te_true_dir)

    # Naive RW: predict next_rv = current_rv → always "same" → direction = 0
    naive_rw_pred = te_cur_rv.copy()
    naive_rw_dir = np.zeros(n_te, dtype=int)  # pred = cur → pred_dir always 0
    naive_rw_da = (naive_rw_dir == te_true_dir).mean()
    naive_rw_r2 = r2_score(te_true_rv, naive_rw_pred)
    naive_rw_rmse = np.sqrt(mean_squared_error(te_true_rv, naive_rw_pred))

    # Historical mean: predict next_rv = mean(train_rv)
    train_rvs = np.exp(y_a[:tr_a])
    hist_mean = train_rvs.mean()
    hist_pred = np.full(n_te, hist_mean)
    hist_dir = (hist_mean > te_cur_rv).astype(int)
    hist_da = (hist_dir == te_true_dir).mean()
    hist_r2 = r2_score(te_true_rv, hist_pred)
    hist_rmse = np.sqrt(mean_squared_error(te_true_rv, hist_pred))

    print(f"  Naive (RW — predict current):   DA={naive_rw_da*100:.1f}%  "
          f"R²={naive_rw_r2:.3f}  RMSE={naive_rw_rmse:.6f}")
    print(f"  Historical Mean (={hist_mean:.6f}):  DA={hist_da*100:.1f}%  "
          f"R²={hist_r2:.3f}  RMSE={hist_rmse:.6f}")

    # GARCH
    garch_res = None
    if HAS_ARCH:
        print("  Fitting GARCH(1,1)...")
        try:
            train_rvs_ts = np.exp(y_a[:tr_a]) * 10000  # scale for GARCH
            am = arch_model(train_rvs_ts, vol='GARCH', p=1, q=1, mean='Constant')
            garch_fit = am.fit(disp='off')
            garch_forecasts = []
            all_rvs_scaled = np.exp(y_a) * 10000
            for i in range(va_a, len(y_a)):
                window = all_rvs_scaled[:i]
                am_i = arch_model(window, vol='GARCH', p=1, q=1, mean='Constant')
                fit_i = am_i.fit(disp='off', last_obs=len(window))
                fc = fit_i.forecast(horizon=1)
                garch_forecasts.append(fc.variance.values[-1, 0])
            garch_pred_rv = np.sqrt(np.array(garch_forecasts)) / 10000
            garch_dir = (garch_pred_rv > te_cur_rv).astype(int)
            garch_da = (garch_dir == te_true_dir).mean()
            garch_r2 = r2_score(te_true_rv, garch_pred_rv)
            garch_rmse = np.sqrt(mean_squared_error(te_true_rv, garch_pred_rv))
            garch_res = {"da": float(garch_da), "r2": float(garch_r2),
                         "rmse": float(garch_rmse)}
            print(f"  GARCH(1,1):   DA={garch_da*100:.1f}%  "
                  f"R²={garch_r2:.3f}  RMSE={garch_rmse:.6f}")
        except Exception as e:
            print(f"  GARCH failed: {e}")
    else:
        print("  [SKIP] GARCH — arch package not installed")

    # ══════════════════════════════════════════════════════
    #  MODELS C & D: LightGBM
    # ══════════════════════════════════════════════════════
    res_c = None; res_d = None
    mdl_c = None; mdl_d = None; fn_c = []; fn_d = []

    if HAS_LGB:
        for tag, include_daily in [("C", False), ("D", True)]:
            print(f"\n{'='*70}")
            print(f"  MODEL {tag}: LightGBM ({'block+daily' if include_daily else 'block'})")
            print(f"{'='*70}")

            t0 = time.time()
            X_lg, y_reg_lg, y_cls_lg, cur_lg, meta_lg, fnames_lg = build_lgb_features(
                all_data, all_daily, include_daily=include_daily)
            print(f"  Samples: {len(y_cls_lg):,}  features: {len(fnames_lg)}")

            if len(X_lg) == 0:
                continue

            tr_lg, va_lg = split3(len(y_cls_lg))
            meta_te_lg = meta_lg[va_lg:]

            # ── Classification approach ──
            mdl_cls = lgb.LGBMClassifier(
                n_estimators=1000, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
                reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
            mdl_cls.fit(X_lg[:tr_lg], y_cls_lg[:tr_lg],
                        eval_set=[(X_lg[tr_lg:va_lg], y_cls_lg[tr_lg:va_lg])],
                        callbacks=[lgb.early_stopping(50, verbose=False),
                                   lgb.log_evaluation(0)])
            cls_probs = mdl_cls.predict_proba(X_lg[va_lg:])[:, 1]
            cls_preds = (cls_probs >= 0.5).astype(int)
            cls_da = (cls_preds == y_cls_lg[va_lg:]).mean()

            # ── Regression approach ──
            mdl_reg = lgb.LGBMRegressor(
                n_estimators=1000, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
                reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
            mdl_reg.fit(X_lg[:tr_lg], y_reg_lg[:tr_lg],
                        eval_set=[(X_lg[tr_lg:va_lg], y_reg_lg[tr_lg:va_lg])],
                        callbacks=[lgb.early_stopping(50, verbose=False),
                                   lgb.log_evaluation(0)])
            reg_pred_log = mdl_reg.predict(X_lg[va_lg:])
            reg_pred_rv = np.exp(reg_pred_log)
            reg_dir = (reg_pred_rv > cur_lg[va_lg:]).astype(int)
            te_dir = y_cls_lg[va_lg:].astype(int)
            reg_da = (reg_dir == te_dir).mean()

            print(f"  Classification DA: {cls_da*100:.1f}%")
            print(f"  Regression DA:     {reg_da*100:.1f}%")

            # Pick the better one
            if reg_da >= cls_da:
                print(f"  → Using REGRESSION approach")
                res_lg = eval_regression(reg_pred_log,
                                         y_reg_lg[va_lg:], cur_lg[va_lg:],
                                         meta_te_lg,
                                         f"{tag}: LightGBM ({'block+daily' if include_daily else 'block'}) [reg]")
                best_mdl = mdl_reg
            else:
                print(f"  → Using CLASSIFICATION approach")
                res_lg = eval_classification(cls_probs, te_dir, meta_te_lg,
                                             f"{tag}: LightGBM ({'block+daily' if include_daily else 'block'}) [cls]")
                best_mdl = mdl_cls

            print(f"  Best DA: {res_lg['da']*100:.1f}%  ({time.time()-t0:.1f}s)")

            if tag == "C":
                res_c = res_lg; mdl_c = best_mdl; fn_c = fnames_lg
            else:
                res_d = res_lg; mdl_d = best_mdl; fn_d = fnames_lg

    # ══════════════════════════════════════════════════════
    #  COMPARISON TABLE
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ VOLATILITY PREDICTION COMPARISON (regression on log(RV))")
    print(f"{'='*70}")

    all_results = {}

    # Naive baselines
    all_results["Naive (RW)"] = {
        "name": "Naive (RW)", "da": float(naive_rw_da),
        "r2": float(naive_rw_r2), "rmse": float(naive_rw_rmse),
        "f1": 0, "n": n_te,
    }
    all_results["Historical Mean"] = {
        "name": "Historical Mean", "da": float(hist_da),
        "r2": float(hist_r2), "rmse": float(hist_rmse),
        "f1": 0, "n": n_te,
    }
    if garch_res:
        all_results["GARCH(1,1)"] = {
            "name": "GARCH(1,1)", "da": garch_res["da"],
            "r2": garch_res["r2"], "rmse": garch_res["rmse"],
            "f1": 0, "n": n_te,
        }

    all_results["A: Original LSTM"] = res_a
    if res_b:
        all_results["B: Multi-Scale LSTM"] = res_b
    if res_c:
        all_results[res_c["name"]] = res_c
    if res_d:
        all_results[res_d["name"]] = res_d

    print(f"\n  ┌{'─'*71}┐")
    print(f"  │ {'Model':<36} │ {'DA':>7} │ {'R²':>7} │ "
          f"{'RMSE':>8} │ {'N':>5} │")
    print(f"  ├{'─'*71}┤")

    best_da = 0; best_name = ""
    for name, r in all_results.items():
        da = r["da"]; r2 = r.get("r2", 0); rmse = r.get("rmse", 0); n = r.get("n", 0)
        if da > best_da and "Naive" not in name and "Historical" not in name:
            best_da = da; best_name = name
        r2_s = f"{r2:>7.3f}" if r2 != 0 else "    —"
        rmse_s = f"{rmse:>8.6f}" if rmse != 0 else "      —"
        marker = " ★" if name == best_name else ""
        print(f"  │ {name:<36} │ {da*100:>6.1f}% │ {r2_s} │ "
              f"{rmse_s} │ {n:>5} │{marker}")
    print(f"  └{'─'*71}┘")

    # ══════════════════════════════════════════════════════
    #  PER-TICKER (best model)
    # ══════════════════════════════════════════════════════
    best_r = all_results.get(best_name, {})
    pt = best_r.get("per_ticker", {})
    if pt:
        print(f"\n  Per-Ticker ({best_name}):")
        print(f"  {'Ticker':<8} {'DA':>8} {'N':>6}")
        print(f"  {'-'*24}")
        das = []
        for ticker in CFG["TICKERS"]:
            td = pt.get(ticker, {})
            if td:
                print(f"  {ticker:<8} {td['da']*100:>7.1f}% {td['n']:>6}")
                das.append(td['da'])
        if das:
            print(f"  {'Average':<8} {np.mean(das)*100:>7.1f}%")

    # ══════════════════════════════════════════════════════
    #  FEATURE IMPORTANCE (best LightGBM)
    # ══════════════════════════════════════════════════════
    best_lgb = mdl_d if mdl_d else mdl_c
    best_lgb_fn = fn_d if mdl_d else fn_c
    best_lgb_tag = "D" if mdl_d else "C"

    if best_lgb is not None:
        imp = best_lgb.feature_importances_
        fi = sorted(zip(best_lgb_fn, imp), key=lambda x: -x[1])

        print(f"\n  Feature Importance (Model {best_lgb_tag} — top 15):")
        print(f"  {'#':>4} {'Feature':<40} {'Imp':>8}")
        print(f"  {'-'*55}")
        for rank, (fn, fi_v) in enumerate(fi[:15], 1):
            cat = ""
            if "rv" in fn.lower() and not fn.startswith("tech_"):
                cat = " [RV]"
            elif fn.startswith("d") and fn[1:2].isdigit():
                cat = " [daily]"
            elif fn.startswith("tech_"):
                cat = " [tech]"
            elif fn.startswith("b") and fn[1:2].isdigit():
                cat = " [block]"
            elif "spy" in fn.lower():
                cat = " [cross]"
            print(f"  {rank:>4} {fn:<40} {fi_v:>8}{cat}")

        # Category totals
        cats = {"RV sequence": 0, "Block per-bar": 0, "Tech indicators": 0,
                "Daily": 0, "Cross-asset": 0, "Other": 0}
        for fn, fi_v in fi:
            if ("rv" in fn.lower() or fn.startswith("rv_")) and not fn.startswith("tech_"):
                cats["RV sequence"] += fi_v
            elif fn.startswith("b") and fn[1:2].isdigit() and "rv" not in fn:
                cats["Block per-bar"] += fi_v
            elif fn.startswith("tech_"):
                cats["Tech indicators"] += fi_v
            elif fn.startswith("d") and fn[1:2].isdigit():
                cats["Daily"] += fi_v
            elif "spy" in fn.lower():
                cats["Cross-asset"] += fi_v
            else:
                cats["Other"] += fi_v
        total = sum(cats.values()) + 1e-10
        print(f"\n  Category breakdown:")
        for cat, val in sorted(cats.items(), key=lambda x: -x[1]):
            if val > 0:
                print(f"    {cat:<20} {val/total*100:>5.1f}%")

    # ══════════════════════════════════════════════════════
    #  ANALYSIS
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ ANALYSIS")
    print(f"{'='*70}")

    print(f"\n  Naive RW baseline:       DA={naive_rw_da*100:.1f}%")
    print(f"  Historical Mean baseline: DA={hist_da*100:.1f}%  ← TRUE baseline (mean reversion)")
    if garch_res:
        print(f"  GARCH(1,1):              DA={garch_res['da']*100:.1f}%")

    da_a = res_a["da"]
    print(f"\n  Model A (original LSTM): DA={da_a*100:.1f}%")
    print(f"    vs Historical Mean: {(da_a - hist_da)*100:+.1f}pp")
    print(f"    vs Naive RW:        {(da_a - naive_rw_da)*100:+.1f}pp")

    if res_b:
        da_b = res_b["da"]
        print(f"\n  Model B (multi-scale):  DA={da_b*100:.1f}%")
        print(f"    vs Model A: {(da_b - da_a)*100:+.1f}pp")
        if da_b > da_a + 0.01:
            print(f"    → Daily context improves vol prediction ✓")
        elif da_b > da_a - 0.01:
            print(f"    → Comparable to single-scale baseline")
        else:
            print(f"    → Multi-scale did not help")

    if res_c:
        print(f"\n  Model C (LGB block):    DA={res_c['da']*100:.1f}%")
    if res_d:
        print(f"  Model D (LGB full):     DA={res_d['da']*100:.1f}%")
        if res_c:
            d = res_d["da"] - res_c["da"]
            print(f"    Daily features add: {d*100:+.1f}pp")

    # Best overall
    all_das = [(n, r["da"]) for n, r in all_results.items()
               if "Naive" not in n and "Historical" not in n and "GARCH" not in n]
    if all_das:
        bn, bd = max(all_das, key=lambda x: x[1])
        print(f"\n  ★ Best model: {bn}  DA={bd*100:.1f}%")
        if bd > hist_da + 0.01:
            print(f"    Beats Historical Mean baseline by {(bd-hist_da)*100:+.1f}pp ✓")
        else:
            print(f"    Does not clearly beat Historical Mean baseline")

    elapsed = time.time() - t_start
    print(f"\n  Total runtime: {elapsed:.1f}s")

    # ── Save ──
    jp = os.path.join(CFG["RESULTS_DIR"], "vol_prediction_v2_results.json")
    save = {}
    for name, r in all_results.items():
        save[name] = {k: v for k, v in r.items()
                      if isinstance(v, (int, float, str, dict))}
    save["naive_baselines"] = {
        "random_walk_da": float(naive_rw_da),
        "historical_mean_da": float(hist_da),
        "historical_mean_value": float(hist_mean),
    }
    if garch_res:
        save["garch"] = garch_res
    save["config"] = {
        "seq_len": CFG["SEQ_LEN"], "hidden": CFG["HIDDEN"],
        "layers": CFG["NUM_LAYERS"], "dropout": CFG["DROPOUT"],
        "lr": CFG["LR"], "loss": "MSE on log(RV)",
    }
    with open(jp, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"  Saved: {jp}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

"""
vol_prediction_v3.py — Model E: LightGBM + LSTM Embeddings
===========================================================
Combines Model D's 102 flat features with Model B's Multi-Scale LSTM
internal embeddings (32+32=64 dims) for ~166 total features.

Prerequisite: vol_prediction_v2.py must have been run first to produce
  results/vol_multiscale_lstm_regression.pt (Model B weights)

Usage:
    !python vol_prediction_v3.py
"""

import os, sys, time, json, warnings, math
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, r2_score, mean_squared_error,
                             f1_score, precision_score, recall_score)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[ERROR] lightgbm required"); sys.exit(1)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = {
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "VOL_DATA_DIR": "vol_data",
    "BLOCK_SIZE": 12,
    "SEQ_LEN": 10,
    "DAILY_LOOKBACK": 20,
    "LR": 5e-4,
    "WD": 1e-4,
    "BATCH": 64,
    "EPOCHS": 200,
    "PATIENCE": 30,
    "SCHED_PATIENCE": 10,
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.85,
}


# ═══════════════════════════════════════════════════════════════
#  UTILITIES (same as v2)
# ═══════════════════════════════════════════════════════════════

def load_features_csv(ticker, freq):
    p = os.path.join(CFG["FEATURES_DIR"], f"{ticker}_{freq}_features.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    df = pd.read_csv(p)
    for col in ["timestamp", "ts_event", "datetime", "date"]:
        if col in df.columns:
            df["timestamp"] = pd.to_datetime(df[col], utc=True)
            break
    if "timestamp" not in df.columns:
        first = df.columns[0]
        if first not in df.select_dtypes(include=[np.number]).columns:
            df["timestamp"] = pd.to_datetime(df[first], utc=True)
    return df.sort_values("timestamp").reset_index(drop=True)


def numeric_cols(df):
    ex = {"timestamp", "ts_event", "datetime", "date", "time",
          "unnamed: 0", "symbol", "ticker"}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c.lower() not in ex]


def split3(n):
    return int(n * CFG["TRAIN_RATIO"]), int(n * CFG["VAL_RATIO"])


# ═══════════════════════════════════════════════════════════════
#  DATA PIPELINE (matches v2 exactly)
# ═══════════════════════════════════════════════════════════════

def compute_blocks(df_1h, block_size=12):
    close = df_1h["close"].values.astype(np.float64)
    ts = df_1h["timestamp"].values
    lr = np.concatenate([[0], np.diff(np.log(close + 1e-10))])
    n_blocks = len(close) // block_size
    rows = []
    for b in range(n_blocks):
        s, e = b * block_size, (b + 1) * block_size
        rv = np.std(lr[s:e]) * np.sqrt(block_size)
        rows.append({"block_idx": b, "bar_start": s, "bar_end": e,
                      "rv": rv, "log_rv": np.log(rv + 1e-10),
                      "ts_start": ts[s], "ts_end": ts[min(e-1, len(close)-1)]})
    return pd.DataFrame(rows)


def compute_daily_bars(df_1h):
    df = df_1h.copy()
    df["_date"] = df["timestamp"].dt.date
    has_vol = "volume" in df.columns
    agg = {"open": ("open", "first"), "high": ("high", "max"),
           "low": ("low", "min"), "close": ("close", "last"),
           "n_bars": ("close", "count")}
    if has_vol:
        agg["volume"] = ("volume", "sum")
    daily = df.groupby("_date").agg(**agg).reset_index()
    daily.rename(columns={"_date": "date"}, inplace=True)

    c = daily["close"].values.astype(np.float64)
    h = daily["high"].values.astype(np.float64)
    lo = daily["low"].values.astype(np.float64)
    o = daily["open"].values.astype(np.float64)
    n = len(daily)

    daily["daily_return"] = (c - o) / (o + 1e-10)
    daily["daily_range"] = (h - lo) / (c + 1e-10)

    if has_vol:
        v = daily["volume"].values.astype(np.float64)
        daily["vol_change"] = np.concatenate([[1.0], v[1:] / (v[:-1] + 1e-10)])
    else:
        daily["vol_change"] = 1.0

    drv = []
    for _, grp in df.groupby("_date"):
        cc = grp["close"].values.astype(np.float64)
        if len(cc) > 1:
            lr_ = np.diff(np.log(cc + 1e-10))
            drv.append(np.std(lr_) * np.sqrt(len(cc)))
        else:
            drv.append(0.0)
    daily["daily_rv"] = drv

    # RSI-14
    delta = np.diff(c, prepend=c[0])
    gain = np.where(delta > 0, delta, 0.0)
    loss_a = np.where(delta < 0, -delta, 0.0)
    ag = np.zeros(n); al = np.zeros(n)
    if n > 14:
        ag[14] = np.mean(gain[1:15]); al[14] = np.mean(loss_a[1:15])
        for i in range(15, n):
            ag[i] = (ag[i-1]*13 + gain[i]) / 14
            al[i] = (al[i-1]*13 + loss_a[i]) / 14
    daily["rsi_14"] = 100 - 100 / (1 + ag / (al + 1e-10))

    # ATR-14
    tr = np.zeros(n)
    for i in range(1, n):
        tr[i] = max(h[i]-lo[i], abs(h[i]-c[i-1]), abs(lo[i]-c[i-1]))
    tr[0] = h[0] - lo[0]
    atr = np.zeros(n)
    if n > 14:
        atr[14] = np.mean(tr[1:15])
        for i in range(15, n):
            atr[i] = (atr[i-1]*13 + tr[i]) / 14
    daily["atr_14"] = atr

    # BB bandwidth
    bb = np.zeros(n)
    for i in range(20, n):
        sma = np.mean(c[i-20:i]); std = np.std(c[i-20:i])
        bb[i] = (4 * std) / (sma + 1e-10)
    daily["bb_bandwidth_20"] = bb
    daily["log_close"] = np.log(c + 1e-10)

    return daily


def load_all_data():
    """Load and build blocks + daily for all tickers."""
    BS = CFG["BLOCK_SIZE"]
    all_blocks = {}   # ticker → (dfb, block_feats, feat_names)
    all_daily = {}

    for ticker in CFG["TICKERS"]:
        try:
            df_1h = load_features_csv(ticker, "1hour")
        except FileNotFoundError:
            continue
        fc = numeric_cols(df_1h)
        fv = np.nan_to_num(df_1h[fc].values.astype(np.float32), nan=0.0)
        dfb = compute_blocks(df_1h, BS)
        nb = len(dfb)
        bf = np.zeros((nb, len(fc)), dtype=np.float32)
        for i in range(nb):
            s, e = dfb.loc[i, "bar_start"], dfb.loc[i, "bar_end"]
            bf[i] = np.mean(fv[s:e], axis=0)
        all_blocks[ticker] = (dfb, bf, fc)
        all_daily[ticker] = compute_daily_bars(df_1h)

    return all_blocks, all_daily


# ═══════════════════════════════════════════════════════════════
#  BUILD ALIGNED DATASETS
# ═══════════════════════════════════════════════════════════════

def build_model_b_sequences(all_blocks, all_daily):
    """
    Build short+long sequences for Model B. Returns arrays + (ticker, block_idx) keys.
    Matches v2 logic exactly: short = 10 blocks of (feat+log_rv), long = 20 days.
    """
    SL = CFG["SEQ_LEN"]; DL = CFG["DAILY_LOOKBACK"]
    dcols = ["daily_rv", "daily_range", "daily_return", "vol_change",
             "rsi_14", "atr_14", "bb_bandwidth_20", "log_close"]

    Xs, Xl, keys = [], [], []
    n_sf = n_df = None

    for ticker, (dfb, bf, fc) in all_blocks.items():
        log_rvs = dfb["log_rv"].values.astype(np.float32)
        bf_ext = np.column_stack([bf, log_rvs.reshape(-1, 1)])
        if n_sf is None:
            n_sf = bf_ext.shape[1]

        daily = all_daily.get(ticker)
        if daily is None:
            continue
        dc_v = [c for c in dcols if c in daily.columns]
        if n_df is None:
            n_df = len(dc_v)
        df_vals = np.nan_to_num(daily[dc_v].values.astype(np.float32), nan=0.0)
        ddates = daily["date"].values
        nb = len(dfb)

        for i in range(SL, nb - 1):
            short_seq = bf_ext[i - SL:i]

            block_end = pd.Timestamp(dfb.loc[i, "ts_end"])
            bd = block_end.date() if hasattr(block_end, 'date') else block_end
            day_idx = -1
            for di in range(len(ddates) - 1, -1, -1):
                if ddates[di] <= bd:
                    day_idx = di; break
            if day_idx < DL:
                continue
            long_seq = df_vals[day_idx - DL + 1:day_idx + 1]
            if len(long_seq) != DL:
                continue

            Xs.append(short_seq)
            Xl.append(long_seq)
            keys.append((ticker, i))

    return (np.array(Xs, dtype=np.float32),
            np.array(Xl, dtype=np.float32),
            keys, n_sf, n_df)


def build_model_d_features(all_blocks, all_daily):
    """
    Build Model D flat features (block + daily). Returns X, y_reg, y_cls, cur_rv, keys, fnames.
    Matches v2 logic exactly.
    """
    SL = CFG["SEQ_LEN"]; DAILY_LB = 5
    key_cols_names = ["close", "volume", "atr_14", "rsi_14"]
    daily_cols = ["daily_rv", "daily_range", "daily_return", "vol_change"]

    X_rows, y_reg, y_cls, cur_rv, keys = [], [], [], [], []

    for ticker, (dfb, bf, fc) in all_blocks.items():
        rvs = dfb["rv"].values.astype(np.float64)
        log_rvs = dfb["log_rv"].values.astype(np.float64)
        fc_list = list(fc)
        nb = len(dfb)

        key_idxs = {}
        for kc in key_cols_names:
            matches = [j for j, c in enumerate(fc_list) if kc.lower() in c.lower()]
            if matches:
                key_idxs[kc] = matches[0]

        daily = all_daily.get(ticker)
        if daily is not None:
            dc_v = [c for c in daily_cols if c in daily.columns]
            df_vals = np.nan_to_num(daily[dc_v].values.astype(np.float32), nan=0.0)
            ddates = daily["date"].values
            n_dc = len(dc_v)
        else:
            dc_v = []; n_dc = 0; df_vals = None; ddates = None

        spy_data = all_blocks.get("SPY")
        spy_rvs = spy_data[0]["rv"].values if spy_data else None

        for i in range(SL, nb - 1):
            feats = {}
            for off in range(SL):
                bi = i - SL + off
                px = f"b{off}"
                feats[f"{px}_rv"] = rvs[bi]
                feats[f"{px}_log_rv"] = log_rvs[bi]
                for kc, ki in key_idxs.items():
                    feats[f"{px}_{kc}"] = float(bf[bi, ki])

            rv_win = rvs[max(0, i-SL):i]
            rv4 = rvs[max(0, i-4):i]
            feats["rv_mean_4"] = np.mean(rv4) if len(rv4) else 0
            feats["rv_std_4"] = np.std(rv4) if len(rv4) > 1 else 0
            feats["rv_mean_8"] = np.mean(rv_win) if len(rv_win) else 0
            feats["rv_std_8"] = np.std(rv_win) if len(rv_win) > 1 else 0
            feats["rv_ratio"] = rvs[i-1] / (feats["rv_mean_8"] + 1e-10)
            if len(rv_win) >= 3:
                feats["rv_trend"] = float(np.polyfit(np.arange(len(rv_win)), rv_win, 1)[0])
            else:
                feats["rv_trend"] = 0.0
            su = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] > rvs[k-1]: su += 1
                else: break
            feats["rv_streak_up"] = su
            sd = 0
            for k in range(i-1, max(0, i-SL)-1, -1):
                if k > 0 and rvs[k] < rvs[k-1]: sd += 1
                else: break
            feats["rv_streak_down"] = sd
            feats["current_log_rv"] = log_rvs[i-1]

            for j, cn in enumerate(fc_list):
                feats[f"tech_{cn}"] = float(bf[i-1, j])

            if ticker != "SPY" and spy_rvs is not None and i < len(spy_rvs):
                feats["spy_rv_cur"] = spy_rvs[max(0, i-1)]
                s4 = spy_rvs[max(0, i-4):i]
                feats["spy_rv_mean_4"] = np.mean(s4) if len(s4) else 0
            else:
                feats["spy_rv_cur"] = 0; feats["spy_rv_mean_4"] = 0

            # Daily features
            if daily is not None and df_vals is not None:
                bets = pd.Timestamp(dfb.loc[i, "ts_end"])
                bd = bets.date() if hasattr(bets, 'date') else bets
                day_idx = -1
                for di in range(len(ddates)-1, -1, -1):
                    if ddates[di] <= bd: day_idx = di; break
                if day_idx >= DAILY_LB:
                    for d_off in range(DAILY_LB):
                        di2 = day_idx - DAILY_LB + 1 + d_off
                        for ci, cn in enumerate(dc_v[:n_dc]):
                            feats[f"d{d_off}_{cn}"] = float(df_vals[di2, ci])
                    drv_w = df_vals[day_idx-DAILY_LB+1:day_idx+1, 0]
                    feats["daily_rv_mean_5"] = float(np.mean(drv_w))
                    feats["daily_rv_std_5"] = float(np.std(drv_w))
                    if len(drv_w) >= 3:
                        feats["daily_rv_trend"] = float(
                            np.polyfit(np.arange(len(drv_w)), drv_w, 1)[0])
                    else:
                        feats["daily_rv_trend"] = 0.0

            X_rows.append(feats)
            y_reg.append(log_rvs[i])
            y_cls.append(1 if rvs[i] > rvs[i-1] else 0)
            cur_rv.append(rvs[i-1])
            keys.append((ticker, i))

    if not X_rows:
        return np.array([]), np.array([]), np.array([]), np.array([]), keys, []
    fnames = list(X_rows[0].keys())
    X = np.array([[r.get(fn, 0.0) for fn in fnames] for r in X_rows], dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return (X, np.array(y_reg, dtype=np.float32),
            np.array(y_cls, dtype=np.float32),
            np.array(cur_rv, dtype=np.float64), keys, fnames)


# ═══════════════════════════════════════════════════════════════
#  MODEL B ARCHITECTURE (must match v2 exactly)
# ═══════════════════════════════════════════════════════════════

class VolLSTM_B(nn.Module):
    def __init__(self, n_sfeat, n_dfeat):
        super().__init__()
        self.short_lstm = nn.LSTM(n_sfeat, 32, 1, batch_first=True)
        self.long_lstm = nn.LSTM(n_dfeat, 32, 1, batch_first=True)
        self.fuse = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, xs, xl):
        so, _ = self.short_lstm(xs)
        lo, _ = self.long_lstm(xl)
        return self.fuse(torch.cat([so[:, -1, :], lo[:, -1, :]], dim=1)).squeeze(-1)

    def extract_embeddings(self, xs, xl):
        """Extract 32-dim embeddings from each branch (before fusion)."""
        so, _ = self.short_lstm(xs)
        lo, _ = self.long_lstm(xl)
        return so[:, -1, :], lo[:, -1, :]  # (batch,32), (batch,32)


class PairDS(Dataset):
    def __init__(self, Xs, Xl):
        self.Xs = torch.tensor(Xs, dtype=torch.float32)
        self.Xl = torch.tensor(Xl, dtype=torch.float32)
    def __len__(self): return len(self.Xs)
    def __getitem__(self, i): return self.Xs[i], self.Xl[i]


class PairLabelDS(Dataset):
    def __init__(self, Xs, Xl, y):
        self.Xs = torch.tensor(Xs, dtype=torch.float32)
        self.Xl = torch.tensor(Xl, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.Xs[i], self.Xl[i], self.y[i]


# ═══════════════════════════════════════════════════════════════
#  RETRAIN MODEL B IF NEEDED
# ═══════════════════════════════════════════════════════════════

def retrain_model_b(Xs, Xl, y_reg, tr, va, n_sf, n_df):
    """Retrain Model B with same config as v2 (regression on log(RV), MSE)."""
    print("  Retraining Model B (regression on log(RV))...")

    # Normalize
    flat_s = Xs[:tr].reshape(-1, n_sf)
    mu_s, sig_s = flat_s.mean(0), flat_s.std(0) + 1e-8
    Xs_n = np.nan_to_num((Xs - mu_s) / sig_s, nan=0.0, posinf=0.0, neginf=0.0)

    flat_l = Xl[:tr].reshape(-1, n_df)
    mu_l, sig_l = flat_l.mean(0), flat_l.std(0) + 1e-8
    Xl_n = np.nan_to_num((Xl - mu_l) / sig_l, nan=0.0, posinf=0.0, neginf=0.0)

    model = VolLSTM_B(n_sf, n_df).to(DEVICE)
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=CFG["SCHED_PATIENCE"], factor=0.5)

    tr_dl = DataLoader(PairLabelDS(Xs_n[:tr], Xl_n[:tr], y_reg[:tr]),
                       batch_size=CFG["BATCH"], shuffle=True)
    va_dl = DataLoader(PairLabelDS(Xs_n[tr:va], Xl_n[tr:va], y_reg[tr:va]),
                       batch_size=512)

    best_vl = float("inf"); best_st = None; pat = 0
    for ep in range(CFG["EPOCHS"]):
        model.train(); tl = 0
        for xs, xl, yb in tr_dl:
            xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xs, xl), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(yb)
        tl /= tr

        model.eval(); vl = 0
        with torch.no_grad():
            for xs, xl, yb in va_dl:
                xs, xl, yb = xs.to(DEVICE), xl.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xs, xl), yb).item() * len(yb)
        vl /= (va - tr)
        sched.step(vl)

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if (ep+1) % 40 == 0 or pat == CFG["PATIENCE"]:
            print(f"    E{ep+1:3d} t={tl:.6f} v={vl:.6f} p={pat}")
        if pat >= CFG["PATIENCE"]:
            print(f"    Early stop")
            break

    if best_st:
        model.load_state_dict(best_st)
    model.to(DEVICE).eval()

    sp = os.path.join(CFG["RESULTS_DIR"], "vol_multiscale_lstm_regression.pt")
    torch.save(model.state_dict(), sp)
    print(f"  Saved: {sp}")
    return model, Xs_n, Xl_n


# ═══════════════════════════════════════════════════════════════
#  EVALUATION
# ═══════════════════════════════════════════════════════════════

def eval_regression(pred_log, true_log, cur_rv, meta_te, name):
    pred_rv = np.exp(pred_log)
    true_rv = np.exp(true_log)
    pred_dir = (pred_rv > cur_rv).astype(int)
    true_dir = (true_rv > cur_rv).astype(int)

    da = (pred_dir == true_dir).mean()
    r2 = r2_score(true_rv, pred_rv)
    rmse = np.sqrt(mean_squared_error(true_rv, pred_rv))
    n = len(true_dir)
    z = (da - 0.5) / np.sqrt(0.5*0.5/n)
    from scipy.stats import norm
    p_val = 2 * (1 - norm.cdf(abs(z)))
    f1 = f1_score(true_dir, pred_dir, zero_division=0)

    result = {"name": name, "da": float(da), "r2": float(r2),
              "rmse": float(rmse), "z": float(z), "p": float(p_val),
              "f1": float(f1), "n": n}

    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                td_da = float((pred_dir[mask] == true_dir[mask]).mean())
                tn = int(mask.sum())
                tz = (td_da - 0.5) / np.sqrt(0.5*0.5/tn)
                pt[ticker] = {"da": td_da, "n": tn, "z": float(tz)}
        result["per_ticker"] = pt
    return result


def eval_classification(pred_prob, true_dir, meta_te, name):
    preds = (pred_prob >= 0.5).astype(int)
    da = (preds == true_dir).mean()
    n = len(true_dir)
    z = (da - 0.5) / np.sqrt(0.5*0.5/n)
    auc = roc_auc_score(true_dir, pred_prob) if true_dir.sum() > 0 and (1-true_dir).sum() > 0 else 0.5
    f1 = f1_score(true_dir, preds, zero_division=0)
    result = {"name": name, "da": float(da), "r2": 0.0, "rmse": 0.0,
              "z": float(z), "f1": float(f1), "n": n, "auc": float(auc)}
    if meta_te:
        pt = {}
        tickers = [m[0] for m in meta_te]
        for ticker in CFG["TICKERS"]:
            mask = np.array([t == ticker for t in tickers])
            if mask.sum() > 0:
                pt[ticker] = {"da": float((preds[mask] == true_dir[mask]).mean()),
                              "n": int(mask.sum())}
        result["per_ticker"] = pt
    return result


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("  Vol Prediction v3 — Model E: LightGBM + LSTM Embeddings")
    print(f"  Device: {DEVICE}")
    print("=" * 70)
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    t_start = time.time()

    # ══════════════════════════════════════════════════════
    #  STEP 1: Load all data
    # ══════════════════════════════════════════════════════
    print("\n  STEP 1: Loading data...")
    all_blocks, all_daily = load_all_data()
    print(f"  Tickers loaded: {list(all_blocks.keys())}")

    # Build Model B sequences (with keys for alignment)
    print("\n  Building Model B sequences...")
    Xs_b, Xl_b, keys_b, n_sf, n_df = build_model_b_sequences(all_blocks, all_daily)
    print(f"  Model B: {len(keys_b):,} samples  "
          f"short=({CFG['SEQ_LEN']},{n_sf})  long=({CFG['DAILY_LOOKBACK']},{n_df})")

    # Build Model D features (with keys for alignment)
    print("  Building Model D features...")
    X_d, y_reg_d, y_cls_d, cur_d, keys_d, fnames_d = build_model_d_features(
        all_blocks, all_daily)
    print(f"  Model D: {len(keys_d):,} samples  {len(fnames_d)} features")

    # ── ALIGN datasets using (ticker, block_idx) keys ──
    print("\n  Aligning datasets...")
    key_set_b = set(keys_b)
    key_set_d = set(keys_d)
    common_keys = sorted(key_set_b & key_set_d)
    print(f"  Model B: {len(keys_b):,}  Model D: {len(keys_d):,}  "
          f"Common: {len(common_keys):,}")

    # Build lookup indices
    b_idx_map = {k: i for i, k in enumerate(keys_b)}
    d_idx_map = {k: i for i, k in enumerate(keys_d)}

    b_indices = np.array([b_idx_map[k] for k in common_keys])
    d_indices = np.array([d_idx_map[k] for k in common_keys])

    Xs_aligned = Xs_b[b_indices]
    Xl_aligned = Xl_b[b_indices]
    X_d_aligned = X_d[d_indices]
    y_reg_aligned = y_reg_d[d_indices]
    y_cls_aligned = y_cls_d[d_indices]
    cur_aligned = cur_d[d_indices]
    meta_aligned = common_keys

    n_total = len(common_keys)
    tr, va = split3(n_total)
    print(f"  Aligned: {n_total:,}  train={tr} val={va-tr} test={n_total-va}")

    # ══════════════════════════════════════════════════════
    #  STEP 2: Load / retrain Model B
    # ══════════════════════════════════════════════════════
    print("\n  STEP 2: Loading Model B checkpoint...")
    ckpt_path = os.path.join(CFG["RESULTS_DIR"], "vol_multiscale_lstm_regression.pt")

    model_b = VolLSTM_B(n_sf, n_df).to(DEVICE)
    retrained = False

    if os.path.exists(ckpt_path):
        try:
            state = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
            model_b.load_state_dict(state)
            print(f"  ✓ Loaded: {ckpt_path}")
        except Exception as e:
            print(f"  ✗ Failed to load: {e}")
            print(f"  Retraining Model B...")
            model_b, Xs_n_cache, Xl_n_cache = retrain_model_b(
                Xs_aligned, Xl_aligned, y_reg_aligned, tr, va, n_sf, n_df)
            retrained = True
    else:
        print(f"  ✗ Checkpoint not found: {ckpt_path}")
        print(f"  Retraining Model B...")
        model_b, Xs_n_cache, Xl_n_cache = retrain_model_b(
            Xs_aligned, Xl_aligned, y_reg_aligned, tr, va, n_sf, n_df)
        retrained = True

    model_b.eval()

    # Normalize sequences for embedding extraction
    if not retrained:
        flat_s = Xs_aligned[:tr].reshape(-1, n_sf)
        mu_s, sig_s = flat_s.mean(0), flat_s.std(0) + 1e-8
        Xs_n = np.nan_to_num((Xs_aligned - mu_s) / sig_s, nan=0.0, posinf=0.0, neginf=0.0)

        flat_l = Xl_aligned[:tr].reshape(-1, n_df)
        mu_l, sig_l = flat_l.mean(0), flat_l.std(0) + 1e-8
        Xl_n = np.nan_to_num((Xl_aligned - mu_l) / sig_l, nan=0.0, posinf=0.0, neginf=0.0)
    else:
        Xs_n = Xs_n_cache
        Xl_n = Xl_n_cache

    # ── Extract embeddings ──
    print("  Extracting LSTM embeddings...")
    dl = DataLoader(PairDS(Xs_n, Xl_n), batch_size=512)
    all_short_emb, all_long_emb = [], []
    with torch.no_grad():
        for xs, xl in dl:
            se, le = model_b.extract_embeddings(xs.to(DEVICE), xl.to(DEVICE))
            all_short_emb.append(se.cpu().numpy())
            all_long_emb.append(le.cpu().numpy())
    short_emb = np.concatenate(all_short_emb)  # (N, 32)
    long_emb = np.concatenate(all_long_emb)    # (N, 32)

    # Normalize embeddings from training set only
    emb_mu_s = short_emb[:tr].mean(0); emb_sig_s = short_emb[:tr].std(0) + 1e-8
    emb_mu_l = long_emb[:tr].mean(0); emb_sig_l = long_emb[:tr].std(0) + 1e-8
    short_emb_n = np.nan_to_num((short_emb - emb_mu_s) / emb_sig_s, nan=0.0)
    long_emb_n = np.nan_to_num((long_emb - emb_mu_l) / emb_sig_l, nan=0.0)

    print(f"  Short emb: {short_emb_n.shape}  Long emb: {long_emb_n.shape}")

    # ── Evaluate Model B on test set ──
    pred_b_dl = DataLoader(PairDS(Xs_n[va:], Xl_n[va:]), batch_size=512)
    pred_b_list = []
    with torch.no_grad():
        for xs, xl in pred_b_dl:
            pred_b_list.append(model_b(xs.to(DEVICE), xl.to(DEVICE)).cpu().numpy())
    pred_b_log = np.concatenate(pred_b_list)
    res_b = eval_regression(pred_b_log, y_reg_aligned[va:], cur_aligned[va:],
                            meta_aligned[va:], "B: Multi-Scale LSTM")
    print(f"  Model B Test: DA={res_b['da']*100:.1f}%  R²={res_b['r2']:.3f}  "
          f"z={res_b['z']:+.2f}")

    # ══════════════════════════════════════════════════════
    #  STEP 3: Build Model E feature matrix
    # ══════════════════════════════════════════════════════
    print("\n  STEP 3: Building Model E features...")

    emb_short_names = [f"emb_short_{i}" for i in range(32)]
    emb_long_names = [f"emb_long_{i}" for i in range(32)]
    fnames_e = fnames_d + emb_short_names + emb_long_names

    X_e = np.column_stack([X_d_aligned, short_emb_n, long_emb_n])
    print(f"  Model E features: {X_d_aligned.shape[1]} (D) + 32 (short_emb) "
          f"+ 32 (long_emb) = {X_e.shape[1]}")

    # ══════════════════════════════════════════════════════
    #  STEP 4: Train Model E + re-run Model D (aligned)
    # ══════════════════════════════════════════════════════
    print("\n  STEP 4: Training LightGBM models...")

    te_y_reg = y_reg_aligned[va:]
    te_y_cls = y_cls_aligned[va:].astype(int)
    te_cur = cur_aligned[va:]
    meta_te = meta_aligned[va:]

    # ── Model D (aligned) — classification ──
    print("\n  Model D (aligned, classification)...")
    mdl_d_cls = lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_d_cls.fit(X_d_aligned[:tr], y_cls_aligned[:tr],
                  eval_set=[(X_d_aligned[tr:va], y_cls_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    d_cls_prob = mdl_d_cls.predict_proba(X_d_aligned[va:])[:, 1]
    d_cls_da = ((d_cls_prob >= 0.5).astype(int) == te_y_cls).mean()

    # ── Model D (aligned) — regression ──
    print("  Model D (aligned, regression)...")
    mdl_d_reg = lgb.LGBMRegressor(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_d_reg.fit(X_d_aligned[:tr], y_reg_aligned[:tr],
                  eval_set=[(X_d_aligned[tr:va], y_reg_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    d_reg_pred = mdl_d_reg.predict(X_d_aligned[va:])
    d_reg_dir = (np.exp(d_reg_pred) > te_cur).astype(int)
    d_reg_da = (d_reg_dir == te_y_cls).mean()

    print(f"  Model D cls DA: {d_cls_da*100:.1f}%  reg DA: {d_reg_da*100:.1f}%")
    if d_reg_da >= d_cls_da:
        res_d = eval_regression(d_reg_pred, te_y_reg, te_cur, meta_te,
                                "D: LightGBM (block+daily) [reg]")
        d_approach = "regression"
    else:
        res_d = eval_classification(d_cls_prob, te_y_cls, meta_te,
                                    "D: LightGBM (block+daily) [cls]")
        d_approach = "classification"
    print(f"  Model D best: {d_approach}  DA={res_d['da']*100:.1f}%")

    # ── Model E — classification ──
    print("\n  Model E (classification)...")
    mdl_e_cls = lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_e_cls.fit(X_e[:tr], y_cls_aligned[:tr],
                  eval_set=[(X_e[tr:va], y_cls_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    e_cls_prob = mdl_e_cls.predict_proba(X_e[va:])[:, 1]
    e_cls_da = ((e_cls_prob >= 0.5).astype(int) == te_y_cls).mean()

    # ── Model E — regression ──
    print("  Model E (regression)...")
    mdl_e_reg = lgb.LGBMRegressor(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.1, reg_lambda=1.0, verbose=-1, random_state=SEED)
    mdl_e_reg.fit(X_e[:tr], y_reg_aligned[:tr],
                  eval_set=[(X_e[tr:va], y_reg_aligned[tr:va])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(0)])
    e_reg_pred = mdl_e_reg.predict(X_e[va:])
    e_reg_dir = (np.exp(e_reg_pred) > te_cur).astype(int)
    e_reg_da = (e_reg_dir == te_y_cls).mean()

    print(f"  Model E cls DA: {e_cls_da*100:.1f}%  reg DA: {e_reg_da*100:.1f}%")
    if e_reg_da >= e_cls_da:
        res_e = eval_regression(e_reg_pred, te_y_reg, te_cur, meta_te,
                                "E: LightGBM + LSTM emb [reg]")
        best_e_mdl = mdl_e_reg
        e_approach = "regression"
    else:
        res_e = eval_classification(e_cls_prob, te_y_cls, meta_te,
                                    "E: LightGBM + LSTM emb [cls]")
        best_e_mdl = mdl_e_cls
        e_approach = "classification"
    print(f"  Model E best: {e_approach}  DA={res_e['da']*100:.1f}%")

    # ── Naive baselines on aligned test set ──
    true_rv_te = np.exp(te_y_reg)
    naive_rw_da = float((te_y_cls == 0).mean())  # predict "down" = predict cur
    train_rvs = np.exp(y_reg_aligned[:tr])
    hist_mean = float(train_rvs.mean())
    hist_dir = (hist_mean > te_cur).astype(int)
    hist_da = float((hist_dir == te_y_cls).mean())

    # ══════════════════════════════════════════════════════
    #  STEP 5: Comparison table
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ COMPARISON TABLE")
    print(f"{'='*70}")

    all_r = {
        "Naive (RW)": {"da": naive_rw_da, "r2": 0, "rmse": 0, "n": len(te_y_cls)},
        "Historical Mean": {"da": hist_da, "r2": 0, "rmse": 0, "n": len(te_y_cls)},
    }
    all_r["B: Multi-Scale LSTM"] = res_b
    all_r[res_d["name"]] = res_d
    all_r[res_e["name"]] = res_e

    print(f"\n  ┌{'─'*71}┐")
    print(f"  │ {'Model':<38} │ {'DA':>7} │ {'R²':>7} │ {'N':>6} │")
    print(f"  ├{'─'*71}┤")

    for name, r in all_r.items():
        da = r["da"]; r2v = r.get("r2", 0); n = r.get("n", 0)
        r2_s = f"{r2v:>7.3f}" if r2v != 0 else "     —"
        print(f"  │ {name:<38} │ {da*100:>6.1f}% │ {r2_s} │ {n:>6} │")

    delta = res_e["da"] - res_d["da"]
    marker = "✓ LSTM emb helps" if delta > 0.005 else ("~ marginal" if delta > -0.005 else "✗ no gain")
    print(f"  ├{'─'*71}┤")
    print(f"  │ {'Δ (E vs D)':<38} │ {delta*100:>+6.1f}p │ {'':>7} │ {marker:>6} │")
    print(f"  └{'─'*71}┘")

    # ══════════════════════════════════════════════════════
    #  STEP 6: Feature importance
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ FEATURE IMPORTANCE (Model E — top 20)")
    print(f"{'='*70}")

    imp = best_e_mdl.feature_importances_
    fi = sorted(zip(fnames_e, imp), key=lambda x: -x[1])

    print(f"\n  {'#':>4} {'Feature':<42} {'Imp':>8} {'Cat':>8}")
    print(f"  {'-'*65}")

    emb_in_top20 = 0
    for rank, (fn, fv) in enumerate(fi[:20], 1):
        if fn.startswith("emb_short_"):
            cat = "[emb_s]"; emb_in_top20 += 1
        elif fn.startswith("emb_long_"):
            cat = "[emb_l]"; emb_in_top20 += 1
        elif ("rv" in fn.lower() or fn.startswith("rv_")) and not fn.startswith("tech_"):
            cat = "[rv]"
        elif fn.startswith("d") and fn[1:2].isdigit():
            cat = "[daily]"
        elif fn.startswith("tech_"):
            cat = "[tech]"
        elif fn.startswith("b") and fn[1:2].isdigit():
            cat = "[block]"
        elif "spy" in fn.lower():
            cat = "[cross]"
        else:
            cat = "[other]"
        print(f"  {rank:>4} {fn:<42} {fv:>8} {cat:>8}")

    # Category breakdown
    cats = {"Block features": 0, "Daily features": 0, "Tech indicators": 0,
            "LSTM short emb": 0, "LSTM long emb": 0,
            "RV sequence": 0, "Cross-asset": 0, "Other": 0}
    for fn, fv in fi:
        if fn.startswith("emb_short_"):
            cats["LSTM short emb"] += fv
        elif fn.startswith("emb_long_"):
            cats["LSTM long emb"] += fv
        elif ("rv" in fn.lower() or fn.startswith("rv_")) and not fn.startswith("tech_"):
            cats["RV sequence"] += fv
        elif fn.startswith("d") and fn[1:2].isdigit():
            cats["Daily features"] += fv
        elif fn.startswith("tech_"):
            cats["Tech indicators"] += fv
        elif fn.startswith("b") and fn[1:2].isdigit():
            cats["Block features"] += fv
        elif "spy" in fn.lower():
            cats["Cross-asset"] += fv
        else:
            cats["Other"] += fv

    total_imp = sum(cats.values()) + 1e-10
    print(f"\n  Category importance breakdown:")
    for cat, val in sorted(cats.items(), key=lambda x: -x[1]):
        bar = "█" * int(val/total_imp*40)
        print(f"    {cat:<20} {val/total_imp*100:>5.1f}%  {bar}")

    lstm_total = (cats["LSTM short emb"] + cats["LSTM long emb"]) / total_imp * 100
    print(f"\n  LSTM embedding total: {lstm_total:.1f}%")
    print(f"  LSTM embedding features in top 20: {emb_in_top20}/20")

    if emb_in_top20 >= 3:
        print(f"  → LSTM captures patterns LightGBM can't learn from flat features ✓")
    elif emb_in_top20 >= 1:
        print(f"  → LSTM provides some complementary signal")
    else:
        print(f"  → Flat features already subsume LSTM's knowledge")

    # ══════════════════════════════════════════════════════
    #  STEP 7: Per-ticker breakdown
    # ══════════════════════════════════════════════════════
    pt_e = res_e.get("per_ticker", {})
    if pt_e:
        print(f"\n  Per-Ticker (Model E):")
        print(f"  {'Ticker':<8} {'DA':>8} {'N':>6} {'z':>7}")
        print(f"  {'-'*32}")
        das = []
        for ticker in CFG["TICKERS"]:
            td = pt_e.get(ticker, {})
            if td:
                z_s = f"{td.get('z', 0):+.2f}" if "z" in td else "—"
                print(f"  {ticker:<8} {td['da']*100:>7.1f}% {td['n']:>6} {z_s:>7}")
                das.append(td["da"])
        if das:
            print(f"  {'Average':<8} {np.mean(das)*100:>7.1f}%")

    # ══════════════════════════════════════════════════════
    #  ANALYSIS
    # ══════════════════════════════════════════════════════
    print(f"\n{'='*70}")
    print(f"  ★ ANALYSIS")
    print(f"{'='*70}")

    print(f"\n  Model B (LSTM only):         DA={res_b['da']*100:.1f}%")
    print(f"  Model D (LGB flat only):     DA={res_d['da']*100:.1f}%")
    print(f"  Model E (LGB + LSTM emb):    DA={res_e['da']*100:.1f}%")
    print(f"\n  Δ(E vs D): {delta*100:+.1f}pp")
    print(f"  Δ(E vs B): {(res_e['da']-res_b['da'])*100:+.1f}pp")

    if delta > 0.01:
        print(f"\n  → LSTM embeddings provide meaningful boost over flat features")
        print(f"    Sequential patterns captured by LSTM complement tabular features")
    elif delta > 0:
        print(f"\n  → Small positive contribution from LSTM embeddings")
        print(f"    Most vol prediction power comes from flat RV/tech features")
    else:
        print(f"\n  → LSTM embeddings do not improve over flat features")
        print(f"    LightGBM with engineered features already captures available signal")

    elapsed = time.time() - t_start
    print(f"\n  Total runtime: {elapsed:.1f}s")

    # ── Save ──
    jp = os.path.join(CFG["RESULTS_DIR"], "vol_prediction_v3_results.json")
    save = {}
    for name, r in all_r.items():
        save[name] = {k: v for k, v in r.items()
                      if isinstance(v, (int, float, str, dict))}
    save["config"] = {
        "model_d_features": len(fnames_d),
        "lstm_emb_features": 64,
        "total_features": len(fnames_e),
        "model_e_approach": e_approach,
        "model_d_approach": d_approach,
    }
    save["category_importance"] = {cat: float(val/total_imp)
                                   for cat, val in cats.items()}
    with open(jp, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"  Saved: {jp}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

"""
meta_label_v2.py
================
Improved meta-label with MFE/MAE analysis and adaptive parameters.

Previous v1 issues:
  - 47 trades in 2 years (bottom), 0-2 trades (top)
  - TP/SL (0.5%/0.3%) didn't match 15min dynamics
  - CNN threshold=0.7 too strict
  - Same TP/SL for long and short

5-step pipeline:
  1. MFE/MAE analysis → understand price dynamics after CNN signals
  2. Auto-calculate optimal TP/SL from data (not guessed)
  3. Build features + labels with optimal parameters
  4. Train LightGBM (separate bottom/top + vol-gated top)
  5. Evaluate with baselines, per-ticker, feature importance
  6. 3-way comparison: fixed 0.5/0.3 vs MFE-optimized vs ATR-adaptive

Key metric: at threshold where N_trades > 100, win rate > 55%.

Usage (Colab):
    !pip install lightgbm --quiet
    !python meta_label_v2.py
"""

import os, warnings, time, json
import numpy as np
import pandas as pd
from scipy import stats as sp_stats

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print("  [WARN] lightgbm not installed: pip install lightgbm")
    HAS_LGB = False

# ─────────────────────────── CONFIG ───────────────────────────

CONFIG = {
    "FEATURES_DIR": "features",
    "RESULTS_DIR": "results",
    "TICKERS": ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"],
    "FREQ": "15min",
    "CNN_BOTTOM_PATH": "results/cnn_bottom_predictions.csv",
    "CNN_TOP_PATH": "results/cnn_top_predictions.csv",
    "CNN_THRESHOLD": 0.5,
    "MAX_BARS_OPTIONS": [12, 24, 48],
    "TRAIN_RATIO": 0.7,
    # ATR-adaptive barriers (for comparison)
    "ATR_TP_MULT": 1.5,
    "ATR_SL_MULT": 1.0,
    "ATR_PERIOD": 14,
    # LightGBM
    "LGB_PARAMS": {
        "objective": "binary",
        "metric": "binary_logloss",
        "n_estimators": 500,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_samples": 20,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "verbose": -1,
        "random_state": 42,
    },
    "LGB_THRESHOLDS": [0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    "OUTPUT_DIR": "results",
}


# ═══════════════════════════════════════════════════════════════
#  UTILITIES
# ═══════════════════════════════════════════════════════════════

def load_features(ticker, cfg):
    for freq in [cfg["FREQ"], "1hour"]:
        p = os.path.join(cfg["FEATURES_DIR"], f"{ticker}_{freq}_features.csv")
        if os.path.exists(p):
            df = pd.read_csv(p)
            for col in ["timestamp", "ts_event", "datetime", "date"]:
                if col in df.columns:
                    df["timestamp"] = pd.to_datetime(df[col], utc=True)
                    break
            if "timestamp" not in df.columns:
                first = df.columns[0]
                if first not in df.select_dtypes(include=[np.number]).columns:
                    df["timestamp"] = pd.to_datetime(df[first], utc=True)
            df = df.sort_values("timestamp").reset_index(drop=True)
            return df
    raise FileNotFoundError(f"No features for {ticker}")


def load_cnn_preds(cfg, task):
    path = cfg["CNN_BOTTOM_PATH"] if task == "bottom" else cfg["CNN_TOP_PATH"]
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def map_cnn_to_features(ticker_cnn, n_feat, test_frac=0.15):
    """Map CNN prediction rows → feature-file indices in test period."""
    n_cnn = len(ticker_cnn)
    test_start = int(n_feat * (1.0 - test_frac))
    test_idx = np.arange(test_start, n_feat)
    n_test = len(test_idx)
    if n_cnn == n_test:
        return test_idx
    elif n_cnn > 0:
        scale = n_test / n_cnn
        return test_idx[np.clip((np.arange(n_cnn) * scale).astype(int),
                                0, n_test - 1)]
    return np.array([], dtype=int)


def compute_atr(high, low, close, period=14):
    n = len(close)
    tr = np.zeros(n)
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i] - low[i],
                     abs(high[i] - close[i - 1]),
                     abs(low[i] - close[i - 1]))
    atr = np.full(n, np.nan)
    if n >= period:
        atr[period - 1] = np.mean(tr[:period])
        for i in range(period, n):
            atr[i] = (atr[i - 1] * (period - 1) + tr[i]) / period
    return atr


def get_candidates(cfg, task):
    """Return list of (ticker, feature_df, cand_feat_idx, cand_probs)."""
    cnn_df = load_cnn_preds(cfg, task)
    th = cfg["CNN_THRESHOLD"]
    out = []
    for ticker in cfg["TICKERS"]:
        tc = cnn_df[cnn_df["ticker"] == ticker].reset_index(drop=True)
        if len(tc) == 0:
            continue
        try:
            df = load_features(ticker, cfg)
        except FileNotFoundError:
            continue
        feat_all = map_cnn_to_features(tc, len(df))
        mask = tc["cnn_prob"].values >= th
        pos = np.where(mask)[0]
        if len(pos) == 0:
            continue
        out.append((ticker, df, feat_all[pos], tc["cnn_prob"].values[mask]))
    return out


# ═══════════════════════════════════════════════════════════════
#  TRIPLE-BARRIER LABELLING
# ═══════════════════════════════════════════════════════════════

def triple_barrier(close, high, low, idx, direction,
                   tp_pct, sl_pct, max_bars):
    """Returns (label, exit_type, pnl_pct)."""
    entry = close[idx]
    n = len(close)
    for i in range(1, min(max_bars + 1, n - idx)):
        bi = idx + i
        if direction == "long":
            tp_hit = (high[bi] - entry) / entry >= tp_pct
            sl_hit = (entry - low[bi]) / entry >= sl_pct
        else:
            tp_hit = (entry - low[bi]) / entry >= tp_pct
            sl_hit = (high[bi] - entry) / entry >= sl_pct
        if tp_hit and sl_hit:
            return 0, "SL", -sl_pct
        elif tp_hit:
            return 1, "TP", tp_pct
        elif sl_hit:
            return 0, "SL", -sl_pct
    # timeout
    if idx + max_bars < n:
        ep = close[min(idx + max_bars, n - 1)]
        pnl = ((ep - entry) / entry if direction == "long"
               else (entry - ep) / entry)
    else:
        pnl = 0.0
    return 0, "timeout", pnl


# ═══════════════════════════════════════════════════════════════
#  STEP 1: MFE / MAE ANALYSIS
# ═══════════════════════════════════════════════════════════════

def compute_mfe_mae_single(close, high, low, idx, direction, max_bars):
    n = len(close)
    end = min(idx + max_bars + 1, n)
    if idx + 1 >= end:
        return 0.0, 0.0
    entry = close[idx]
    seg_h = high[idx + 1:end]
    seg_l = low[idx + 1:end]
    if len(seg_h) == 0:
        return 0.0, 0.0
    if direction == "long":
        mfe = max((np.max(seg_h) - entry) / entry, 0.0)
        mae = max((entry - np.min(seg_l)) / entry, 0.0)
    else:
        mfe = max((entry - np.min(seg_l)) / entry, 0.0)
        mae = max((np.max(seg_h) - entry) / entry, 0.0)
    return mfe, mae


def run_mfe_mae_analysis(cfg):
    """Step 1: Analyze MFE/MAE across all candidates."""
    print(f"\n{'='*78}")
    print(f"  STEP 1: MFE/MAE ANALYSIS  (CNN threshold={cfg['CNN_THRESHOLD']})")
    print(f"{'='*78}")

    results = {}
    for task in ["bottom", "top"]:
        direction = "long" if task == "bottom" else "short"
        print(f"\n  ── {task.upper()} ({direction}) ──")
        candidates = get_candidates(cfg, task)
        results[task] = {}

        for mb in cfg["MAX_BARS_OPTIONS"]:
            all_mfe, all_mae = [], []
            ticker_data = {}

            for ticker, df, cand_idx, cand_probs in candidates:
                close = df["close"].values
                high_ = df["high"].values if "high" in df.columns else close
                low_ = df["low"].values if "low" in df.columns else close
                t_mfe, t_mae = [], []
                for idx in cand_idx:
                    mfe, mae = compute_mfe_mae_single(
                        close, high_, low_, idx, direction, mb)
                    t_mfe.append(mfe)
                    t_mae.append(mae)
                all_mfe.extend(t_mfe)
                all_mae.extend(t_mae)
                if t_mfe:
                    ticker_data[ticker] = {
                        "mfe_med": float(np.median(t_mfe)),
                        "mae_med": float(np.median(t_mae)),
                        "n": len(t_mfe),
                    }

            all_mfe = np.array(all_mfe)
            all_mae = np.array(all_mae)
            results[task][mb] = {
                "mfe": all_mfe, "mae": all_mae,
                "ticker_data": ticker_data,
            }

            ptiles = [10, 25, 40, 50, 60, 75, 90]
            print(f"\n  {task.upper()} MFE (MAX_BARS={mb}, N={len(all_mfe):,}):")
            mfe_p = [np.percentile(all_mfe, p) * 100 for p in ptiles]
            print(f"    " + "  ".join(
                f"P{p}={v:.3f}%" for p, v in zip(ptiles, mfe_p)))

            print(f"  {task.upper()} MAE (MAX_BARS={mb}, N={len(all_mae):,}):")
            mae_p = [np.percentile(all_mae, p) * 100 for p in ptiles]
            print(f"    " + "  ".join(
                f"P{p}={v:.3f}%" for p, v in zip(ptiles, mae_p)))

            mfe_50 = np.percentile(all_mfe, 50)
            mae_50 = np.percentile(all_mae, 50)
            ratio = mfe_50 / mae_50 if mae_50 > 0 else 999
            print(f"  MFE/MAE ratio at median: {ratio:.2f}")

            print(f"\n  Per-ticker medians (MAX_BARS={mb}):")
            print(f"  {'Ticker':<8} {'MFE_med':>8} {'MAE_med':>8} "
                  f"{'Ratio':>6} {'N':>6}")
            print(f"  {'-'*42}")
            for t in sorted(ticker_data.keys()):
                d = ticker_data[t]
                r = d["mfe_med"] / d["mae_med"] if d["mae_med"] > 0 else 999
                print(f"  {t:<8} {d['mfe_med']*100:>7.3f}% "
                      f"{d['mae_med']*100:>7.3f}% {r:>6.2f} {d['n']:>6}")

    return results


# ═══════════════════════════════════════════════════════════════
#  STEP 2: AUTO-CALCULATE OPTIMAL TP/SL
# ═══════════════════════════════════════════════════════════════

def compute_base_wr(cfg, task, tp_pct, sl_pct, max_bars):
    """Base win rate across all CNN candidates (no model filtering)."""
    direction = "long" if task == "bottom" else "short"
    candidates = get_candidates(cfg, task)
    total, wins = 0, 0
    for ticker, df, cand_idx, _ in candidates:
        close = df["close"].values
        high_ = df["high"].values if "high" in df.columns else close
        low_ = df["low"].values if "low" in df.columns else close
        for idx in cand_idx:
            if idx < 1 or idx >= len(close) - 1:
                continue
            lab, _, _ = triple_barrier(close, high_, low_, idx,
                                       direction, tp_pct, sl_pct, max_bars)
            total += 1
            wins += lab
    return (wins / total if total > 0 else 0), total, wins


def run_optimal_params(cfg, mfe_mae):
    """Step 2: Grid search for optimal TP/SL per task."""
    print(f"\n{'='*78}")
    print(f"  STEP 2: AUTO-CALCULATE OPTIMAL TP/SL")
    print(f"{'='*78}")

    optimal = {}
    for task in ["bottom", "top"]:
        direction = "long" if task == "bottom" else "short"
        print(f"\n  ── {task.upper()} ({direction}) ──")

        best_edge = -999
        best_params = None
        rows = []

        for mb in cfg["MAX_BARS_OPTIONS"]:
            data = mfe_mae[task].get(mb)
            if data is None or len(data["mfe"]) == 0:
                continue
            mfe_arr = data["mfe"]
            mae_arr = data["mae"]

            tp_opts = [
                ("MFE_P25", float(np.percentile(mfe_arr, 25))),
                ("MFE_P40", float(np.percentile(mfe_arr, 40))),
                ("MFE_P50", float(np.percentile(mfe_arr, 50))),
            ]
            sl_opts = [
                ("MAE_P50", float(np.percentile(mae_arr, 50))),
                ("MAE_P60", float(np.percentile(mae_arr, 60))),
                ("MAE_P75", float(np.percentile(mae_arr, 75))),
            ]

            for tp_name, tp_pct in tp_opts:
                for sl_name, sl_pct in sl_opts:
                    if tp_pct <= 0 or sl_pct <= 0:
                        continue
                    be_wr = sl_pct / (tp_pct + sl_pct)
                    rr = tp_pct / sl_pct
                    wr, n_total, n_wins = compute_base_wr(
                        cfg, task, tp_pct, sl_pct, mb)
                    edge = wr - be_wr

                    rows.append({
                        "mb": mb, "tp_name": tp_name, "sl_name": sl_name,
                        "tp_pct": tp_pct, "sl_pct": sl_pct,
                        "rr": rr, "be_wr": be_wr,
                        "base_wr": wr, "edge": edge,
                        "n_total": n_total, "n_wins": n_wins,
                    })
                    if edge > best_edge and n_total >= 50:
                        best_edge = edge
                        best_params = rows[-1]

        rows.sort(key=lambda r: r["edge"], reverse=True)
        print(f"\n  TP/SL Grid ({task}):")
        print(f"  {'MB':>3} {'TP':>10} {'SL':>10} {'TP%':>7} {'SL%':>7} "
              f"{'RR':>5} {'BE_WR':>7} {'BaseWR':>7} {'Edge':>8} {'N':>6}")
        print(f"  {'-'*80}")
        for r in rows[:15]:
            star = " ★" if r is best_params else ""
            print(f"  {r['mb']:>3} {r['tp_name']:>10} {r['sl_name']:>10} "
                  f"{r['tp_pct']*100:>6.3f}% {r['sl_pct']*100:>6.3f}% "
                  f"{r['rr']:>5.2f} {r['be_wr']*100:>6.1f}% "
                  f"{r['base_wr']*100:>6.1f}% "
                  f"{r['edge']*100:>+7.1f}pp "
                  f"{r['n_total']:>6}{star}")

        if best_params:
            print(f"\n  ★ OPTIMAL {task.upper()}: "
                  f"TP={best_params['tp_pct']*100:.3f}% "
                  f"SL={best_params['sl_pct']*100:.3f}% "
                  f"MB={best_params['mb']} "
                  f"RR={best_params['rr']:.2f}:1")
            print(f"    Base WR={best_params['base_wr']*100:.1f}% "
                  f"BE={best_params['be_wr']*100:.1f}% "
                  f"Edge={best_params['edge']*100:+.1f}pp "
                  f"N={best_params['n_total']}")
            optimal[task] = best_params
        else:
            print(f"  [WARN] No positive edge, using MFE_P40/MAE_P60 from MB=24")
            d = mfe_mae[task].get(24, mfe_mae[task].get(
                cfg["MAX_BARS_OPTIONS"][0]))
            tp = float(np.percentile(d["mfe"], 40))
            sl = float(np.percentile(d["mae"], 60))
            optimal[task] = {
                "tp_pct": max(tp, 0.001), "sl_pct": max(sl, 0.001),
                "mb": 24, "rr": tp / sl if sl > 0 else 1.0,
                "be_wr": sl / (tp + sl) if (tp + sl) > 0 else 0.5,
                "base_wr": 0.5, "edge": 0.0, "n_total": 0,
            }

    return optimal


# ═══════════════════════════════════════════════════════════════
#  STEP 3: BUILD FEATURES AND LABELS
# ═══════════════════════════════════════════════════════════════

def build_candidate_features(df, cand_idx, cand_probs, tech_cols):
    """Build feature matrix for LightGBM at candidate indices."""
    close = df["close"].values
    high_ = df["high"].values if "high" in df.columns else close
    low_ = df["low"].values if "low" in df.columns else close
    vol = df["volume"].values if "volume" in df.columns else np.ones(len(df))

    rows = []
    for k, idx in enumerate(cand_idx):
        if idx < 20 or idx >= len(close):
            continue
        r = {}
        c = close[idx]

        # --- CNN features (2) ---
        r["cnn_prob"] = cand_probs[k]
        start = max(0, k - 200)
        recent = cand_probs[start:k + 1]
        r["cnn_prob_rank"] = np.sum(recent <= cand_probs[k]) / max(len(recent), 1)

        # --- Price action (8) ---
        for lb in [5, 10, 20]:
            r[f"ret_{lb}"] = (c - close[max(0, idx - lb)]) / (
                abs(close[max(0, idx - lb)]) + 1e-10)
        for lb in [5, 10]:
            seg = close[max(0, idx - lb):idx + 1]
            rets = (np.diff(seg) / (np.abs(seg[:-1]) + 1e-10)
                    if len(seg) > 1 else [0])
            r[f"rvol_{lb}"] = float(np.std(rets))
        h20 = np.max(high_[max(0, idx - 20):idx + 1])
        l20 = np.min(low_[max(0, idx - 20):idx + 1])
        r["dist_high_20"] = (c - h20) / (c + 1e-10)
        r["dist_low_20"] = (c - l20) / (c + 1e-10)
        r["bar_range"] = (high_[idx] - low_[idx]) / (c + 1e-10)

        # --- Technical indicators from CSV ---
        for col in tech_cols:
            v = df[col].iloc[idx]
            r[col] = float(v) if not pd.isna(v) else 0.0

        # --- Bollinger position ---
        bb_u = bb_l = None
        for col in tech_cols:
            cl = col.lower()
            if any(s in cl for s in ["bbu", "bb_upper", "upper_band"]):
                bb_u = df[col].iloc[idx]
            elif any(s in cl for s in ["bbl", "bb_lower", "lower_band"]):
                bb_l = df[col].iloc[idx]
        if bb_u is not None and bb_l is not None:
            rng = bb_u - bb_l
            r["bb_position"] = (c - bb_l) / rng if abs(rng) > 1e-10 else 0.5

        # --- Volume ratio ---
        vol_20 = np.mean(vol[max(0, idx - 20):idx]) if idx >= 1 else 1
        r["volume_ratio"] = vol[idx] / (vol_20 + 1e-10)

        # --- Time features (2) ---
        if "timestamp" in df.columns:
            ts = df["timestamp"].iloc[idx]
            r["hour"] = ts.hour
            r["day_of_week"] = ts.dayofweek

        r["_idx"] = idx
        rows.append(r)
    return pd.DataFrame(rows)


def build_dataset(cfg, task, tp_pct, sl_pct, max_bars, label_tag=""):
    """Build labeled dataset with given TP/SL/MAX_BARS."""
    direction = "long" if task == "bottom" else "short"
    print(f"\n  Building dataset: {task} {label_tag}"
          f"  TP={tp_pct*100:.3f}% SL={sl_pct*100:.3f}% MB={max_bars}")

    exclude = {"timestamp", "ts_event", "datetime", "date", "time",
               "unnamed: 0", "symbol", "ticker",
               "open", "high", "low", "close", "volume"}

    candidates = get_candidates(cfg, task)
    frames = []
    stats = {"total": 0, "tp": 0, "sl": 0, "timeout": 0}

    for ticker, df, cand_idx, cand_probs in candidates:
        close = df["close"].values
        high_ = df["high"].values if "high" in df.columns else close
        low_ = df["low"].values if "low" in df.columns else close

        tech_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                     if c.lower() not in exclude]
        feat_df = build_candidate_features(df, cand_idx, cand_probs, tech_cols)
        if len(feat_df) == 0:
            continue

        feat_set = set(feat_df["_idx"].values)
        labels, exits, pnls = [], [], []
        for idx in cand_idx:
            if idx not in feat_set or idx < 1 or idx >= len(close) - 1:
                labels.append(np.nan); exits.append("skip"); pnls.append(0)
                continue
            lab, ex, pnl = triple_barrier(close, high_, low_, idx,
                                          direction, tp_pct, sl_pct, max_bars)
            labels.append(lab); exits.append(ex); pnls.append(pnl)

        aligned_l, aligned_e, aligned_p = [], [], []
        for k_idx in range(len(cand_idx)):
            if cand_idx[k_idx] in feat_set:
                aligned_l.append(labels[k_idx])
                aligned_e.append(exits[k_idx])
                aligned_p.append(pnls[k_idx])

        ml = min(len(aligned_l), len(feat_df))
        feat_df = feat_df.iloc[:ml].copy()
        feat_df["label"] = aligned_l[:ml]
        feat_df["exit_type"] = aligned_e[:ml]
        feat_df["pnl_pct"] = aligned_p[:ml]
        feat_df["ticker"] = ticker
        feat_df = feat_df.dropna(subset=["label"]).reset_index(drop=True)
        feat_df["label"] = feat_df["label"].astype(int)

        nt = len(feat_df)
        n_tp = (feat_df["exit_type"] == "TP").sum()
        n_sl = (feat_df["exit_type"] == "SL").sum()
        n_to = (feat_df["exit_type"] == "timeout").sum()
        stats["total"] += nt; stats["tp"] += n_tp
        stats["sl"] += n_sl; stats["timeout"] += n_to
        wr = n_tp / nt if nt > 0 else 0
        print(f"    {ticker}: {nt:>5}  WR={wr*100:>5.1f}%  "
              f"TP={n_tp} SL={n_sl} TO={n_to}")
        frames.append(feat_df)

    if not frames:
        return pd.DataFrame(), stats
    combined = pd.concat(frames, ignore_index=True)
    t = stats["total"]
    wr = stats["tp"] / t if t > 0 else 0
    print(f"\n  TOTAL: {t:,}  base WR={wr*100:.1f}%  "
          f"(TP={stats['tp']} SL={stats['sl']} TO={stats['timeout']})")
    return combined, stats


def build_atr_dataset(cfg, task, max_bars):
    """Build ATR-adaptive barrier dataset for comparison."""
    direction = "long" if task == "bottom" else "short"
    exclude = {"timestamp", "ts_event", "datetime", "date", "time",
               "unnamed: 0", "symbol", "ticker",
               "open", "high", "low", "close", "volume"}
    candidates = get_candidates(cfg, task)
    frames = []
    stats = {"total": 0, "tp": 0, "sl": 0, "timeout": 0}

    for ticker, df, cand_idx, cand_probs in candidates:
        close = df["close"].values
        high_ = df["high"].values if "high" in df.columns else close
        low_ = df["low"].values if "low" in df.columns else close
        atr = compute_atr(high_, low_, close, cfg["ATR_PERIOD"])

        tech_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                     if c.lower() not in exclude]
        feat_df = build_candidate_features(df, cand_idx, cand_probs, tech_cols)
        if len(feat_df) == 0:
            continue
        feat_set = set(feat_df["_idx"].values)
        labels, exits, pnls = [], [], []
        for idx in cand_idx:
            if idx not in feat_set or idx < 1 or idx >= len(close) - 1:
                labels.append(np.nan); exits.append("skip"); pnls.append(0)
                continue
            a = atr[idx]
            if np.isnan(a) or a <= 0:
                a = close[idx] * 0.003
            tp = (a * cfg["ATR_TP_MULT"]) / close[idx]
            sl = (a * cfg["ATR_SL_MULT"]) / close[idx]
            lab, ex, pnl = triple_barrier(close, high_, low_, idx,
                                          direction, tp, sl, max_bars)
            labels.append(lab); exits.append(ex); pnls.append(pnl)

        aligned_l, aligned_e, aligned_p = [], [], []
        for k in range(len(cand_idx)):
            if cand_idx[k] in feat_set:
                aligned_l.append(labels[k])
                aligned_e.append(exits[k])
                aligned_p.append(pnls[k])
        ml = min(len(aligned_l), len(feat_df))
        feat_df = feat_df.iloc[:ml].copy()
        feat_df["label"] = aligned_l[:ml]
        feat_df["exit_type"] = aligned_e[:ml]
        feat_df["pnl_pct"] = aligned_p[:ml]
        feat_df["ticker"] = ticker
        feat_df = feat_df.dropna(subset=["label"]).reset_index(drop=True)
        feat_df["label"] = feat_df["label"].astype(int)
        nt = len(feat_df)
        stats["total"] += nt
        stats["tp"] += (feat_df["exit_type"] == "TP").sum()
        stats["sl"] += (feat_df["exit_type"] == "SL").sum()
        stats["timeout"] += (feat_df["exit_type"] == "timeout").sum()
        frames.append(feat_df)

    if not frames:
        return pd.DataFrame(), stats
    return pd.concat(frames, ignore_index=True), stats


# ═══════════════════════════════════════════════════════════════
#  STEP 4: TRAIN LightGBM
# ═══════════════════════════════════════════════════════════════

def get_lgb_cols(df):
    skip = {"label", "exit_type", "pnl_pct", "ticker", "task", "_idx"}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in skip]


def train_lgb(train_df, feat_cols, cfg, label):
    X = np.nan_to_num(train_df[feat_cols].values, nan=0.0,
                      posinf=0.0, neginf=0.0)
    y = train_df["label"].values
    vs = int(len(y) * 0.8)
    X_t, y_t = X[:vs], y[:vs]
    X_v, y_v = X[vs:], y[vs:]
    print(f"    LGB [{label}]: train={len(y_t)} val={len(y_v)} "
          f"pos_tr={y_t.mean():.3f} pos_va={y_v.mean():.3f}")
    params = dict(cfg["LGB_PARAMS"])
    n_est = params.pop("n_estimators", 500)
    model = lgb.LGBMClassifier(n_estimators=n_est, **params)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)],
              callbacks=[lgb.early_stopping(30, verbose=False),
                         lgb.log_evaluation(0)])
    print(f"    Best iter: {getattr(model, 'best_iteration_', n_est)}")
    return model


# ═══════════════════════════════════════════════════════════════
#  STEP 5: EVALUATE
# ═══════════════════════════════════════════════════════════════

def eval_thresholds(probs, y, pnls, thresholds, base_rate):
    out = {}
    for th in thresholds:
        m = probs >= th
        n = int(m.sum())
        if n == 0:
            out[th] = dict(win_rate=0, n_trades=0, n_wins=0,
                           profit_factor=0, total_pnl=0, avg_pnl=0,
                           z_score=0, p_value=1)
            continue
        ys, ps = y[m], pnls[m]
        nw = int(ys.sum())
        wr = nw / n
        gp = ps[ps > 0].sum()
        gl = abs(ps[ps < 0].sum())
        pf = gp / gl if gl > 0 else (999.0 if gp > 0 else 0.0)
        se = np.sqrt(base_rate * (1 - base_rate) / n) if 0 < base_rate < 1 else 1
        z = (wr - base_rate) / se if se > 0 else 0
        pv = 2 * (1 - sp_stats.norm.cdf(abs(z)))
        out[th] = dict(win_rate=wr, n_trades=n, n_wins=nw,
                       profit_factor=pf, total_pnl=float(ps.sum()),
                       avg_pnl=float(ps.sum()) / n,
                       z_score=z, p_value=pv)
    return out


def eval_baseline(test_df, mask, base_rate, name):
    y = test_df["label"].values
    pnls = test_df["pnl_pct"].values
    n = int(mask.sum())
    if n == 0:
        return dict(name=name, win_rate=0, n_trades=0, n_wins=0,
                    profit_factor=0, total_pnl=0, z_score=0, p_value=1)
    ys, ps = y[mask], pnls[mask]
    nw = int(ys.sum()); wr = nw / n
    gp = ps[ps > 0].sum(); gl = abs(ps[ps < 0].sum())
    pf = gp / gl if gl > 0 else (999.0 if gp > 0 else 0.0)
    se = np.sqrt(base_rate * (1 - base_rate) / n) if 0 < base_rate < 1 else 1
    z = (wr - base_rate) / se if se > 0 else 0
    pv = 2 * (1 - sp_stats.norm.cdf(abs(z)))
    return dict(name=name, win_rate=wr, n_trades=n, n_wins=nw,
                profit_factor=pf, total_pnl=float(ps.sum()),
                z_score=z, p_value=pv)


# ── Printing helpers ──

def print_threshold_table(results, task, base_rate, be_wr, barrier_tag):
    print(f"\n  ┌{'─'*80}┐")
    print(f"  │  {task.upper()} — LightGBM [{barrier_tag}]  "
          f"base={base_rate*100:.1f}%  BE={be_wr*100:.1f}%"
          f"{' '*(80-len(task)-len(barrier_tag)-42)}│")
    print(f"  ├{'─'*80}┤")
    print(f"  │ {'Th':>5} │ {'WinRate':>8} │ {'N_trade':>8} │ "
          f"{'N_win':>6} │ {'PF':>7} │ {'TotPnL':>9} │ "
          f"{'AvgPnL':>9} │ {'z':>7} │")
    print(f"  ├{'─'*80}┤")
    for th in sorted(results.keys()):
        r = results[th]
        sig = ("***" if r["p_value"] < 0.001 else "**" if r["p_value"] < 0.01
               else "*" if r["p_value"] < 0.05 else "")
        ws = f"{r['win_rate']*100:.1f}%" if r["n_trades"] > 0 else "—"
        print(f"  │ {th:>5.2f} │ {ws:>8} │ {r['n_trades']:>8} │ "
              f"{r['n_wins']:>6} │ {r['profit_factor']:>7.2f} │ "
              f"{r['total_pnl']*100:>8.3f}% │ "
              f"{r['avg_pnl']*100:>8.4f}% │ "
              f"{r['z_score']:>+6.2f}{sig:>1} │")
    print(f"  └{'─'*80}┘")


def print_baselines(baselines, task):
    print(f"\n  {task.upper()} — BASELINES:")
    print(f"  {'Name':<22} {'WR':>8} {'N':>7} {'PF':>7} "
          f"{'TotPnL':>9} {'z':>7}")
    print(f"  {'-'*62}")
    for b in baselines:
        sig = "*" if b["p_value"] < 0.05 else ""
        ws = f"{b['win_rate']*100:.1f}%" if b["n_trades"] > 0 else "—"
        print(f"  {b['name']:<22} {ws:>8} {b['n_trades']:>7} "
              f"{b['profit_factor']:>7.2f} "
              f"{b['total_pnl']*100:>8.3f}% "
              f"{b['z_score']:>+6.2f}{sig}")


def print_importance(model, feat_cols, top_n=20):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1]
    print(f"\n  TOP {top_n} FEATURES:")
    print(f"  {'#':>3} {'Feature':<35} {'Imp':>8}")
    print(f"  {'-'*50}")
    for i in range(min(top_n, len(feat_cols))):
        print(f"  {i+1:>3} {feat_cols[idx[i]]:<35} {imp[idx[i]]:>8}")


def print_per_ticker(test_df, probs, threshold, task):
    y = test_df["label"].values
    pnls = test_df["pnl_pct"].values
    print(f"\n  {task.upper()} — PER-TICKER @ threshold={threshold}:")
    print(f"  {'Ticker':<8} {'FiltWR':>8} {'N_filt':>7} {'BaseWR':>8} "
          f"{'N_all':>7} {'PF':>7} {'TotPnL':>9}")
    print(f"  {'-'*62}")
    for ticker in sorted(test_df["ticker"].unique()):
        mt = (test_df["ticker"] == ticker).values
        pt, yt, pnlt = probs[mt], y[mt], pnls[mt]
        bwr = yt.mean()
        fm = pt >= threshold
        nf = int(fm.sum())
        if nf == 0:
            print(f"  {ticker:<8} {'—':>8} {0:>7} "
                  f"{bwr*100:>7.1f}% {int(mt.sum()):>7}")
            continue
        fwr = yt[fm].mean()
        pf_ = pnlt[fm]
        gp = pf_[pf_ > 0].sum(); gl = abs(pf_[pf_ < 0].sum())
        pf = gp / gl if gl > 0 else 999.0
        print(f"  {ticker:<8} {fwr*100:>7.1f}% {nf:>7} "
              f"{bwr*100:>7.1f}% {int(mt.sum()):>7} "
              f"{pf:>7.2f} {pf_.sum()*100:>8.3f}%")


# ═══════════════════════════════════════════════════════════════
#  PIPELINE: run one barrier mode for one task
# ═══════════════════════════════════════════════════════════════

def run_barrier_mode(cfg, task, dataset, stats, tp_pct, sl_pct,
                     max_bars, barrier_tag):
    """Train LightGBM and evaluate for one barrier mode."""
    if len(dataset) < 50:
        print(f"  [SKIP] {barrier_tag}: only {len(dataset)} samples")
        return {}

    dataset = dataset.sort_values("_idx").reset_index(drop=True)
    n = len(dataset)
    sp = int(n * cfg["TRAIN_RATIO"])
    train_df = dataset.iloc[:sp].reset_index(drop=True)
    test_df = dataset.iloc[sp:].reset_index(drop=True)

    base_rate = test_df["label"].mean()
    be_wr = sl_pct / (tp_pct + sl_pct) if (tp_pct + sl_pct) > 0 else 0.5
    edge = base_rate - be_wr

    print(f"\n  [{barrier_tag}] Train={len(train_df):,} Test={len(test_df):,}")
    print(f"  Base WR={base_rate*100:.1f}%  BE={be_wr*100:.1f}%  "
          f"Edge={edge*100:+.1f}pp")

    feat_cols = get_lgb_cols(dataset)
    print(f"  Features: {len(feat_cols)}")

    if not HAS_LGB or len(train_df) < 50:
        return {"base_rate": float(base_rate), "be_wr": float(be_wr),
                "n_test": len(test_df)}

    model = train_lgb(train_df, feat_cols, cfg, f"{task}-{barrier_tag}")

    X_test = np.nan_to_num(test_df[feat_cols].values, nan=0.0,
                           posinf=0.0, neginf=0.0)
    probs = model.predict_proba(X_test)[:, 1]
    y_test = test_df["label"].values
    pnls_test = test_df["pnl_pct"].values

    lgb_res = eval_thresholds(probs, y_test, pnls_test,
                              cfg["LGB_THRESHOLDS"], base_rate)
    print_threshold_table(lgb_res, task, base_rate, be_wr, barrier_tag)

    # Baselines
    nt = len(test_df)
    baselines = [
        eval_baseline(test_df, np.ones(nt, dtype=bool), base_rate, "CNN alone"),
        eval_baseline(test_df, np.random.RandomState(42).random(nt) > 0.5,
                      base_rate, "Random 50%"),
    ]
    rsi_cols = [c for c in feat_cols if c.lower() in ("rsi_14", "rsi")]
    if rsi_cols:
        rv = test_df[rsi_cols[0]].values
        if task == "bottom":
            baselines.append(eval_baseline(test_df, rv < 30, base_rate, "RSI<30"))
        else:
            baselines.append(eval_baseline(test_df, rv > 70, base_rate, "RSI>70"))
    vol_cols = [c for c in feat_cols
                if "rvol" in c.lower() or (
                    "vol" in c.lower() and "volume" not in c.lower())]
    if vol_cols:
        vv = test_df[vol_cols[0]].values
        baselines.append(eval_baseline(test_df, vv > np.median(vv),
                                       base_rate, "High Vol"))
    print_baselines(baselines, task)

    print_importance(model, feat_cols)

    # Best threshold with N>100 and N>20
    best_100 = max(
        cfg["LGB_THRESHOLDS"],
        key=lambda th: (lgb_res[th]["win_rate"]
                        if lgb_res[th]["n_trades"] >= 100 else 0))
    best_20 = max(
        cfg["LGB_THRESHOLDS"],
        key=lambda th: (lgb_res[th]["win_rate"]
                        if lgb_res[th]["n_trades"] >= 20 else 0))

    print_per_ticker(test_df, probs, best_100 if lgb_res[best_100]["n_trades"] >= 100
                     else best_20, task)

    return {
        "base_rate": float(base_rate),
        "be_wr": float(be_wr),
        "edge_no_model": float(edge),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "tp_pct": float(tp_pct),
        "sl_pct": float(sl_pct),
        "max_bars": int(max_bars),
        "thresholds": {
            str(k): {kk: (float(vv) if isinstance(vv, (float, np.floating))
                          else int(vv) if isinstance(vv, (int, np.integer))
                          else vv)
                     for kk, vv in v.items()}
            for k, v in lgb_res.items()
        },
        "baselines": {b["name"]: {k: (float(v) if isinstance(v, (float, np.floating))
                                       else int(v) if isinstance(v, (int, np.integer))
                                       else v)
                                   for k, v in b.items() if k != "name"}
                      for b in baselines},
        "feature_importance": {
            feat_cols[i]: float(model.feature_importances_[i])
            for i in np.argsort(model.feature_importances_)[::-1][:20]
        },
        "best_th_100": float(best_100),
        "best_th_20": float(best_20),
    }


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════

def main():
    cfg = CONFIG
    print("=" * 78)
    print("  Meta-Label v2: MFE/MAE → Optimal TP/SL → LightGBM Filter")
    print(f"  CNN threshold: {cfg['CNN_THRESHOLD']}")
    print(f"  Tickers: {cfg['TICKERS']}")
    print("=" * 78)

    os.makedirs(cfg["OUTPUT_DIR"], exist_ok=True)

    # ──── STEP 1: MFE/MAE ────
    mfe_mae = run_mfe_mae_analysis(cfg)

    # ──── STEP 2: Optimal TP/SL ────
    optimal = run_optimal_params(cfg, mfe_mae)

    # ──── STEPS 3-5 per task: 3-way comparison ────
    all_results = {}
    for task in ["bottom", "top"]:
        direction = "LONG" if task == "bottom" else "SHORT"
        print(f"\n{'#'*78}")
        print(f"  TASK: {task.upper()} → {direction}")
        print(f"{'#'*78}")

        opt = optimal[task]
        opt_tp = opt["tp_pct"]
        opt_sl = opt["sl_pct"]
        opt_mb = opt.get("mb", 24)

        task_res = {}

        # ── Mode 1: MFE-optimized ──
        print(f"\n{'='*60}")
        print(f"  BARRIER MODE 1: MFE-OPTIMIZED")
        print(f"  TP={opt_tp*100:.3f}% SL={opt_sl*100:.3f}% MB={opt_mb} "
              f"RR={opt_tp/opt_sl:.2f}:1")
        print(f"{'='*60}")
        ds_opt, st_opt = build_dataset(cfg, task, opt_tp, opt_sl, opt_mb,
                                       label_tag="MFE-opt")
        task_res["mfe_optimized"] = run_barrier_mode(
            cfg, task, ds_opt, st_opt, opt_tp, opt_sl, opt_mb, "MFE-opt")

        # ── Mode 2: Fixed 0.5%/0.3% ──
        fix_tp, fix_sl, fix_mb = 0.005, 0.003, 48
        print(f"\n{'='*60}")
        print(f"  BARRIER MODE 2: FIXED")
        print(f"  TP={fix_tp*100:.1f}% SL={fix_sl*100:.1f}% MB={fix_mb} "
              f"RR={fix_tp/fix_sl:.2f}:1")
        print(f"{'='*60}")
        ds_fix, st_fix = build_dataset(cfg, task, fix_tp, fix_sl, fix_mb,
                                       label_tag="Fixed")
        task_res["fixed"] = run_barrier_mode(
            cfg, task, ds_fix, st_fix, fix_tp, fix_sl, fix_mb, "Fixed")

        # ── Mode 3: ATR-adaptive ──
        # Use median ATR from training data as proxy for be_wr calculation
        atr_tp_mult = cfg["ATR_TP_MULT"]
        atr_sl_mult = cfg["ATR_SL_MULT"]
        print(f"\n{'='*60}")
        print(f"  BARRIER MODE 3: ATR-ADAPTIVE")
        print(f"  TP={atr_tp_mult}×ATR  SL={atr_sl_mult}×ATR  MB={opt_mb} "
              f"RR={atr_tp_mult/atr_sl_mult:.2f}:1")
        print(f"{'='*60}")
        ds_atr, st_atr = build_atr_dataset(cfg, task, opt_mb)
        # For ATR, be_wr uses the multiplier ratio
        atr_tp_proxy = atr_tp_mult * 0.003  # rough proxy
        atr_sl_proxy = atr_sl_mult * 0.003
        task_res["atr"] = run_barrier_mode(
            cfg, task, ds_atr, st_atr,
            atr_tp_proxy, atr_sl_proxy, opt_mb, "ATR")

        # ── Vol-gated top model (top task only) ──
        if task == "top" and len(ds_opt) > 50:
            print(f"\n{'='*60}")
            print(f"  VOL-GATED TOP MODEL (MFE-opt, high vol only)")
            print(f"{'='*60}")
            vol_cols = [c for c in ds_opt.columns
                        if "rvol" in c.lower() or (
                            "vol" in c.lower() and "volume" not in c.lower()
                            and c.lower() not in ("volume_ratio",))]
            if vol_cols:
                vc = vol_cols[0]
                med = ds_opt[vc].median()
                ds_vg = ds_opt[ds_opt[vc] > med].copy()
                print(f"  Vol filter on '{vc}': {len(ds_vg)}/{len(ds_opt)} "
                      f"candidates retained")
                if len(ds_vg) > 50:
                    st_vg = {"total": len(ds_vg),
                             "tp": (ds_vg["exit_type"] == "TP").sum(),
                             "sl": (ds_vg["exit_type"] == "SL").sum(),
                             "timeout": (ds_vg["exit_type"] == "timeout").sum()}
                    task_res["vol_gated"] = run_barrier_mode(
                        cfg, task, ds_vg, st_vg, opt_tp, opt_sl, opt_mb,
                        "Vol-gated")

        all_results[task] = task_res

    # ═══════════════════════════════════════════════════════════
    #  FINAL SUMMARY: 3-WAY COMPARISON
    # ═══════════════════════════════════════════════════════════
    print(f"\n{'='*78}")
    print(f"  ★ FINAL SUMMARY: 3-WAY BARRIER COMPARISON")
    print(f"{'='*78}")

    for task in ["bottom", "top"]:
        direction = "LONG" if task == "bottom" else "SHORT"
        tr = all_results.get(task, {})
        if not tr:
            continue

        print(f"\n  {task.upper()} ({direction}):")
        print(f"  {'Mode':<16} │ {'TP/SL':>14} │ {'BaseWR':>7} │ {'BE':>6} │ "
              f"{'Edge':>7} │ {'BestWR':>7} │ {'@N≥100':>7} │ {'N_test':>7} │")
        print(f"  {'─'*16}─┼{'─'*16}┼{'─'*9}┼{'─'*8}┼"
              f"{'─'*9}┼{'─'*9}┼{'─'*9}┼{'─'*9}┤")

        for mode_name, mode_label in [("mfe_optimized", "MFE-opt"),
                                       ("fixed", "Fixed 0.5/0.3"),
                                       ("atr", "ATR-adaptive"),
                                       ("vol_gated", "Vol-gated")]:
            r = tr.get(mode_name, {})
            if not r or "base_rate" not in r:
                continue
            base = r["base_rate"]
            be = r.get("be_wr", 0)
            edge = base - be
            n_test = r.get("n_test", 0)
            tp_s = f"{r.get('tp_pct',0)*100:.2f}/{r.get('sl_pct',0)*100:.2f}%"

            # Best WR at any threshold with N>0
            ths = r.get("thresholds", {})
            best_wr = 0
            best_wr_100 = 0
            for th_k, th_v in ths.items():
                if th_v.get("n_trades", 0) > 0:
                    best_wr = max(best_wr, th_v["win_rate"])
                if th_v.get("n_trades", 0) >= 100:
                    best_wr_100 = max(best_wr_100, th_v["win_rate"])

            bw_s = f"{best_wr*100:.1f}%" if best_wr > 0 else "—"
            bw100_s = f"{best_wr_100*100:.1f}%" if best_wr_100 > 0 else "—"

            print(f"  {mode_label:<16} │ {tp_s:>14} │ "
                  f"{base*100:>6.1f}% │ {be*100:>5.1f}% │ "
                  f"{edge*100:>+6.1f}pp │ {bw_s:>7} │ "
                  f"{bw100_s:>7} │ {n_test:>7} │")

    # KEY METRIC
    print(f"\n  {'─'*60}")
    print(f"  KEY METRIC: Win Rate where N_trades ≥ 100")
    print(f"  Target: >55% win rate with >100 trades\n")

    for task in ["bottom", "top"]:
        tr = all_results.get(task, {})
        direction = "LONG" if task == "bottom" else "SHORT"
        for mode_name in ["mfe_optimized", "fixed", "atr", "vol_gated"]:
            r = tr.get(mode_name, {})
            if not r:
                continue
            ths = r.get("thresholds", {})
            for th_k in sorted(ths.keys()):
                th_v = ths[th_k]
                if th_v.get("n_trades", 0) >= 100:
                    wr = th_v["win_rate"]
                    nt = th_v["n_trades"]
                    mark = "✓" if wr > 0.55 else "~" if wr > 0.50 else "✗"
                    print(f"    {mark} {task.upper()} {mode_name:<16} "
                          f"th={th_k}: WR={wr*100:.1f}% N={nt}")

    # CONCLUSIONS
    print(f"\n  {'─'*60}")
    print(f"  CONCLUSIONS:")

    for task in ["bottom", "top"]:
        tr = all_results.get(task, {})
        r_opt = tr.get("mfe_optimized", {})
        if not r_opt:
            continue
        base = r_opt.get("base_rate", 0)
        be = r_opt.get("be_wr", 0)
        edge = base - be
        direction = "LONG" if task == "bottom" else "SHORT"

        if edge > 0.05:
            print(f"    ✓ {task.upper()} ({direction}): strong edge "
                  f"{edge*100:+.1f}pp over breakeven")
        elif edge > 0:
            print(f"    ~ {task.upper()} ({direction}): marginal edge "
                  f"{edge*100:+.1f}pp — LGB filter can help")
        else:
            print(f"    ✗ {task.upper()} ({direction}): negative edge "
                  f"{edge*100:+.1f}pp — signal not profitable without filter")

        # Check if LGB improves at N>=100
        ths = r_opt.get("thresholds", {})
        improved = False
        for th_k, th_v in ths.items():
            if th_v.get("n_trades", 0) >= 100:
                if th_v["win_rate"] > base + 0.02:
                    improved = True
                    print(f"      LGB filter improves: "
                          f"th={th_k} WR={th_v['win_rate']*100:.1f}% "
                          f"(+{(th_v['win_rate']-base)*100:.1f}pp) "
                          f"N={th_v['n_trades']}")
                    break
        if not improved:
            print(f"      LGB filter: no significant improvement at N≥100")

    # ── Save ──
    save = {}
    for task in all_results:
        save[task] = {}
        for mode in all_results[task]:
            r = all_results[task][mode]
            if not r:
                continue
            save[task][mode] = {
                k: v for k, v in r.items()
                if k != "feature_importance" or True
            }
    json_path = os.path.join(cfg["OUTPUT_DIR"], "meta_label_v2_results.json")
    with open(json_path, "w") as f:
        json.dump(save, f, indent=2, default=str)
    print(f"\n  Saved: {json_path}")
    print("\n  Done!")


if __name__ == "__main__":
    main()

[ref] Using AAPL_1hour_features.csv as reference → 41 indicator columns

Resampling 1min → 15min + computing indicators
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  AAPL:   574,798 1min rows →  43,349 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  MSFT:   498,006 1min rows →  42,922 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  GOOGL:   404,100 1min rows →  39,076 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  GOOG:   372,249 1min rows →  38,153 15min rows → 61 indicator cols
[WARN] pandas_ta not installed – pip install pandas_ta
  [INFO] Low match rate (8/41), adding all manual indicators
  NVDA:   511,269 1min 

In [15]:
import os
# 检查 vol 相关权重文件
for root, dirs, files in os.walk("/content/drive/MyDrive/毕设"):
    for f in files:
        if f.endswith((".joblib", ".pkl", ".txt")) and "vol" in root.lower():
            print(os.path.join(root, f))

In [8]:
import joblib, torch, json, os

results_dir = "results"

# ── LightGBM 模型 ──
lgb_models = [
    "models/layer1/volatility/lightgbm_v3/weights.joblib",
    "models/layer1/volatility/lightgbm_v3_flat/weights.joblib",
    "models/layer3/trade_filter/lgb_bottom_v1/weights.joblib",
    "models/layer3/trade_filter/lgb_top_v1/weights.joblib",
]

for path in lgb_models:
    full = os.path.join(results_dir, path)
    if not os.path.exists(full):
        print(f"❌ {path}")
        continue
    m = joblib.load(full)
    print(f"✅ {path}")
    print(f"   type: {type(m).__name__}")
    print(f"   n_features: {m.n_features_in_}")
    print(f"   n_estimators: {m.n_estimators_}")
    print()

# ── PyTorch 模型 ──
pt_models = [
    "multiscale_cnn_bottom.pt",
    "multiscale_cnn_top.pt",
    "vol_multiscale_lstm_regression.pt",
    "vol_lstm_regression.pt",
]

for path in pt_models:
    full = os.path.join(results_dir, path)
    if not os.path.exists(full):
        print(f"❌ {path}")
        continue
    state = torch.load(full, map_location="cpu", weights_only=True)
    print(f"✅ {path}")
    for k, v in state.items():
        print(f"   {k}: {tuple(v.shape)}")
    print()

# ── meta.json ──
for path in lgb_models:
    meta_path = os.path.join(results_dir, os.path.dirname(path), "meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            print(f"📄 {meta_path}")
            print(f"   {json.load(f)}")

✅ models/layer1/volatility/lightgbm_v3/weights.joblib
   type: LGBMRegressor
   n_features: 183
   n_estimators: 561

✅ models/layer1/volatility/lightgbm_v3_flat/weights.joblib
   type: LGBMRegressor
   n_features: 119
   n_estimators: 496

✅ models/layer3/trade_filter/lgb_bottom_v1/weights.joblib
   type: LGBMClassifier
   n_features: 490
   n_estimators: 138

✅ models/layer3/trade_filter/lgb_top_v1/weights.joblib
   type: LGBMClassifier
   n_features: 490
   n_estimators: 84

✅ multiscale_cnn_bottom.pt
   short_branch.0.weight: (32, 66, 5)
   short_branch.0.bias: (32,)
   short_branch.2.weight: (32,)
   short_branch.2.bias: (32,)
   short_branch.2.running_mean: (32,)
   short_branch.2.running_var: (32,)
   short_branch.2.num_batches_tracked: ()
   short_branch.3.weight: (64, 32, 3)
   short_branch.3.bias: (64,)
   short_branch.5.weight: (64,)
   short_branch.5.bias: (64,)
   short_branch.5.running_mean: (64,)
   short_branch.5.running_var: (64,)
   short_branch.5.num_batches_tracked:

AttributeError: 'collections.OrderedDict' object has no attribute 'shape'

In [9]:
import torch, json

state = torch.load("results/multiscale_cnn_top.pt", map_location="cpu", weights_only=False)
print(f"type: {type(state)}")
print(f"keys: {list(state.keys())}")

# 如果是嵌套 dict，递归打印
def print_state(d, indent=0):
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            print(" " * indent + f"{k}: {tuple(v.shape)}")
        elif isinstance(v, dict):
            print(" " * indent + f"{k}:")
            print_state(v, indent + 2)
        else:
            print(" " * indent + f"{k}: {type(v).__name__} = {v}")

print_state(state)

type: <class 'dict'>
keys: ['model_state_dict', 'n_short_feat', 'n_long_feat', 'short_win', 'long_win', 's_mu', 's_sigma', 'l_mu', 'l_sigma', 'task']
model_state_dict:
  short_branch.0.weight: (32, 66, 5)
  short_branch.0.bias: (32,)
  short_branch.2.weight: (32,)
  short_branch.2.bias: (32,)
  short_branch.2.running_mean: (32,)
  short_branch.2.running_var: (32,)
  short_branch.2.num_batches_tracked: ()
  short_branch.3.weight: (64, 32, 3)
  short_branch.3.bias: (64,)
  short_branch.5.weight: (64,)
  short_branch.5.bias: (64,)
  short_branch.5.running_mean: (64,)
  short_branch.5.running_var: (64,)
  short_branch.5.num_batches_tracked: ()
  short_branch.6.weight: (32, 64, 3)
  short_branch.6.bias: (32,)
  long_branch.0.weight: (32, 53, 7)
  long_branch.0.bias: (32,)
  long_branch.2.weight: (32,)
  long_branch.2.bias: (32,)
  long_branch.2.running_mean: (32,)
  long_branch.2.running_var: (32,)
  long_branch.2.num_batches_tracked: ()
  long_branch.3.weight: (64, 32, 5)
  long_branch.3.b

In [11]:
# ── 重新构建数据 + 训练 bottom + 保存新格式 ──
import torch, os, numpy as np

task = "bottom"
cfg = CFG  # 确保 CFG 还在，如果不在需要重新定义

# Step 1: 重新构建数据
print("Building paired windows...")
data = build_paired_windows(cfg)

# Step 2: Normalize
print("Normalizing...")
tr = data["train_mask"]
va = data["val_mask"]
te = data["test_mask"]

X_s_tr, X_s_va, X_s_te, s_mu, s_sigma = normalize_windows(
    data["X_short"][tr], data["X_short"][va], data["X_short"][te])
X_l_tr, X_l_va, X_l_te, l_mu, l_sigma = normalize_windows(
    data["X_long"][tr], data["X_long"][va], data["X_long"][te])

# Step 3: 训练
print("Training bottom CNN...")
model = train_model(X_s_tr, X_l_tr, data["y_bottom"][tr],
                    X_s_va, X_l_va, data["y_bottom"][va],
                    data["n_short_feat"], data["n_long_feat"],
                    cfg, task)

# Step 4: 保存新格式
ckpt_path = os.path.join(cfg["OUTPUT_DIR"], "multiscale_cnn_bottom.pt")
checkpoint = {
    "model_state_dict": model.state_dict(),
    "n_short_feat": data["n_short_feat"],
    "n_long_feat": data["n_long_feat"],
    "short_win": cfg["SHORT_WIN"],
    "long_win": cfg["LONG_WIN"],
    "s_mu": torch.tensor(s_mu, dtype=torch.float32),
    "s_sigma": torch.tensor(s_sigma, dtype=torch.float32),
    "l_mu": torch.tensor(l_mu, dtype=torch.float32),
    "l_sigma": torch.tensor(l_sigma, dtype=torch.float32),
    "task": task,
}
torch.save(checkpoint, ckpt_path)

# Step 5: 验证
ck = torch.load(ckpt_path, map_location="cpu", weights_only=True)
nan_found = any(torch.isnan(v).any()
                for v in ck["model_state_dict"].values()
                if v.is_floating_point())
print(f"{'❌ NaN found' if nan_found else '✅ Clean'}: {ckpt_path}")
print(f"keys: {list(ck.keys())}")
print(f"n_short_feat={ck['n_short_feat']}  n_long_feat={ck['n_long_feat']}")

Building paired windows...

  Building paired windows...
    AAPL: 42,919 samples, bottoms=17195 tops=17203  s_feat=66 l_feat=53
    MSFT: 42,496 samples, bottoms=17327 tops=17347  s_feat=66 l_feat=53
    GOOGL: 38,701 samples, bottoms=15460 tops=15496  s_feat=66 l_feat=53
    GOOG: 37,779 samples, bottoms=15212 tops=15246  s_feat=66 l_feat=53
    NVDA: 42,281 samples, bottoms=16570 tops=16572  s_feat=66 l_feat=53
    TSLA: 42,977 samples, bottoms=17486 tops=17504  s_feat=66 l_feat=53
    SPY: 42,817 samples, bottoms=17039 tops=17076  s_feat=66 l_feat=53
    QQQ: 42,914 samples, bottoms=16596 tops=16621  s_feat=66 l_feat=53

  Total: 332,884 samples
  Short shape: (332884, 30, 66)  Long shape: (332884, 48, 53)
  Bottom pos rate: 0.3957  Top pos rate: 0.3962
  Train: 233,014  Val: 49,933  Test: 49,937
Normalizing...
Training bottom CNN...

    Training MultiScaleCNN (bottom)...
    After undersample: 233,014 (pos=92638, neg=140376)
    Params: 53,857
      E  1 t=0.715319 v=0.674931 lr=

In [12]:
state = torch.load("results/multiscale_cnn_bottom.pt", map_location="cpu", weights_only=False)
print(f"keys: {list(state.keys())}")
for k, v in state.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}")
    elif isinstance(v, dict):
        for kk, vv in v.items():
            print(f"  {k}.{kk}: {tuple(vv.shape)}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

keys: ['model_state_dict', 'n_short_feat', 'n_long_feat', 'short_win', 'long_win', 's_mu', 's_sigma', 'l_mu', 'l_sigma', 'task']
  model_state_dict.short_branch.0.weight: (32, 66, 5)
  model_state_dict.short_branch.0.bias: (32,)
  model_state_dict.short_branch.2.weight: (32,)
  model_state_dict.short_branch.2.bias: (32,)
  model_state_dict.short_branch.2.running_mean: (32,)
  model_state_dict.short_branch.2.running_var: (32,)
  model_state_dict.short_branch.2.num_batches_tracked: ()
  model_state_dict.short_branch.3.weight: (64, 32, 3)
  model_state_dict.short_branch.3.bias: (64,)
  model_state_dict.short_branch.5.weight: (64,)
  model_state_dict.short_branch.5.bias: (64,)
  model_state_dict.short_branch.5.running_mean: (64,)
  model_state_dict.short_branch.5.running_var: (64,)
  model_state_dict.short_branch.5.num_batches_tracked: ()
  model_state_dict.short_branch.6.weight: (32, 64, 3)
  model_state_dict.short_branch.6.bias: (32,)
  model_state_dict.long_branch.0.weight: (32, 53, 7)
